# FranchiseOps AI - Milestone 2

A multi-user franchise operations platform: **FastAPI + SQLite + JWT + RBAC**, six working agents and a SQL-only AI Copilot.

**How to run:** add the secrets below in the Colab sidebar (key icon, *Notebook access* ON), then **Runtime -> Run all**. The last cell prints **Open the app** (an ngrok link) and a backup Colab link.

| Secret | Used for |
|---|---|
| `JWT_SECRET` | signing login tokens (required) |
| `ADMIN_PASSWORD` | password of the first Admin account (required) |
| `ADMIN_EMAIL_ID` | e-mail of the first Admin (optional, default `admin@infosys.com`) |
| `EMAIL_ADDRESS`, `EMAIL_PASSWORD` | Gmail sender and app password for password-recovery codes (from Milestone 1) |
| `NGROK_AUTHTOKEN` | the public link (recommended: Colab's own link is sometimes blocked) |

Sign in as `admin` (or the Admin e-mail) with your `ADMIN_PASSWORD`. Everyone else creates an account on the sign-up page and picks their own role; an Admin can promote or demote them in the Admin Dashboard.
Never type a password or token into a cell.

In [1]:
!pip install -q fastapi 'uvicorn[standard]' python-multipart pandas numpy scikit-learn scipy matplotlib vaderSentiment requests pyngrok bcrypt pyjwt pdfplumber

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
import os
os.makedirs('/content/app/static', exist_ok=True)
os.chdir('/content/app')

In [3]:
# Colab Secrets -> environment variables (the server runs as a separate process and cannot read Colab Secrets itself)
import os
NAMES = ["JWT_SECRET", "ADMIN_PASSWORD", "ADMIN_EMAIL_ID", "EMAIL_ADDRESS", "EMAIL_PASSWORD", "NGROK_AUTHTOKEN"]
try:
    from google.colab import userdata
    for n in NAMES:
        try:
            v = userdata.get(n)
        except Exception:
            v = None
        if v:
            os.environ[n] = str(v).strip(); print(f"[OK]      secret {n} is set")
        else:
            print(f"[MISSING] secret {n}" + ("  <- required" if n in ("JWT_SECRET", "ADMIN_PASSWORD") else ""))
except ImportError:
    print("Not running in Colab: using the environment variables that are already set.")
if not os.environ.get("ADMIN_PASSWORD"):
    print("\nNo ADMIN_PASSWORD secret: the server will create a random Admin password and print it in the server log (server.log).")

[OK]      secret JWT_SECRET is set
[OK]      secret ADMIN_PASSWORD is set
[MISSING] secret ADMIN_EMAIL_ID
[OK]      secret EMAIL_ADDRESS is set
[OK]      secret EMAIL_PASSWORD is set
[OK]      secret NGROK_AUTHTOKEN is set


In [4]:
%%writefile /content/app/accounts.py
"""Accounts: the Milestone 1 login brought into FastAPI (signup, login, forgot/reset), the users table, JWT sessions,
the Admin dashboard API (add, delete, promote/demote, lock/unlock, activity log) and the Profile API.

Rules: passwords are hashed with bcrypt and never returned; the JWT carries only the user id (role, outlet and lock state are read
from the database on EVERY request, so a promotion, demotion or lock takes effect immediately); an unknown user and a wrong password
give the same error; 5 wrong passwords lock the account; the last Admin can never be removed, demoted or locked."""
import re, time, secrets, sqlite3, datetime
import bcrypt, jwt
from fastapi import APIRouter, Request, HTTPException
from fastapi.responses import JSONResponse
import config, rbac, mailer, validators as V
import live_engine as LE

router = APIRouter()
COOKIE = "fo_token"                       # EventSource (live pulse) cannot send headers, so GET requests may also use this cookie
TOKEN_HOURS = 8
MAX_FAILS = 5
OTP_MINUTES, OTP_TRIES, OTP_COOLDOWN = 10, 5, 30
SIGNUP_ROLES = [r for r in rbac.ROLES if r != "Admin"]      # a person may pick their own role at signup, but never Admin (only an Admin can promote)
GENERIC = "Invalid email/username or password."
_DUMMY = bcrypt.hashpw(b"timing-equaliser", bcrypt.gensalt())

def db():
    c = sqlite3.connect(config.DB_PATH, timeout=30); c.row_factory = sqlite3.Row; c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;"); return c
def now(): return datetime.datetime.now().isoformat(timespec="seconds")
def hash_pw(p): return bcrypt.hashpw(p.encode(), bcrypt.gensalt()).decode()
def check_pw(p, h):
    try: return bcrypt.checkpw(p.encode(), (h or "").encode())
    except Exception: return False

# ------------------------------------------------------------------ users table (extends the table db.py creates)
NEW_COLS = {"outlet": "TEXT", "locked": "INTEGER NOT NULL DEFAULT 0", "last_login": "TEXT", "token_version": "INTEGER NOT NULL DEFAULT 0",
            "otp_hash": "TEXT", "otp_expires": "REAL", "otp_tries": "INTEGER NOT NULL DEFAULT 0", "otp_sent": "REAL"}
def init():
    with db() as c:
        c.execute("CREATE TABLE IF NOT EXISTS users(id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE, password_hash TEXT, role TEXT, security_question TEXT, security_answer TEXT, profile_picture TEXT, failed_attempts INTEGER DEFAULT 0, lock_until DATETIME DEFAULT NULL, account_status TEXT DEFAULT 'active', created_at DATETIME DEFAULT CURRENT_TIMESTAMP)")
        have = {r[1] for r in c.execute("PRAGMA table_info(users)").fetchall()}
        for col, ddl in NEW_COLS.items():
            if col not in have: c.execute(f"ALTER TABLE users ADD COLUMN {col} {ddl}")
        c.execute("CREATE UNIQUE INDEX IF NOT EXISTS ix_users_email_lc ON users(lower(email))")
        c.execute("CREATE UNIQUE INDEX IF NOT EXISTS ix_users_username_lc ON users(lower(username))")
        seed = not c.execute("SELECT 1 FROM users WHERE role='Admin'").fetchone()
        if seed: c.execute("INSERT INTO users(username,email,password_hash,role,locked,created_at) VALUES(?,?,?,?,0,?)", ("admin", config.ADMIN_EMAIL_ID.lower(), hash_pw(config.ADMIN_PASSWORD), "Admin", now()))
    if seed: LE.audit("system", "admin_seeded", config.ADMIN_EMAIL_ID.lower(), "first Admin created from the ADMIN_PASSWORD secret")

PUBLIC_COLS = "id, email, username, role, outlet, locked, failed_attempts, created_at, last_login"
def public(r): return {"id": r["id"], "email": r["email"], "username": r["username"], "role": r["role"], "outlet": r["outlet"] if r["role"] in rbac.SCOPED_ROLES else None, "locked": bool(r["locked"]), "created_at": r["created_at"], "last_login": r["last_login"]}
def outlets():
    with db() as c: return [r[0] for r in c.execute("SELECT outlet_id FROM outlets ORDER BY outlet_id").fetchall()]

# ------------------------------------------------------------------ sessions
def make_token(r): return jwt.encode({"sub": str(r["id"]), "tv": r["token_version"], "iat": int(time.time()), "exp": int(time.time()) + TOKEN_HOURS * 3600}, config.JWT_SECRET_KEY, algorithm="HS256")
def current_user(request: Request):
    """user_of(): reads the Bearer token, checks the user still exists and is not locked, and returns the REAL user. 401 otherwise."""
    h = request.headers.get("authorization", ""); tok = h[7:].strip() if h.lower().startswith("bearer ") else (request.cookies.get(COOKIE) if request.method == "GET" else None)
    if not tok: raise HTTPException(401, "Sign in to continue.")
    try: p = jwt.decode(tok, config.JWT_SECRET_KEY, algorithms=["HS256"])
    except jwt.ExpiredSignatureError: raise HTTPException(401, "Your session expired. Sign in again.")
    except Exception: raise HTTPException(401, "Invalid session. Sign in again.")
    with db() as c: r = c.execute(f"SELECT {PUBLIC_COLS}, token_version FROM users WHERE id=?", (p.get("sub"),)).fetchone()
    if not r or r["token_version"] != p.get("tv"): raise HTTPException(401, "Your session is no longer valid. Sign in again.")
    if r["locked"]: raise HTTPException(401, "This account is locked. Contact an administrator.")
    return {k: r[k] for k in ("id", "email", "username", "role", "outlet", "created_at", "last_login")}

def body_of(d, *keys): return [str(d.get(k) if d.get(k) is not None else "") for k in keys]
async def jbody(request):
    try: d = await request.json()
    except Exception: raise HTTPException(400, "Send the form as JSON.")
    if not isinstance(d, dict): raise HTTPException(400, "Send the form as JSON.")
    return d
def ok(msg="Done", **kw): return {"ok": True, "msg": msg, **kw}
def fail(msg, code=400): raise HTTPException(code, msg)
def check(r):
    good, msg = r
    if not good: fail(msg)

def validate_new_account(email, username, password, confirm, role, outlet, allowed_roles):
    check(V.validate_mandatory_fields({"Email": email, "Username": username, "Password": password, "Confirm Password": confirm, "Role": role}))
    check(V.validate_email_format(email)); check(V.validate_username(username))
    check(V.validate_password_complexity(password)); check(V.validate_password_confirmation(password, confirm))
    if role not in allowed_roles: fail("Choose a valid role.")
    outlet = (outlet or "").strip().upper()
    if role in rbac.SCOPED_ROLES:
        if not outlet: fail(f"Choose the outlet for a {role}.")
        if outlet not in outlets(): fail("That outlet does not exist.")
    else: outlet = None
    return email.strip().lower(), username.strip(), outlet
def insert_user(email, username, password, role, outlet):
    with db() as c:
        if c.execute("SELECT 1 FROM users WHERE lower(email)=?", (email,)).fetchone(): fail("This email is already registered.")
        if c.execute("SELECT 1 FROM users WHERE lower(username)=lower(?)", (username,)).fetchone(): fail("This username is taken.")
        cur = c.execute("INSERT INTO users(username,email,password_hash,role,outlet,locked,failed_attempts,created_at) VALUES(?,?,?,?,?,0,0,?)", (username, email, hash_pw(password), role, outlet, now()))
        return cur.lastrowid

# ------------------------------------------------------------------ public auth routes
@router.get("/api/auth/options")
def auth_options(): return {"roles": SIGNUP_ROLES, "scoped_roles": sorted(rbac.SCOPED_ROLES), "outlets": outlets()}

@router.post("/api/auth/signup")
async def signup(request: Request):
    d = await jbody(request); email, username, password, confirm, role, outlet = body_of(d, "email", "username", "password", "confirm_password", "role", "outlet")
    email, username, outlet = validate_new_account(email, username, password, confirm, role, outlet, SIGNUP_ROLES)      # Admin is not offered and is refused here
    uid = insert_user(email, username, password, role, outlet); LE.audit(email, "signup", str(uid), f"role={role} outlet={outlet or '-'}")
    return ok("Account created. You can sign in now.")

@router.post("/api/auth/login")
async def login(request: Request):
    d = await jbody(request); ident, password = body_of(d, "identity", "password"); ident = ident.strip()
    if not ident or not password: fail("Enter your email or username and password.")
    locked_now = False
    with db() as c:
        r = c.execute(f"SELECT {PUBLIC_COLS}, password_hash, token_version FROM users WHERE lower(email)=lower(?) OR lower(username)=lower(?)", (ident, ident)).fetchone()
        if not r: bcrypt.checkpw(password.encode(), _DUMMY); fail(GENERIC, 401)            # same text and similar time as a wrong password
        if not check_pw(password, r["password_hash"]):
            n = (r["failed_attempts"] or 0) + 1; lock = 0
            if n >= MAX_FAILS:
                others = c.execute("SELECT COUNT(*) FROM users WHERE role='Admin' AND locked=0 AND id<>?", (r["id"],)).fetchone()[0]
                lock = 0 if r["role"] == "Admin" and not others else 1        # the last active Admin is never auto-locked
            c.execute("UPDATE users SET failed_attempts=?, locked=? WHERE id=?", (n, 1 if (lock or r["locked"]) else 0, r["id"])); c.commit()       # commit: raising below would roll the count back
            locked_now = bool(lock) and not r["locked"]
        elif r["locked"]: fail("This account is locked. Contact an administrator.", 403)
        else:
            c.execute("UPDATE users SET failed_attempts=0, last_login=? WHERE id=?", (now(), r["id"])); token = make_token(r); u = public(r); u["last_login"] = now()
    if not check_pw(password, r["password_hash"]):
        if locked_now: LE.audit(r["email"], "auto_lock", str(r["id"]), f"{MAX_FAILS} wrong passwords")
        fail(GENERIC, 401)
    LE.audit(r["email"], "login", str(r["id"]), r["role"])
    resp = JSONResponse({"ok": True, "msg": "Signed in.", "token": token, "user": u})
    resp.set_cookie(COOKIE, token, max_age=TOKEN_HOURS * 3600, httponly=True, samesite="lax", path="/"); return resp

@router.post("/api/auth/logout")
def logout():
    resp = JSONResponse({"ok": True, "msg": "Signed out."}); resp.delete_cookie(COOKIE, path="/"); return resp

# ---- password recovery by e-mail code: forgot -> verify -> reset
def _user_by_email(c, email): return c.execute("SELECT id, email, role, locked, otp_hash, otp_expires, otp_tries, otp_sent FROM users WHERE lower(email)=?", (email.strip().lower(),)).fetchone()
def _otp_ok(c, r, code, consume):
    """True when the code is right. Counts tries (OTP_TRIES wrong codes burn it) and enforces expiry."""
    if not r or not r["otp_hash"] or (r["otp_expires"] or 0) < time.time(): return False
    if (r["otp_tries"] or 0) >= OTP_TRIES: c.execute("UPDATE users SET otp_hash=NULL, otp_expires=NULL WHERE id=?", (r["id"],)); c.commit(); return False
    if not re.fullmatch(r"\d{6}", code or "") or not check_pw(code, r["otp_hash"]):
        c.execute("UPDATE users SET otp_tries=otp_tries+1 WHERE id=?", (r["id"],)); c.commit(); return False       # commit: the caller raises, which would roll the counter back
    if consume: c.execute("UPDATE users SET otp_hash=NULL, otp_expires=NULL, otp_tries=0 WHERE id=?", (r["id"],)); c.commit()
    return True

@router.post("/api/auth/forgot")
async def forgot(request: Request):
    d = await jbody(request); (email,) = body_of(d, "email"); check(V.validate_email_format(email.strip()))
    msg = "If an account exists for that email, a 6-digit code has been sent. It is valid for %d minutes." % OTP_MINUTES
    with db() as c:
        r = _user_by_email(c, email)
        if r and not r["locked"] and time.time() - (r["otp_sent"] or 0) >= OTP_COOLDOWN:
            code = f"{secrets.randbelow(10**6):06d}"
            c.execute("UPDATE users SET otp_hash=?, otp_expires=?, otp_tries=0, otp_sent=? WHERE id=?", (hash_pw(code), time.time() + OTP_MINUTES * 60, time.time(), r["id"]))
            sent, info = mailer.send(r["email"], "Your FranchiseOps AI verification code", f"Your verification code is {code}. It expires in {OTP_MINUTES} minutes. If you did not ask for it, ignore this e-mail.")
            if not sent: print(f"[forgot-password] e-mail not sent ({info}); code for {r['email']}: {code}")       # visible in the server log when Gmail is not configured
    return ok(msg)

@router.post("/api/auth/verify")
async def verify(request: Request):
    d = await jbody(request); email, otp = body_of(d, "email", "otp")
    with db() as c:
        if not _otp_ok(c, _user_by_email(c, email), otp.strip(), False): fail("That code is wrong or has expired.")
    return ok("Code verified.")

@router.post("/api/auth/reset")
async def reset(request: Request):
    d = await jbody(request); email, otp, pw, cf = body_of(d, "email", "otp", "password", "confirm_password")
    check(V.validate_password_complexity(pw)); check(V.validate_password_confirmation(pw, cf))
    with db() as c:
        r = _user_by_email(c, email)
        if not _otp_ok(c, r, otp.strip(), True): fail("That code is wrong or has expired.")
        c.execute("UPDATE users SET password_hash=?, failed_attempts=0, token_version=token_version+1 WHERE id=?", (hash_pw(pw), r["id"]))     # old sessions end; a locked account stays locked
    LE.audit(r["email"], "password_reset", str(r["id"]), "e-mail code"); return ok("Password changed. You can sign in with the new password.")

# ------------------------------------------------------------------ profile
def _server():
    import server; return server
def me_ctx(request):
    s = _server(); u = s.user_of(request); return u, s.ctx(u)

@router.get("/api/profile")
def profile(request: Request):
    u, c = me_ctx(request); rbac.need(c, "profile")
    with db() as cn: r = cn.execute(f"SELECT {PUBLIC_COLS} FROM users WHERE id=?", (u["id"],)).fetchone()
    return {"ok": True, "profile": public(r)}

@router.post("/api/profile/username")
async def profile_username(request: Request):
    u, c = me_ctx(request); rbac.need(c, "profile"); d = await jbody(request); (name,) = body_of(d, "username"); name = name.strip(); check(V.validate_username(name))
    with db() as cn:
        if cn.execute("SELECT 1 FROM users WHERE lower(username)=lower(?) AND id<>?", (name, u["id"])).fetchone(): fail("This username is taken.")
        cn.execute("UPDATE users SET username=? WHERE id=?", (name, u["id"]))
    LE.audit(u["email"], "username_changed", str(u["id"]), name); return ok("Username updated.", username=name)

@router.post("/api/profile/password")
async def profile_password(request: Request):
    u, c = me_ctx(request); rbac.need(c, "profile"); d = await jbody(request); cur, new, cf = body_of(d, "current_password", "new_password", "confirm_password")
    check(V.validate_mandatory_fields({"Current password": cur, "New password": new, "Confirm password": cf}))
    with db() as cn:
        r = cn.execute("SELECT password_hash, token_version FROM users WHERE id=?", (u["id"],)).fetchone()
        if not check_pw(cur, r["password_hash"]): fail("The current password is wrong.")
        check(V.validate_password_complexity(new)); check(V.validate_password_confirmation(new, cf))
        if check_pw(new, r["password_hash"]): fail("Choose a password you have not used just now.")
        cn.execute("UPDATE users SET password_hash=?, token_version=token_version+1 WHERE id=?", (hash_pw(new), u["id"])); tv = r["token_version"] + 1
        row = cn.execute(f"SELECT {PUBLIC_COLS}, token_version FROM users WHERE id=?", (u["id"],)).fetchone()
    LE.audit(u["email"], "password_changed", str(u["id"]), ""); token = make_token(row)
    resp = JSONResponse({"ok": True, "msg": "Password changed.", "token": token}); resp.set_cookie(COOKIE, token, max_age=TOKEN_HOURS * 3600, httponly=True, samesite="lax", path="/"); return resp

# ------------------------------------------------------------------ admin dashboard (every route needs the "users" capability)
def admin_ctx(request):
    u, c = me_ctx(request); rbac.need(c, "users"); return u, c
def _active_admins(cn, excluding=None):
    return cn.execute("SELECT COUNT(*) FROM users WHERE role='Admin' AND locked=0 AND id<>?", (excluding or 0,)).fetchone()[0]
def _target(cn, uid):
    r = cn.execute(f"SELECT {PUBLIC_COLS} FROM users WHERE id=?", (uid,)).fetchone()
    if not r: raise HTTPException(404, "User not found.")
    return r

@router.get("/api/admin/overview")
def admin_overview(request: Request):
    admin_ctx(request)
    with db() as cn:
        rows = cn.execute(f"SELECT {PUBLIC_COLS} FROM users ORDER BY id").fetchall(); per = {r: 0 for r in rbac.ROLES}
        for r in rows: per[r["role"]] = per.get(r["role"], 0) + 1
        log = [dict(x) for x in cn.execute("SELECT ts, actor, event, target, detail FROM audit_log ORDER BY audit_id DESC LIMIT 50").fetchall()]
    return {"ok": True, "users": [{**public(r), "saved_outlet": r["outlet"]} for r in rows], "kpis": {"total": len(rows), "locked": sum(1 for r in rows if r["locked"]), "active": sum(1 for r in rows if not r["locked"]), "per_role": per},
            "activity": log, "roles": rbac.ROLES, "scoped_roles": sorted(rbac.SCOPED_ROLES), "outlets": outlets(), "matrix": rbac.permission_table()}

@router.post("/api/admin/users")
async def admin_add(request: Request):
    u, c = admin_ctx(request); d = await jbody(request); email, username, password, role, outlet = body_of(d, "email", "username", "password", "role", "outlet")
    email, username, outlet = validate_new_account(email, username, password, password, role, outlet, rbac.ROLES)
    uid = insert_user(email, username, password, role, outlet); LE.audit(u["email"], "user_added", email, f"role={role} outlet={outlet or '-'}"); return ok(f"User {username} added.", id=uid)

@router.post("/api/admin/users/{uid}/role")
async def admin_role(uid: int, request: Request):
    u, c = admin_ctx(request); d = await jbody(request); role, outlet = body_of(d, "role", "outlet"); outlet = outlet.strip().upper()
    if role not in rbac.ROLES: fail("Choose a valid role.")
    with db() as cn:
        t = _target(cn, uid)
        if t["id"] == u["id"]: fail("You cannot change your own role.")
        if t["role"] == "Admin" and role != "Admin" and not _active_admins(cn, t["id"]): fail("The last remaining Admin cannot be demoted.")
        if role in rbac.SCOPED_ROLES:
            outlet = outlet or (t["outlet"] or "")
            if not outlet: fail(f"Choose the outlet for a {role}.")
            if outlet not in outlets(): fail("That outlet does not exist.")
        else: outlet = t["outlet"]                         # keep it: the person may go back to a one-outlet role later
        cn.execute("UPDATE users SET role=?, outlet=? WHERE id=?", (role, outlet, uid))
    LE.audit(u["email"], "role_changed", t["email"], f"{t['role']} -> {role}" + (f" ({outlet})" if outlet and role in rbac.SCOPED_ROLES else "")); return ok(f"{t['username']} is now {role}.")

@router.post("/api/admin/users/{uid}/lock")
def admin_lock(uid: int, request: Request): return _set_lock(uid, request, 1)
@router.post("/api/admin/users/{uid}/unlock")
def admin_unlock(uid: int, request: Request): return _set_lock(uid, request, 0)
def _set_lock(uid, request, lock):
    u, c = admin_ctx(request)
    with db() as cn:
        t = _target(cn, uid)
        if lock and t["id"] == u["id"]: fail("You cannot lock your own account.")
        if lock and t["role"] == "Admin" and not _active_admins(cn, t["id"]): fail("The last active Admin cannot be locked.")
        cn.execute("UPDATE users SET locked=?, failed_attempts=0 WHERE id=?", (lock, uid))
    LE.audit(u["email"], "user_locked" if lock else "user_unlocked", t["email"], ""); return ok(f"{t['username']} is now {'locked' if lock else 'unlocked'}.")

@router.post("/api/admin/users/{uid}/delete")
async def admin_delete(uid: int, request: Request):
    u, c = admin_ctx(request); d = await jbody(request)
    if d.get("confirm") is not True: fail("Confirm the deletion first.")
    with db() as cn:
        t = _target(cn, uid)
        if t["id"] == u["id"]: fail("You cannot delete your own account.")
        if t["role"] == "Admin" and not _active_admins(cn, t["id"]): fail("The last remaining Admin cannot be removed.")
        cn.execute("DELETE FROM users WHERE id=?", (uid,))
    LE.audit(u["email"], "user_deleted", t["email"], f"role={t['role']}"); return ok(f"{t['username']} was deleted.")


Writing /content/app/accounts.py


In [5]:
%%writefile /content/app/alts.py
"""Alternatives shown in the detail panels: other sea routes for a shipment, nearby lower-congestion ports for a port, nearby outlets that can transfer stock to an outlet."""
import json, numpy as np, pandas as pd
import blocks as B
from blocks import Q, TEAL, BLUE, AMBER, RED, VIOLET
COLORS = [TEAL, BLUE, AMBER, VIOLET, "#d6409f", "#8b8d98"]
def hav(la1, lo1, la2, lo2, R=6371.0):
    p = np.pi / 180; a = np.sin((np.asarray(la2) - la1) * p / 2) ** 2 + np.cos(la1 * p) * np.cos(np.asarray(la2) * p) * np.sin((np.asarray(lo2) - lo1) * p / 2) ** 2; return 2 * R * np.arcsin(np.sqrt(a))
def usd(x): return "${:,.0f}".format(x)
VARIANTS = [((), "Default"), (("suez",), "Avoid Suez"), (("panama",), "Avoid Panama"), (("malacca",), "Avoid Malacca"), (("hormuz",), "Avoid Hormuz"), (("storm",), "Avoid storms")]
def route_alternatives(dom, o_xy, d_xy, tonnes=20.0, interp=None, rate=0.012, day_cost=120.0, dwell=0.0):
    """-> list of options sorted by total cost; each {label, name, nm, days, freight, delay_cost, total, wind, chain, avoid, flags}"""
    seen, out = {}, []
    for av, lab in VARIANTS:
        try: opts = dom.route_options(o_xy, d_xy, avoid=av)
        except Exception: continue
        if not opts: continue
        o = opts[0]; key = o["name"]
        if key in seen: continue
        seen[key] = 1; ch = o["chain"]; wind = 0.0
        if interp:
            try: wind = max(dom.weather_at(*interp(ch, f), allow_live=False)["wind"] for f in (.15, .35, .55, .75, .95))
            except Exception: pass
        days = o["nm"] / (14 * 24) + dwell; fr = o["nm"] * tonnes * rate; dl = days * day_cost; risk_cost = max(0, wind - 15) * 0.02 * days * day_cost * 3
        out.append({"label": lab, "name": o["name"], "nm": o["nm"], "days": round(days, 1), "freight": fr, "delay_cost": dl + risk_cost, "total": fr + dl + risk_cost, "wind": round(wind, 1), "chain": ch, "avoid": list(av)})
    if not out: return out
    out.sort(key=lambda r: r["total"]); fast = min(out, key=lambda r: r["days"]); short = min(out, key=lambda r: r["nm"])
    for r in out: r["tag"] = ", ".join(t for t, c in (("Recommended", r is out[0]), ("Fastest", r is fast), ("Shortest", r is short)) if c)
    return out
def route_blocks(opts, sid=None, title="Route alternatives", cur=None):
    if not opts: return [B.callout("No alternative sea route found for this lane.", "info")]
    lines = [{"pts": r["chain"], "color": COLORS[i % len(COLORS)], "w": 4 if i == 0 else 2.5, "dash": None if i == 0 else "6 6"} for i, r in enumerate(opts)]
    rows = [{"option": f"{i + 1}. {r['label']}", "route": r["name"], "nm": r["nm"], "days": r["days"], "freight": r["freight"], "delay": r["delay_cost"], "total": r["total"], "wind": r["wind"], "tag": r["tag"]} for i, r in enumerate(opts)]
    best, worst = opts[0], opts[-1]; pts = [p for r in opts for p in r["chain"]]; la = [p[0] for p in pts]; lo = [p[1] for p in pts]
    view = {"bounds": [[min(la) - 4, min(lo) - 6], [max(la) + 4, max(lo) + 6]]}
    ends = [{"id": "o", "lat": opts[0]["chain"][0][0], "lon": opts[0]["chain"][0][1], "color": "#1f2430", "r": 6, "tip": "start"}, {"id": "d", "lat": opts[0]["chain"][-1][0], "lon": opts[0]["chain"][-1][1], "color": RED, "r": 6, "tip": "destination"}]
    bl = [B.card(title, B.map_([{"type": "lines", "lines": lines}, {"type": "points", "id": "alt_ends", "pts": ends}], h=300, view=view, key="alt_map", legend=[(str(rows[i]["option"]) + (" (solid = recommended)" if i == 0 else " (dashed)"), COLORS[i % len(COLORS)]) for i in range(min(len(rows), len(COLORS)))] + [("Start", "#1f2430"), ("Destination", RED)]), B.table(rows, ["option", "route", B.num_col("nm", "nm", "int"), B.num_col("days", "days", "num"), B.num_col("freight", "freight cost", "money"), B.num_col("delay", "time + weather cost", "money"), B.num_col("total", "total", "money"), B.num_col("wind", "max wind m/s", "num"), "tag"], h=min(60 + 40 * len(rows), 330), key="alt_t"),
                       sub=f"Best: {best['label']} ({best['name']}), {best['days']} days, {usd(best['total'])}. Worst: {usd(worst['total'])}. Cost = $0.012 per tonne-nm + $120 per day + weather penalty above 15 m/s.")]
    return bl
def port_alternatives(port, k=6):
    p = Q("SELECT port_name, country, region, lat, lon, congestion_index ci, avg_dwell_days dw FROM ports"); me = p[p.port_name == port]
    if not len(me): return []
    me = me.iloc[0]; p["km"] = hav(me.lat, me.lon, p.lat.values, p.lon.values); c = p[(p.port_name != port) & (p.km < 1800)].copy()
    c["better"] = (me.ci - c.ci); c["score"] = c.ci * 12 + c.dw * 4 + c.km / 100; c = c[c.ci < me.ci].sort_values("score").head(k)
    rows = [{"port": r.port_name, "country": r.country, "km": r.km, "ci": r.ci, "dwell": r.dw, "days_saved": max(0, me.dw - r.dw), "haul_days": r.km / 600 / 1 if r.country == me.country else r.km / 450} for r in c.itertuples()]
    if not rows: return [B.callout(f"No less-congested port within 1,800 km of {port}.", "info")]
    return [B.card("Alternative ports nearby", B.table(rows, ["port", "country", B.num_col("km", "km away", "int"), B.num_col("ci", "congestion", "num"), B.num_col("dwell", "dwell days", "num"), B.num_col("days_saved", "dwell saved", "num"), B.num_col("haul_days", "extra haul days (approx.)", "num")], h=min(70 + 40 * len(rows), 340), open={"kind": "port", "col": "port"}, key="alt_p"),
                    sub="Cargo for the port could be diverted here; haul by road/rail is estimated at 600 km/day domestic, 450 km/day cross-border. Check customs and carrier agreements before diverting.")]
def outlet_alternatives(oid, max_km=400):
    g = Q("SELECT outlet_id, lat, lon FROM outlet_geo"); me = g[g.outlet_id == oid]
    if not len(me): return []
    me = me.iloc[0]; g["km"] = hav(me.lat, me.lon, g.lat.values, g.lon.values); near = g[(g.outlet_id != oid) & (g.km <= max_km)].sort_values("km").head(12)
    need = Q("SELECT sku_name, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5 ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 6", (oid,))
    if not len(need): return [B.callout("This outlet has no SKUs at or near the reorder level (within 150%), so no transfers are suggested.", "info")]
    if not len(near): return [B.callout(f"No outlet within {max_km} km to transfer stock from.", "info")]
    inv = Q(f"SELECT outlet_id, sku_name, current_stock st, reorder_threshold thr, weekly_demand wk FROM inventory WHERE outlet_id IN ({','.join('?' * len(near))}) AND sku_name IN ({','.join('?' * len(need))})", tuple(near.outlet_id) + tuple(need.sku_name))
    inv["spare"] = inv.st - (inv.thr + inv.wk); inv = inv[inv.spare > 0].merge(near[["outlet_id", "km"]], on="outlet_id"); rows = []
    for r in need.itertuples():
        want = max(1, int(r.thr + r.wk - r.st)); c = inv[inv.sku_name == r.sku_name].sort_values("km")
        if len(c):
            s = c.iloc[0]; q = int(min(want, s.spare)); days = max(.5, s.km * 1.3 / 320); rows.append({"sku": r.sku_name, "short_by": want, "from": s.outlet_id, "km": s.km, "units": q, "days": days, "vs_supplier": r.lead - days, "cost": s.km * 1.3 * 14})
        else: rows.append({"sku": r.sku_name, "short_by": want, "from": "none nearby has spare", "km": None, "units": 0, "days": None, "vs_supplier": None, "cost": None})
    d = pd.DataFrame(rows); mp = [{"type": "lines", "lines": [{"pts": [[float(me.lat), float(me.lon)], [float(g[g.outlet_id == r["from"]].lat.iloc[0]), float(g[g.outlet_id == r["from"]].lon.iloc[0])]], "color": COLORS[i % 6], "w": 3} for i, r in enumerate(rows) if r["units"]]},
                                {"type": "points", "id": "alt_o", "pts": [{"id": oid, "lat": float(me.lat), "lon": float(me.lon), "color": "#1f2430", "r": 8, "tip": oid}] + [{"id": r["from"], "lat": float(g[g.outlet_id == r["from"]].lat.iloc[0]), "lon": float(g[g.outlet_id == r["from"]].lon.iloc[0]), "color": TEAL, "r": 6, "tip": r["from"]} for r in rows if r["units"]]}]
    return [B.card("Suggested stock transfers from nearby outlets", B.map_(mp, h=240, key="alt_om", legend=[("This outlet (needs stock)", "#1f2430"), ("Outlet that can send stock", TEAL), ("Transfer line (colour per supplier outlet)", BLUE)]) if any(r["units"] for r in rows) else B.md("No nearby outlet has spare stock for these SKUs."), B.table(d, ["sku", B.num_col("short_by", "short by", "int"), "from", B.num_col("km", "km (straight line)", "int"), B.num_col("units", "transfer units", "int"), B.num_col("days", "transfer days (approx.)", "num"), B.num_col("vs_supplier", "days faster than supplier", "num"), B.num_col("cost", "est. cost ₹", "money")], h=min(80 + 40 * len(d), 330), open={"kind": "outlet", "col": "from"}, key="alt_ot"),
                    sub="Spare = stock above reorder level plus one week of demand. Distance is straight-line; road distance ≈ 1.3×. Cost ₹14 per road km. Compare with the supplier lead time before moving stock.")]


Writing /content/app/alts.py


In [6]:
%%writefile /content/app/analyst.py
"""Question understanding for the Copilot, ahead of the generic table query (qa.py). Each handler computes the answer from live data and returns
{"answer": lead text (markdown), "df": optional table, "sql": how it was computed, "extra": more markdown}. Nothing here asks the language model for a number."""
import re, json, datetime
import numpy as np, pandas as pd
from blocks import Q
import qa as QA, alts
CITY_STATE = {"Mumbai": "Maharashtra", "Pune": "Maharashtra", "Nashik": "Maharashtra", "Nagpur": "Maharashtra", "Delhi": "Delhi", "Bengaluru": "Karnataka", "Mysuru": "Karnataka", "Hyderabad": "Telangana", "Chennai": "Tamil Nadu", "Coimbatore": "Tamil Nadu", "Madurai": "Tamil Nadu",
              "Kolkata": "West Bengal", "Ahmedabad": "Gujarat", "Surat": "Gujarat", "Vadodara": "Gujarat", "Jaipur": "Rajasthan", "Lucknow": "Uttar Pradesh", "Kochi": "Kerala", "Thiruvananthapuram": "Kerala", "Chandigarh": "Chandigarh", "Bhopal": "Madhya Pradesh", "Indore": "Madhya Pradesh",
              "Patna": "Bihar", "Ranchi": "Jharkhand", "Raipur": "Chhattisgarh", "Bhubaneswar": "Odisha", "Guwahati": "Assam", "Goa": "Goa", "Visakhapatnam": "Andhra Pradesh", "Vijayawada": "Andhra Pradesh", "Kanpur": "Uttar Pradesh", "Dehradun": "Uttarakhand", "Amritsar": "Punjab", "Ludhiana": "Punjab"}
CHOKE = {"suez": ("Suez Canal", 30.5, 32.3), "panama": ("Panama Canal", 9.1, -79.7), "malacca": ("Strait of Malacca", 2.5, 101.5), "hormuz": ("Strait of Hormuz", 26.5, 56.5), "bosphorus": ("Bosphorus", 41.1, 29.0), "cape": ("Cape of Good Hope", -34.4, 18.5), "red sea": ("Red Sea", 18.0, 40.0)}
def _w(q, t): return re.search(r"(?<![a-z0-9])" + re.escape(t) + r"(?![a-z0-9])", q) is not None
def _md(df, n=10): return QA.md(df.head(n)) if df is not None and len(df) else ""
def _health(dec): return (((dec or {}).get("ev") or {}).get("health")) or {}
def _agent_scores(e):
    import entities as EN; out = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: out[a] = round(float(v), 1)
    return out
NEXT = {"staffing_shortage": "add a cover shift or move staff from a nearby outlet", "health_alert": "review the lowest agent score below and fix that first", "reorder": "place the reorder or transfer stock from a nearby outlet", "stockout": "reorder now or transfer stock",
        "corrective_action": "complete the corrective plan before the re-audit", "anomaly_review": "check the POS feed and the day's sales for this outlet", "campaign_review": "pause or re-target the campaign", "retention": "talk to the staff at risk and review overtime"}
# ------------------------------------------------------------------------------------------------ Franchise
def explain_outlet(e, dec):
    o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return None
    r = o.iloc[0]; h = _health(dec).get(e); sc = _agent_scores(e); tk = Q("SELECT action_id id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT 8", (e,))
    avg = Q("SELECT AVG(revenue) rv, AVG(customer_satisfaction) cs FROM outlets").iloc[0]; low = Q("SELECT COUNT(*) n FROM inventory WHERE outlet_id=? AND current_stock<=reorder_threshold*1.5", (e,)).n[0]
    att = Q("SELECT AVG(predicted_attrition_prob) p, COUNT(*) n FROM staff WHERE outlet_id=?", (e,)).iloc[0]
    lines = []
    if h is not None: lines.append(f"Health score **{h:.1f}**/100 ({'Critical' if h < 60 else 'Needs attention' if h < 80 else 'Healthy'}).")
    if sc: worst = sorted(sc.items(), key=lambda kv: kv[1])[:2]; lines.append("Weakest areas: " + ", ".join(f"**{a}** {v}" for a, v in worst) + " (each agent scores 0-100, higher is better).")
    lines.append(f"Revenue {r.revenue:,.0f} ({(r.revenue / avg.rv - 1):+.0%} vs chain average), satisfaction {r.customer_satisfaction:.2f} (chain {avg.cs:.2f}), {int(att.n)} staff with average attrition risk {att.p:.0%}, {int(low)} SKUs at or near reorder level.")
    why = "\n".join(f"- {t.type} ({t.severity}): {t.detail}" + (f" → {NEXT[t.type]}" if t.type in NEXT else "") for t in tk.itertuples()) or "- no open tickets"
    return {"answer": (f"**{e}** ({r.outlet_name}, {r.location}) is flagged because:\n" if (h is None or h < 80 or len(tk)) else f"**{e}** ({r.outlet_name}, {r.location}) is not flagged right now; here is its status:\n") + "\n".join("- " + l for l in lines) + "\n\n**Open tickets:**\n" + why, "df": pd.DataFrame([{"agent": a, "score": v} for a, v in sc.items()]) if sc else None, "sql": f"decision health + agent per_outlet scores + agent_actions for {e}"}
def risk_rank(q, dec, scope, n=5):
    hl = _health(dec)
    if not hl: return None
    s = pd.Series(hl).astype(float); worst = bool(re.search(r"\b(highest risk|most at risk|worst|lowest|riskiest|critical|weakest|needs? attention|problem|bottom)\b", q)); best = bool(re.search(r"\b(best|top|highest health|strongest|healthiest)\b", q)) and not re.search(r"highest risk", q)
    both = re.search(r"best.*worst|worst.*best", q) is not None
    if scope: s = s[s.index == scope]
    o = Q("SELECT outlet_id, outlet_name, location, revenue, customer_satisfaction csat FROM outlets").set_index("outlet_id")
    def frame(ix):
        rows = []
        for e in ix:
            sc = _agent_scores(e); wk = min(sc, key=sc.get) if sc else "-"; ct = int(Q("SELECT COUNT(*) n FROM agent_actions WHERE target=? AND status IN ('open','proposed') AND severity='Critical'", (e,)).n[0])
            rows.append({"outlet": e, "name": o.name if False else (o.outlet_name.get(e, "") if e in o.index else ""), "location": o.location.get(e, ""), "health": round(float(s[e]), 1), "weakest_area": f"{wk} {sc.get(wk, '')}", "critical_tickets": ct, "revenue": o.revenue.get(e), "csat": o.csat.get(e)})
        return pd.DataFrame(rows)
    low = s.sort_values().head(n).index; hi = s.sort_values(ascending=False).head(n).index
    if both or (best and worst):
        a, b = frame(hi), frame(low); return {"answer": f"Best outlet: **{a.outlet[0]}** (health {a.health[0]}). Worst outlet: **{b.outlet[0]}** (health {b.health[0]}). Health combines outlet, inventory, staff, marketing, audit and feedback agent scores.", "df": None, "sql": "decision health per outlet", "extra": "**Top " + str(n) + " by health**\n" + QA.md(a) + "\n\n**Bottom " + str(n) + " by health**\n" + QA.md(b)}
    if best and not worst: d = frame(hi); return {"answer": f"Healthiest outlet: **{d.outlet[0]}** (health {d.health[0]}, {d.location[0]}). Top {n} below.", "df": d, "sql": "decision health per outlet, descending"}
    d = frame(low); crit = int((s < 60).sum()); return {"answer": f"Highest risk: **{d.outlet[0]}** (health {d.health[0]}, weakest area {d.weakest_area[0]}). {crit} outlets are below 60 (critical). Lowest {n} below; ask \"why is {d.outlet[0]} flagged\" for the reasons.", "df": d, "sql": "decision health per outlet, ascending"}
def inventory_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name, category, current_stock st, reorder_threshold thr, weekly_demand wk, lead_time_days lead, stockout_risk_prob p FROM inventory" + w, p)
    if not len(d): return None
    d["cover_d"] = d.st / (d.wk / 7).replace(0, np.nan); below = d[d.st <= d.thr]; wk1 = d[d.cover_d < 7]; lt = d[d.cover_d < d.lead]
    po = Q("SELECT status, COUNT(*) n FROM purchase_orders" + w + " GROUP BY status", p); cat = d.groupby("category").agg(skus=("sku_name", "count"), below_reorder=("st", lambda s: int((d.loc[s.index, "st"] <= d.loc[s.index, "thr"]).sum())), avg_cover_days=("cover_d", "mean"), avg_risk=("p", "mean")).reset_index().sort_values("avg_risk", ascending=False)
    worst = d.sort_values("cover_d").head(8)[["outlet_id", "sku_name", "st", "cover_d", "lead", "p"]].rename(columns={"st": "stock", "cover_d": "days_of_cover", "lead": "lead_days", "p": "stockout_risk"})
    return {"answer": f"Inventory: **{len(d):,}** SKU records; **{len(below):,}** at or below reorder level, **{len(wk1):,}** have under 7 days of cover, **{len(lt):,}** will run out before a new order can arrive (cover shorter than lead time). Average stock-out risk {d.p.mean():.0%}." + (" Open purchase orders: " + ", ".join(f"{r.status} {r.n}" for r in po.itertuples()) + "." if len(po) else ""), "df": worst, "sql": "inventory: stock vs reorder threshold, cover = stock ÷ daily demand", "extra": "**By category**\n" + QA.md(cat.round(2))}
def sales_trend(scope):
    w = " AND outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q(f"SELECT substr(ts,1,10) day, SUM(amount) amount, SUM(tx_count) tx FROM sales WHERE ts IS NOT NULL{w} GROUP BY day ORDER BY day DESC LIMIT 14", p)
    if len(d) < 2: return None
    d = d.iloc[::-1].reset_index(drop=True); last7, prev7 = d.tail(7).amount.sum(), d.iloc[:-7].tail(7).amount.sum() if len(d) > 7 else np.nan
    ch = f" ({(last7 / prev7 - 1):+.1%} vs the 7 days before)" if prev7 == prev7 and prev7 else ""
    return {"answer": f"Sales over the last 7 recorded days: **{last7:,.0f}**{ch}. Best day {d.loc[d.amount.idxmax(), 'day']} ({d.amount.max():,.0f}), weakest {d.loc[d.amount.idxmin(), 'day']} ({d.amount.min():,.0f}).", "df": d.tail(10), "sql": "sales grouped by day"}

def _place(ql):
    """Find a state or city named in the question. Returns ("state"|"city", name) or None."""
    flat = re.sub(r"[^a-z]", "", ql); cities = Q("SELECT DISTINCT location FROM outlets").location.dropna().tolist()
    for st in sorted(set(CITY_STATE.values()), key=len, reverse=True):
        if re.sub(r"[^a-z]", "", st.lower()) in flat: return "state", st
    for ct in sorted(cities, key=len, reverse=True):
        if re.search(r"\b" + re.escape(ct.lower()) + r"\b", ql): return "city", ct
    return None
def outlets_in_place(ql, scope=None):
    pl = _place(ql)
    if not pl: return None
    kind, name = pl; o = Q("SELECT outlet_id, outlet_name name, location, tier, revenue, customer_satisfaction csat, staff_headcount staff FROM outlets")
    o["state"] = o.location.map(CITY_STATE).fillna("Other"); d = o[o.state == name] if kind == "state" else o[o.location == name]
    if scope: d = d[d.outlet_id == scope]           # RBAC: a scoped role only sees its own outlet
    if not len(d): return {"answer": f"No outlets found in {name}.", "df": None, "sql": "outlets filtered by place"}
    by = d.groupby("location").size().sort_values(ascending=False); txt = ", ".join(f"{k} {v}" for k, v in by.items())
    return {"answer": f"**{len(d)}** outlets in {name}" + (f": {txt}." if kind == "state" else "."), "df": d.drop(columns=["state"]).sort_values(["location", "outlet_id"]), "sql": f"outlets where {'state (from city)' if kind == 'state' else 'location'} = {name}"}
def staff_by_place(ql, scope=None):
    s = Q("SELECT s.staff_id, s.outlet_id, o.location FROM staff s LEFT JOIN outlets o USING(outlet_id)")
    if not len(s): return None
    if scope: s = s[s.outlet_id == scope]           # RBAC: a scoped role only sees its own outlet
    s["state"] = s.location.map(CITY_STATE).fillna("Other"); pl = _place(ql)
    if pl:
        kind, name = pl; d = s[s.state == name] if kind == "state" else s[s.location == name]; g = d.groupby("location").size().sort_values(ascending=False).reset_index(name="employees")
        return {"answer": f"**{len(d):,}** employees in {name}" + (f"; most in **{g.location.iloc[0]}** ({int(g.employees.iloc[0]):,})." if len(g) else "."), "df": g, "sql": f"staff joined to outlets, place = {name}"}
    by_state = bool(re.search(r"\bstates?\b", ql)); by_out = bool(re.search(r"\boutlets?\b", ql)); col = "outlet_id" if by_out else ("state" if by_state else "location")
    g = s.groupby(col).size().sort_values(ascending=False).head(15).reset_index(name="employees"); low = bool(re.search(r"\b(fewest|least|lowest|less|low number)\b", ql)); r = (s.groupby(col).size().sort_values().reset_index(name="employees").iloc[0] if low else g.iloc[0])
    return {"answer": f"**{r[col]}** has the {'fewest' if low else 'most'} employees: **{int(r.employees):,}** (of {len(s):,} in total, by {'outlet' if by_out else 'state' if by_state else 'city'}).", "df": g, "sql": f"staff joined to outlets, grouped by {col}"}
def by_state(q):
    o = Q("SELECT outlet_id, location, revenue FROM outlets"); o["state"] = o.location.map(CITY_STATE).fillna("Other"); g = o.groupby("state").agg(outlets=("outlet_id", "count"), revenue=("revenue", "sum")).reset_index().sort_values("outlets", ascending=False)
    low = bool(re.search(r"\b(fewest|least|lowest)\b", q)); r = g.iloc[-1] if low else g.iloc[0]
    return {"answer": f"**{r.state}** has the {'fewest' if low else 'most'} outlets: **{int(r.outlets)}**. Your outlets table stores the city, so states are derived from the city name.", "df": g, "sql": "outlets grouped by state (city → state mapping)"}
def staff_status(scope):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    s = Q("SELECT outlet_id, role, overtime_hrs ot, predicted_attrition_prob pr FROM staff" + w, p); at = Q("SELECT status, COUNT(*) n FROM attendance" + (" WHERE outlet_id=?" if scope else "") + " GROUP BY status", p)
    if not len(s): return None
    hi = s[s.pr > .6]; g = s.groupby("outlet_id").pr.mean().sort_values(ascending=False).head(8).reset_index().rename(columns={"pr": "avg_attrition_risk"})
    return {"answer": f"Workforce: **{len(s):,}** staff, **{len(hi):,}** at high attrition risk (>60%), average overtime {s.ot.mean():.1f} h" + (". Attendance: " + ", ".join(f"{r.status} {r.n:,}" for r in at.itertuples()) if len(at) else "") + ".", "df": g, "sql": "staff + attendance", }
# ------------------------------------------------------------------------------------------------ Freight
STOP = {"long", "port", "new", "fort", "saint", "south", "north", "east", "west", "city", "bay", "cape", "grand", "great", "royal", "santa", "puerto", "king", "bandar"}
def _ports_in(q):
    p = Q("SELECT port_name FROM ports").port_name.tolist(); spans = []; first = {}
    for name in p:
        core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and len(core[0]) >= 4 and core[0] not in STOP: first.setdefault(core[0], []).append(name)
    for name in p:
        als = set(QA._alias(name)); core = [w for w in re.sub(r"\(.*?\)", "", name).lower().split() if w not in QA.GENERIC]
        if core and core[0] not in STOP and len(first.get(core[0], [])) == 1 and len(core[0]) >= 4: als.add(core[0])
        for al in sorted(als, key=len, reverse=True):
            m = re.search(r"(?<![a-z0-9])" + re.escape(al) + r"(?![a-z0-9])", q)
            if m: spans.append((m.start(), m.end(), len(al), name)); break
    spans.sort(key=lambda x: (-x[2], x[0])); kept = []
    for s_ in spans:
        if not any(s_[0] < k[1] and k[0] < s_[1] for k in kept): kept.append(s_)
    kept.sort(); return [k[3] for k in kept], kept
def plan_lane(q, dom):
    ps, hits = _ports_in(re.sub(r"how long", "how_long", q.lower()))
    if len(ps) < 2: return None
    pre = q.lower(); o, d = ps[0], ps[1]; m = re.search(r"from\s+(.+?)\s+to\s+", pre)
    cargo = next((k for k in dom.CARGO if k.lower() in pre or k.lower().rstrip("s") in pre), "General"); wm = re.search(r"(\d+(?:\.\d+)?)\s*(t|tons?|tonnes?|mt|kg|kgs)\b", pre); wt = 20.0
    if wm: wt = float(wm.group(1)) / (1000 if wm.group(2).startswith("kg") else 1)
    po, pd_ = dom.port_xy(o), dom.port_xy(d)
    if not po or not pd_: return None
    cust = Q("SELECT customer_id FROM customers LIMIT 1").customer_id[0]; req = {"origin": o, "dest": d, "cargo": cargo, "weight_t": wt, "hs_code": "HS-8501", "customer_id": cust, "docs_provided": None}
    pq = dom.preview_quote(req); opts = alts.route_alternatives(dom, (po[0], po[1]), (pd_[0], pd_[1]), tonnes=wt, interp=dom.interp, dwell=(po[3] + pd_[3]) * .5)
    if not pq and not opts: return None
    rows = [{"option": f"{i + 1}. {x['label']}", "route": x["name"], "nm": x["nm"], "days": round(x["days"], 1), "freight_cost": round(x["freight"]), "time_cost": round(x["delay_cost"]), "total": round(x["total"]), "tag": x["tag"]} for i, x in enumerate(opts)]
    head = f"**{o} → {d}**, {wt:g} t of {cargo}: "
    if pq: head += f"recommended route **{pq['route']}**, about **{pq['days']} days** at sea and in port, quoted price **${pq['price']:,.0f}** (margin {pq['margin_pct']}%), weather risk {pq['weather']}, decision: {pq['decision']}."
    elif opts: head += f"best route **{opts[0]['name']}**, about **{opts[0]['days']} days**."
    ex = (pq["narrative"] if pq and pq.get("narrative") else "") + ("\n\nOnly one sea route exists for this lane in the routing model (it does not pass a closable canal or strait), so the alternatives are commercial: another carrier, or diverting to a nearby less-congested port (open the port in the map to see candidates)." if len(opts) == 1 else "")
    return {"answer": head, "df": pd.DataFrame(rows) if rows else None, "sql": "dom.preview_quote (route, weather, ML price, customs, margin) + sea-route alternatives; nothing was saved", "extra": ex, "map": opts}
def choke_ports(q):
    k = next((k for k in CHOKE if _w(q, k)), None)
    if not k: return None
    name, la, lo = CHOKE[k]; p = Q("SELECT port_name, country, congestion_index ci, avg_dwell_days dw, lat, lon FROM ports"); p["km"] = alts.hav(la, lo, p.lat.values, p.lon.values); near = p[p.km <= 1800].sort_values("ci", ascending=False)
    v = Q("SELECT COUNT(*) n FROM vessels WHERE route_name LIKE ?", (f"%{name.split()[0]}%",)).n[0]; sh = Q("SELECT COUNT(*) n FROM shipments s JOIN vessels v USING(shipment_id) WHERE v.route_name LIKE ? AND s.status LIKE '%Delay%'", (f"%{name.split()[0]}%",)).n[0]
    if not len(near): return None
    return {"answer": f"Ports within 1,800 km of the **{name}**: **{len(near)}**; **{int((near.ci >= 4).sum())}** are congested (index ≥ 4). Most congested: **{near.port_name.iloc[0]}** ({near.ci.iloc[0]:.1f}, dwell {near.dw.iloc[0]:.1f} days). {int(v)} tracked vessels are routed via this passage, {int(sh)} of them delayed.", "df": near[["port_name", "country", "ci", "dw", "km"]].rename(columns={"ci": "congestion", "dw": "dwell_days", "km": "km_from_passage"}), "sql": f"ports within 1,800 km of {name}"}
def freight_summary(q):
    if re.search(r"\bcustoms\b", q) and re.search(r"\b(hold|held|status|summary|how many|why|docs?|documents?)\b", q):
        d = Q("SELECT dest_port, COUNT(*) n FROM shipments WHERE status='Customs Hold' GROUP BY dest_port ORDER BY n DESC LIMIT 8"); t = int(Q("SELECT COUNT(*) n FROM shipments WHERE status='Customs Hold'").n[0]); tk = Q("SELECT detail FROM agent_actions WHERE type='customs_docs' AND status IN ('open','proposed') LIMIT 400")
        miss = pd.Series([x for dd in tk.detail for x in re.findall(r"missing ([^:]+?)(?:\.|$)", dd)[0].split(", ")] if len(tk) and all("missing" in x for x in tk.detail) else []).value_counts().head(6)
        return {"answer": f"**{t}** shipments are on customs hold. Most held at **{d.dest_port.iloc[0] if len(d) else '-'}**." + (" Most often missing: " + ", ".join(f"{k} ({v})" for k, v in miss.items()) + "." if len(miss) else ""), "df": d.rename(columns={"n": "held"}), "sql": "shipments with status Customs Hold; customs_docs tickets"}
    return None
def greeting(q, kind):
    ql = re.sub(r"[^a-z0-9 ?']", " ", q.lower()).strip()
    hi = re.match(r"^(hi+|hello+|hey+|hola|namaste|good (morning|afternoon|evening)|greetings|yo|sup|howdy)\b( there| team| copilot| buddy)?$", ql)
    thanks = re.match(r"^(thanks?|thank you|thx|ok(ay)?|cool|great|nice|got it|awesome)( a lot| so much)?$", ql)
    who = re.search(r"\b(who are you|what are you|what can you do|what do you do|help me|how can you help|your capabilities|what is this)\b", ql) or ql in ("help", "?")
    how = re.match(r"^how are you", ql)
    if not (hi or thanks or who or how): return None
    ex = (["Which stock needs immediate refill?", "Which outlets are at highest risk and why?", "Top 5 outlets by revenue", "Why is OUT-007 flagged?", "Which state has the most outlets?"] if kind == "Franchise" else
          ["Mumbai to Singapore 50 t electronics: days, cost and best route", "Which ports are most congested?", "How many shipments are on customs hold?", "Which quotes have the lowest margin?"])
    what = ("outlets, stock, staff, marketing, audits, customer feedback and anomalies" if kind == "Franchise" else "routes, ports, weather, customs, quotes and carriers")
    if thanks: return {"answer": "You're welcome. Ask me anything else about " + what + ".", "df": None, "sql": ""}
    head = "Hello! " if hi else ("I'm doing well, thank you. " if how else "")
    return {"answer": head + f"I'm your operations copilot. I answer from your live data on {what}, and I can explain *why* something is flagged, suggest actions and send them to the approval queue. I understand 37 languages.\n\nTry one of these:\n" + "\n".join(f"- {e}" for e in ex), "df": None, "sql": ""}

def refill(scope, n=10):
    w = " WHERE outlet_id=?" if scope else ""; p = (scope,) if scope else ()
    d = Q("SELECT outlet_id, sku_name sku, category, current_stock stock, reorder_threshold thr, weekly_demand wk, lead_time_days lead FROM inventory" + w, p)
    if not len(d): return None
    d["daily"] = d.wk / 7.0; d["cover"] = d.stock / d.daily.replace(0, np.nan); d["cover"] = d.cover.fillna(999)
    d["order_qty"] = np.maximum(d.thr * 3, np.ceil(d.daily * (d.lead + 7)) - d.stock).astype(int)
    d["gap"] = d.cover - d.lead
    d["urgency"] = np.where(d.cover < d.lead, "Critical: runs out before delivery", np.where(d.stock <= d.thr, "High: below reorder level", np.where(d.cover < 7, "Medium: under 7 days", "OK")))
    d["rank"] = d.urgency.str[:4].map({"Crit": 0, "High": 1, "Medi": 2}).fillna(3); need = d[d.urgency != "OK"].sort_values(["rank", "gap", "cover"])
    if not len(need): return {"answer": "No SKU needs an urgent refill right now: all have stock above the reorder level and cover longer than the supplier lead time.", "df": None, "sql": "inventory: cover = stock / daily demand vs lead time"}
    crit = need[need.urgency.str.startswith("Critical")]; po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status IN ('open','ordered','pending','Pending','Ordered','Open')" + (" AND outlet_id=?" if scope else ""), p) if "purchase_orders" in [r[0] for r in Q("SELECT name FROM sqlite_master WHERE type='table'").values] else None
    top = need.head(n)[["outlet_id", "sku", "stock", "thr", "cover", "lead", "order_qty", "urgency"]].rename(columns={"thr": "reorder_level", "cover": "days_of_cover", "lead": "lead_days"}); top["days_of_cover"] = top.days_of_cover.round(1)
    pool = crit if len(crit) else need[need.urgency.str.startswith("High")] if (need.urgency.str.startswith("High")).any() else need; by = pool.groupby("outlet_id").size().sort_values(ascending=False).head(5)
    outs = ", ".join(f"{k} ({v} SKUs)" for k, v in by.items()) if len(by) else "none"
    return {"answer": f"**{len(crit):,}** SKU records will run out *before a new order can arrive* (days of cover < supplier lead time) and **{len(need) - len(crit):,}** more are low. Most urgent outlets: {outs}. The table lists the {len(top)} most urgent SKUs, with the quantity to order.",
            "df": top, "sql": "inventory: days_of_cover = stock / (weekly_demand/7); urgent when cover < lead_time_days; order_qty = max(3 x reorder level, daily demand x (lead + 7 days) - stock)",
            "extra": "**How to read it:** *days_of_cover* is how long current stock lasts at current demand; *lead_days* is how long the supplier takes. When cover is shorter than lead time, stock-out is certain unless you order now (or transfer stock from a nearby outlet). The matching *reorder* tickets are raised by the agent."}

# ------------------------------------------------------------------------------------------------ router
def answer(kind, q, scope=None, dec=None, dom=None, mask=False):
    if kind == "Franchise":
        import intents                                   # the 23 SQL question patterns (spelling tolerant, scope and mask aware)
        r = intents.answer(q, scope, mask)
        if r: return r
    r = _answer(kind, q, scope, dec, dom)
    if r: r["complete"] = True
    return r
def _answer(kind, q, scope=None, dec=None, dom=None):
    g = greeting(q, kind)
    if g: return g
    ql = q.lower().strip(); ids = re.findall(r"\b(OUT-\d+|STF-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+)\b", q, re.I)
    try:
        if kind == "Franchise":
            if ids and ids[0].upper().startswith("OUT") and re.search(r"\b(why|reason|flag\w*|wrong|problem|issue|explain|risk)\b", ql):
                if scope and ids[0].upper() != scope: return {"answer": f"{ids[0].upper()} is outside your outlet scope ({scope}).", "df": None, "sql": ""}
                return explain_outlet(ids[0].upper(), dec)
            _pl = (bool(re.search(r"\b(where|which|list|all|show|located|locations?|states?|cities|city|how many|number)\b", ql)) or bool(_place(ql))) and not re.search(r"\b(risk\w*|health\w*|revenue|sales|csat|satisfaction|inventory|stock|attrition|salary|overtime|best|worst|weakest|critical)\b", ql)
            if _pl and re.search(r"\b(employees?|staff|workers?|people|headcount|manpower|workforce)\b", ql) and (_place(ql) or re.search(r"\b(where|which|most|highest|more|largest|big|fewest|least|lowest|number)\b", ql)): return staff_by_place(ql, scope)
            if _pl and re.search(r"\b(outlets?|stores?|branch\w*|franchises?)\b", ql) and _place(ql): return outlets_in_place(ql, scope)
            if re.search(r"\b(state|states)\b", ql) and re.search(r"\b(outlets?|stores?|branch\w*)\b", ql): return by_state(ql) if not scope else {"answer": "State-wise comparisons cover the whole chain and are not available for your outlet role.", "df": None, "sql": ""}
            if re.search(r"\b(outlets?|stores?|branch\w*)\b", ql) and re.search(r"\b(risk\w*|health\w*|best|worst|weakest|strongest|critical|attention|problem\w*|performing|performance)\b", ql) and not re.search(r"\b(revenue|sales|csat|satisfaction|headcount)\b", ql): return risk_rank(ql, dec, scope)
            if re.search(r"\b(inventory|stock|stocks|stocked|sku|skus|items?|products?|ingredients?)\b", ql) and re.search(r"\b(refill\w*|restock\w*|re-?order\w*|replenish\w*|immediate\w*|urgent\w*|attention|run(ning)? (out|low)|shortage\w*|low|critical|need\w*|order now)\b", ql) and not re.search(r"\b(how many|count|average|total)\b", ql): return refill(scope)
            if re.search(r"\b(inventory|stock)\b", ql) and re.search(r"\b(how|status|holding|summary|overview|situation|look|doing|week|today|health)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return inventory_status(scope)
            if re.search(r"\b(sales|revenue|takings)\b", ql) and re.search(r"\b(trend|this week|last week|today|yesterday|last 7|past week|recent|how (is|are|did))\b", ql): return sales_trend(scope)
            if re.search(r"\b(staff|workforce|employees?|team)\b", ql) and re.search(r"\b(how|status|summary|overview|situation|doing|look)\b", ql) and not re.search(r"\b(how many|count|average|total|top|lowest|highest|which)\b", ql): return staff_status(scope)
        else:
            if dom is not None and re.search(r"\b(days?|time|transit|cost|price|quote|route|how long|how much|eta)\b", ql):
                r = plan_lane(q, dom)
                if r: return r
            if re.search(r"\b(congest\w*|delay\w*|busy|ports?)\b", ql) and any(_w(ql, k) for k in CHOKE): return choke_ports(ql)
            r = freight_summary(ql)
            if r: return r
    except Exception as e:
        import sys; print("analyst failed:", repr(e), file=sys.stderr)
    return None


Writing /content/app/analyst.py


In [7]:
%%writefile /content/app/blocks.py
"""Server-driven UI blocks. Pages are Python functions returning JSON blocks; the single-page front end renders them and live-updates only what changed.
Charts are ECharts option dicts built here with one consistent palette."""
import math, json, datetime
import numpy as np, pandas as pd
import live_engine as LE

TEAL, BLUE, AMBER, RED, VIOLET, GREEN, PINK, GREY = "#0d9488", "#2563eb", "#f59e0b", "#e5484d", "#7c3aed", "#16a34a", "#db2777", "#64748b"
PRI = "#0f766e"
PAL = [PRI, BLUE, AMBER, VIOLET, RED, GREEN, PINK, TEAL, GREY]
TONE = {"Critical": RED, "High": RED, "Warning": AMBER, "Medium": AMBER, "Info": BLUE, "Low": TEAL, "Pass": TEAL, "Fail": RED}

def Q(sql, p=()):
    with LE._DB_LOCK:
        return pd.read_sql(sql, LE.conn(), params=p)

def clean(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating, float)): return None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))) or (isinstance(v, np.floating) and (np.isnan(v) or np.isinf(v)))) else round(float(v), 4)
    if isinstance(v, (np.bool_,)): return bool(v)
    if isinstance(v, (pd.Timestamp, datetime.datetime, datetime.date)): return str(v)
    if isinstance(v, (list, tuple)): return [clean(x) for x in v]
    if isinstance(v, dict): return {str(k): clean(x) for k, x in v.items()}
    if v is pd.NA or v is pd.NaT: return None
    return v
def L(x): return [clean(v) for v in (x.tolist() if hasattr(x, "tolist") else list(x))]

# ------------------------------------------------------------------ charts (ECharts options)
def _base(extra=None, legend=False, grid=None):
    o = {"animation": True, "animationDuration": 350, "color": PAL, "tooltip": {"trigger": "axis", "confine": True}, "grid": grid or {"left": 8, "right": 14, "top": 28 if legend else 14, "bottom": 6, "containLabel": True}}
    if legend: o["legend"] = {"top": 0, "type": "scroll"}
    if extra: o.update(extra)
    return o
def bar(cats, series, horizontal=False, stack=False, colors=None, fmt=None, max_=None, pct=False, itemcolors=None, sort=False):
    """series: {name: values} or a list of values."""
    series = {"": series} if not isinstance(series, dict) else series; cats = L(cats)
    if sort and len(series) == 1:
        v = L(list(series.values())[0]); order = sorted(range(len(v)), key=lambda i: (v[i] is None, v[i] or 0)); cats = [cats[i] for i in order]; series = {k: [L(x)[i] for i in order] for k, x in series.items()}
    ser = []
    for i, (n, v) in enumerate(series.items()):
        d = L(v); s = {"type": "bar", "name": n, "data": d, "barMaxWidth": 22, "itemStyle": {"borderRadius": [0, 4, 4, 0] if horizontal else [4, 4, 0, 0], "color": (colors or PAL)[i % len(colors or PAL)]}, "emphasis": {"focus": "series"}}
        if itemcolors and i == 0: s["data"] = [{"value": x, "itemStyle": {"color": c}} for x, c in zip(d, itemcolors)]
        if stack: s["stack"] = "s"
        ser.append(s)
    cat = {"type": "category", "data": cats, "axisLabel": {"hideOverlap": True}}; val = {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if max_: val["max"] = max_
    if pct: val["axisLabel"] = {"formatter": "{value}%"}
    o = _base(legend=len(series) > 1 and any(series)); o.update({"xAxis": val if horizontal else cat, "yAxis": cat if horizontal else val, "series": ser})
    return o
def line(x, series, area=True, band=None, colors=None, dashed=(), ymin=None, ymax=None, step=False, markers=False):
    series = {"": series} if not isinstance(series, dict) else series; ser = []
    for i, (n, v) in enumerate(series.items()):
        c = (colors or PAL)[i % len(colors or PAL)]; s = {"type": "line", "name": n, "data": L(v), "smooth": not step, "showSymbol": markers, "symbolSize": 5, "lineStyle": {"width": 2, "color": c, "type": "dashed" if n in dashed else "solid"}, "itemStyle": {"color": c}}
        if step: s["step"] = "end"
        if area and i == 0 and len(series) <= 2: s["areaStyle"] = {"color": {"type": "linear", "x": 0, "y": 0, "x2": 0, "y2": 1, "colorStops": [{"offset": 0, "color": c + "55"}, {"offset": 1, "color": c + "00"}]}}
        ser.append(s)
    if band:
        lo, hi = L(band[0]), L(band[1]); ser += [{"type": "line", "data": lo, "stack": "b", "lineStyle": {"opacity": 0}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}, {"type": "line", "data": [None if (a is None or b is None) else b - a for a, b in zip(lo, hi)], "stack": "b", "lineStyle": {"opacity": 0}, "areaStyle": {"color": "rgba(251,191,36,.16)"}, "showSymbol": False, "silent": True, "tooltip": {"show": False}}]
    y = {"type": "value", "scale": True, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    if ymin is not None: y["min"] = ymin
    if ymax is not None: y["max"] = ymax
    o = _base(legend=len([s for s in series if s]) > 1); o.update({"xAxis": {"type": "category", "data": L(x), "boundaryGap": False, "axisLabel": {"hideOverlap": True}}, "yAxis": y, "series": ser}); return o
def pie(items, donut=True, colors=None):
    """items: {name: value} or list of (name, value, color?)"""
    its = list(items.items()) if isinstance(items, dict) else items; data = []
    for i, it in enumerate(its):
        d = {"name": str(it[0]), "value": clean(it[1])}
        if (colors and str(it[0]) in colors) or len(it) > 2: d["itemStyle"] = {"color": (colors or {}).get(str(it[0])) or it[2]}
        data.append(d)
    return {"animation": True, "color": PAL, "tooltip": {"trigger": "item", "confine": True}, "legend": {"bottom": 0, "type": "scroll"}, "series": [{"type": "pie", "radius": ["52%", "78%"] if donut else "75%", "center": ["50%", "44%"], "avoidLabelOverlap": True, "label": {"show": False}, "itemStyle": {"borderColor": "#ffffff", "borderWidth": 2, "borderRadius": 4}, "data": data}]}
def scatter(groups, xname="", yname="", colors=None, sized=False):
    """groups: {name: [ {x,y,label,size?} ]}. Points are [x, y, label, size]; the client resolves the {"__fn": ...} formatters."""
    ser = []
    for i, (n, pts) in enumerate(groups.items()):
        ser.append({"type": "scatter", "name": n, "symbolSize": {"__fn": "size3"} if sized else 10, "data": [[clean(p["x"]), clean(p["y"]), p.get("label", ""), clean(p.get("size", 10))] for p in pts],
                    "itemStyle": {"color": (colors or {}).get(n, PAL[i % len(PAL)]), "opacity": .85}, "emphasis": {"focus": "series"}})
    o = _base(legend=len(groups) > 1); o["tooltip"] = {"trigger": "item", "confine": True, "formatter": {"__fn": "scatterTip"}}
    ax = lambda n: {"type": "value", "name": n, "scale": True, "nameTextStyle": {"color": "#6b7080"}, "splitLine": {"lineStyle": {"color": "#eceef6"}}}
    o.update({"xAxis": ax(xname), "yAxis": ax(yname), "series": ser}); return o
def hist(values, bins=24, color=TEAL, pct=False):
    v = np.asarray([x for x in values if x is not None and not (isinstance(x, float) and math.isnan(x))], dtype=float)
    if not len(v): return bar([], [])
    h, e = np.histogram(v, bins=bins); labels = [f"{(e[i] * (100 if pct else 1)):.{0 if pct else 1}f}" + ("%" if pct else "") for i in range(len(h))]
    return bar(labels, [int(x) for x in h], colors=[color])
def waterfall(labels, deltas, total_label="Total"):
    base, run, up = [], 0.0, []
    for d in deltas: base.append(run if d >= 0 else run + d); up.append(abs(d)); run += d
    cats = list(labels) + [total_label]; base.append(0); up.append(run)
    o = _base(); o.update({"xAxis": {"type": "category", "data": cats}, "yAxis": {"type": "value", "splitLine": {"lineStyle": {"color": "#eceef6"}}},
                           "series": [{"type": "bar", "stack": "w", "data": [clean(x) for x in base], "itemStyle": {"color": "transparent"}, "tooltip": {"show": False}}, {"type": "bar", "stack": "w", "data": [{"value": clean(u), "itemStyle": {"color": TEAL if i == len(up) - 1 else (BLUE if deltas[i] >= 0 else RED)}} for i, u in enumerate(up)], "barMaxWidth": 34, "itemStyle": {"borderRadius": 4}}]}); return o
def radar(names, series, mx=100):
    o = {"animation": True, "color": PAL, "tooltip": {"confine": True}, "legend": {"bottom": 0}, "radar": {"indicator": [{"name": n, "max": mx} for n in names], "splitArea": {"show": False}, "axisName": {"color": "#6b7080"}}, "series": [{"type": "radar", "data": [{"name": k, "value": L(v), "areaStyle": {"opacity": .18}} for k, v in series.items()]}]}; return o
def gauge(value, name="", mx=100):
    c = RED if value < 60 else AMBER if value < 80 else TEAL
    return {"series": [{"type": "gauge", "startAngle": 210, "endAngle": -30, "min": 0, "max": mx, "progress": {"show": True, "width": 14, "itemStyle": {"color": c}}, "axisLine": {"lineStyle": {"width": 14, "color": [[1, "rgba(148,163,184,.15)"]]}}, "axisTick": {"show": False}, "splitLine": {"show": False}, "axisLabel": {"show": False}, "pointer": {"show": False}, "anchor": {"show": False},
                         "detail": {"valueAnimation": True, "fontSize": 30, "fontWeight": 700, "offsetCenter": [0, "8%"], "color": "#1c2024", "formatter": "{value}"}, "title": {"offsetCenter": [0, "62%"], "color": "#6b7080", "fontSize": 12}, "data": [{"value": clean(round(value, 1)), "name": name}]}]}
def graph(nodes, links, h=520):
    """nodes: [{id,name,kind,size,color,tip}] ; links: [(a,b)]"""
    cat = sorted({n.get("kind", "x") for n in nodes}); return {"animation": True, "tooltip": {"confine": True, "formatter": {"__fn": "nodeTip"}}, "legend": {"data": cat, "bottom": 0, "textStyle": {"color": "#6b7080"}}, "color": PAL,
        "series": [{"type": "graph", "layout": "force", "roam": True, "draggable": True, "force": {"repulsion": 260, "edgeLength": [40, 120], "gravity": .08}, "label": {"show": True, "position": "right", "color": "#454a58", "fontSize": 10, "formatter": {"__fn": "nodeLabel"}},
                    "categories": [{"name": c} for c in cat], "lineStyle": {"color": "#cfd3e3", "width": 1}, "emphasis": {"focus": "adjacency", "lineStyle": {"width": 2}},
                    "data": [{"id": n["id"], "name": n["id"], "category": cat.index(n.get("kind", "x")), "symbolSize": n.get("size", 12), "itemStyle": {"color": n.get("color")} if n.get("color") else {}, "tip": n.get("tip", ""), "showLabel": n.get("label", False), "open": n.get("open")} for n in nodes],
                    "links": [{"source": a, "target": b} for a, b in links]}]}

# ------------------------------------------------------------------ blocks
def kpis(*items):
    out = []
    for it in items:
        d = {"label": it[0], "value": clean(it[1])}
        if len(it) > 2 and it[2] is not None: d["sub"] = str(it[2])
        if len(it) > 3 and it[3]: d["tone"] = it[3]
        out.append(d)
    return {"t": "kpis", "items": out}
def chart(opt, title=None, h=290, open=None, sub=None, key=None): return {"t": "chart", "opt": opt, "title": title, "h": h, "open": open, "sub": sub, "key": key or title}
def table(df, cols=None, title=None, h=None, open=None, key=None, sub=None, search=False, page=None):
    """cols: list of names or dicts {k,label,type: text|num|int|pct|pill|bar|money|time|id, max}. open = {"kind":"outlet","col":"outlet_id"} makes rows clickable."""
    if isinstance(df, list): df = pd.DataFrame(df)
    cols = cols or list(df.columns); cs = [{"k": c, "label": c.replace("_", " ")} if isinstance(c, str) else c for c in cols]
    rows = [{c["k"]: clean(r.get(c["k"])) for c in cs} for r in df.to_dict("records")] if len(df) else []
    return {"t": "table", "cols": cs, "rows": rows, "title": title, "h": h, "open": open, "key": key or title, "sub": sub, "search": search}
def callout(text, tone="info", title=None): return {"t": "callout", "text": text, "tone": tone, "title": title}
def md(text): return {"t": "md", "text": text}
def row(*cols, ratio=None): return {"t": "row", "cols": list(cols), "ratio": ratio}
def col(*blocks): return {"t": "col", "blocks": list(blocks)}
def card(title, *blocks, sub=None): return {"t": "card", "title": title, "sub": sub, "blocks": list(blocks)}
def form(id, fields, submit="Submit", url=None, title=None, tone="primary", result=True, sub=None):
    """fields: {name,label,type: select|number|text|textarea|range|checks|multi|file|date, options, value, min,max,step}"""
    return {"t": "form", "id": id, "fields": fields, "submit": submit, "url": url, "title": title, "tone": tone, "sub": sub}
def button(id, label, url, tone="primary", confirm=None, payload=None): return {"t": "button", "id": id, "label": label, "url": url, "tone": tone, "confirm": confirm, "payload": payload or {}}
def map_(layers, h=470, view=None, open=None, key="map", legend=None, live=None): return {"t": "map", "layers": layers, "h": h, "view": view, "open": open, "key": key, "legend": legend, "live": live}
def tickets(items, key="tickets", title=None, sub=None): return {"t": "tickets", "items": items, "key": key, "title": title, "sub": sub}
def hero(agent, title, blurb, r, can_run=True):
    return {"t": "hero", "agent": agent, "title": title, "blurb": blurb, "can_run": can_run, "score": None if not r else clean(r["score"]), "risk": None if not r else r["risk"], "summary": None if not r else r["summary"], "ts": None if not r else r["ts"], "ms": None if not r else r["ms"]}
def tabs(items): return {"t": "tabs", "tabs": items}
def page(title, sub, blocks, tabs=None, live=8, actions=None): return {"title": title, "sub": sub, "blocks": blocks, "tabs": tabs, "live": live, "actions": actions or []}
def pill_col(k, label=None): return {"k": k, "label": label or k.replace("_", " "), "type": "pill"}
def num_col(k, label=None, fmt="num", mx=None): return {"k": k, "label": label or k.replace("_", " "), "type": fmt, "max": mx}


Writing /content/app/blocks.py


In [8]:
%%writefile /content/app/config.py
"""Shared configuration. APP_KIND (Franchise | Freight) is set by the launcher. No Google Drive prompts: data lives in DATA_DIR (set PERSIST_DIR to keep it between sessions)."""
import os
KIND = os.environ.get("APP_KIND", "Franchise")
APP_DIR = os.path.dirname(os.path.abspath(__file__))
_default = "/content/app_data" if os.path.isdir("/content") else os.path.join(APP_DIR, "runtime_data")
DATA_DIR = os.environ.get("PERSIST_DIR") or os.environ.get("APP_DATA_DIR") or os.environ.get("FRANCHISEOPS_DATA_DIR") or os.environ.get("FREIGHTQUOTE_DATA_DIR") or _default
os.makedirs(DATA_DIR, exist_ok=True)
DB_PATH = os.path.join(DATA_DIR, "franchise_database.db" if KIND == "Franchise" else "freight_database.db")
def _secret(key, default=""):
    try:
        from google.colab import userdata
        v = userdata.get(key)
        if v: return str(v).strip()
    except Exception: pass
    return str(os.environ.get(key, default) or "").strip()
HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGINGFACE_TOKEN")
def _persisted_secret():
    """No JWT_SECRET_KEY given: generate a random one once and keep it beside the database, so sessions survive restarts and are never signed with a public default."""
    import secrets as _s
    p = os.path.join(DATA_DIR, ".jwt_secret")
    try:
        if os.path.exists(p): return open(p).read().strip()
        v = _s.token_urlsafe(48); fd = os.open(p, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600); os.write(fd, v.encode()); os.close(fd); return v
    except Exception: return _s.token_urlsafe(48)
JWT_SECRET_KEY = _secret("JWT_SECRET") or _secret("JWT_SECRET_KEY") or _persisted_secret()
ADMIN_EMAIL_ID = _secret("ADMIN_EMAIL_ID", "admin@infosys.com")
def _admin_pw():
    """ADMIN_PASSWORD comes from Colab Secrets / env. There is no built-in default: without it a random password is generated once, stored beside the database and printed in the server log."""
    v = _secret("ADMIN_PASSWORD")
    if v: return v
    import secrets as _s; p = os.path.join(DATA_DIR, ".admin_initial")
    if os.path.exists(p): return open(p).read().strip()
    v = _s.token_urlsafe(9) + "1aA"; open(p, "w").write(v); print("*** ADMIN_PASSWORD secret not set: generated initial admin password:", v, "(stored in", p, ") ***"); return v
ADMIN_PASSWORD = _admin_pw()
EMAIL_ID = _secret("EMAIL_ADDRESS") or _secret("EMAIL_ID"); EMAIL_PASSWORD = _secret("EMAIL_PASSWORD").replace(" ", "")
MODEL_NAME = os.environ.get("LLM_MODEL", "")
NLLB_MODEL = os.environ.get("NLLB_MODEL", "facebook/nllb-200-distilled-600M")


Writing /content/app/config.py


In [9]:
%%writefile /content/app/db.py
import sqlite3, os
import pandas as pd
from config import DB_PATH

def get_conn():
    """Establishes and returns a connection to the SQLite database with WAL enabled."""
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.execute("PRAGMA journal_mode=WAL;")
    conn.execute("PRAGMA synchronous=NORMAL;")
    return conn

def save_chat_message(username, role, message):
    """Saves a user or assistant chat message to the history ledger."""
    try:
        with get_conn() as conn:
            conn.execute("INSERT INTO chat_history (username, role, message) VALUES (?, ?, ?);", (username, role, message))
            conn.commit()
    except Exception:
        pass

def load_chat_history(username=None, limit=100):
    """Loads chat history for a specific user or the global system up to the specified limit."""
    try:
        with get_conn() as conn:
            if username:
                df = pd.read_sql("SELECT role, message FROM chat_history WHERE username=? ORDER BY id ASC LIMIT ?;", conn, params=(username, limit))
                if df.empty:
                    df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))
            else:
                df = pd.read_sql("SELECT role, message FROM chat_history ORDER BY id ASC LIMIT ?;", conn, params=(limit,))

            res = []
            for _, r in df.iterrows():
                content_val = str(r.get("message") or r.get("content") or "")
                res.append({
                    "role": str(r.get("role", "assistant")),
                    "content": content_val,
                    "message": content_val
                })
            return res
    except Exception:
        return []

def clear_chat_history(username=None):
    """Clears stored chat history entries."""
    try:
        with get_conn() as conn:
            if username:
                conn.execute("DELETE FROM chat_history WHERE username=?;", (username,))
            else:
                conn.execute("DELETE FROM chat_history;")
            conn.commit()
    except Exception:
        pass

def init_db():
    """Initializes all database tables and executes migrations for existing database schemas."""
    with get_conn() as conn:
        conn.execute("""
        CREATE TABLE IF NOT EXISTS chat_history (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT,
            role TEXT,
            message TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE,
            email TEXT UNIQUE,
            password_hash TEXT,
            role TEXT,
            security_question TEXT,
            security_answer TEXT,
            profile_picture TEXT,
            failed_attempts INTEGER DEFAULT 0,
            lock_until DATETIME DEFAULT NULL,
            account_status TEXT DEFAULT 'active',
            created_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)

        # Safe migration for DBs created before these columns existed
        existing_cols = {r[1] for r in conn.execute("PRAGMA table_info(users);").fetchall()}
        migrations = {
            "username": "ALTER TABLE users ADD COLUMN username TEXT;",
            "security_question": "ALTER TABLE users ADD COLUMN security_question TEXT;",
            "security_answer": "ALTER TABLE users ADD COLUMN security_answer TEXT;",
            "profile_picture": "ALTER TABLE users ADD COLUMN profile_picture TEXT;",
            "failed_attempts": "ALTER TABLE users ADD COLUMN failed_attempts INTEGER DEFAULT 0;",
            "lock_until": "ALTER TABLE users ADD COLUMN lock_until DATETIME DEFAULT NULL;",
            "account_status": "ALTER TABLE users ADD COLUMN account_status TEXT DEFAULT 'active';",
        }
        for col, ddl in migrations.items():
            if col not in existing_cols:
                try:
                    conn.execute(ddl)
                except Exception:
                    pass

        conn.execute("""
        CREATE TABLE IF NOT EXISTS alerts (
            alert_id INTEGER PRIMARY KEY AUTOINCREMENT,
            shipment_id TEXT,
            outlet_id TEXT,
            severity TEXT,
            category TEXT,
            message TEXT,
            date TEXT,
            resolved INT DEFAULT 0
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS shipments (
            shipment_id TEXT PRIMARY KEY,
            origin_port TEXT,
            dest_port TEXT,
            carrier TEXT,
            status TEXT,
            eta TEXT,
            weight_kg REAL,
            hs_code TEXT,
            predicted_delay_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS freight_quotes (
            quote_id TEXT PRIMARY KEY,
            shipment_id TEXT,
            customer_id TEXT,
            carrier TEXT,
            base_cost REAL,
            insurance REAL,
            customs_fee REAL,
            fuel_surcharge REAL,
            final_price REAL,
            margin_pct REAL,
            status TEXT,
            created_at TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ports (
            port_id TEXT PRIMARY KEY,
            port_name TEXT,
            country TEXT,
            congestion_index REAL,
            avg_dwell_days REAL,
            lat REAL,
            lon REAL,
            region TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS carriers (
            carrier_id TEXT PRIMARY KEY,
            name TEXT,
            rating REAL,
            on_time_pct REAL,
            avg_cost_index REAL,
            risk_level TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            customer_id TEXT PRIMARY KEY,
            name TEXT,
            industry TEXT,
            priority_tier TEXT,
            credit_risk REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS ml_metrics (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            module TEXT,
            model_name TEXT,
            metric_name TEXT,
            metric_value REAL,
            trained_at DATETIME DEFAULT CURRENT_TIMESTAMP
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS outlets (
            outlet_id TEXT PRIMARY KEY,
            outlet_name TEXT,
            location TEXT,
            tier TEXT,
            revenue REAL,
            operating_costs REAL,
            customer_satisfaction REAL,
            staff_headcount INT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS staff (
            staff_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            name TEXT,
            role TEXT,
            salary REAL,
            overtime_hrs REAL,
            job_satisfaction INT,
            age INT,
            tenure_years INT,
            work_life_balance INT,
            predicted_attrition_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS inventory (
            record_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            sku_name TEXT,
            category TEXT,
            current_stock INT,
            reorder_threshold INT,
            weekly_demand REAL,
            lead_time_days INT,
            stockout_risk_prob REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS marketing (
            campaign_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            campaign_name TEXT,
            channel TEXT,
            budget REAL,
            actual_roi REAL,
            reach INT,
            conversions INT,
            start_date TEXT,
            end_date TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS feedback (
            feedback_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            rating INT,
            comment TEXT,
            date TEXT,
            sentiment_score REAL
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS audits (
            audit_id TEXT PRIMARY KEY,
            outlet_id TEXT,
            audit_date TEXT,
            score REAL,
            violations INT,
            category TEXT,
            status TEXT,
            notes TEXT
        );
        """)
        conn.execute("""
        CREATE TABLE IF NOT EXISTS weather_risks (
            port_name TEXT PRIMARY KEY,
            current_severity INT,
            forecast TEXT,
            wind_speed REAL,
            wave_height REAL,
            temperature REAL
        );
        """)
        conn.commit()


Writing /content/app/db.py


In [10]:
%%writefile /content/app/domain_franchise.py
"""FranchiseOps AI live domain: simulator, 5 agents, intelligence engine, action handlers."""
import os, re, json, math, random, time, datetime, uuid
import numpy as np, pandas as pd, requests
from live_engine import AgentResult, risk_label, conn, now, _DB_LOCK, set_state, get_state, emit

KIND = "Franchise"; NAME = "FranchiseOps AI"; ENTITY_COL = "outlet_id"; SMS = True; ESCALATE_TO = "VP Operations"
OWNER = {"reorder": "Store Manager", "pause_promo": "Marketing Manager", "corrective_action": "Regional Manager", "retention_plan": "HR Lead",
         "outlet_review": "Regional Manager", "staffing_shortage": "Store Manager", "reallocate_budget": "Marketing Manager"}
ACTION_AGENT = {"reorder": "inventory", "pause_promo": "inventory", "corrective_action": "audit", "retention_plan": "staff",
                "outlet_review": "outlet", "staffing_shortage": "staff", "reallocate_budget": "marketing"}
SIM_DAY_SEC = 60          # one simulated day = 60 s of wall-clock (5 s ticks -> 12 ticks per day)
TICKS_PER_DAY = 12
def Q(sql, p=()):
    with _DB_LOCK, conn() as c: return pd.read_sql(sql, c, params=p)

# ---------------------------------------------------------------- bootstrap / simulator
def bootstrap():
    from scale_data import scale_franchise; scale_franchise()
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS sales(sale_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT, ts TEXT, amount REAL, tx_count INT, source TEXT);
        CREATE TABLE IF NOT EXISTS attendance(att_id INTEGER PRIMARY KEY AUTOINCREMENT, staff_id TEXT, outlet_id TEXT, ts TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS purchase_orders(po_id INTEGER PRIMARY KEY AUTOINCREMENT, outlet_id TEXT, sku_name TEXT, qty INT, ts TEXT, eta_ts TEXT, status TEXT);
        CREATE TABLE IF NOT EXISTS ml_models(model_id INTEGER PRIMARY KEY AUTOINCREMENT, model_name TEXT, metrics TEXT, trained_at TEXT, is_active INT);
        CREATE INDEX IF NOT EXISTS ix_sales ON sales(outlet_id, ts);
        """)
        if c.execute("SELECT COUNT(*) FROM sales").fetchone()[0] == 0:       # 90 days of seeded history so baselines exist
            rng = np.random.default_rng(7); today = datetime.date.today()
            for oid, rev in c.execute("SELECT outlet_id, revenue FROM outlets").fetchall():
                base = (rev or 100000) / 30
                for d in range(90, 0, -1):
                    dt = today - datetime.timedelta(days=d)
                    c.execute("INSERT INTO sales(outlet_id,ts,amount,tx_count,source) VALUES(?,?,?,?,'seed')", (oid, dt.isoformat(), float(base * rng.normal(1, .07) * (1.15 if dt.weekday() >= 5 else 1)), int(base / 250)))
        active = c.execute("SELECT COUNT(*) FROM marketing WHERE end_date>=date('now')").fetchone()[0]
        if active == 0:                                                       # launch a live promo set (seed campaigns are all expired)
            ids = [r[0] for r in c.execute("SELECT campaign_id FROM marketing ORDER BY RANDOM() LIMIT 14").fetchall()]
            for i in ids: c.execute("UPDATE marketing SET start_date=date('now','-3 day'), end_date=date('now','+14 day') WHERE campaign_id=?", (i,))
        c.execute("UPDATE inventory SET weekly_demand=MAX(weekly_demand,1)")
        for rid, st_, wk, lead in c.execute("SELECT record_id, current_stock, weekly_demand, lead_time_days FROM inventory").fetchall():   # (re)start from a healthy, varied stock position
            if st_ / max(wk / 7, 1) < lead + 1.5: c.execute("UPDATE inventory SET current_stock=? WHERE record_id=?", (int(wk / 7 * (lead + random.uniform(1.5, 14))), rid))

_incident = {}; _tickn = [0]
def tick():
    reasons = []; _tickn[0] += 1; n = _tickn[0]; mult = get_state("demand_mult", 1.0) or 1.0
    ts = now()
    with _DB_LOCK, conn() as c:
        outlets = c.execute("SELECT outlet_id, revenue, operating_costs, customer_satisfaction FROM outlets").fetchall()
        real = set()
        for oid, rev, cost, csat in outlets:
            if oid in real: continue
            if oid not in _incident and random.random() < 0.002: _incident[oid] = random.randint(8, 14)
            f = 0.35 if oid in _incident else 1.0
            if oid in _incident:
                _incident[oid] -= 1
                if _incident[oid] <= 0: _incident.pop(oid)
            base = 3000000 / 30 / TICKS_PER_DAY if not rev else (rev / 30 / TICKS_PER_DAY)
            amt = float(base * random.uniform(.8, 1.2) * mult * f)
            c.execute("INSERT INTO sales(outlet_id,ts,amount,tx_count,source) VALUES(?,?,?,?,'sim')", (oid, ts, amt, max(1, int(amt / 250))))
            c.execute("UPDATE outlets SET revenue=revenue+?, operating_costs=operating_costs+? WHERE outlet_id=?", (amt, amt * (cost / rev if rev else .65), oid))
        # stock drawdown + deliveries
        for rid, oid, sku, stock, thr, wk, lead in c.execute("SELECT record_id,outlet_id,sku_name,current_stock,reorder_threshold,weekly_demand,lead_time_days FROM inventory").fetchall():
            use = max(0, int(round(wk / 7 / TICKS_PER_DAY * mult * random.uniform(.6, 1.6))))
            new = max(0, stock - use); cover = new / max(wk / 7, 1)
            risk = float(min(1, max(0, 1 - cover / (lead + 3))))
            c.execute("UPDATE inventory SET current_stock=?, stockout_risk_prob=? WHERE record_id=?", (new, round(risk, 2), rid))
            if stock / max(wk / 7, 1) >= lead + 2 > cover: reasons.append(f"stock-low:{oid}")
        for po, oid, sku, qty in c.execute("SELECT po_id,outlet_id,sku_name,qty FROM purchase_orders WHERE status='ordered' AND eta_ts<=?", (ts,)).fetchall():
            c.execute("UPDATE inventory SET current_stock=current_stock+? WHERE outlet_id=? AND sku_name=?", (qty, oid, sku))
            c.execute("UPDATE purchase_orders SET status='delivered' WHERE po_id=?", (po,)); emit("po_delivered", oid, {"sku": sku, "qty": qty})
        # staff: attendance, overtime drift, attrition recompute (same heuristic the app uses)
        for sid, oid, sal, sat, ot in c.execute("SELECT staff_id,outlet_id,salary,job_satisfaction,overtime_hrs FROM staff ORDER BY RANDOM() LIMIT 40").fetchall():
            st = random.choices(["Present", "Late", "Absent"], [.86, .09, .05 + (.25 if oid in _incident else 0)])[0]
            c.execute("INSERT INTO attendance(staff_id,outlet_id,ts,status) VALUES(?,?,?,?)", (sid, oid, ts, st))
            ot2 = float(min(45, max(0, ot + random.uniform(-1.5, 2) + (1.5 if st == "Absent" else 0))))
            c.execute("UPDATE staff SET overtime_hrs=? WHERE staff_id=?", (round(ot2, 1), sid))
        # feedback + csat moving average
        for oid, csat in random.sample([(o[0], o[3]) for o in outlets], 6):
            r = int(min(5, max(1, round(random.gauss(csat if csat else 4, .9) - (1.5 if oid in _incident else 0)))))
            c.execute("INSERT INTO feedback(feedback_id,outlet_id,rating,comment,date,sentiment_score) VALUES(?,?,?,?,?,?)",
                      ("FB-L"+uuid.uuid4().hex[:10], oid, r, "Live feedback", datetime.date.today().isoformat(), (r - 3) / 2))
            c.execute("UPDATE outlets SET customer_satisfaction=ROUND(0.95*customer_satisfaction+0.05*?,3) WHERE outlet_id=?", (r, oid))
        if random.random() < 0.05:
            oid = random.choice(outlets)[0]; sc = float(np.clip(random.gauss(84, 12), 40, 100))
            c.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,?,?,?,?,?,?)",
                      ("AUD-L"+uuid.uuid4().hex[:10], oid, datetime.date.today().isoformat(), sc, int((100 - sc) / 6), "Food Safety", "Pass" if sc >= 85 else "Conditional Pass" if sc >= 70 else "Fail", "Live inspection"))
            if sc < 70: reasons.append(f"audit-fail:{oid}")
    return list(dict.fromkeys(reasons))[:5]

def process(): pass
def feed(): _external()
DEFAULT_POLICY = {"pause_promo": "approval"}

def _external():
    """Real feeds: Open-Meteo weather (city list from the app), Nager.Date public holidays. Falls back silently when offline."""
    mult, feeds, rain = 1.0, {}, []
    try:
        from weather_context import CITY_COORDS
        for city, (la, lo) in list(CITY_COORDS.items())[:5]:
            r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={la}&longitude={lo}&current=temperature_2m,precipitation,wind_speed_10m", timeout=4).json()["current"]
            rain.append(r.get("precipitation", 0) or 0); feeds[city] = {"temp": r["temperature_2m"], "rain": r["precipitation"], "wind": r["wind_speed_10m"]}
        if np.mean(rain) > 1.5: mult *= .92
        feeds_ok = True
    except Exception: feeds_ok = False
    hol = False
    try:
        hol = requests.get("https://date.nager.at/api/v3/IsTodayPublicHoliday/IN", timeout=4).status_code == 200
        if hol: mult *= 1.15
    except Exception: pass
    set_state("demand_mult", round(mult, 3)); set_state("weather", feeds); set_state("feeds", {"weather": feeds_ok, "holiday": hol, "ts": now()})

def feed_status():
    f = get_state("feeds", {}) or {}
    return [("Open-Meteo weather", "Real · live" if f.get("weather") else "Offline · no demand adjustment"),
            ("Public holidays (Nager.Date)", "Real · today is a holiday" if f.get("holiday") else "Real · no holiday today" if f else "Waiting"),
            ("Outlet POS, stock, attendance, feedback", "Simulated feed (replace via Data Feed Center)")]

# ---------------------------------------------------------------- agents (PRS v3: 7 agents + Intelligence Engine)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, silhouette_score
TIER_BASE = {"Excellent": 95, "Good": 80, "Needs Attention": 58, "Critical": 35}
def _reg_model(name, metrics, active=1):
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE ml_models SET is_active=0 WHERE model_name=?", (name,))
        c.execute("INSERT INTO ml_models(model_name,metrics,trained_at,is_active) VALUES(?,?,?,?)", (name, json.dumps(metrics), now(), active))

def outlet_agent(s):
    """Agent 1: KMeans (k=4) on 30-day revenue/orders/customers, ranked into Excellent/Good/Needs Attention/Critical."""
    d = Q("SELECT outlet_id, source, AVG(amount) rev, AVG(tx_count) orders FROM sales GROUP BY outlet_id, source")
    d.loc[d.source == "sim", ["rev", "orders"]] *= TICKS_PER_DAY
    d = d.sort_values("source").drop_duplicates("outlet_id", keep="first")            # sim rows preferred once present ('sim' < 'seed')
    d["cust"] = d.orders * .8; X = StandardScaler().fit_transform(d[["rev", "orders", "cust"]])
    km = KMeans(4, n_init=10, random_state=42).fit(X); d["cl"] = km.labels_
    order = d.groupby("cl").rev.mean().sort_values(ascending=False).index.tolist(); tmap = dict(zip(order, ["Excellent", "Good", "Needs Attention", "Critical"]))
    d["tier"] = d.cl.map(tmap); dist = np.linalg.norm(X - km.cluster_centers_[km.labels_], axis=1); d["conf"] = (1 / (1 + dist)).round(2)
    d["score"] = d.tier.map(TIER_BASE) + d.groupby("tier").rev.rank(pct=True) * 8 - 4
    sil = float(silhouette_score(X, km.labels_)) if len(d) > 4 else 0
    _reg_model("kmeans_outlet_clusters", {"silhouette": round(sil, 3), "n": len(d)})
    crit = d[d.tier == "Critical"]
    return AgentResult("outlet", round(float(d.score.mean()), 1), risk_label(d.score.mean()), f"KMeans tiers: " + ", ".join(f"{t} {int((d.tier == t).sum())}" for t in TIER_BASE),
        {"per_outlet": d.set_index("outlet_id").score.round(1).to_dict(), "tiers": d.set_index("outlet_id").tier.to_dict(), "silhouette": round(sil, 3), "weak": crit.outlet_id.tolist()},
        [{"type": "outlet_review", "target": r.outlet_id, "detail": f"KMeans tier Critical (revenue {r.rev:,.0f}/day, confidence {r.conf})", "severity": "Warning"} for r in crit.itertuples()])

def forecast_series(outlet, horizon=14, method=None):
    """Demand forecast for one outlet. Prophet when installed and USE_PROPHET != 0 (daily + weekly seasonality), else linear trend x weekday-seasonal index."""
    h = Q("SELECT substr(ts,1,10) d, SUM(amount) a FROM sales WHERE outlet_id=? AND source='seed' GROUP BY d ORDER BY d", (outlet,))
    if len(h) < 21: return h.assign(kind="actual"), pd.DataFrame(), "insufficient history"
    h["d"] = pd.to_datetime(h.d); fut_d = pd.date_range(h.d.iloc[-1] + pd.Timedelta(days=1), periods=horizon)
    if (method or os.environ.get("USE_PROPHET", "1")) not in ("0", "trend"):
        try:
            from prophet import Prophet
            m = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False); m.fit(h.rename(columns={"d": "ds", "a": "y"}))
            f = m.predict(pd.DataFrame({"ds": fut_d})); return h, pd.DataFrame({"d": f.ds, "a": f.yhat, "lo": f.yhat_lower, "hi": f.yhat_upper}), "Prophet"
        except Exception: pass
    y = h.a.values; x = np.arange(len(y)); k, b = np.polyfit(x, y, 1); trend = k * x + b; ratio = pd.Series(y / np.maximum(trend, 1)).groupby(h.d.dt.weekday.values).mean()
    xf = np.arange(len(y), len(y) + horizon); fa = (k * xf + b) * pd.Series(fut_d.weekday).map(ratio).values; sd = float(np.std(y - trend * h.d.dt.weekday.map(ratio).values))
    return h, pd.DataFrame({"d": fut_d, "a": fa, "lo": fa - 1.28 * sd, "hi": fa + 1.28 * sd}), "trend x weekday seasonality"

def _forecast_ratio(outlet):
    """Agent 2 forecast: Prophet when installed (USE_PROPHET=1), otherwise weekday-seasonal linear trend. Returns next-7-day / last-7-day demand ratio."""
    h, f, _ = forecast_series(outlet, 7, method="trend")                 # the fleet-wide loop stays on the fast method; Prophet is used on the Inventory page per outlet
    if not len(f): return 1.0
    return float(np.clip(f.a.mean() / max(h.a.iloc[-7:].mean(), 1), .8, 1.3))

def inventory_agent(s):
    mult = get_state("demand_mult", 1.0) or 1.0
    d = Q("SELECT outlet_id, sku_name, current_stock, reorder_threshold mn, weekly_demand, lead_time_days lead FROM inventory")
    fr = get_state("forecast_ratio", {}) or {}
    if not fr or now()[:13] != (get_state("forecast_hour") or ""):
        fr = {o: _forecast_ratio(o) for o in d.outlet_id.unique()}; set_state("forecast_ratio", fr); set_state("forecast_hour", now()[:13])
    d["daily"] = d.weekly_demand.clip(lower=1) / 7 * mult * d.outlet_id.map(fr).fillna(1.0); d["cover"] = d.current_stock / d.daily
    po = Q("SELECT outlet_id, sku_name FROM purchase_orders WHERE status='ordered'"); open_po = set(zip(po.outlet_id, po.sku_name))
    d["short"] = (d.current_stock <= d.mn) | (d.cover < d.lead + 2)
    per = (100 - d.groupby("outlet_id").short.mean() * 200).clip(0, 100).round(1)
    acts = []
    for r in d[d.short].sort_values("cover").itertuples():
        if (r.outlet_id, r.sku_name) in open_po: continue
        qty = int(max(r.mn * 3, math.ceil(r.daily * (r.lead + 7)) - r.current_stock))               # PRS: reorder_qty = min_stock x 3 (or forecast need if larger)
        acts.append({"type": "reorder", "target": r.outlet_id, "detail": f"{r.sku_name}: {r.cover:.1f} days cover, order {qty} units", "severity": "Critical" if r.cover < r.lead else "Warning", "sku": r.sku_name, "qty": qty, "lead": int(r.lead)})
    n = int(d.short.sum())
    return AgentResult("inventory", round(float(per.mean()), 1), risk_label(per.mean()), f"{n} SKUs flagged of {len(d)}, demand x{mult}", {"per_outlet": per.to_dict(), "outlets": d[d.short].outlet_id.unique().tolist(), "demand_mult": mult, "forecast": "weekday-seasonal trend (Prophet optional)"}, acts[:40])

_LR = {}
def _attrition_model():
    """Agent 3 ML: LogisticRegression. Uses a fed HR dataset (table hr_training, e.g. IBM HR Analytics) when present; otherwise a bootstrap model on the staff table labelled by the rule score."""
    d = Q("SELECT st.staff_id, st.outlet_id, st.salary, st.overtime_hrs, st.job_satisfaction sat, st.tenure_years ten, st.work_life_balance wlb, st.age, st.predicted_attrition_prob p FROM staff st")
    att = Q("SELECT staff_id, AVG(status='Present') pres FROM attendance GROUP BY staff_id").set_index("staff_id").pres
    d["att"] = d.staff_id.map(att).fillna(.9) * 100; feats = ["salary", "overtime_hrs", "sat", "ten", "wlb", "age", "att"]
    src = "bootstrap (labels from rule score)"
    try:
        h = Q("SELECT * FROM hr_training")
        if len(h) >= 300 and "attrition" in h.columns:
            src = "fed HR dataset"; tr = h.rename(columns={"job_satisfaction": "sat", "tenure_years": "ten", "work_life_balance": "wlb", "monthly_income": "salary", "overtime_hrs": "overtime_hrs"})
            for f in feats:
                if f not in tr: tr[f] = d[f].median()
            X, y = tr[feats].fillna(0), tr.attrition.astype(int)
    except Exception: pass
    if src.startswith("boot"): X, y = d[feats], (d.p + np.random.default_rng(1).normal(0, .05, len(d)) > .5).astype(int)
    sc = StandardScaler().fit(X); Xtr, Xte, ytr, yte = train_test_split(sc.transform(X), y, test_size=.2, random_state=42, stratify=y if y.nunique() > 1 else None)
    lr = LogisticRegression(max_iter=500).fit(Xtr, ytr)
    m = {"accuracy": round(accuracy_score(yte, lr.predict(Xte)), 3), "auc": round(roc_auc_score(yte, lr.predict_proba(Xte)[:, 1]), 3) if yte.nunique() > 1 else None, "source": src}
    _reg_model("logreg_attrition", m); _LR.update(model=lr, sc=sc, feats=feats, metrics=m, trained=time.time()); return d

def _staff_frame():
    d = Q("SELECT st.staff_id, st.outlet_id, st.name, st.role, st.salary, st.overtime_hrs, st.job_satisfaction sat, st.tenure_years ten, st.work_life_balance wlb, st.age, st.predicted_attrition_prob p FROM staff st")
    att = Q("SELECT staff_id, AVG(status='Present') pres FROM attendance GROUP BY staff_id").set_index("staff_id").pres
    d["att"] = d.staff_id.map(att).fillna(.9) * 100; return d

def attrition_table(scope=None, n=25):
    """Per-staff attrition probability with the single biggest driver (coefficient x standardised value): what the Workforce page shows."""
    if not _LR: return pd.DataFrame()
    d = _staff_frame()
    if scope: d = d[d.outlet_id == scope]
    if not len(d): return d
    X = _LR["sc"].transform(d[_LR["feats"]]); d["prob"] = _LR["model"].predict_proba(X)[:, 1].round(3)
    contrib = X * _LR["model"].coef_[0]; d["driver"] = [_LR["feats"][i] for i in contrib.argmax(1)]
    return d.sort_values("prob", ascending=False).head(n)
def attrition_importance():
    if not _LR: return {}
    return dict(zip(_LR["feats"], _LR["model"].coef_[0].round(3).tolist()))

def staff_agent(s):
    if not _LR or time.time() - _LR["trained"] > 900: _attrition_model()
    d = _staff_frame()
    d["prob"] = _LR["model"].predict_proba(_LR["sc"].transform(d[_LR["feats"]]))[:, 1].round(3)
    with _DB_LOCK, conn() as c: c.executemany("UPDATE staff SET predicted_attrition_prob=? WHERE staff_id=?", list(zip(d.prob.tolist(), d.staff_id)))
    d["perf"] = d.sat * 20; hi = d[d.prob > .6]; low = d[(d.perf < 60) | (d.att < 75)]
    per = (100 - (d.groupby("outlet_id").prob.mean() * 100 * .6 + (100 - d.groupby("outlet_id").att.mean()) * .4)).clip(0, 100).round(1)
    a = [{"type": "retention_plan", "target": o, "detail": f"{n} staff above 60% predicted attrition (LogReg {_LR['metrics']['source']})", "severity": "Warning"} for o, n in hi.groupby("outlet_id").size().items() if n >= 4]
    a += [{"type": "staffing_shortage", "target": o, "detail": f"{n} staff below 75% attendance or performance 60", "severity": "Warning"} for o, n in low.groupby("outlet_id").size().items() if n >= 6]
    return AgentResult("staff", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(hi)} high attrition risk, {len(low)} low performers or attendance (attrition model AUC {_LR['metrics'].get('auc', 0):.2f})", {"per_outlet": per.to_dict(), "short_staffed": [x["target"] for x in a if x["type"] == "staffing_shortage"], "model": _LR["metrics"]}, a[:25])

def marketing_agent(s):
    """Agent 4: ROI = (revenue-budget)/budget; KMeans (k=3) customer segments: High-Value, Frequent, Churned."""
    m = Q("SELECT outlet_id, campaign_name, channel, actual_roi, end_date FROM marketing"); m["roi_pct"] = (m.actual_roi - 1) * 100
    cu = Q("SELECT outlet_id, total_spend_90d sp, visit_freq vf, avg_order_value aov, days_since_last_visit dl FROM customers_fr")
    churn = {}; seg_counts = {}
    if len(cu) > 30:
        X = StandardScaler().fit_transform(cu[["sp", "vf", "aov", "dl"]]); km = KMeans(3, n_init=10, random_state=42).fit(X); cu["cl"] = km.labels_
        g = cu.groupby("cl").agg(sp=("sp", "mean"), dl=("dl", "mean")); names = {g.sp.idxmax(): "High-Value", g.dl.idxmax(): "Churned"}
        cu["seg"] = cu.cl.map(lambda k: names.get(k, "Frequent")); churn = cu.assign(ch=cu.seg == "Churned").groupby("outlet_id").ch.mean().to_dict(); seg_counts = cu.seg.value_counts().to_dict()
    per = (m.groupby("outlet_id").roi_pct.mean().clip(0, 300) / 3).round(1); best = m.groupby("channel").actual_roi.mean().idxmax()
    live = m[m.end_date >= datetime.date.today().isoformat()]; low = live[live.actual_roi < 2.0]
    acts = [{"type": "reallocate_budget", "target": r.outlet_id, "detail": f"{r.campaign_name}: ROI {r.roi_pct:.0f}% below 100%; shift budget to {best}", "severity": "Info"} for r in low.head(10).itertuples()]
    acts += [{"type": "win_back_campaign", "target": o, "detail": f"{v*100:.0f}% of customers in Churned segment", "severity": "Warning"} for o, v in churn.items() if v > .5][:10]
    return AgentResult("marketing", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(live)} live campaigns, best channel {best}, {sum(v > .5 for v in churn.values())} outlets with high churn", {"per_outlet": per.to_dict(), "active_outlets": live.outlet_id.unique().tolist(), "best_channel": best, "segments": seg_counts, "churn_share": {k: round(v, 3) for k, v in churn.items()}}, acts)

def audit_agent(s):
    """Agent 5: latest audit per outlet. Pass >= 80, Warning >= 60, Fail < 60. SOP hint from the FAISS/keyword RAG when available."""
    d = Q("SELECT outlet_id, score, category FROM audits ORDER BY audit_date DESC, rowid DESC").drop_duplicates("outlet_id")
    per = d.set_index("outlet_id").score.clip(0, 100).round(1); fail = d[d.score < 60]; warn = d[(d.score >= 60) & (d.score < 80)]
    hint = ""
    try:
        from rag_engine import answer_with_citation
        ctx, _ = answer_with_citation("food safety corrective action procedure"); hint = (str(ctx)[:140] + "...") if ctx else ""
    except Exception: pass
    acts = [{"type": "corrective_action", "target": r.outlet_id, "detail": f"Audit FAIL {r.score:.0f} ({r.category}): corrective plan within 72 h. {hint}", "severity": "Critical", "sla_h": 72} for r in fail.itertuples()]
    acts += [{"type": "corrective_action", "target": r.outlet_id, "detail": f"Audit WARNING {r.score:.0f} ({r.category}): re-check in 14 days", "severity": "Warning", "sla_h": 336} for r in warn.itertuples()][:15]
    return AgentResult("audit", round(float(per.mean()), 1), risk_label(per.mean()), f"Pass {int((d.score >= 80).sum())}, Warning {len(warn)}, Fail {len(fail)}", {"per_outlet": per.to_dict(), "failed": fail.outlet_id.tolist()}, acts)

POS = {"great", "good", "clean", "tasty", "friendly", "fast", "love", "excellent", "value", "fresh"}; NEG = {"slow", "rude", "dirty", "cold", "bad", "long", "wait", "stale", "expensive", "late"}
ISSUES = {"food quality": ("cold", "stale", "taste", "food"), "staff behavior": ("rude", "staff"), "wait time": ("slow", "wait", "long", "late"), "hygiene": ("dirty", "clean", "hygiene"), "value": ("expensive", "value", "price")}
def feedback_agent(s):
    """Agent 7: VADER when installed, else lexicon + rating blend; keyword issue categorisation."""
    f = Q("SELECT outlet_id, rating, comment FROM feedback ORDER BY rowid DESC LIMIT 6000")
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer; v = SentimentIntensityAnalyzer(); f["sent"] = f.comment.map(lambda t: v.polarity_scores(str(t))["compound"]); method = "VADER"
    except Exception:
        f["sent"] = f.comment.map(lambda t: (sum(w in POS for w in str(t).lower().split()) - sum(w in NEG for w in str(t).lower().split())) / 2); method = "lexicon"
    f["sent"] = (.6 * (f.rating - 3) / 2 + .4 * f.sent.clip(-1, 1)); per = ((f.groupby("outlet_id").sent.mean() + 1) / 2 * 100).round(1)
    cat = {k: int(f.comment.str.lower().str.contains("|".join(v)).sum()) for k, v in ISSUES.items()}
    low = per[per < 45]
    return AgentResult("feedback", round(float(per.mean()), 1), risk_label(per.mean()), f"{len(low)} outlets with poor sentiment ({method}); top issue {max(cat, key=cat.get)}", {"per_outlet": per.to_dict(), "issues": cat},
        [{"type": "service_recovery", "target": o, "detail": f"Sentiment score {v:.0f}/100; main issue: {max(cat, key=cat.get)}", "severity": "Warning"} for o, v in low.items()][:15])

def digest_agent(s):
    """Agent 6: weekly digest + immediate alerts. Runs on the weekly trigger or on request; stores the digest and emails the owner when SMTP is set."""
    r = s["results"]; per = r.get("outlet").findings["per_outlet"] if "outlet" in r else {}
    top = sorted(per, key=per.get, reverse=True)[:3]; bot = sorted(per, key=per.get)[:3]
    txt = f"Weekly digest {now()[:10]}: top {top}, bottom {bot}. " + " ".join(f"{k}: {v.summary}." for k, v in r.items() if k != "digest")
    with _DB_LOCK, conn() as c:
        c.execute("CREATE TABLE IF NOT EXISTS digests(digest_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, body TEXT)"); c.execute("INSERT INTO digests(ts,body) VALUES(?,?)", (now(), txt))
    return AgentResult("digest", 100, "Low", "weekly digest generated", {"digest": txt}, [{"type": "digest_sent", "target": "OWNER", "detail": txt[:300], "severity": "Info"}])


def anomaly_agent(s):
    """Live anomaly detection: IsolationForest over each outlet's recent sales-vs-baseline ratio, volatility and feedback sentiment."""
    from sklearn.ensemble import IsolationForest
    sl = Q("SELECT outlet_id, amount FROM sales WHERE source='sim' ORDER BY sale_id DESC LIMIT 4000")
    if len(sl) < 600: return AgentResult("anomaly", 100, "Low", "collecting live data (needs a few minutes of ticks)", {"flagged": []})
    base = Q("SELECT outlet_id, AVG(amount)/? b FROM sales WHERE source='seed' GROUP BY outlet_id", (TICKS_PER_DAY,)).set_index("outlet_id").b
    g = sl.groupby("outlet_id").amount; d = pd.DataFrame({"ratio": g.agg(lambda x: x.iloc[:5].mean()) / base, "vol": g.std() / g.mean()}).dropna()
    if len(d) < 10: return AgentResult("anomaly", 100, "Low", "waiting for enough outlets with both live and baseline sales", {"flagged": []})
    fb = Q("SELECT outlet_id, AVG(sentiment_score) s FROM (SELECT * FROM feedback ORDER BY rowid DESC LIMIT 1500) GROUP BY outlet_id").set_index("outlet_id").s
    d["sent"] = fb.reindex(d.index).fillna(0); iso = IsolationForest(contamination=.05, random_state=42).fit(d); d["flag"] = iso.predict(d) == -1
    bad = d[d.flag | (d.ratio < .6)]; sc = 100 - len(bad) / max(len(d), 1) * 300
    return AgentResult("anomaly", round(max(sc, 0), 1), risk_label(sc), f"{len(bad)} outlets with anomalous sales/feedback pattern (IsolationForest)", {"flagged": bad.index.tolist(), "detail": bad.round(2).reset_index().to_dict("records")[:30], "all": d.round(3).reset_index().to_dict("records")},
        [{"type": "anomaly_review", "target": o, "detail": f"Unusual pattern: sales at {r.ratio*100:.0f}% of baseline, volatility {r.vol:.2f}, sentiment {r.sent:.2f}", "severity": "Critical" if r.ratio < .6 else "Warning"} for o, r in bad.iterrows()][:20])

AGENTS = {"outlet": outlet_agent, "inventory": inventory_agent, "staff": staff_agent, "marketing": marketing_agent, "audit": audit_agent, "feedback": feedback_agent, "anomaly": anomaly_agent, "digest": digest_agent}
REPLAN = {"inventory": lambda r: ["marketing"] if r.risk != "Low" else [], "audit": lambda r: ["staff", "feedback"] if r.risk == "High" else [], "outlet": lambda r: ["inventory", "staff"] if r.risk != "Low" else []}
KW = {"outlet": ["outlet", "store", "revenue", "sales", "performance", "health"], "inventory": ["stock", "inventory", "reorder", "demand"], "staff": ["staff", "attrition", "workforce", "attendance", "employee"],
      "marketing": ["campaign", "roi", "marketing", "promo", "customer"], "audit": ["audit", "compliance", "violation", "sop"], "feedback": ["feedback", "sentiment", "review", "complaint"], "digest": ["digest", "weekly report"], "anomaly": ["anomal", "unusual", "spike", "fraud", "outlier"]}
ACTION_AGENT.update({"digest_sent": "digest", "anomaly_review": "anomaly", "win_back_campaign": "marketing", "service_recovery": "feedback", "health_alert": "outlet"}); OWNER.update({"anomaly_review": "Regional Manager", "win_back_campaign": "Marketing Manager", "service_recovery": "Store Manager", "health_alert": "Regional Manager", "digest_sent": "Franchise Owner"})
W = {"outlet": .30, "inventory": .20, "staff": .20, "marketing": .15, "audit": .15}     # PRS v3 weighted health score
def plan(req):
    q = (req or "").lower(); picked = [a for a, ks in KW.items() if any(re.search(r"\b" + k, q) for k in ks)]
    base = [a for a in AGENTS if a != "digest"]
    return (picked + (["digest"] if "digest" in picked and len(picked) == 1 else [])) if picked else base

def engine(s):
    r = s["results"]; used = [k for k in W if k in r and "per_outlet" in r[k].findings]
    outlets = set().union(*[set(r[k].findings.get("per_outlet", {})) for k in used]) if used else set()
    health = {o: round(sum(W[k] * r[k].findings["per_outlet"].get(o, r[k].score) for k in used) / sum(W[k] for k in used), 1) for o in outlets}
    score = float(np.mean(list(health.values()))) if health else 0.0
    actions = [a for res in r.values() for a in res.actions]
    for o, h in health.items():
        if h < 60: actions.append({"type": "health_alert", "target": o, "detail": f"Outlet health score {h} below 60 (perf, stock, staff, ROI, audit)", "severity": "Critical"})
    if "inventory" in r and "marketing" in r:
        for o in set(r["inventory"].findings.get("outlets", [])) & set(r["marketing"].findings.get("active_outlets", [])):
            actions.append({"type": "pause_promo", "target": o, "detail": "Stock shortage during active campaign: promo paused", "severity": "Critical"})
    if "staff" in r and "inventory" in r:
        for o in set(r["staff"].findings.get("short_staffed", [])) & set(r["inventory"].findings.get("outlets", [])):
            actions.append({"type": "staffing_shortage", "target": o, "detail": "Short-staffed while stock is low: add cover shift", "severity": "Critical"})
    order = {"Critical": 0, "Warning": 1, "Info": 2}; actions.sort(key=lambda a: order.get(a.get("severity"), 3))
    ev = {"franchise_health": round(score, 1), "outlets": len(health), "outlets_below_60": sum(h < 60 for h in health.values()), "agents": {k: {"score": v.score, "risk": v.risk, "summary": v.summary} for k, v in r.items()},
          "critical": sum(a.get("severity") == "Critical" for a in actions), "total_actions": len(actions)}
    return {"score": round(score, 1), "risk": risk_label(score), "narrative": _narrate(ev, actions), "actions": actions, "agents": ev["agents"], "health": health, "evidence": ev}

def _narrate(ev, actions):
    tmpl = f"Franchise health {ev['franchise_health']}/100 across {ev['outlets']} outlets, {ev['outlets_below_60']} below 60. " + " ".join(f"{k.title()}: {v['summary']}." for k, v in ev["agents"].items()) + f" {ev['critical']} critical of {ev['total_actions']} actions."
    try:
        from llm_engine import is_llm_loaded, generate_text
        if not is_llm_loaded(): return tmpl
        txt = generate_text([{"role": "system", "content": "Write a 3-line executive recommendation for a franchise owner. Use only numbers present in the JSON."}, {"role": "user", "content": json.dumps(ev)}], max_new_tokens=160, temperature=0.1)
        ok = all(n in json.dumps(ev) for n in re.findall(r"\d+(?:\.\d+)?", txt or ""))
        return txt.strip() if txt and ok else tmpl
    except Exception: return tmpl

def apply(a, c):
    if a["type"] == "reorder":
        eta = (datetime.datetime.now() + datetime.timedelta(seconds=SIM_DAY_SEC * a.get("lead", 3))).isoformat(timespec="seconds")
        cur = c.execute("INSERT INTO purchase_orders(outlet_id,sku_name,qty,ts,eta_ts,status) VALUES(?,?,?,?,?,'ordered')", (a["target"], a["sku"], a["qty"], now(), eta))
        return f"PO-{cur.lastrowid} created, ETA {eta[11:19]}"
    if a["type"] == "pause_promo":
        n = c.execute("UPDATE marketing SET end_date=date('now','-1 day') WHERE outlet_id=? AND end_date>=date('now')", (a["target"],)).rowcount
        return f"{n} campaign(s) paused"
    return ""

# ---------------------------------------------------------------- UI hooks
def kpi_extra():
    s = Q("SELECT COALESCE(SUM(amount),0) v FROM sales WHERE source='sim'").v[0]; po = Q("SELECT COUNT(*) n FROM purchase_orders WHERE status='ordered'").n[0]
    return [("Live sales since start", f"Rs {s:,.0f}"), ("Purchase orders in transit", int(po))]


Writing /content/app/domain_franchise.py


In [11]:
%%writefile /content/app/entities.py
"""One description of any entity (outlet, staff, quote, shipment, vessel, port): master data, relations, and everything flagged on it. Feeds the drawer, the knowledge graph and the Copilot."""
import json
from blocks import Q, clean

def _tickets(e, n=30):
    return Q("SELECT action_id id, type, severity, status, detail, owner, assignee, due FROM agent_actions WHERE target=? ORDER BY action_id DESC LIMIT ?", (e, n))
def latest_out(agent):
    r = Q("SELECT output FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,))
    try: return json.loads(r.output[0]) if len(r) else {}
    except Exception: return {}
def kind_of(e):
    e = str(e)
    if e.startswith("OUT-"): return "outlet"
    if e.startswith("QTE"): return "quote"
    if e.startswith("SHP"): return "shipment"
    if e.startswith("EMP") or e.startswith("STF"): return "staff"
    return "port"

def describe(kind, dom, e):
    """-> {"kind","info":{label:value},"rel":[(name,kind,tip)], "tickets": DataFrame}"""
    k = kind_of(e); info, rel = {}, []
    if kind == "Franchise" and k == "outlet":
        o = Q("SELECT * FROM outlets WHERE outlet_id=?", (e,))
        if len(o):
            r = o.iloc[0]; info = {"Outlet": r.outlet_name, "Location": r.location, "Revenue": f"{r.revenue:,.0f}", "Customer satisfaction": round(float(r.customer_satisfaction), 2), "Headcount": int(r.staff_headcount)}
            for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
                v = latest_out(a).get("per_outlet", {}).get(e)
                if v is not None: rel.append((f"{a} agent: {v:.0f}", "agent", f"{a} score for {e}"))
            for s in Q("SELECT staff_id, predicted_attrition_prob p FROM staff WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.staff_id, "staff", f"attrition {s.p:.0%}"))
            for s in Q("SELECT sku_name, stockout_risk_prob p FROM inventory WHERE outlet_id=? ORDER BY p DESC LIMIT 4", (e,)).itertuples(): rel.append((s.sku_name, "sku", f"stock-out risk {s.p:.0%}"))
            for s in Q("SELECT campaign_name, channel FROM marketing WHERE outlet_id=? AND end_date>=date('now')", (e,)).itertuples(): rel.append((s.campaign_name, "campaign", f"live via {s.channel}"))
    elif kind == "Freight" and k == "quote":
        q = Q("SELECT * FROM freight_quotes WHERE quote_id=?", (e,))
        if len(q):
            r = q.iloc[0]; info = {"Quote": e, "Price": f"${r.final_price:,.0f}", "Margin": f"{r.margin_pct}%", "Status": r.status}
            for n, tip in ((r.shipment_id, "shipment"), (r.customer_id, "customer"), (r.carrier, "carrier")): rel.append((n, tip, tip))
    elif kind == "Freight" and k == "shipment":
        s = Q("SELECT * FROM shipments WHERE shipment_id=?", (e,))
        if len(s):
            r = s.iloc[0]; info = {"Shipment": e, "Route": f"{r.origin_port} to {r.dest_port}", "Carrier": r.carrier, "Status": r.status, "Delay risk": r.predicted_delay_risk}
            for n, tip in ((r.origin_port, "port"), (r.dest_port, "port"), (r.carrier, "carrier")): rel.append((n, tip, tip))
            v = Q("SELECT lat, lon, route_name, progress FROM vessels WHERE shipment_id=?", (e,))
            if len(v): info["Vessel"] = f"{v.lat[0]:.1f}, {v.lon[0]:.1f} · {v.route_name[0]} · {v.progress[0]:.0%}"
    elif kind == "Freight":
        p = Q("SELECT * FROM ports WHERE port_name=?", (e,))
        if len(p):
            r = p.iloc[0]; info = {"Port": e, "Country": r.country, "Congestion index": round(float(r.congestion_index), 2), "Dwell days": r.avg_dwell_days}
            for s in Q("SELECT shipment_id FROM shipments WHERE origin_port=? OR dest_port=? LIMIT 6", (e, e)).itertuples(): rel.append((s.shipment_id, "shipment", "via port"))
    return {"kind": k, "info": {a: clean(b) for a, b in info.items()}, "rel": rel, "tickets": _tickets(e)}

def facts(kind, dom, text):
    import re
    out = []
    for e in dict.fromkeys(m.upper() for m in re.findall(r"\b(OUT-\d+|QTE-L\d+|SHP-[A-Z0-9-]+)\b", text, re.I)):
        d = describe(kind, dom, e); t = d["tickets"]; t = t[t.status.isin(["open", "proposed"])].head(8)
        if d["info"] or len(t): out.append(f"{e}: " + "; ".join(f"{k}={v}" for k, v in d["info"].items()) + (f". Related: {', '.join(n for n, _, _ in d['rel'][:8])}." if d["rel"] else "") + (" Open tickets: " + " | ".join(f"{r.type} ({r.severity}): {r.detail}" for r in t.itertuples()) if len(t) else " No open tickets."))
    return out


Writing /content/app/entities.py


In [12]:
%%writefile /content/app/gpu.py
"""Minimal stub (no language model in this version). Kept so hardening.py can report health."""
def status(): return {"cuda": False, "gpu": None, "models": {}, "stats": {}}


Writing /content/app/gpu.py


In [13]:
%%writefile /content/app/hardening.py
"""Production hardening for the FastAPI app: request ids + JSON access log, security headers (CSP, frame, permissions), per-IP rate limits on auth routes,
a global JSON error handler that never leaks tracebacks, and Prometheus-style metrics. Install with hardening.install(app)."""
import os, time, json, uuid, threading, logging, traceback, collections
from logging.handlers import RotatingFileHandler
from fastapi import Request
from fastapi.responses import JSONResponse, PlainTextResponse
import config

_LOG = logging.getLogger("access"); _LOG.setLevel(logging.INFO); _LOG.propagate = False
try:
    _h = RotatingFileHandler(os.path.join(config.DATA_DIR, "access.log"), maxBytes=2_000_000, backupCount=3); _h.setFormatter(logging.Formatter("%(message)s")); _LOG.addHandler(_h)
except Exception: pass
_ERR = logging.getLogger("errors"); _ERR.propagate = False
try:
    _e = RotatingFileHandler(os.path.join(config.DATA_DIR, "errors.log"), maxBytes=1_000_000, backupCount=2); _e.setFormatter(logging.Formatter("%(asctime)s %(message)s")); _ERR.addHandler(_e)
except Exception: pass

M = {"requests": collections.Counter(), "status": collections.Counter(), "lat_sum": collections.Counter(), "lat_n": collections.Counter(), "errors": 0, "limited": 0, "started": time.time()}
_L = threading.Lock(); _BUCKET = {}
CSP = "default-src 'self'; img-src 'self' data: blob:; style-src 'self' 'unsafe-inline'; script-src 'self' 'unsafe-inline'; connect-src 'self'; font-src 'self' data:; frame-ancestors 'none'; base-uri 'self'; form-action 'self'"
# path prefix -> (max requests, per seconds). Everything else: 900 per minute per IP.
LIMITS = {"/api/login": (10, 60), "/api/register": (6, 600), "/api/otp": (5, 300), "/api/reset": (8, 600), "/api/profile/password": (8, 600), "/api/copilot": (40, 60), "/api/translate": (60, 60)}
def _ip(r):
    f = r.headers.get("x-forwarded-for"); return (f.split(",")[0].strip() if f else (r.client.host if r.client else "?"))
def _allow(ip, path):
    key, (n, per) = next(((p, v) for p, v in LIMITS.items() if path.startswith(p)), ("*", (900, 60))); k = (ip, key); now = time.time()
    with _L:
        q = _BUCKET.setdefault(k, collections.deque())
        while q and now - q[0] > per: q.popleft()
        if len(q) >= n: return False, int(per - (now - q[0])) + 1
        q.append(now)
        if len(_BUCKET) > 20000: _BUCKET.clear()
    return True, 0
def _route(path):
    parts = path.split("/")
    return "/".join(parts[:3]) if path.startswith("/api/") else "static"

def install(app, kind="app"):
    @app.middleware("http")
    async def guard(request: Request, call_next):
        rid = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]; t0 = time.time(); path = request.url.path
        if os.environ.get("RATE_LIMIT_OFF") != "1" and (path.startswith("/api/") and request.method != "GET" or path.startswith("/api/login")):
            ok, wait = _allow(_ip(request), path)
            if not ok:
                M["limited"] += 1; return JSONResponse({"ok": False, "msg": f"Too many requests. Try again in {wait}s.", "request_id": rid}, status_code=429, headers={"Retry-After": str(wait), "X-Request-ID": rid})
        try: r = await call_next(request)
        except Exception as e:
            M["errors"] += 1; _ERR.error("%s %s %s\n%s", rid, request.method, path, traceback.format_exc())
            return JSONResponse({"ok": False, "msg": "Something went wrong on the server. Quote this id to the admin: " + rid, "request_id": rid}, status_code=500, headers={"X-Request-ID": rid})
        ms = (time.time() - t0) * 1000; rt = _route(path)
        with _L: M["requests"][rt] += 1; M["status"][r.status_code // 100] += 1; M["lat_sum"][rt] += ms; M["lat_n"][rt] += 1
        h = r.headers; h["X-Request-ID"] = rid; h["X-Frame-Options"] = "DENY"; h["Content-Security-Policy"] = CSP; h["Permissions-Policy"] = "microphone=(self), camera=(), geolocation=()"; h["Cross-Origin-Opener-Policy"] = "same-origin"
        if request.headers.get("x-forwarded-proto") == "https": h["Strict-Transport-Security"] = "max-age=31536000"
        if path.startswith("/api/") and path not in ("/api/stream", "/api/pulse"): _LOG.info(json.dumps({"t": round(t0), "id": rid, "ip": _ip(request), "m": request.method, "p": path, "s": r.status_code, "ms": round(ms)}))
        return r

    @app.get("/api/ready")
    def ready(): return {"ok": True, "kind": kind, "uptime_s": round(time.time() - M["started"])}

    @app.get("/api/metrics")
    def metrics(request: Request):
        """Prometheus text format. Set METRICS_TOKEN to require ?token=..., otherwise only logged-in admins (cookie) may read it."""
        tok = os.environ.get("METRICS_TOKEN")
        if tok and request.query_params.get("token") != tok: return PlainTextResponse("forbidden", status_code=403)
        if not tok:
            try:
                u = request.app.state.user_of(request)          # server.py registers its user_of here; no registered user_of means forbidden
                if not u or u.get("role") != "Admin": return PlainTextResponse("forbidden", status_code=403)
            except Exception: return PlainTextResponse("forbidden", status_code=403)
        L = ["# TYPE app_requests_total counter"] + [f'app_requests_total{{route="{k}"}} {v}' for k, v in M["requests"].items()] + ["# TYPE app_responses_total counter"] + [f'app_responses_total{{class="{k}xx"}} {v}' for k, v in M["status"].items()]
        L += ["# TYPE app_latency_ms_avg gauge"] + [f'app_latency_ms_avg{{route="{k}"}} {M["lat_sum"][k] / max(1, M["lat_n"][k]):.1f}' for k in M["lat_n"]] + [f"app_errors_total {M['errors']}", f"app_rate_limited_total {M['limited']}", f"app_uptime_seconds {time.time() - M['started']:.0f}"]
        try:
            import live_engine as LE; c = LE.conn(); L += [f"app_open_tickets {c.execute('SELECT COUNT(*) FROM agent_actions WHERE status=?', ('open',)).fetchone()[0]}"]
        except Exception: pass
        try:
            import gpu; g = gpu.status(); L += [f'app_model_ready{{model="{k}"}} {1 if v["state"] == "ready" else 0}' for k, v in g["models"].items()]
        except Exception: pass
        return PlainTextResponse("\n".join(L) + "\n")


Writing /content/app/hardening.py


In [14]:
%%writefile /content/app/intent_router.py
import re as _re_live
import pandas as pd
import re, math
from db import get_conn

def df_to_markdown_safe(df):
    """Safely converts a pandas DataFrame to Markdown format."""
    if df is None or df.empty:
        return ""
    try:
        return df.to_markdown(index=False)
    except Exception:
        headers = list(df.columns)
        lines = ["| " + " | ".join([str(h) for h in headers]) + " |"]
        lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
        for _, row in df.iterrows():
            vals = [str(v) if v is not None else "" for v in row.values]
            lines.append("| " + " | ".join(vals) + " |")
        return "\n".join(lines)

# Use meaningful phrases/whole words. Avoid tiny substrings such as
# "ad", "item", and "location" accidentally hijacking unrelated questions.
INTENT_MAP = {
    "staff": ["staff", "employee", "employees", "attrition", "salary", "workforce", "headcount", "hiring", "hire"],
    "outlet": ["outlet", "store", "branch", "revenue", "margin", "sales", "profit", "csat", "customer satisfaction"],
    "inventory": ["inventory", "stock", "stockout", "sku", "reorder", "demand", "supply", "safety stock"],
    "marketing": ["marketing", "campaign", "roi", "conversion", "promotion", "budget", "channel", "advertising", "advertisement", "ads"],
    "audit": ["audit", "compliance", "violation", "inspection", "policy", "standard", "fssai", "food safety"],
    "feedback": ["feedback", "review", "sentiment", "rating", "complaint", "customer review"],
}

def _contains_term(q, term):
    return bool(re.search(r"(?<!\w)" + re.escape(term) + r"(?!\w)", q))

def classify_intent(query):
    """Score meaningful whole-word/phrase matches and return one primary domain."""
    q = query.lower().strip()
    scores = {}
    for intent, keywords in INTENT_MAP.items():
        score = 0
        for kw in keywords:
            if _contains_term(q, kw):
                score += 2 if " " in kw else 1
        scores[intent] = score
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "general"

def handle_franchise_intent(query):
    """Return only data for the most relevant franchise domain."""
    q_low = query.lower()
    intent = classify_intent(query)

    try:
        with get_conn() as conn:
            if intent == "outlet":
                total = conn.execute("SELECT COUNT(*) FROM outlets;").fetchone()[0]
                revenue = conn.execute("SELECT SUM(revenue) FROM outlets;").fetchone()[0] or 0
                csat = conn.execute("SELECT AVG(customer_satisfaction) FROM outlets;").fetchone()[0] or 0

                if any(_contains_term(q_low, k) for k in ["margin", "profit"]):
                    sql = """SELECT outlet_id, outlet_name, location, tier,
                             ROUND(revenue, 0) AS revenue_rs,
                             ROUND(operating_costs, 0) AS costs_rs,
                             ROUND(((revenue - operating_costs)/NULLIF(revenue,0))*100, 2) AS net_margin_pct,
                             customer_satisfaction
                             FROM outlets ORDER BY net_margin_pct DESC LIMIT 10;"""
                else:
                    sql = """SELECT outlet_id, outlet_name, location, tier,
                             ROUND(revenue, 0) AS revenue_rs,
                             ROUND(((revenue - operating_costs)/NULLIF(revenue,0))*100, 2) AS net_margin_pct,
                             customer_satisfaction
                             FROM outlets ORDER BY revenue_rs DESC LIMIT 10;"""
                df = pd.read_sql(sql, conn)
                return f"### Franchise Outlet Data\nActive outlets: **{total}** | Total revenue: **Rs. {revenue:,.0f}** | Average CSAT: **{csat:.2f}/5**\n\n{df_to_markdown_safe(df)}", "Outlets DB"

            if intent == "staff":
                total = conn.execute("SELECT COUNT(*) FROM staff;").fetchone()[0]
                high_risk = conn.execute("SELECT COUNT(*) FROM staff WHERE predicted_attrition_prob > 0.6;").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, role, COUNT(*) AS staff_count,
                                    ROUND(AVG(salary), 0) AS avg_salary_rs,
                                    ROUND(AVG(predicted_attrition_prob), 2) AS avg_attrition_risk
                                    FROM staff GROUP BY outlet_id, role
                                    ORDER BY avg_attrition_risk DESC LIMIT 10;""", conn)
                return f"### Workforce Data\nTotal staff: **{total}** | High attrition risk (>60%): **{high_risk}**\n\n{df_to_markdown_safe(df)}", "Staff DB"

            if intent == "inventory":
                total_skus = conn.execute("SELECT COUNT(*) FROM inventory;").fetchone()[0]
                risk_items = conn.execute("SELECT COUNT(*) FROM inventory WHERE stockout_risk_prob > 0.7;").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, sku_name, category, current_stock,
                                    reorder_threshold, stockout_risk_prob
                                    FROM inventory ORDER BY stockout_risk_prob DESC LIMIT 12;""", conn)
                return f"### Inventory Data\nTracked SKUs: **{total_skus}** | SKUs above 70% stockout risk: **{risk_items}**\n\n{df_to_markdown_safe(df)}", "Inventory DB"

            if intent == "marketing":
                avg_roi = conn.execute("SELECT AVG(actual_roi) FROM marketing;").fetchone()[0] or 0
                df = pd.read_sql("""SELECT channel, COUNT(*) AS campaigns,
                                    ROUND(AVG(actual_roi), 2) AS avg_roi,
                                    SUM(conversions) AS conversions
                                    FROM marketing GROUP BY channel ORDER BY avg_roi DESC;""", conn)
                return f"### Marketing Data\nAverage campaign ROI: **{avg_roi:.2f}x**\n\n{df_to_markdown_safe(df)}", "Marketing DB"

            if intent == "feedback":
                avg_rating = conn.execute("SELECT AVG(rating) FROM feedback;").fetchone()[0] or 0
                df = pd.read_sql("""SELECT outlet_id, rating, comment, sentiment_score, date
                                    FROM feedback ORDER BY sentiment_score ASC LIMIT 10;""", conn)
                return f"### Customer Feedback Data\nAverage rating: **{avg_rating:.2f}/5**\n\n{df_to_markdown_safe(df)}", "Feedback DB"

            if intent == "audit":
                avg_score = conn.execute("SELECT AVG(score) FROM audits;").fetchone()[0] or 0
                open_items = conn.execute("SELECT COUNT(*) FROM audits WHERE status != 'Pass';").fetchone()[0]
                df = pd.read_sql("""SELECT outlet_id, audit_date, score, violations, category, status
                                    FROM audits ORDER BY score ASC LIMIT 10;""", conn)
                return f"### Audit & Compliance Data\nAverage score: **{avg_score:.1f}/100** | Non-pass audits: **{open_items}**\n\n{df_to_markdown_safe(df)}", "Audits DB"

    except Exception:
        return None

    return None

def handle_port_intent(query):
    try:
        with get_conn() as conn:
            tot_ports = conn.execute("SELECT COUNT(*) FROM ports;").fetchone()[0]
            df_ports = pd.read_sql("""SELECT port_name AS 'Port Name', country AS 'Country',
                                      region AS 'Region',
                                      congestion_index AS 'Congestion (1-5)',
                                      avg_dwell_days AS 'Avg Dwell Days'
                                      FROM ports ORDER BY congestion_index ASC LIMIT 10;""", conn)
        return f"### Global Ports Data ({tot_ports} monitored)\n\n{df_to_markdown_safe(df_ports)}", f"Ports Ledger DB ({tot_ports} Hubs)"
    except Exception:
        return None, None


_AGENTIC_Q = _re_live.compile(r"(need[s]? attention|what should i do|action plan|prioriti[sz]e|overall (status|health)|run (the )?agents?|full analysis|full status|why is .{0,40}(low|down|failing|underperform|risky)|what is going wrong)")
def _agentic_query(query):
    """Open-ended 'what should I do / what needs attention' questions run the whole agent graph instead of a single table lookup."""
    if not _AGENTIC_Q.search(query.lower()): return None
    try:
        from live_engine import run_cycle
        import domain_franchise as _dom
        dec = run_cycle(_dom, "copilot", query)
        top = "\n".join(f"- **{a['severity']}** · {a['type']} · {a['target']}: {a['detail']}" for a in dec["actions"][:8])
        return f"### Agent orchestration result\n\n{dec['narrative']}\n\n**Top actions**\n{top}", "Agent Orchestrator (live)"
    except Exception:
        return None

def _run_centralized_orig(query):
    """Retrieve one relevant evidence set. Do not mix unrelated data sources."""
    q_low = query.lower()

    franchise = handle_franchise_intent(query)
    if franchise:
        return franchise

    if any(_contains_term(q_low, k) for k in ["port", "ports", "harbor", "terminal", "congestion"]):
        return handle_port_intent(query)

    if any(_contains_term(q_low, k) for k in ["shipment", "shipments", "carrier"]):
        try:
            with get_conn() as conn:
                tot_shipments = conn.execute("SELECT COUNT(*) FROM shipments;").fetchone()[0]
                df_ship = pd.read_sql("""SELECT shipment_id, origin_port, dest_port, carrier,
                                         status, predicted_delay_risk
                                         FROM shipments ORDER BY predicted_delay_risk DESC LIMIT 10;""", conn)
            return f"### Active Shipments ({tot_shipments})\n\n{df_to_markdown_safe(df_ship)}", "Shipments Ledger DB"
        except Exception:
            pass

    # Only invoke RAG for questions that clearly look like document/SOP/policy questions.
    rag_terms = ["sop", "document", "manual", "policy", "procedure", "fssai", "hygiene", "food safety", "guideline"]
    if any(_contains_term(q_low, k) for k in rag_terms):
        try:
            from rag_engine import answer_with_citation
            ctx, src = answer_with_citation(query)
            if ctx and str(ctx).strip():
                return ctx, src
        except Exception:
            pass

    return None, None

def run_grounded_query(query):
    return run_centralized_brain_query(query)

def text_to_sql(query):
    return run_centralized_brain_query(query)

def run_centralized_brain_query(query):
    _ag = _agentic_query(query)
    return _ag if _ag else _run_centralized_orig(query)


Writing /content/app/intent_router.py


In [15]:
%%writefile /content/app/intents.py
"""Copilot question patterns (Step 7). Every pattern is an intent that runs ONE fixed, read-only, parameterised SQL query over the live tables;
the answer is built from the rows that query returns, so no number is ever made up. Table and column names are fixed in this file, values are bound parameters.

- Spelling mistakes and other wordings: normalize() maps synonyms (store/shop/branch -> outlet, employees/workers -> staff, csat -> satisfaction ...)
  and corrects near-misses against the vocabulary ("custmer", "revanue", "campain", "reoder", "audt" ...).
- RBAC: every query takes the caller's outlet scope (Store Manager and Staff only ever see their own outlet) and the PII mask (staff names are hidden from Staff and Auditor).
- Every answer carries the SQL that produced it and the number of rows it returned."""
import re, difflib
import pandas as pd
from blocks import Q

CANON = {  # synonym -> canonical word
    "outlets": "outlet", "store": "outlet", "stores": "outlet", "shop": "outlet", "shops": "outlet", "branch": "outlet", "branches": "outlet", "franchise": "outlet", "franchises": "outlet",
    "employee": "staff", "employees": "staff", "worker": "staff", "workers": "staff", "personnel": "staff", "workforce": "staff", "people": "staff", "staffs": "staff", "manpower": "staff",
    "csat": "satisfaction", "satisfied": "satisfaction", "turnover": "revenue", "income": "revenue", "earnings": "revenue", "takings": "revenue",
    "stock": "inventory", "stocks": "inventory", "sku": "inventory", "skus": "inventory", "item": "inventory", "items": "inventory", "product": "inventory", "products": "inventory", "ingredient": "inventory", "ingredients": "inventory",
    "campaigns": "campaign", "promotion": "campaign", "promotions": "campaign", "promo": "campaign", "promos": "campaign", "ads": "campaign",
    "reviews": "review", "feedback": "review", "feedbacks": "review", "comment": "review", "comments": "review", "complaint": "review", "complaints": "review",
    "audits": "audit", "inspection": "audit", "inspections": "audit", "compliance": "audit", "failing": "fail", "failed": "fail", "fails": "fail", "failures": "fail",
    "tickets": "ticket", "alerts": "ticket", "alert": "ticket", "violations": "violation", "ratings": "rating", "orders": "order", "costs": "cost", "expenses": "cost",
    "biggest": "highest", "largest": "highest", "greatest": "highest", "maximum": "highest", "max": "highest", "top": "highest", "most": "highest", "best": "best",
    "smallest": "lowest", "least": "lowest", "minimum": "lowest", "min": "lowest", "poorest": "lowest", "weakest": "lowest", "bottom": "lowest",
    "restock": "reorder", "restocking": "reorder", "reordering": "reorder", "replenish": "reorder", "reorders": "reorder", "overtime": "overtime", "mean": "average", "avg": "average",
}
VOCAB = set(CANON.values()) | set(CANON) | {w for w in """customer satisfaction revenue inventory reorder level threshold review negative positive audit violation rating average total count many number list show which what who how
 highest lowest best worst campaign roi return channel staff attrition risk likely leave quit employee outlet purchase order pending open critical urgent ticket sales week today last
 stockout shortage operating cost running attendance salary city live active fail score percent items need below under above more less than with for the of in at by per each all every has have does are is
 give tell find display top first""".split()}
STOP = {"works", "work", "working", "worked", "earns", "earn", "does", "the", "and", "for", "with", "that", "this", "are", "was", "were", "has", "have", "does", "did", "you", "your", "our", "can", "please", "could", "would", "tell", "give", "show", "list", "which", "what", "who", "how", "when", "where", "there"}

def normalize(q):
    """lower-case, fix near-miss spellings, map synonyms to one canonical word (outlet ids and numbers are kept as they are)."""
    out = []
    for tok in re.findall(r"out-\d+|[a-z]+|\d+(?:\.\d+)?|[<>=%]", q.lower()):
        if tok in CANON: tok = CANON[tok]
        elif tok.isalpha() and len(tok) >= 4 and tok not in VOCAB and tok not in STOP:
            m = difflib.get_close_matches(tok, VOCAB, n=1, cutoff=0.78)
            if m: tok = CANON.get(m[0], m[0])
        out.append(tok)
    return " ".join(out)

# ------------------------------------------------------------------ helpers
def _n(q, default):
    m = re.search(r"\b(?:highest|lowest|best|worst|first|top|bottom)\s+(\d{1,2})\b|\b(\d{1,2})\s+(?:outlet|staff|inventory|campaign|review|audit|ticket|order)", q)
    return min(25, int(next(g for g in m.groups() if g))) if m else default
def _scope(scope, col="outlet_id", pre=" WHERE "):
    return (f"{pre}{col}=?", [scope]) if scope else ("", [])
def _sql_shown(sql, params):
    s = sql
    for p in params: s = s.replace("?", repr(p) if isinstance(p, str) else str(p), 1)
    return s
def _run(sql, params):
    return Q(sql, tuple(params))
def _out(intent, answer, df, sql, params):
    return {"intent": intent, "answer": answer, "df": df if df is not None and len(df) else None, "sql": _sql_shown(sql, params), "rows": 0 if df is None else int(len(df)), "params": list(params), "complete": True}
def _none(intent, what, sql, params, scope):
    return _out(intent, f"No {what} found" + (f" for {scope}." if scope else "."), None, sql, params)
def _tail(q): return re.search(r"out-(\d+)", q)
def _oid(q, scope):
    m = _tail(q); return scope or (f"OUT-{int(m.group(1)):03d}" if m else None)
def _pct(x): return f"{float(x):.0%}"
def _money(x): return f"{float(x):,.0f}"

# ------------------------------------------------------------------ intents: (id, example question, regex on the normalized question, handler(q, scope, mask))
def csat(q, scope, mask, low):
    w, p = _scope(scope); n = _n(q, 5); sql = f"SELECT outlet_id, outlet_name, location, customer_satisfaction FROM outlets{w} ORDER BY customer_satisfaction {'ASC' if low else 'DESC'} LIMIT ?"; p += [n]
    d = _run(sql, p)
    if not len(d): return _none("lowest_csat" if low else "highest_csat", "outlets", sql, p, scope)
    r = d.iloc[0]; return _out("lowest_csat" if low else "highest_csat", f"**{r.outlet_id}** ({r.outlet_name}, {r.location}) has the {'lowest' if low else 'highest'} customer satisfaction: **{r.customer_satisfaction:.2f}** out of 5.", d, sql, p)

def revenue(q, scope, mask, low):
    w, p = _scope(scope); n = _n(q, 5); sql = f"SELECT outlet_id, outlet_name, location, revenue FROM outlets{w} ORDER BY revenue {'ASC' if low else 'DESC'} LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("lowest_revenue" if low else "highest_revenue", "outlets", sql, p, scope)
    r = d.iloc[0]; return _out("lowest_revenue" if low else "highest_revenue", f"**{r.outlet_id}** ({r.outlet_name}) has the {'lowest' if low else 'highest'} revenue: **{_money(r.revenue)}**.", d, sql, p)

def staff_count(q, scope, mask):
    o = _oid(q, scope); w, p = (" WHERE outlet_id=?", [o]) if o else ("", []); sql = f"SELECT COUNT(*) AS staff FROM staff{w}"; d = _run(sql, p); n = int(d.staff[0])
    return _out("staff_count", f"**{o}** has **{n:,}** staff members." if o else f"There are **{n:,}** staff members in total.", d, sql, p)

def most_staff(q, scope, mask):
    w, p = _scope(scope); n = _n(q, 5); sql = f"SELECT outlet_id, COUNT(*) AS staff FROM staff{w} GROUP BY outlet_id ORDER BY staff DESC, outlet_id LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("highest_staff_count", "staff", sql, p, scope)
    r = d.iloc[0]; return _out("highest_staff_count", f"**{r.outlet_id}** has the most staff: **{int(r.staff)}**.", d, sql, p)

def attrition(q, scope, mask):
    w, p = _scope(scope, pre=" AND "); n = _n(q, 10); cols = "staff_id, role, outlet_id, predicted_attrition_prob" if mask else "staff_id, name, role, outlet_id, predicted_attrition_prob"
    sql = f"SELECT {cols} FROM staff WHERE predicted_attrition_prob>=0.6{w} ORDER BY predicted_attrition_prob DESC LIMIT ?"; p += [n]; d = _run(sql, p)
    cs, cp = f"SELECT COUNT(*) AS n FROM staff WHERE predicted_attrition_prob>=0.6{w}", ([scope] if scope else []); tot = int(_run(cs, cp).n[0])
    if not tot: return _none("high_attrition", "employees with a high attrition risk (60% or more)", sql, p, scope)
    d = d.assign(predicted_attrition_prob=d.predicted_attrition_prob.map(_pct)); return _out("high_attrition", f"**{tot:,}** employees have a high attrition risk (60% or more){' at ' + scope if scope else ''}. The {len(d)} highest are below." + (" Names are hidden for your role." if mask else ""), d, sql, p)

def below_reorder(q, scope, mask):
    w, p = _scope(scope, pre=" AND "); n = _n(q, 15); sql = f"SELECT outlet_id, sku_name, current_stock, reorder_threshold FROM inventory WHERE current_stock<=reorder_threshold{w} ORDER BY current_stock*1.0/MAX(reorder_threshold,1) LIMIT ?"; p += [n]; d = _run(sql, p)
    tot = int(_run(f"SELECT COUNT(*) n FROM inventory WHERE current_stock<=reorder_threshold{w}", [scope] if scope else []).n[0])
    if not tot: return _none("below_reorder", "inventory items at or below their reorder level", sql, p, scope)
    return _out("below_reorder", f"**{tot:,}** inventory items are at or below their reorder level{' at ' + scope if scope else ''}. The {len(d)} most urgent are below.", d, sql, p)

def reorder_count(q, scope, mask):
    w, p = _scope(scope, pre=" AND "); sql = f"SELECT COUNT(*) AS items_to_reorder FROM inventory WHERE current_stock<=reorder_threshold{w}"; d = _run(sql, p)
    return _out("reorder_count", f"**{int(d.items_to_reorder[0]):,}** inventory items need reordering (stock at or below the reorder level){' at ' + scope if scope else ''}.", d, sql, p)

def stockout(q, scope, mask):
    w, p = _scope(scope); n = _n(q, 10); sql = f"SELECT outlet_id, sku_name, current_stock, weekly_demand, stockout_risk_prob FROM inventory{w} ORDER BY stockout_risk_prob DESC LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("stockout_risk", "inventory items", sql, p, scope)
    r = d.iloc[0]; d = d.assign(stockout_risk_prob=d.stockout_risk_prob.map(_pct)); return _out("stockout_risk", f"Highest stock-out risk: **{r.sku_name}** at **{r.outlet_id}** ({_pct(r.stockout_risk_prob)}). Top {len(d)} below.", d, sql, p)

def roi(q, scope, mask, low):
    w, p = _scope(scope); n = _n(q, 5); sql = f"SELECT campaign_id, campaign_name, outlet_id, channel, budget, actual_roi FROM marketing{w} ORDER BY actual_roi {'ASC' if low else 'DESC'} LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("worst_campaign_roi" if low else "best_campaign_roi", "campaigns", sql, p, scope)
    r = d.iloc[0]; return _out("worst_campaign_roi" if low else "best_campaign_roi", f"The campaign with the {'lowest' if low else 'best'} ROI is **{r.campaign_name}** ({r.campaign_id}, {r.channel}, {r.outlet_id}): ROI **{r.actual_roi:.2f}**x.", d, sql, p)

def avg_roi(q, scope, mask):
    w, p = _scope(scope); sql = f"SELECT channel, ROUND(AVG(actual_roi),2) AS avg_roi, COUNT(*) AS campaigns FROM marketing{w} GROUP BY channel ORDER BY avg_roi DESC"; d = _run(sql, p)
    if not len(d): return _none("average_campaign_roi", "campaigns", sql, p, scope)
    sql2 = f"SELECT ROUND(AVG(actual_roi),2) AS v FROM marketing{w}"; v = float(_run(sql2, p).v[0])
    return _out("average_campaign_roi", f"The average campaign ROI is **{v:.2f}**x across {int(d.campaigns.sum()):,} campaigns. Best channel: **{d.channel[0]}** ({d.avg_roi[0]:.2f}x).", d, sql, p)

def negative_reviews(q, scope, mask):
    week = bool(re.search(r"\bweek\b|\brecent\b|\blatest\b", q)); w, p = _scope(scope, pre=" AND "); n = _n(q, 10)
    recent = " AND date>=date((SELECT MAX(date) FROM feedback),'-7 day')" if week else ""
    sql = f"SELECT feedback_id, outlet_id, rating, date, comment FROM feedback WHERE rating<=2{recent}{w} ORDER BY date DESC, rating LIMIT ?"; p += [n]; d = _run(sql, p)
    tot = int(_run(f"SELECT COUNT(*) n FROM feedback WHERE rating<=2{recent}{w}", [scope] if scope else []).n[0])
    if not tot: return _none("negative_reviews", "negative reviews" + (" in the last week of data" if week else ""), sql, p, scope)
    return _out("negative_reviews", f"**{tot:,}** negative reviews (rating 1-2){' in the last week of data' if week else ''}{' at ' + scope if scope else ''}. The latest {len(d)} are below.", d, sql, p)

def most_negative(q, scope, mask):
    w, p = _scope(scope, pre=" AND "); n = _n(q, 5); sql = f"SELECT outlet_id, COUNT(*) AS negative_reviews FROM feedback WHERE rating<=2{w} GROUP BY outlet_id ORDER BY negative_reviews DESC, outlet_id LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("most_negative_reviews", "negative reviews", sql, p, scope)
    r = d.iloc[0]; return _out("most_negative_reviews", f"**{r.outlet_id}** has the most negative reviews: **{int(r.negative_reviews)}**.", d, sql, p)

def avg_rating(q, scope, mask):
    w, p = _scope(scope); sql = f"SELECT ROUND(AVG(rating),2) AS average_rating, COUNT(*) AS reviews FROM feedback{w}"; d = _run(sql, p)
    if not int(d.reviews[0]): return _none("average_rating", "reviews", sql, p, scope)
    return _out("average_rating", f"The average customer rating is **{d.average_rating[0]:.2f}** out of 5, from {int(d.reviews[0]):,} reviews{' at ' + scope if scope else ''}.", d, sql, p)

def failed_audits(q, scope, mask):
    w, p = _scope(scope, pre=" AND "); n = _n(q, 10); sql = f"SELECT audit_id, outlet_id, audit_date, score, violations, category, status FROM audits WHERE score<60{w} ORDER BY audit_date DESC, score LIMIT ?"; p += [n]; d = _run(sql, p)
    tot = int(_run(f"SELECT COUNT(*) n FROM audits WHERE score<60{w}", [scope] if scope else []).n[0])
    if not tot: return _none("failed_audits", "failed audits (score below 60)", sql, p, scope)
    return _out("failed_audits", f"**{tot:,}** audits failed (score below 60){' at ' + scope if scope else ''}. The latest {len(d)} are below.", d, sql, p)

def violations(q, scope, mask):
    w, p = _scope(scope); n = _n(q, 5); sql = f"SELECT outlet_id, SUM(violations) AS violations, COUNT(*) AS audits FROM audits{w} GROUP BY outlet_id ORDER BY violations DESC, outlet_id LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("highest_violations", "audits", sql, p, scope)
    r = d.iloc[0]; return _out("highest_violations", f"**{r.outlet_id}** has the most audit violations: **{int(r.violations)}** across {int(r.audits)} audits.", d, sql, p)

def total_outlets(q, scope, mask):
    w, p = _scope(scope); sql = f"SELECT COUNT(*) AS outlets FROM outlets{w}"; d = _run(sql, p); n = int(d.outlets[0])
    return _out("total_outlets", f"You can see **{n:,}** outlet{'s' if n != 1 else ''}" + (f" ({scope})." if scope else " in the franchise."), d, sql, p)

def op_cost(q, scope, mask):
    w, p = _scope(scope); n = _n(q, 5); sql = f"SELECT outlet_id, outlet_name, location, operating_costs, revenue FROM outlets{w} ORDER BY operating_costs DESC LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("highest_operating_cost", "outlets", sql, p, scope)
    r = d.iloc[0]; return _out("highest_operating_cost", f"**{r.outlet_id}** ({r.outlet_name}) has the highest operating costs: **{_money(r.operating_costs)}**.", d, sql, p)

def tickets(q, scope, mask):
    w, p = _scope(scope, col="target", pre=" AND "); n = _n(q, 10); crit = " AND severity='Critical'" if re.search(r"critical|urgent", q) else ""
    sql = f"SELECT action_id, severity, type, target, detail FROM agent_actions WHERE status IN ('open','proposed'){crit}{w} ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT ?"; p += [n]; d = _run(sql, p)
    tot = int(_run(f"SELECT COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed'){crit}{w}", [scope] if scope else []).n[0])
    if not tot: return _none("open_tickets", ("critical " if crit else "") + "open tickets", sql, p, scope)
    return _out("open_critical_tickets" if crit else "open_tickets", f"**{tot:,}** {'critical ' if crit else ''}tickets are open{' for ' + scope if scope else ''}. The {len(d)} most severe are below.", d, sql, p)

def overtime(q, scope, mask):
    w, p = _scope(scope); n = _n(q, 10); cols = "staff_id, role, outlet_id, overtime_hrs" if mask else "staff_id, name, role, outlet_id, overtime_hrs"
    sql = f"SELECT {cols} FROM staff{w} ORDER BY overtime_hrs DESC LIMIT ?"; p += [n]; d = _run(sql, p)
    if not len(d): return _none("highest_overtime", "staff", sql, p, scope)
    return _out("highest_overtime", f"The highest overtime is **{d.overtime_hrs[0]:.1f} hours** (staff {d.staff_id[0]}, {d.outlet_id[0]}). Top {len(d)} below." + (" Names are hidden for your role." if mask else ""), d, sql, p)

def pending_po(q, scope, mask):
    w, p = _scope(scope, pre=" AND "); n = _n(q, 10); sql = f"SELECT po_id, outlet_id, sku_name, qty, eta_ts, status FROM purchase_orders WHERE LOWER(status) NOT IN ('delivered','received','closed'){w} ORDER BY eta_ts LIMIT ?"; p += [n]; d = _run(sql, p)
    tot = int(_run(f"SELECT COUNT(*) n FROM purchase_orders WHERE LOWER(status) NOT IN ('delivered','received','closed'){w}", [scope] if scope else []).n[0])
    if not tot: return _none("pending_purchase_orders", "pending purchase orders", sql, p, scope)
    return _out("pending_purchase_orders", f"**{tot:,}** purchase orders are still on their way{' for ' + scope if scope else ''}. The {len(d)} arriving first are below.", d, sql, p)

def sales_week(q, scope, mask):
    w, p = _scope(scope, pre=" AND "); sql = f"SELECT SUBSTR(ts,1,10) AS day, ROUND(SUM(amount)) AS sales FROM sales WHERE ts>=date((SELECT MAX(ts) FROM sales),'-6 day'){w} GROUP BY day ORDER BY day DESC"; d = _run(sql, p)
    if not len(d): return _none("sales_last_7_days", "sales", sql, p, scope)
    return _out("sales_last_7_days", f"Sales over the last 7 days of data: **{_money(d.sales.sum())}**{' at ' + scope if scope else ''}. Best day {d.loc[d.sales.idxmax(), 'day']} ({_money(d.sales.max())}).", d, sql, p)

# (intent id, example question, regex on the NORMALIZED question, handler). First match wins, so the specific patterns come before the general ones.
INTENTS = [
    ("lowest_csat",             "Which outlet has the lowest customer satisfaction?",   r"lowest.*satisfaction|satisfaction.*lowest|worst.*satisfaction",             lambda q, s, m: csat(q, s, m, True)),
    ("highest_csat",            "Which outlet has the best customer satisfaction?",     r"(highest|best).*satisfaction|satisfaction.*(highest|best)",                 lambda q, s, m: csat(q, s, m, False)),
    ("lowest_revenue",          "Which outlet has the lowest revenue?",                 r"(lowest|worst).*revenue",                                                   lambda q, s, m: revenue(q, s, m, True)),
    ("highest_revenue",         "Which outlet has the highest revenue?",                r"(highest|best).*revenue|revenue.*(highest|best)",                           lambda q, s, m: revenue(q, s, m, False)),
    ("highest_overtime",        "Who works the most overtime?",                         r"overtime",                                                                   overtime),
    ("highest_staff_count",     "Which outlet has the most staff?",                     r"(highest|most).*(staff|outlet).*(staff|headcount)|(highest|most) staff|staff.*(highest|most)", most_staff),
    ("staff_count",             "How many staff does OUT-010 have?",                    r"how many staff|staff count|number of staff|count.*staff",                   staff_count),
    ("high_attrition",          "Which employees have a high attrition risk?",          r"attrition|likely (to )?(leave|quit)|leave soon|churn|resign",                       attrition),
    ("reorder_count",           "How many inventory items need reordering?",            r"how many.*reorder|number.*reorder|count.*reorder|reorder.*how many",         reorder_count),
    ("below_reorder",           "List inventory items below reorder level",             r"reorder|below.*level|inventory.*(low|short)|(low|short).*inventory",                                below_reorder),
    ("stockout_risk",           "Which inventory items have the highest stockout risk?", r"stockout|inventory out|run out|shortage",                                       stockout),
    ("worst_campaign_roi",      "Which campaign has the worst ROI?",                    r"(lowest|worst).*(roi|return)",                                               lambda q, s, m: roi(q, s, m, True)),
    ("average_campaign_roi",    "What is the average campaign ROI?",                    r"average.*(roi|return)|(roi|return).*average",                                avg_roi),
    ("best_campaign_roi",       "Which campaign has the best ROI?",                     r"(highest|best).*(roi|return)|(roi|return).*(highest|best)|campaign.*roi",   lambda q, s, m: roi(q, s, m, False)),
    ("most_negative_reviews",   "Which outlet has the most negative reviews?",          r"(highest|most).*negative|negative.*(highest|most)",                          most_negative),
    ("negative_reviews",        "Show negative reviews this week",                      r"negative|bad review|poor review|unhappy|low rating",                         negative_reviews),
    ("average_rating",          "What is the average customer rating?",                 r"average.*rating|rating.*average",                                            avg_rating),
    ("failed_audits",           "Show failed audits",                                   r"fail.*audit|audit.*fail",                                                    failed_audits),
    ("highest_violations",      "Which outlet has the most audit violations?",          r"violation",                                                                  violations),
    ("highest_operating_cost",  "Which outlet has the highest operating costs?",        r"(highest|most).*(operating|running)? ?cost|cost.*(highest|most)|operating cost", op_cost),
    ("open_critical_tickets",   "Show critical open tickets",                           r"ticket",                                                                     tickets),
    ("pending_purchase_orders", "Which purchase orders are still pending?",             r"(pending|open|outstanding|undelivered|transit).*order|order.*(pending|outstanding)", pending_po),
    ("sales_last_7_days",       "What were the total sales in the last 7 days?",        r"(total|how much).*sales|sales.*(week|7 day|last)",                           sales_week),
    ("total_outlets",           "How many outlets are there?",                          r"how many outlet|number of outlet|count.*outlet|total outlet",                total_outlets),
]
EXAMPLES = [e for _, e, _, _ in INTENTS]
RANKING = {"lowest_csat", "highest_csat", "lowest_revenue", "highest_revenue", "highest_staff_count", "most_negative_reviews", "highest_violations", "highest_operating_cost", "best_campaign_roi", "worst_campaign_roi"}

def answer(q, scope=None, mask=False):
    """-> dict(intent, answer, df, sql, rows) for a recognised question, else None."""
    nq = normalize(q)
    for iid, _ex, rx, fn in INTENTS:
        if re.search(rx, nq):
            r = fn(nq, scope, mask)
            if scope and r and iid in RANKING and r.get("df") is not None:       # one visible outlet is not a ranking: say so instead of calling it "the highest"
                r["answer"] = f"Rankings compare outlets across the chain, so for your role this shows {scope} only. " + r["answer"]
            return r
    return None

def suggestions(scope=None, seed=""):
    """Three valid questions for the polite fallback (own-outlet wording for a scoped role)."""
    import random
    pool = ["Which outlet has the lowest customer satisfaction?", "How many inventory items need reordering?", "Which campaign has the best ROI?", "Show negative reviews this week", "Show failed audits",
            "Which employees have a high attrition risk?", "What is the average customer rating?", "Show critical open tickets"]
    if scope: pool = ["How many staff does " + scope + " have?", "Show inventory items below reorder level", "Show negative reviews this week", "Show failed audits", "What is the average customer rating?", "Show critical open tickets"]
    return random.Random(seed).sample(pool, 3)


Writing /content/app/intents.py


In [16]:
%%writefile /content/app/live_engine.py
"""Shared live agent runtime (FranchiseOps AI + FreightQuote AI).
Domain modules (domain_franchise.py / domain_freight.py) plug in agents, a simulator and action handlers."""
import os, re, json, time, sqlite3, threading, datetime, smtplib, ssl
from dataclasses import dataclass, field
from email.message import EmailMessage
import requests
from config import DB_PATH

_DB_LOCK = threading.RLock()
_TL = threading.local()
def conn():
    """One SQLite connection per thread: nested helpers (emit/set_state) share the caller's transaction instead of deadlocking."""
    c = getattr(_TL, "c", None)
    if c is None:
        c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("PRAGMA busy_timeout=30000;"); _TL.c = c
    return c
def now(): return datetime.datetime.now().isoformat(timespec="seconds")

@dataclass
class AgentResult:
    agent: str
    score: float = 100.0
    risk: str = "Low"
    summary: str = ""
    findings: dict = field(default_factory=dict)
    actions: list = field(default_factory=list)   # {"type","target","detail","severity","owner"?,"sla_h"?}
    confidence: float = 0.8

def risk_label(score): return "High" if score < 60 else "Medium" if score < 80 else "Low"

def init_live_tables():
    with _DB_LOCK, conn() as c:
        c.executescript("""
        CREATE TABLE IF NOT EXISTS agent_runs(run_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, agent TEXT, risk TEXT, score REAL, latency_ms INT, summary TEXT, output TEXT);
        CREATE TABLE IF NOT EXISTS agent_actions(action_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, type TEXT, target TEXT, detail TEXT, severity TEXT, owner TEXT, status TEXT DEFAULT 'open', due TEXT, resolved_ts TEXT, source_cycle TEXT, note TEXT);
        CREATE TABLE IF NOT EXISTS live_events(event_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, etype TEXT, entity TEXT, payload TEXT);
        CREATE TABLE IF NOT EXISTS live_state(k TEXT PRIMARY KEY, v TEXT, ts TEXT);
        CREATE TABLE IF NOT EXISTS decisions(decision_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, cycle_id TEXT, trigger_src TEXT, score REAL, risk TEXT, narrative TEXT, evidence TEXT);
        CREATE TABLE IF NOT EXISTS audit_log(audit_id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, event TEXT, target TEXT, detail TEXT);
        CREATE TABLE IF NOT EXISTS ticket_comments(comment_id INTEGER PRIMARY KEY AUTOINCREMENT, action_id INT, ts TEXT, author TEXT, body TEXT);
        CREATE INDEX IF NOT EXISTS ix_act_status ON agent_actions(status, type, target);
        CREATE INDEX IF NOT EXISTS ix_runs_agent ON agent_runs(agent, run_id);
        CREATE INDEX IF NOT EXISTS ix_runs_cycle ON agent_runs(cycle_id);
        CREATE INDEX IF NOT EXISTS ix_cm_act ON ticket_comments(action_id);
        CREATE INDEX IF NOT EXISTS ix_audit_ts ON audit_log(audit_id);
        """)
        for col in ("assignee TEXT", "approval TEXT"):
            try: c.execute(f"ALTER TABLE agent_actions ADD COLUMN {col}")
            except sqlite3.OperationalError: pass
        for t, cols in (("sales", "outlet_id, ts"), ("attendance", "outlet_id, ts"), ("feedback", "outlet_id, date")):
            try: c.execute(f"CREATE INDEX IF NOT EXISTS ix_{t}_o ON {t}({cols})")
            except sqlite3.OperationalError: pass

def audit(actor, event, target="", detail=""):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO audit_log(ts,actor,event,target,detail) VALUES(?,?,?,?,?)", (now(), str(actor), event, str(target), str(detail)[:500]))

RETAIN = {"sales": ("ts", 6000), "attendance": ("ts", 3000), "feedback": ("date", 2000), "agent_runs": ("ts", 4000), "decisions": ("ts", 800), "live_events": ("ts", 2000), "audit_log": ("ts", 20000)}
def maintenance():
    """Keep the live DB bounded: prune to the newest N rows per table (never touches seed/master data)."""
    n = 0
    with _DB_LOCK, conn() as c:
        for t, (col, keep) in RETAIN.items():
            try:
                pk = c.execute(f"PRAGMA table_info({t})").fetchall()
                if not pk: continue
                if t in ("sales", "attendance"):
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 4: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep * 2,)).rowcount
                else:
                    cnt = c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                    if cnt > keep * 2: n += c.execute(f"DELETE FROM {t} WHERE rowid IN (SELECT rowid FROM {t} ORDER BY rowid ASC LIMIT ?)", (cnt - keep,)).rowcount
            except sqlite3.OperationalError: pass
    return n

def set_state(k, v):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT OR REPLACE INTO live_state(k,v,ts) VALUES(?,?,?)", (k, json.dumps(v, default=str), now()))
def get_state(k, default=None):
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT v FROM live_state WHERE k=?", (k,)).fetchone()
    return json.loads(r[0]) if r else default
def emit(etype, entity, payload=None):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO live_events(ts,etype,entity,payload) VALUES(?,?,?,?)", (now(), etype, str(entity), json.dumps(payload or {}, default=str)))

# ---------------- notifications (real channels, all optional via env) ----------------
_SENT = {}
def _nk(s): return re.sub(r"\d+(?:\.\d+)?", "#", str(s))
def notify(dom, target, severity, category, message):
    with _DB_LOCK, conn() as c:
        c.execute(f"INSERT INTO alerts ({dom.ENTITY_COL}, severity, category, message, date) VALUES (?,?,?,?,?)",
                  (target, severity, category, message, datetime.datetime.now().strftime("%Y-%m-%d %H:%M")))
    if severity not in ("Critical", "High"): return
    key = (target, category, _nk(message))
    if time.time() - _SENT.get(key, 0) < 3600: return          # no alert spam
    _SENT[key] = time.time()
    text = f"[{dom.NAME}] {severity} · {target} · {category}: {message}"
    for var in ("SLACK_WEBHOOK_URL", "TEAMS_WEBHOOK_URL"):
        if os.environ.get(var):
            try: requests.post(os.environ[var], json={"text": text}, timeout=5)
            except Exception: pass
    em, pw, to = os.environ.get("EMAIL_ADDRESS") or os.environ.get("EMAIL_ID"), (os.environ.get("EMAIL_APP_PASSWORD") or os.environ.get("EMAIL_PASSWORD") or "").replace(" ", ""), os.environ.get("ALERT_EMAIL_TO")
    if em and pw and to:
        try:
            m = EmailMessage(); m["Subject"] = text[:90]; m["From"] = em; m["To"] = to; m.set_content(text)
            with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=15) as s:
                s.login(em, pw); s.send_message(m)
        except Exception: pass
    def _mail_subscribers():
        try:
            import mailer
            if not mailer.configured(): return
            with conn() as cc: rows = [r[0] for r in cc.execute("SELECT email FROM user_prefs WHERE email_alerts=1")]
            for e in rows: mailer.send(e, text[:90], text)
        except Exception: pass
    threading.Thread(target=_mail_subscribers, daemon=True).start()
    sid, tok, frm, tph = (os.environ.get(k) for k in ("TWILIO_SID", "TWILIO_TOKEN", "TWILIO_FROM", "ALERT_SMS_TO"))
    if sid and tok and frm and tph and getattr(dom, "SMS", False):
        try: requests.post(f"https://api.twilio.com/2010-04-01/Accounts/{sid}/Messages.json", auth=(sid, tok),
                           data={"From": frm, "To": tph, "Body": text[:300]}, timeout=8)
        except Exception: pass

# ---------------- action layer ----------------
SLA_H = {"Critical": 24, "Warning": 72, "Info": 168}
def policy(dom):
    """Autonomy policy per action type: 'auto' (agent executes) or 'approval' (human approves first). Admin-editable."""
    return {}      # no approval queue in this version: agents act on their own
def set_policy(t, mode, actor="admin"):
    p = get_state("policy", {}) or {}; p[t] = mode; set_state("policy", p); audit(actor, "policy", t, mode)
# ---------------------------------------------------------------- multi-level approval chains
# Each ticket that needs a human gets an ordered chain of steps (table approval_steps). Every step names a role; the next step opens only after the previous one is approved.
# Four-eyes rule: nobody can approve two steps of the same ticket, and nobody can approve a request they raised. Any step rejection rejects the whole ticket.
LADDER = ["Operations Manager", "Franchise Owner", "Admin"]
def _role_ok(role, need): return role == "Admin" or role == need or (need == "Operations Manager" and role == "Franchise Owner")
def chain_for(severity, typ, levels=None):
    cfg = get_state("approval_chain", {}) or {}
    if levels: return LADDER[:1] + (["Admin"] if int(levels) >= 2 else [])
    if typ in cfg and cfg[typ]: return list(cfg[typ])
    return ["Operations Manager", "Admin"] if severity == "Critical" else ["Operations Manager"]
def _steps_ddl(c): c.execute("CREATE TABLE IF NOT EXISTS approval_steps(action_id INT, step INT, role TEXT, status TEXT DEFAULT 'pending', by TEXT, ts TEXT, note TEXT, PRIMARY KEY(action_id, step))")
def _ensure_chain(c, aid, severity, typ):
    _steps_ddl(c)
    if not c.execute("SELECT 1 FROM approval_steps WHERE action_id=?", (aid,)).fetchone():
        for i, r in enumerate(chain_for(severity, typ)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
def chains(ids):
    """{action_id: [{step, role, status, by}]} for the ticket list (creates the default chain for tickets that have none yet)."""
    out = {}
    if not ids: return out
    with _DB_LOCK, conn() as c:
        _steps_ddl(c)
        for aid, sev, typ in c.execute("SELECT action_id, severity, type FROM agent_actions WHERE action_id IN (%s) AND (status='proposed' OR type='quote_approval')" % ",".join("?" * len(ids)), list(ids)).fetchall(): _ensure_chain(c, aid, sev, typ)
        for r in c.execute("SELECT action_id, step, role, status, by FROM approval_steps WHERE action_id IN (%s) ORDER BY action_id, step" % ",".join("?" * len(ids)), list(ids)).fetchall(): out.setdefault(r[0], []).append({"step": r[1], "role": r[2], "status": r[3], "by": r[4] or ""})
    return out
def next_role(chain):
    for s in chain or []:
        if s["status"] == "pending": return s["role"]
    return None
def advance(aid, actor, role, approve=True, note=""):
    """Record one decision. Returns ('final'|'partial'|'rejected'|'denied'|'missing', message). Does not execute anything."""
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT severity, type, note FROM agent_actions WHERE action_id=? AND (status='proposed' OR type='quote_approval')", (aid,)).fetchone()
        if not r: return "missing", "nothing to approve on this ticket"
        _ensure_chain(c, aid, r[0], r[1]); steps = c.execute("SELECT step, role, status, by FROM approval_steps WHERE action_id=? ORDER BY step", (aid,)).fetchall()
        cur = next((s for s in steps if s[2] == "pending"), None)
        if not cur: return "missing", "this ticket has no pending approval step"
        if not _role_ok(role, cur[1]): return "denied", f"step {cur[0]} needs a {cur[1]} (you are {role})"
        if approve:
            if actor in [s[3] for s in steps if s[3]]: return "denied", "four-eyes rule: you already approved an earlier step of this ticket"
            if f"requested by {actor}" in (r[2] or ""): return "denied", "four-eyes rule: you cannot approve your own request"
        c.execute("UPDATE approval_steps SET status=?, by=?, ts=?, note=? WHERE action_id=? AND step=?", ("approved" if approve else "rejected", actor, now(), note, aid, cur[0]))
        left = [s for s in steps if s[2] == "pending" and s[0] != cur[0]]
        if not approve: return "rejected", f"rejected at step {cur[0]} ({cur[1]})"
        if left:
            c.execute("UPDATE agent_actions SET note=COALESCE(note,'')||? WHERE action_id=?", (f" step {cur[0]}/{len(steps)} approved by {actor};", aid)); audit(actor, "approve_step", aid, f"step {cur[0]}/{len(steps)} {note}".strip())
            return "partial", f"step {cur[0]} of {len(steps)} approved; waiting for {left[0][1]}"
    return "final", "all steps approved"
def request_approval(dom, typ, target, detail, severity, requester, levels=None):
    """Anyone with access can ask for a decision: creates a proposed ticket with its own approval chain."""
    a = {"type": "manual_request", "target": target, "detail": detail, "request": typ}
    with _DB_LOCK, conn() as c:
        cur = c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,status,note,approval) VALUES(?,?,?,?,?,?,'proposed',?,?)", (now(), "manual_request", target, f"{typ}: {detail}", severity, requester, f"requested by {requester};", json.dumps(a)))
        aid = cur.lastrowid; _steps_ddl(c)
        for i, r in enumerate(chain_for(severity, "manual_request", levels)): c.execute("INSERT INTO approval_steps(action_id,step,role) VALUES(?,?,?)", (aid, i + 1, r))
    audit(requester, "request_approval", f"{typ}:{target}", detail[:120]); return aid
def approve_action(dom, aid, actor="user", approve=True, note="", role="Admin"):
    """Human-in-the-loop: runs the approval chain; the action executes only when the last step is approved. Returns (state, message)."""
    st, msg = advance(aid, actor, role, approve, note)
    if st in ("denied", "missing", "partial"): return st, msg
    with _DB_LOCK, conn() as c:
        r = c.execute("SELECT type,target,detail,severity,approval FROM agent_actions WHERE action_id=?", (aid,)).fetchone()
        if st == "final":
            a = json.loads(r[4] or "{}") or {"type": r[0], "target": r[1], "detail": r[2]}
            try: res = dom.apply(a, c) or ""
            except Exception as ex: res = f"apply failed: {ex}"
            if r[0] == "manual_request": c.execute("UPDATE agent_actions SET status='resolved', resolved_ts=?, note=COALESCE(note,'')||?, approval='approved' WHERE action_id=?", (now(), f" approved by {actor}; request granted", aid))
            else: c.execute("UPDATE agent_actions SET status='open', note=COALESCE(note,'')||?, approval=? WHERE action_id=?", (f" approved by {actor}; {res}", "approved", aid))
        else:
            c.execute("UPDATE agent_actions SET status='rejected', resolved_ts=?, note=COALESCE(note,'')||?, approval='rejected' WHERE action_id=?", (now(), f" rejected by {actor} {note}", aid))
    audit(actor, "approve" if approve else "reject", f"{r[0]}:{r[1]}", note); return st, msg
def assign_action(aid, assignee, actor="user"):
    with _DB_LOCK, conn() as c: c.execute("UPDATE agent_actions SET assignee=? WHERE action_id=?", (assignee, aid))
    audit(actor, "assign", aid, assignee)
def comment_action(aid, author, body):
    with _DB_LOCK, conn() as c: c.execute("INSERT INTO ticket_comments(action_id,ts,author,body) VALUES(?,?,?,?)", (now(), author, body) if False else (aid, now(), author, body))
    audit(author, "comment", aid, body[:80])
def act(dom, decision, cid, results):
    raised = set(); sent = 0; extra = 0; pol = policy(dom)
    for a in decision.get("actions", []):
        key = (a["type"], str(a["target"]), _nk(a["detail"])); raised.add(key)
        sev = a.get("severity", "Warning")
        due = (datetime.datetime.now() + datetime.timedelta(hours=a.get("sla_h", SLA_H.get(sev, 72)))).isoformat(timespec="seconds")
        with _DB_LOCK, conn() as c:
            hit = next((r[0] for r in c.execute("SELECT action_id, detail FROM agent_actions WHERE type=? AND target=? AND status IN ('open','proposed')", key[:2]).fetchall() if _nk(r[1].replace("OVERDUE: ", "")) == key[2]), None)
            if hit:
                c.execute("UPDATE agent_actions SET detail=CASE WHEN detail LIKE 'OVERDUE:%' THEN 'OVERDUE: '||? ELSE ? END WHERE action_id=?", (a["detail"], a["detail"], hit)); continue
            note = ""; status = "open"; appr = None
            if pol.get(a["type"]) == "approval": status = "proposed"; note = "awaiting human approval"; appr = json.dumps(a, default=str)
            else:
                try: note = dom.apply(a, c) or ""
                except Exception as ex: note = f"apply failed: {ex}"
            c.execute("INSERT INTO agent_actions(ts,type,target,detail,severity,owner,due,source_cycle,note,status,approval) VALUES(?,?,?,?,?,?,?,?,?,?,?)",
                      (now(), a["type"], str(a["target"]), a["detail"], sev, a.get("owner") or dom.OWNER.get(a["type"], "Operations"), due, cid, note, status, appr))
        if sev == "Critical":
            if sent < 8: notify(dom, a["target"], "Critical", a["type"], a["detail"]); sent += 1
            else: extra += 1
    if extra: notify(dom, "MULTIPLE", "Critical", "Summary", f"{extra} more critical actions raised this cycle; see Command Center")
    with _DB_LOCK, conn() as c:      # auto-close: the owning agent re-ran and no longer raises it
        for aid, t, tg, d in c.execute("SELECT action_id,type,target,detail FROM agent_actions WHERE status IN ('open','proposed')").fetchall():
            if dom.ACTION_AGENT.get(t) in results and (t, tg, _nk(d.replace("OVERDUE: ", ""))) not in raised:
                c.execute("UPDATE agent_actions SET status='auto-closed', resolved_ts=? WHERE action_id=?", (now(), aid))

def escalate_overdue(dom):
    n = 0
    with _DB_LOCK, conn() as c:
        rows = c.execute("SELECT action_id,target,detail,type FROM agent_actions WHERE status='open' AND due<? AND detail NOT LIKE 'OVERDUE:%'", (now(),)).fetchall()
        for aid, tg, d, t in rows:
            nd = (datetime.datetime.now() + datetime.timedelta(hours=24)).isoformat(timespec="seconds")
            c.execute("UPDATE agent_actions SET severity='Critical', owner=?, detail=?, due=? WHERE action_id=?", (dom.ESCALATE_TO, "OVERDUE: " + d, nd, aid)); n += 1
    for aid, tg, d, t in rows: notify(dom, tg, "Critical", "Escalation", "OVERDUE: " + d)
    return n

def resolve_action(aid, status="resolved", note="", actor="user"):
    audit(actor, status, aid, note)
    with _DB_LOCK, conn() as c:
        c.execute("UPDATE agent_actions SET status=?, resolved_ts=?, note=COALESCE(note,'')||? WHERE action_id=?", (status, now(), (" " + note) if note else "", aid))

# ---------------- orchestration (LangGraph when installed, plain loop otherwise) ----------------
def _log_run(cid, trig, res, ms):
    with _DB_LOCK, conn() as c:
        c.execute("INSERT INTO agent_runs(ts,cycle_id,trigger_src,agent,risk,score,latency_ms,summary,output) VALUES(?,?,?,?,?,?,?,?,?)",
                  (now(), cid, trig, res.agent, res.risk, res.score, ms, res.summary, json.dumps(res.findings, default=str)[:30000]))

def make_nodes(dom):
    def planner(s): return {"queue": dom.plan(s["request"]), "results": {}}
    def agent(s):
        name = s["queue"][0]; t = time.time()
        try: res = dom.AGENTS[name](s)
        except Exception as e: res = AgentResult(name, 50.0, "Medium", f"agent error: {e}")
        _log_run(s["cycle_id"], s["trigger"], res, int((time.time() - t) * 1000))
        new = [n for n in dom.REPLAN.get(name, lambda r: [])(res) if n not in s["results"] and n not in s["queue"]]
        if new: _log_run(s["cycle_id"], s["trigger"], AgentResult("re-plan", 100, "Low", f"{name} risk {res.risk}: adding {', '.join(new)}"), 0)
        return {"results": {**s["results"], name: res}, "queue": s["queue"][1:] + new}
    def engine(s):
        d = dom.engine(s)
        with _DB_LOCK, conn() as c:
            c.execute("INSERT INTO decisions(ts,cycle_id,trigger_src,score,risk,narrative,evidence) VALUES(?,?,?,?,?,?,?)",
                      (now(), s["cycle_id"], s["trigger"], d["score"], d["risk"], d["narrative"], json.dumps({k: v for k, v in d.items() if k not in ("narrative", "actions")}, default=str)))
        return {"decision": d}
    def act_node(s): act(dom, s["decision"], s["cycle_id"], s["results"]); return {}
    return planner, agent, engine, act_node

def _compile(dom):
    from typing import TypedDict
    from langgraph.graph import StateGraph, END
    class S(TypedDict, total=False):
        cycle_id: str; trigger: str; request: str; queue: list; results: dict; decision: dict
    planner, agent, engine, act_node = make_nodes(dom)
    g = StateGraph(S)
    for n, f in (("planner", planner), ("agent", agent), ("engine", engine), ("act", act_node)): g.add_node(n, f)
    g.set_entry_point("planner"); g.add_edge("planner", "agent")
    g.add_conditional_edges("agent", lambda s: "agent" if s["queue"] else "engine")
    g.add_edge("engine", "act"); g.add_edge("act", END)
    return g.compile()

_GRAPHS = {}
def run_cycle(dom, trigger="manual", request=""):
    init_live_tables()
    cid = datetime.datetime.now().strftime("%m%d-%H%M%S") + "-" + trigger[:4]
    s = {"cycle_id": cid, "trigger": trigger, "request": request, "queue": [], "results": {}, "decision": {}}
    try:
        if dom.NAME not in _GRAPHS: _GRAPHS[dom.NAME] = _compile(dom)
        out = _GRAPHS[dom.NAME].invoke(s)
    except ImportError:
        planner, agent, engine, act_node = make_nodes(dom)
        s.update(planner(s))
        while s["queue"]: s.update(agent(s))
        s.update(engine(s)); act_node(s); out = s
    return out["decision"]

# ---------------- background runtime ----------------
class Runtime:
    def __init__(self, dom, tick=5, cycle=60, min_gap=12):
        self.dom, self.tick, self.cycle, self.min_gap = dom, tick, cycle, min_gap
        self.stop = threading.Event(); self.ticks = 0; self.cycles = 0; self.last_cycle = 0; self.last_err = ""
        self.th = threading.Thread(target=self._loop, daemon=True); self.th.start()
        if hasattr(dom, "feed"): self.fth = threading.Thread(target=self._feed_loop, daemon=True); self.fth.start()
    def _feed_loop(self):          # network polling lives here so slow/offline APIs never stall the tick
        while not self.stop.is_set():
            try: self.dom.feed()
            except Exception as e: self.last_err = f"feed: {e}"
            self.stop.wait(int(os.environ.get("FEED_SEC", 60)))
    def _loop(self):
        init_live_tables(); set_state("boot", {"stage": "bootstrap", "ts": now()})
        try: self.dom.bootstrap()
        except Exception as e: self.last_err = f"bootstrap: {e}"
        set_state("boot", {"stage": "first_cycle", "ts": now()})
        while not self.stop.is_set():
            try:
                reasons = self.dom.tick() or []; self.ticks += 1
                if self.ticks % 120 == 0: maintenance()
                reasons = reasons + (self.dom.process() or [])
                due = time.time() - self.last_cycle
                if (reasons and due >= self.min_gap) or due >= self.cycle:
                    run_cycle(self.dom, "event" if reasons else "sched", ""); escalate_overdue(self.dom)
                    self.cycles += 1; self.last_cycle = time.time()
                    set_state("runtime", {"ticks": self.ticks, "cycles": self.cycles, "last_cycle": now(), "reasons": reasons[:3], "err": self.last_err})
                    if self.cycles == 1: set_state("boot", {"stage": "ready", "ts": now()})
                elif self.ticks % 6 == 0: set_state("runtime", {**(get_state("runtime", {}) or {}), "ticks": self.ticks})
            except Exception as e: self.last_err = str(e)
            self.stop.wait(self.tick)

_RT = None; _RT_LOCK = threading.Lock()
def start_runtime(dom):
    global _RT
    with _RT_LOCK:
        if _RT is None:
            _RT = Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    return _RT


Writing /content/app/live_engine.py


In [17]:
%%writefile /content/app/mailer.py
"""One place that sends e-mail (OTP codes, reports, alerts). Gmail SMTP with an app password: set EMAIL_ADDRESS + EMAIL_PASSWORD (Colab secrets or env)."""
import os, ssl, smtplib
from email.message import EmailMessage
def creds(): return os.environ.get("EMAIL_ADDRESS") or os.environ.get("EMAIL_ID") or "", (os.environ.get("EMAIL_PASSWORD") or os.environ.get("EMAIL_APP_PASSWORD") or "").replace(" ", "")
def configured(): a, b = creds(); return bool(a and b)
def send(to, subject, body, attachments=(), html=None):
    """attachments: [(filename, bytes, mimetype)]. Returns (ok, message)."""
    em, pw = creds()
    if not (em and pw): return False, "E-mail is not configured (set EMAIL_ADDRESS and EMAIL_PASSWORD)."
    try:
        m = EmailMessage(); m["Subject"] = subject; m["From"] = em; m["To"] = to; m.set_content(body)
        if html: m.add_alternative(html, subtype="html")
        for fn, data, mt in attachments: m.add_attachment(data, maintype=mt.split("/")[0], subtype=mt.split("/")[1], filename=fn)
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=ssl.create_default_context(), timeout=20) as s: s.login(em, pw); s.send_message(m)
        return True, f"Sent to {to}"
    except Exception as e: return False, f"Could not send: {e}"


Writing /content/app/mailer.py


In [18]:
%%writefile /content/app/qa.py
"""Exact-answer engine for the Copilot. Turns a plain-English question into a read-only, parameterised SQL query over the live tables
(counts, top/bottom N, averages, totals, group-by, filters, ID lookups) and returns the real rows. The LLM only explains afterwards;
it never has to guess a number. Table and column names come from a fixed whitelist, values are always bound parameters."""
import re
from blocks import Q

TABLES = {
    "Franchise": {
        "outlets": (["outlet", "store", "branch", "franchise"], "outlet_id", ["outlet_id", "outlet_name", "location", "tier", "revenue", "customer_satisfaction", "staff_headcount"], "revenue"),
        "staff": (["staff", "employee", "worker", "workforce", "people"], "staff_id", ["staff_id", "name", "role", "outlet_id", "salary", "overtime_hrs", "predicted_attrition_prob"], "predicted_attrition_prob"),
        "inventory": (["inventory", "sku", "item", "product", "stock level"], "record_id", ["outlet_id", "sku_name", "category", "current_stock", "reorder_threshold", "stockout_risk_prob"], "stockout_risk_prob"),
        "marketing": (["campaign", "marketing", "promotion", "advert"], "campaign_id", ["campaign_id", "campaign_name", "outlet_id", "channel", "budget", "actual_roi", "conversions"], "actual_roi"),
        "feedback": (["feedback", "review", "complaint", "comment"], "feedback_id", ["feedback_id", "outlet_id", "rating", "sentiment_score", "date", "comment"], "rating"),
        "audits": (["audit", "inspection", "compliance"], "audit_id", ["audit_id", "outlet_id", "audit_date", "score", "violations", "category", "status"], "score"),
        "sales": (["sale", "transaction", "takings"], "sale_id", ["outlet_id", "ts", "amount", "tx_count", "source"], "amount"),
        "purchase_orders": (["purchase order", "reorder order", "po"], "po_id", ["po_id", "outlet_id", "sku_name", "qty", "eta_ts", "status"], "qty"),
        "attendance": (["attendance", "absent", "absence", "present"], "att_id", ["staff_id", "outlet_id", "ts", "status"], None),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
    "Freight": {
        "shipments": (["shipment", "cargo", "container", "consignment"], "shipment_id", ["shipment_id", "origin_port", "dest_port", "carrier", "status", "eta", "weight_kg", "predicted_delay_risk"], "predicted_delay_risk"),
        "freight_quotes": (["quote", "quotation", "price", "pricing"], "quote_id", ["quote_id", "shipment_id", "customer_id", "carrier", "final_price", "margin_pct", "status"], "final_price"),
        "ports": (["port", "terminal", "harbour", "harbor"], "port_id", ["port_name", "country", "region", "congestion_index", "avg_dwell_days"], "congestion_index"),
        "carriers": (["carrier", "shipping line", "liner", "line"], "carrier_id", ["name", "rating", "on_time_pct", "avg_cost_index", "risk_level"], "on_time_pct"),
        "customers": (["customer", "client", "shipper"], "customer_id", ["customer_id", "name", "industry", "priority_tier", "credit_risk"], "credit_risk"),
        "weather_risks": (["weather", "storm", "wave", "wind", "forecast"], "port_name", ["port_name", "current_severity", "forecast", "wind_speed", "wave_height"], "wave_height"),
        "vessels": (["vessel", "ship ", "ais", "voyage"], "shipment_id", ["shipment_id", "route_name", "lat", "lon", "progress", "speed_kn"], "progress"),
        "quote_requests": (["request", "enquiry", "enquiries", "inquiry"], "req_id", ["req_id", "ts", "origin", "dest", "cargo", "weight_t", "status", "quote_id"], None),
        "customs_rules": (["customs", "duty", "hs code", "tariff"], "chapter", ["chapter", "goods", "duty_pct", "docs", "restricted"], "duty_pct"),
        "agent_actions": (["ticket", "action", "task", "approval", "alert"], "action_id", ["action_id", "type", "target", "severity", "status", "assignee", "detail"], None),
    },
}
CSYN = {"revenue": ["revenue", "turnover"], "customer_satisfaction": ["satisfaction", "csat"], "staff_headcount": ["headcount"], "salary": ["salary", "pay", "wage"], "overtime_hrs": ["overtime"],
        "job_satisfaction": ["job satisfaction"], "tenure_years": ["tenure"], "predicted_attrition_prob": ["attrition", "churn", "likely to leave", "quit"], "current_stock": ["current stock", "units in stock"],
        "reorder_threshold": ["reorder level", "threshold"], "weekly_demand": ["demand"], "lead_time_days": ["lead time"], "stockout_risk_prob": ["stockout", "stock-out", "stock out", "shortage", "run out"],
        "actual_roi": ["roi", "return on"], "sentiment_score": ["sentiment"], "score": ["audit score", "score"], "violations": ["violation"], "amount": ["amount", "sales value"], "qty": ["quantity", "qty"],
        "congestion_index": ["congestion", "congested", "crowded"], "avg_dwell_days": ["dwell"], "on_time_pct": ["on-time", "on time", "punctual", "punctuality", "reliab"], "avg_cost_index": ["cost index"],
        "predicted_delay_risk": ["delay risk", "delay", "delayed"], "weight_kg": ["weight"], "final_price": ["final price", "price", "quote value", "worth"], "margin_pct": ["margin"], "base_cost": ["base cost"],
        "insurance": ["insurance"], "customs_fee": ["customs fee"], "fuel_surcharge": ["fuel"], "credit_risk": ["credit risk", "credit"], "duty_pct": ["duty"], "wave_height": ["wave"], "wind_speed": ["wind"],
        "speed_kn": ["speed"], "progress": ["progress", "completed"], "operating_costs": ["operating cost", "costs"], "rating": ["rating", "rated", "stars"], "budget": ["budget", "spend"], "reach": ["reach"], "conversions": ["conversion"]}
BAD = {"predicted_attrition_prob", "stockout_risk_prob", "violations", "congestion_index", "avg_dwell_days", "predicted_delay_risk", "credit_risk", "operating_costs", "overtime_hrs", "lead_time_days", "avg_cost_index", "wave_height", "wind_speed"}
PROB = {"predicted_attrition_prob", "stockout_risk_prob", "predicted_delay_risk", "credit_risk", "progress", "sentiment_score"}
LABEL = {"outlets": "outlet_name", "staff": "name", "carriers": "name", "customers": "name", "ports": "port_name", "inventory": "sku_name", "marketing": "campaign_name"}
GENERIC = {"port", "line", "terminal", "cargo", "ocean", "shipping", "the", "of"}

def _w(q, term): return re.search(r"(?<![a-z0-9])" + re.escape(term.strip()) + r"(?:s|es)?(?![a-z0-9])", q) is not None
def _num(s, unit):
    v = float(s.replace(",", "")); u = (unit or "").lower()
    return v * {"k": 1e3, "m": 1e6, "lakh": 1e5, "lakhs": 1e5, "cr": 1e7, "crore": 1e7, "crores": 1e7}.get(u, 1)
_NUM = {}
def _cols(t):
    df = Q(f"SELECT * FROM {t} LIMIT 20"); _NUM[t] = {c for c in df.columns if str(df[c].dtype)[:3] in ("flo", "int") }
    return list(df.columns)
PL = {"inventory": "inventory records", "staff": "staff", "marketing": "campaigns", "feedback": "feedback entries", "audits": "audits", "sales": "sales records", "attendance": "attendance records", "freight_quotes": "quotes", "ports": "ports", "carriers": "carriers", "customs_rules": "customs rules", "weather_risks": "port weather entries", "agent_actions": "tickets", "purchase_orders": "purchase orders", "quote_requests": "quote requests", "outlets": "outlets", "shipments": "shipments", "customers": "customers", "vessels": "vessels"}
def pct(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns and c in PROB: df[c] = df[c].map(lambda x: f"{x:.0%}" if isinstance(x, (int, float)) and abs(x) <= 1 else x)
    return df
def _alias(v):
    v = str(v); out = {v.lower()}; base = re.sub(r"\(.*?\)", "", v).strip().lower(); out.add(base)
    out.update(m.lower() for m in re.findall(r"\((.*?)\)", v))
    core = " ".join(w for w in base.split() if w not in GENERIC)
    if len(core) >= 3: out.add(core)
    return {a for a in out if len(a) >= 3}
def _fmt(v):
    if isinstance(v, float): return f"{v:,.2f}" if abs(v) < 1000 else f"{v:,.0f}"
    if isinstance(v, int): return f"{v:,}" if abs(v) >= 10000 else str(v)
    return str(v)
def md(df):
    if df is None or not len(df): return ""
    h = "| " + " | ".join(map(str, df.columns)) + " |\n|" + "---|" * len(df.columns) + "\n"
    return h + "\n".join("| " + " | ".join(_fmt(x).replace("|", "/").replace("\n", " ")[:90] for x in r) + " |" for r in df.itertuples(index=False))

def lookup(kind, q, scope=None):
    """ID in the question -> the exact row(s) plus its open tickets."""
    ids = list(dict.fromkeys(re.findall(r"\b(OUT-\d+|STF-\d+|EMP-\d+|SHP-[A-Z0-9-]+|QTE-[A-Z0-9-]+|PORT-\d+|CAR-\d+|CUST-\d+|CAMP-\d+|AUD-\d+|FB-\d+)\b", q, re.I)))
    ids = [i.upper() for i in ids]
    if not ids: return None
    parts, sqls = [], []
    for e in ids[:4]:
        if scope and e != scope and not e.startswith("PORT") and not e.startswith("CAR"): parts.append(f"{e} is outside your outlet scope ({scope})."); continue
        for t, (_, idc, cols, _m) in TABLES[kind].items():
            if t == "agent_actions": continue
            try: df = Q(f"SELECT * FROM {t} WHERE {idc}=?", (e,))
            except Exception: continue
            if len(df):
                r = df.iloc[0]; parts.append(f"**{e}** ({t}): " + "; ".join(f"{c}={_fmt(r[c])}" for c in df.columns if str(r[c]) not in ("None", "nan"))); sqls.append(f"SELECT * FROM {t} WHERE {idc}='{e}'"); break
        tk = Q("SELECT action_id, type, severity, status, detail FROM agent_actions WHERE target=? AND status IN ('open','proposed') ORDER BY action_id DESC LIMIT 5", (e,))
        if len(tk): parts.append(f"Open tickets on {e}:\n" + md(tk)); sqls.append(f"SELECT … FROM agent_actions WHERE target='{e}' AND status IN ('open','proposed')")
    return {"answer": "\n\n".join(parts), "df": None, "sql": "; ".join(sqls)} if parts else None

def answer(kind, q, scope=None, mask=False):
    """-> dict(answer, sql, df, table) when the question maps to the data, else None."""
    ql = q.lower().strip()
    if not ql: return None
    d = lookup(kind, q, scope)
    if d: return d
    T = TABLES[kind]; sc = {}
    for t, (syn, _, _, _) in T.items(): sc[t] = sum(3 for s in syn if _w(ql, s))
    cols_of = {}
    for t in T:
        try: cols_of[t] = _cols(t)
        except Exception: cols_of[t] = []
    hits = {c: any(_w(ql, s) for s in (CSYN.get(c, []) + [c.replace("_", " ")])) for c in {c for v in cols_of.values() for c in v}}
    for t, cs in cols_of.items(): sc[t] += sum(2 for c in cs if hits.get(c) and c not in ("status", "type", "date", "name"))
    if not sc or max(sc.values()) <= 0: return None
    order = list(T); t = max(order, key=lambda x: (sc[x], -order.index(x)))
    syn, idc, show, dmetric = T[t]; cs = cols_of[t]
    # aggregation
    agg = "list"
    if re.search(r"\b(how many|number of|count)\b", ql): agg = "count"
    elif re.search(r"\b(average|avg|mean)\b", ql): agg = "avg"
    elif re.search(r"\b(total|sum|overall|combined)\b", ql): agg = "sum"
    elif re.search(r"\b(highest|most|top|best|largest|biggest|maximum|max|greatest|riskiest|worst|lowest|least|smallest|minimum|min|bottom|weakest|cheapest|fewest|poorest)\b", ql): agg = "rank"
    # metric
    mc = [c for c in cs if hits.get(c) and c in _NUM.get(t, ()) and c not in ("lat", "lon")]
    metric = mc[0] if mc else dmetric
    if metric and metric not in cs: metric = None
    # filters
    where, params, desc = [], [], []
    def val_filter(tab, col, sub=False):
        try: vals = Q(f"SELECT DISTINCT {col} v FROM {tab} WHERE {col} IS NOT NULL LIMIT 600").v.tolist()
        except Exception: return
        found = []
        for v in vals:
            if isinstance(v, (int, float)): continue
            for a in _alias(v):
                if _w(ql, a): found.append((len(a), str(v), a)); break
        found.sort(reverse=True); used, taken = [], ql
        for _, v, a in found:
            if a in taken and not any(a in u or u in a for u in used):
                used.append(a)
                if sub: where.append(f"outlet_id IN (SELECT outlet_id FROM {tab} WHERE {col}=?)")
                else:
                    where.append(f"{col}=?")
                params.append(v); desc.append(f"{col} = {v}")
    for c in cs:
        if c in ("status", "category", "channel", "role", "tier", "severity", "type", "location", "origin_port", "dest_port", "carrier", "region", "country", "risk_level", "industry", "priority_tier", "source", "sku_name", "name", "port_name", "customer_id", "forecast", "route_name", "origin", "dest", "cargo"):
            if c in ("origin_port", "dest_port", "origin", "dest"): continue
            val_filter(t, c)
    if t in ("shipments", "quote_requests"):
        oc, dc = ("origin_port", "dest_port") if t == "shipments" else ("origin", "dest")
        try: pv = Q("SELECT DISTINCT port_name v FROM ports").v.tolist()
        except Exception: pv = []
        for pn in pv:
            for a_ in sorted(_alias(pn), key=len, reverse=True):
                mm = re.search(r"(?<![a-z0-9])" + re.escape(a_) + r"(?![a-z0-9])", ql)
                if not mm: continue
                pre = ql[max(0, mm.start() - 22):mm.start()].strip()
                frm = re.search(r"(from|origin|out of|leaving|departing|ex)$", pre); to = re.search(r"(to|towards|into|destination|dest|bound for|arriving at|arriving in|reaching)$", pre)
                if t == "quote_requests": pn_ = a_.split(" (")[0]; pat = pn_
                else: pat = pn
                if frm: where.append(f"{oc}=?"); params.append(pat); desc.append(f"{oc} = {pn}")
                elif to: where.append(f"{dc}=?"); params.append(pat); desc.append(f"{dc} = {pn}")
                else: where.append(f"({oc}=? OR {dc}=?)"); params += [pat, pat]; desc.append(f"{pn} is origin or destination")
                break
    if "outlet_id" in cs and t != "outlets":
        try: val_filter("outlets", "location", sub=True); val_filter("outlets", "tier", sub=True)
        except Exception: pass
    m = re.search(r"\b(outlet|store|branch)\s*(?:no\.?|number|#)?\s*(\d{1,3})\b", ql)
    if m and "outlet_id" in cs: where.append("outlet_id=?"); params.append(f"OUT-{int(m.group(2)):03d}"); desc.append(f"outlet_id = OUT-{int(m.group(2)):03d}")
    for pat, op in ((r"(?:below|under|less than|lower than|smaller than|<)", "<"), (r"(?:above|over|more than|greater than|higher than|exceed\w*|>)", ">"), (r"(?:at least|minimum of|>=)", ">="), (r"(?:at most|no more than|<=)", "<=")):
        mm = re.search(pat + r"\s*(?:rs\.?|₹|\$|inr|usd)?\s*([\d,]*\.?\d+)\s*(%|k|m|lakhs?|cr(?:ores?)?)?", ql)
        if mm and metric:
            v = _num(mm.group(1), mm.group(2) if mm.group(2) != "%" else "");
            if metric in PROB and v > 1: v /= 100
            where.append(f"{metric}{op}?"); params.append(v); desc.append(f"{metric} {op} {v:g}"); break
    if re.search(r"\b(open|pending|unresolved)\b", ql) and "status" in cs and not any("status" in x for x in desc) and t in ("agent_actions", "purchase_orders"):
        where.append("status IN ('open','proposed')" if t == "agent_actions" else "status NOT IN ('delivered','received','closed')"); desc.append("status = open")
    if re.search(r"\b(critical|high severity|urgent)\b", ql) and "severity" in cs and not any("severity" in x for x in desc): where.append("severity IN ('critical','high')"); desc.append("severity = critical/high")
    if re.search(r"\b(delayed|late)\b", ql) and t == "shipments" and not any("status" in x for x in desc): where.append("(status LIKE '%delay%' OR predicted_delay_risk>=0.5)"); desc.append("delayed or delay risk ≥ 0.5")
    if re.search(r"\b(low stock|running low|below reorder|need reorder|reorder)\b", ql) and t == "inventory": where.append("current_stock<=reorder_threshold"); desc.append("stock ≤ reorder threshold")
    if re.search(r"\b(high risk|at risk)\b", ql) and metric in PROB and not any(metric in x for x in desc): where.append(f"{metric}>=0.5"); desc.append(f"{metric} ≥ 0.5")
    if scope and t != "agent_actions":
        if "outlet_id" in cs: where.append("outlet_id=?"); params.append(scope); desc.append(f"outlet_id = {scope}")
        else: return {"answer": "That data is not available for your outlet role.", "sql": "", "df": None}
    elif scope: return {"answer": "Ticket lists are limited to your outlet; open the Actions page.", "sql": "", "df": None}
    # an unfiltered, unaggregated, unranked question is too vague to answer exactly
    if agg == "list" and not where and not re.search(r"\b(show|list|which|what|who|give)\b", ql): return None
    W = (" WHERE " + " AND ".join(where)) if where else ""
    nm = re.search(r"\b(?:top|bottom|first|best|worst|lowest|highest)\s+(\d{1,2})\b", ql); n = int(nm.group(1)) if nm else (5 if agg == "rank" else 10)
    tdesc = PL.get(t, syn[0] + "s"); fdesc = (" where " + ", ".join(desc)) if desc else ""
    grp = re.search(r"\b(?:by|per|for each|across|grouped by|each)\s+(location|city|tier|role|category|channel|carrier|region|country|status|type|severity|industry|outlet|port|origin|destination|sku|customer)\b", ql)
    gcol = {"city": "location", "outlet": "outlet_id", "port": "port_name" if "port_name" in cs else "origin_port", "origin": "origin_port", "destination": "dest_port", "sku": "sku_name", "customer": "customer_id"}.get(grp.group(1), grp.group(1)) if grp else None
    if gcol and gcol not in cs and "outlet_id" in cs and gcol in ("location", "tier"):
        j = f" JOIN outlets o ON o.outlet_id={t}.outlet_id"; gq = f"o.{gcol}"
    elif gcol in cs: j, gq = "", gcol
    else: j, gq = "", None
    if mask and t == "staff" and (metric in ("salary", "age") or re.search(r"\b(salary|salaries|pay|wage|age)\b", ql)): return {"answer": "Salary and personal details are visible to the Admin, Franchise Owner and Store Manager roles only.", "sql": "", "df": None}     # RBAC: checked before every aggregation, not only for lists
    if gq:
        fn = {"count": "COUNT(*)", "avg": f"AVG({metric})", "sum": f"SUM({metric})"}.get(agg, f"AVG({metric})" if metric else "COUNT(*)")
        if agg == "rank" and re.search(r"\b(lowest|least|smallest|minimum|min|bottom|worst|fewest|weakest|poorest|cheapest)\b", ql) and not re.search(r"\b(worst|riskiest)\b", ql) or (re.search(r"\b(worst)\b", ql) and metric not in BAD): order_dir = "ASC"
        else: order_dir = "DESC"
        if re.search(r"\bhow many|number of|count\b", ql): fn = "COUNT(*)"
        sql = f"SELECT {gq} AS grp, {fn} AS value, COUNT(*) AS rows_n FROM {t}{j}{W} GROUP BY {gq} ORDER BY value {order_dir} LIMIT 25"
        df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        lab = {"COUNT(*)": "count"}.get(fn, fn.lower().replace("(", " of ").replace(")", ""))
        top = df.iloc[0]; df = df.rename(columns={"grp": gcol, "value": lab});
        return {"answer": f"{lab.capitalize()} of {tdesc} by {gcol}{fdesc}. Highest: **{top.iloc[0]}** ({_fmt(top.iloc[1])}); lowest: **{df.iloc[-1, 0]}** ({_fmt(df.iloc[-1, 1])}).", "sql": sql, "df": df}
    if agg == "count":
        sql = f"SELECT COUNT(*) AS n FROM {t}{W}"; n_ = int(Q(sql, tuple(params)).n[0])
        return {"answer": f"**{n_:,}** {tdesc}{fdesc}.", "sql": sql, "df": None}
    if agg in ("avg", "sum") and metric:
        fn = "AVG" if agg == "avg" else "SUM"; sql = f"SELECT {fn}({metric}) AS v, COUNT(*) AS n, MIN({metric}) AS lo, MAX({metric}) AS hi FROM {t}{W}"; r = Q(sql, tuple(params)).iloc[0]
        if not r.n: return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        pv = f"{r.v:.1%}" if metric in PROB and abs(r.v) <= 1 else _fmt(float(r.v))
        return {"answer": f"{'Average' if agg == 'avg' else 'Total'} {metric.replace('_', ' ')} of {tdesc}{fdesc}: **{pv}** across {int(r.n):,} rows (min {_fmt(float(r.lo))}, max {_fmt(float(r.hi))}).", "sql": sql, "df": None}
    if mask and t == "staff" and (metric in ("salary", "age") or re.search(r"\b(salary|salaries|pay|wage|age)\b", ql)): return {"answer": "Salary and personal details are visible to Admin, Owner and Operations roles only.", "sql": "", "df": None}
    sel = list(dict.fromkeys([c for c in show if c in cs] + ([metric] if metric else [])))
    if mask: sel = [c for c in sel if not (t == "staff" and c in ("name", "salary", "age"))]
    if t == "inventory" and "outlet_id" in cs and "sku_name" not in sel: sel.append("sku_name")
    if agg == "rank" and metric:
        low = re.search(r"\b(lowest|least|smallest|minimum|min|bottom|fewest|weakest|poorest|cheapest)\b", ql) is not None; worst = re.search(r"\b(worst|riskiest)\b", ql) is not None
        asc = (low and not worst) or (worst and metric not in BAD) or (re.search(r"\b(best)\b", ql) is not None and metric in BAD)
        sql = f"SELECT {', '.join(sel)} FROM {t}{W} ORDER BY {metric} {'ASC' if asc else 'DESC'} LIMIT {n}"; df = Q(sql, tuple(params))
        if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
        r0 = df.iloc[0]; lab = r0[LABEL[t]] if t in LABEL and LABEL[t] in df.columns else r0[idc] if idc in df.columns else r0.iloc[0]; pv = f"{r0[metric]:.1%}" if metric in PROB and abs(r0[metric]) <= 1 else _fmt(r0[metric])
        return {"answer": f"{'Lowest' if asc else 'Highest'} {metric.replace('_', ' ')} among {tdesc}{fdesc}: **{lab}** at **{pv}**. Top {len(df)} below.", "sql": sql, "df": pct(df, sel)}
    sql = f"SELECT {', '.join(sel)} FROM {t}{W}" + (f" ORDER BY {metric} {'ASC' if metric in BAD and False else 'DESC'}" if metric else "") + f" LIMIT {max(n, 10) if not nm else n}"
    tot = int(Q(f"SELECT COUNT(*) n FROM {t}{W}", tuple(params)).n[0]); df = Q(sql, tuple(params))
    if not len(df): return {"answer": f"No {tdesc}{fdesc} found.", "sql": sql, "df": None}
    return {"answer": f"**{tot:,}** {tdesc}{fdesc}; showing {len(df)}" + (f", sorted by {metric.replace('_', ' ')}" if metric else "") + ".", "sql": sql, "df": pct(df, sel)}


Writing /content/app/qa.py


In [19]:
%%writefile /content/app/rbac.py
"""Role-based access control (Step 5). One place that says who may open which page, which capability each role has and
which outlet a role is limited to. Everything here is enforced on the server; hiding a menu item in the front end is only a convenience.

The user object that user_of(request) returns must contain: email, username, role and (for Store Manager / Staff) outlet.
An unknown role gets nothing, and a scoped role without an outlet is refused: every check fails closed."""
import re
from fastapi import HTTPException

ROLES = ["Admin", "Franchise Owner", "Store Manager", "Staff", "Auditor"]

# page ids (the menu entries of server.py plus the profile and admin pages)
AGENT_PAGES = {"workforce", "outlets", "inventory", "marketing", "sentiment", "audit"}
ALL_PAGES = {"copilot", "profile", "admin"} | AGENT_PAGES
EXTRA_MENU = [("profile", "My Profile", "user"), ("admin", "Admin Dashboard", "shield")]      # appended to V.MENU; the front end draws these two pages itself

# page access matrix (Copilot, Workforce, Outlets, Inventory, Marketing, Sentiment, Audit; Profile for everyone; Users = the admin page)
ROLE_MENU = {
    "Admin":           {"copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit", "profile", "admin"},
    "Franchise Owner": {"copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit", "profile"},
    "Store Manager":   {"copilot", "workforce", "outlets", "inventory", "sentiment", "audit", "profile"},
    "Staff":           {"copilot", "profile"},
    "Auditor":         {"copilot", "workforce", "outlets", "inventory", "marketing", "sentiment", "audit", "profile"},
}

# capabilities: what a role may DO (need(ctx, capability) is called by every action route)
#   copilot  use the AI Copilot and its saved chats      view     read the agent pages and open the outlet detail drawer
#   run      run the agents                              resolve  log an inspection (Audit page)
#   review   add a customer review (Sentiment page)      users    open the admin dashboard and manage users and roles
#   profile  see and edit your own profile               quote, approve, admin: Freight-edition actions, Admin only
ALL_CAPS = {"copilot", "view", "run", "resolve", "review", "users", "profile", "quote", "approve", "admin"}
ROLE_CAPS = {
    "Admin":           set(ALL_CAPS),
    "Franchise Owner": {"copilot", "view", "run", "resolve", "review", "profile"},
    "Store Manager":   {"copilot", "view", "resolve", "review", "profile"},
    "Staff":           {"copilot", "profile"},
    "Auditor":         {"copilot", "view", "profile"},
}

SCOPED_ROLES = {"Store Manager", "Staff"}       # limited to ONE outlet (user["outlet"]); the other roles see every outlet
MASKED_ROLES = {"Staff", "Auditor"}             # the Copilot hides salary and personal details from these roles
OUTLET_RE = re.compile(r"OUT-\d+", re.I)

def role_of(user):
    role = (user or {}).get("role")
    if role not in ROLE_MENU: raise HTTPException(403, "Your account has no valid role. Ask an Admin to assign one.")
    return role

def scope_for(user):
    """None = every outlet. A scoped role returns its outlet id, or is refused when no outlet is assigned."""
    role = role_of(user)
    if role not in SCOPED_ROLES: return None
    outlet = str((user or {}).get("outlet") or "").strip().upper()
    if not OUTLET_RE.fullmatch(outlet): raise HTTPException(403, f"No outlet is assigned to this {role} account. Ask an Admin to assign one.")
    return outlet

def permission_table():
    """The page access matrix as rows ({"Role": ..., "Copilot": "✓", ...}) for the Admin dashboard and the README, so both always match ROLE_MENU / ROLE_CAPS."""
    cols = [("copilot", "Copilot"), ("workforce", "Workforce"), ("outlets", "Outlets"), ("inventory", "Inventory"), ("marketing", "Marketing"), ("sentiment", "Sentiment"), ("audit", "Audit")]
    yes = lambda b: "✓" if b else "✗"
    return [{"Role": r, **{label: yes(pid in ROLE_MENU[r]) for pid, label in cols}, "Run agents": yes("run" in ROLE_CAPS[r]), "Users": yes("users" in ROLE_CAPS[r])} for r in ROLES]

def caps_of(role): return set(ROLE_CAPS.get(role, ()))
def pages_of(role): return set(ROLE_MENU.get(role, ()))
def masked(role): return role in MASKED_ROLES

def menu_for(role, catalog):
    """The menu entries (id, label, icon) this role may open, in the catalog's order."""
    allowed = pages_of(role)
    return [m for m in catalog if m[0] in allowed]

def need(c, cap):
    """Raise 403 when the role lacks the capability."""
    if cap not in ROLE_CAPS.get(c["role"], ()): raise HTTPException(403, f"Forbidden: the {c['role']} role is not allowed to do this ({cap}).")

def need_page(c, pid):
    if pid not in ALL_PAGES: raise HTTPException(404, "unknown page")
    if pid not in ROLE_MENU.get(c["role"], ()): raise HTTPException(403, f"Forbidden: the {c['role']} role cannot open the {pid} page.")

def guard_outlets(c, *texts):
    """A scoped user may not mention another outlet in a request (page parameters, tab, detail id, question)."""
    s = c.get("scope")
    if not s: return
    for t in texts:
        for m in OUTLET_RE.findall(str(t)):
            if m.upper() != s: raise HTTPException(403, f"Forbidden: {m.upper()} is outside your outlet ({s}).")

def own_outlet(c, outlet):
    """Writes (reviews, inspections): a scoped user may only write for their own outlet."""
    s = c.get("scope")
    if s and str(outlet or "").strip().upper() != s: raise HTTPException(403, f"Forbidden: you can only act on your own outlet ({s}).")


Writing /content/app/rbac.py


In [20]:
%%writefile /content/app/scale_data.py
"""Scale both apps to production-like volume. Idempotent: only tops tables up to the target counts.
Coordinates are approximate (1-2 decimals): replace/extend through the Data Feeder with UN/LOCODE or World Port Index data."""
import random, datetime, math
from live_engine import conn, _DB_LOCK, now

PORTS = {
"China": "Dalian 38.9 121.7;Xiamen 24.5 118.1;Fuzhou 26.0 119.4;Lianyungang 34.7 119.4;Yantai 37.6 121.4;Rizhao 35.4 119.5;Nanjing 32.1 118.8;Zhanjiang 21.2 110.4;Haikou 20.0 110.3;Zhuhai 22.2 113.6;Taicang 31.5 121.2;Wuhan 30.6 114.3;Shantou 23.4 116.7;Dongguan 22.9 113.6",
"Japan": "Yokohama 35.4 139.7;Nagoya 35.1 136.9;Osaka 34.6 135.4;Kobe 34.7 135.2;Hakata 33.6 130.4;Chiba 35.6 140.1",
"South Korea": "Incheon 37.4 126.6;Gwangyang 34.9 127.7;Ulsan 35.5 129.4;Pyeongtaek 36.9 126.8",
"Taiwan": "Keelung 25.1 121.7;Taichung 24.3 120.5", "Vietnam": "Hai Phong 20.8 106.7;Da Nang 16.1 108.2;Vung Tau 10.4 107.1;Quy Nhon 13.8 109.3",
"Thailand": "Bangkok 13.7 100.6;Songkhla 7.2 100.6", "Malaysia": "Penang 5.4 100.3;Johor 1.4 103.9;Kuantan 3.97 103.4;Kuching 1.6 110.3",
"Indonesia": "Surabaya -7.2 112.7;Semarang -6.9 110.4;Belawan 3.8 98.7;Makassar -5.1 119.4;Balikpapan -1.3 116.8;Palembang -3.0 104.8;Bitung 1.4 125.2",
"Philippines": "Cebu 10.3 123.9;Davao 7.1 125.6;Subic Bay 14.8 120.3", "Myanmar": "Yangon 16.8 96.2", "Cambodia": "Sihanoukville 10.6 103.5",
"Bangladesh": "Chittagong 22.3 91.8;Mongla 22.5 89.6", "Sri Lanka": "Hambantota 6.1 81.1;Galle 6.0 80.2", "Pakistan": "Port Qasim 24.8 67.3;Gwadar 25.1 62.3", "Iran": "Chabahar 25.3 60.6",
"India": "Mormugao 15.4 73.8;Pipavav 20.9 71.5;Hazira 21.1 72.6;Ennore 13.2 80.3;Krishnapatnam 14.3 80.1;Kakinada 16.9 82.3;Gangavaram 17.6 83.2;Dahej 21.7 72.5;Mumbai Port 18.9 72.8;Navlakhi 22.9 70.5;Karaikal 10.9 79.9;Kamarajar 13.3 80.3;Vizhinjam 8.4 77.0;Okha 22.5 69.1;Porbandar 21.6 69.6",
"UAE": "Khalifa Port 24.8 54.6;Port Rashid 25.3 55.3;Fujairah 25.1 56.4;Khor Fakkan 25.3 56.4", "Oman": "Sohar 24.5 56.6;Duqm 19.7 57.7",
"Saudi Arabia": "Dammam 26.5 50.2;King Abdullah Port 22.5 39.1;Yanbu 24.1 38.1", "Qatar": "Hamad Port 25.0 51.6", "Kuwait": "Shuwaikh 29.4 47.9", "Bahrain": "Khalifa Bin Salman 26.2 50.6",
"Iraq": "Umm Qasr 30.0 47.9", "Jordan": "Aqaba 29.5 35.0", "Israel": "Haifa 32.8 35.0;Ashdod 31.8 34.6", "Turkey": "Mersin 36.8 34.6;Ambarli 41.0 28.7;Izmir 38.4 27.1",
"Yemen": "Aden 12.8 45.0", "Djibouti": "Djibouti 11.6 43.1", "Sudan": "Port Sudan 19.6 37.2",
"Germany": "Bremerhaven 53.6 8.5", "Belgium": "Zeebrugge 51.3 3.2", "Netherlands": "Amsterdam 52.4 4.9", "United Kingdom": "Southampton 50.9 -1.4;London Gateway 51.5 0.5;Liverpool 53.4 -3.0",
"Ireland": "Dublin 53.3 -6.2", "France": "Marseille Fos 43.4 4.9", "Spain": "Barcelona 41.4 2.2;Bilbao 43.3 -3.0", "Portugal": "Lisbon 38.7 -9.1;Sines 37.9 -8.9",
"Italy": "Naples 40.8 14.3;La Spezia 44.1 9.8;Livorno 43.6 10.3;Trieste 45.6 13.8;Gioia Tauro 38.4 15.9;Taranto 40.5 17.2", "Slovenia": "Koper 45.5 13.7", "Croatia": "Rijeka 45.3 14.4",
"Greece": "Thessaloniki 40.6 22.9", "Romania": "Constanta 44.2 28.7", "Bulgaria": "Varna 43.2 27.9", "Ukraine": "Odesa 46.5 30.7", "Sweden": "Gothenburg 57.7 11.9", "Denmark": "Aarhus 56.2 10.2",
"Norway": "Oslo 59.9 10.7", "Finland": "Helsinki 60.2 25.0", "Poland": "Gdynia 54.5 18.5", "Lithuania": "Klaipeda 55.7 21.1", "Latvia": "Riga 57.0 24.1", "Estonia": "Tallinn 59.4 24.8",
"Russia": "St Petersburg 59.9 30.2;Novorossiysk 44.7 37.8", "Malta": "Marsaxlokk 35.8 14.5", "Cyprus": "Limassol 34.7 33.0",
"Egypt": "Alexandria 31.2 29.9;Port Said 31.3 32.3;Damietta 31.5 31.8;Ain Sokhna 29.6 32.4", "Libya": "Tripoli 32.9 13.2", "Tunisia": "Rades 36.8 10.3", "Algeria": "Algiers 36.8 3.1",
"Morocco": "Casablanca 33.6 -7.6", "Senegal": "Dakar 14.7 -17.4", "Ivory Coast": "Abidjan 5.3 -4.0", "Ghana": "Tema 5.6 0.0", "Togo": "Lome 6.1 1.3", "Benin": "Cotonou 6.4 2.4",
"Nigeria": "Onne 4.7 7.2", "Cameroon": "Douala 4.0 9.7", "Angola": "Luanda -8.8 13.2", "Namibia": "Walvis Bay -22.9 14.5", "South Africa": "Cape Town -33.9 18.4;Ngqura -33.8 25.7",
"Mozambique": "Maputo -25.97 32.6;Beira -19.8 34.9", "Tanzania": "Dar es Salaam -6.8 39.3", "Kenya": "Mombasa -4.1 39.7", "Somaliland": "Berbera 10.4 45.0", "Mauritius": "Port Louis -20.2 57.5", "Madagascar": "Toamasina -18.1 49.4",
"USA": "Oakland 37.8 -122.3;Seattle 47.6 -122.3;Tacoma 47.3 -122.4;Houston 29.7 -95.3;New Orleans 29.9 -90.1;Miami 25.8 -80.2;Charleston 32.8 -79.9;Norfolk 36.9 -76.3;Baltimore 39.3 -76.6;Boston 42.4 -71.0",
"Canada": "Vancouver 49.3 -123.1;Prince Rupert 54.3 -130.3;Halifax 44.6 -63.6;Montreal 45.5 -73.5", "Mexico": "Manzanillo 19.1 -104.3;Veracruz 19.2 -96.1;Lazaro Cardenas 17.9 -102.2",
"Panama": "Balboa 8.95 -79.6;Colon 9.35 -79.9", "Colombia": "Cartagena 10.4 -75.5;Buenaventura 3.9 -77.1", "Peru": "Callao -12.05 -77.15", "Ecuador": "Guayaquil -2.3 -79.9",
"Chile": "San Antonio -33.6 -71.6;Valparaiso -33.0 -71.6", "Argentina": "Buenos Aires -34.6 -58.4", "Uruguay": "Montevideo -34.9 -56.2",
"Brazil": "Santos -23.98 -46.3;Paranagua -25.5 -48.5;Rio Grande -32.0 -52.1;Itajai -26.9 -48.7;Suape -8.4 -35.0;Pecem -3.5 -38.8", "Jamaica": "Kingston 17.97 -76.8", "Dominican Republic": "Caucedo 18.4 -69.6", "Bahamas": "Freeport 26.5 -78.8",
"Australia": "Sydney -33.9 151.2;Melbourne -37.8 144.9;Brisbane -27.4 153.2;Fremantle -32.05 115.7;Adelaide -34.8 138.5;Port Hedland -20.3 118.6", "New Zealand": "Auckland -36.8 174.8;Tauranga -37.65 176.2;Lyttelton -43.6 172.7",
"Fiji": "Suva -18.1 178.4", "New Caledonia": "Noumea -22.3 166.4", "Papua New Guinea": "Port Moresby -9.5 147.1"}
CARRIERS = ["Evergreen Marine","HMM","Yang Ming","ZIM Integrated","Pacific International Lines","Wan Hai Lines","OOCL","APL","Hamburg Sud","Matson","SM Line","Sinokor","KMTC","Namsung Shipping","SITC",
"X-Press Feeders","Gold Star Line","Arkas Line","Turkon Line","Grimaldi Lines","Messina Line","CCNI","Seaboard Marine","Crowley Maritime","Tropical Shipping","Swire Shipping","Hoegh Autoliners",
"Wallenius Wilhelmsen","NYK Line","Mitsui OSK Lines","K Line","Safmarine","Emirates Shipping Line","Shipping Corporation of India","Great Eastern Shipping","Samudera Shipping","Salam Pacific",
"Interasia Lines","Heung-A Line","Unifeeder","Containerships","Finnlines","DFDS","Stena Line","Atlantic Container Line","King Ocean","Bengal Tiger Line","Transworld Shipping","Sealand Maersk Asia","Pan Ocean","Cosco Hong Kong"]
def region_of(lat, lon):
    if -170 < lon < -30: return "Americas"
    if -30 <= lon < 50 and lat > 12: return "Europe/Mediterranean" if lat > 30 else "Africa"
    if -30 <= lon < 52 and lat <= 12: return "Africa"
    if 112 < lon and lat < -10: return "Oceania"
    if 25 < lon < 60 and 12 < lat < 33: return "Middle East"
    return "Asia"

def scale_freight(target_ports=220, target_carriers=110, target_shipments=400):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        have = {r[0].lower() for r in c.execute("SELECT port_name FROM ports")}; n = c.execute("SELECT COUNT(*) FROM ports").fetchone()[0]
        for ctry, spec in PORTS.items():
            for e in spec.split(";"):
                if n >= target_ports + 40: break
                nm, la, lo = e.rsplit(" ", 2); name = f"{nm} Port"
                if name.lower() in have: continue
                n += 1; have.add(name.lower())
                c.execute("INSERT OR REPLACE INTO ports(port_id,port_name,country,congestion_index,avg_dwell_days,lat,lon,region) VALUES(?,?,?,?,?,?,?,?)",
                          (f"PORT-{n:03d}", name, ctry, round(R.uniform(1.0, 4.6), 1), round(R.uniform(1.5, 7.0), 1), float(la), float(lo), region_of(float(la), float(lo))))
        k = c.execute("SELECT COUNT(*) FROM carriers").fetchone()[0]; names = {r[0] for r in c.execute("SELECT name FROM carriers")}
        pool = [x for x in CARRIERS if x not in names] + [f"Regional Feeder Line {i:02d}" for i in range(1, 80)]
        for nm in pool:
            if k >= target_carriers: break
            k += 1
            c.execute("INSERT OR REPLACE INTO carriers(carrier_id,name,rating,on_time_pct,avg_cost_index,risk_level) VALUES(?,?,?,?,?,?)",
                      (f"CAR-{k:03d}", nm, round(R.uniform(3.2, 4.9), 2), round(R.uniform(68, 97), 1), round(R.uniform(.8, 1.25), 2), R.choice(["Low", "Low", "Medium", "High"])))
        ports = c.execute("SELECT port_name FROM ports").fetchall(); cars = [r[0] for r in c.execute("SELECT name FROM carriers")]
        s = c.execute("SELECT COUNT(*) FROM shipments").fetchone()[0]
        while s < target_shipments:
            s += 1; a, b = R.sample(ports, 2)
            c.execute("INSERT OR REPLACE INTO shipments(shipment_id,origin_port,dest_port,carrier,status,eta,weight_kg,hs_code,predicted_delay_risk) VALUES(?,?,?,?,?,?,?,?,?)",
                      (f"SHP-{s:04d}", a[0], b[0], R.choice(cars), R.choices(["In Transit", "Delivered", "Customs Hold", "Port Congestion Delay", "Anchorage Pending"], [55, 20, 8, 10, 7])[0],
                       (datetime.date.today() + datetime.timedelta(days=R.randint(2, 40))).isoformat(), round(R.uniform(800, 28000), 1), f"HS-{R.choice([30,84,85,87,10,62,27,39,72,22])}{R.randint(10,99)}", round(R.uniform(.03, .6), 2)))

CITIES = {"Mumbai": (19.08, 72.88), "Delhi": (28.61, 77.21), "Bengaluru": (12.97, 77.59), "Chennai": (13.08, 80.27), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86), "Ahmedabad": (23.03, 72.57),
"Jaipur": (26.91, 75.79), "Kolkata": (22.57, 88.36), "Surat": (21.17, 72.83), "Kochi": (9.93, 76.27), "Coimbatore": (11.02, 76.96), "Lucknow": (26.85, 80.95), "Nagpur": (21.15, 79.09), "Indore": (22.72, 75.86),
"Bhopal": (23.26, 77.41), "Patna": (25.59, 85.14), "Chandigarh": (30.73, 76.78), "Visakhapatnam": (17.69, 83.22), "Vadodara": (22.31, 73.18), "Thiruvananthapuram": (8.52, 76.94), "Mysuru": (12.30, 76.64),
"Guwahati": (26.14, 91.74), "Bhubaneswar": (20.30, 85.82), "Ranchi": (23.34, 85.31), "Raipur": (21.25, 81.63), "Dehradun": (30.32, 78.03), "Goa": (15.50, 73.83), "Madurai": (9.93, 78.12), "Nashik": (19.99, 73.79)}
SKUS = [("Buns", "Food"), ("Cooking Oil", "Food"), ("Sauce", "Food"), ("Frozen Fries", "Food"), ("Chicken", "Food"), ("Cheese", "Food"), ("Lettuce", "Food"), ("Packaging", "Supplies"), ("Beverage Syrup", "Beverage"), ("Potatoes", "Food"), ("Flour", "Food"), ("Spice Mix", "Food")]
def scale_franchise(outlets=120, staff=1250, per_outlet_sku=12):
    R = random.Random(42)
    with _DB_LOCK, conn() as c:
        c.executescript("""CREATE TABLE IF NOT EXISTS outlet_geo(outlet_id TEXT PRIMARY KEY, lat REAL, lon REAL);
        CREATE TABLE IF NOT EXISTS customers_fr(customer_id TEXT PRIMARY KEY, outlet_id TEXT, total_spend_90d REAL, visit_freq REAL, avg_order_value REAL, days_since_last_visit INT);""")
        n = c.execute("SELECT COUNT(*) FROM outlets").fetchone()[0]; tiers = ["Metro Flagship", "Urban", "Suburban", "Highway"]; cities = list(CITIES)
        while n < outlets:
            n += 1; oid = f"OUT-{n:03d}"; city = R.choice(cities); rev = R.uniform(900000, 5200000)
            c.execute("INSERT OR REPLACE INTO outlets VALUES(?,?,?,?,?,?,?,?)", (oid, f"Franchise Outlet {n:03d}", city, R.choice(tiers), rev, rev * R.uniform(.55, .82), round(R.uniform(3.0, 4.9), 2), R.randint(12, 30)))
        for oid, city in c.execute("SELECT outlet_id, location FROM outlets").fetchall():
            if not c.execute("SELECT 1 FROM outlet_geo WHERE outlet_id=?", (oid,)).fetchone():
                la, lo = CITIES.get(city, CITIES["Mumbai"]); c.execute("INSERT INTO outlet_geo VALUES(?,?,?)", (oid, la + R.uniform(-.06, .06), lo + R.uniform(-.06, .06)))
        s = c.execute("SELECT COUNT(*) FROM staff").fetchone()[0]; oids = [r[0] for r in c.execute("SELECT outlet_id FROM outlets")]
        while s < staff:
            s += 1; sal = R.uniform(18000, 80000); sat = R.randint(1, 5); ot = R.uniform(0, 35)
            p = max(.05, min(.95, .9 - sal / 100000 - sat / 10 + ot / 100))
            c.execute("INSERT OR REPLACE INTO staff VALUES(?,?,?,?,?,?,?,?,?,?,?)", (f"STF-{s:04d}", R.choice(oids), f"Employee {s:04d}", R.choice(["Crew", "Crew", "Cashier", "Cook", "Shift Lead", "Store Manager"]), sal, round(ot, 1), sat, R.randint(19, 58), R.randint(0, 12), R.randint(1, 5), round(p, 2)))
        for oid in oids:
            have = {r[0] for r in c.execute("SELECT sku_name FROM inventory WHERE outlet_id=?", (oid,))}
            for sku, cat in SKUS[:per_outlet_sku]:
                if sku in have or sku in {"Oil"} and "Cooking Oil" == sku: continue
                wk = R.uniform(80, 700); lead = R.randint(2, 9)
                c.execute("INSERT OR REPLACE INTO inventory VALUES(?,?,?,?,?,?,?,?,?)", (f"INV-{oid[4:]}-{sku[:3].upper()}", oid, sku, cat, int(wk / 7 * (lead + R.uniform(2, 12))), int(wk / 7 * lead), round(wk, 1), lead, .1))
            if c.execute("SELECT COUNT(*) FROM marketing WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    c.execute("INSERT OR REPLACE INTO marketing VALUES(?,?,?,?,?,?,?,?,?,?)", (f"CMP-{oid[4:]}-{j}", oid, f"Campaign {oid[4:]}-{j}", R.choice(["Social", "Email", "Influencer", "Offline"]), R.uniform(20000, 120000), round(R.uniform(1.2, 5.2), 2), R.randint(8000, 60000), R.randint(200, 2500), "2024-01-01", "2024-12-31"))
            if c.execute("SELECT COUNT(*) FROM audits WHERE outlet_id=?", (oid,)).fetchone()[0] < 2:
                for j in range(2):
                    sc = max(40, min(100, R.gauss(84, 10)))
                    c.execute("INSERT OR REPLACE INTO audits VALUES(?,?,?,?,?,?,?,?)", (f"AUD-{oid[4:]}-{j}", oid, (datetime.date.today() - datetime.timedelta(days=30 * (j + 1))).isoformat(), sc, int((100 - sc) / 6), R.choice(["Food Safety", "Hygiene", "SOP Compliance"]), "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", "Monthly audit"))
            if c.execute("SELECT COUNT(*) FROM feedback WHERE outlet_id=?", (oid,)).fetchone()[0] < 8:
                for j in range(8):
                    r = R.choices([1, 2, 3, 4, 5], [5, 8, 15, 35, 37])[0]
                    c.execute("INSERT OR REPLACE INTO feedback VALUES(?,?,?,?,?,?)", (f"FB-{oid[4:]}-{j}", oid, r, R.choice(["Great taste", "Slow service", "Clean ambience", "Staff was rude", "Cold food", "Good value", "Long wait time", "Dirty tables"]), datetime.date.today().isoformat(), (r - 3) / 2))
            if c.execute("SELECT COUNT(*) FROM customers_fr WHERE outlet_id=?", (oid,)).fetchone()[0] < 30:
                for j in range(30):
                    seg = R.choices(["hv", "freq", "churn"], [.2, .4, .4])[0]
                    f = {"hv": (6, 12), "freq": (3, 8), "churn": (.2, 1.5)}[seg]; aov = R.uniform(250, 900) if seg == "hv" else R.uniform(150, 450)
                    fr = R.uniform(*f); c.execute("INSERT OR REPLACE INTO customers_fr VALUES(?,?,?,?,?,?)", (f"C-{oid[4:]}-{j}", oid, round(fr * aov * 3, 0), round(fr, 1), round(aov, 0), R.randint(60, 120) if seg == "churn" else R.randint(1, 20)))


Writing /content/app/scale_data.py


In [21]:
%%writefile /content/app/seanav.py
"""Sea navigation: an ocean mask built from the world coastline (Natural Earth 110m land, bundled with the app), real canal/strait corridors
(Suez, Panama, Malacca, Gibraltar, Dover, Danish straits, Bosphorus, Hormuz, Sunda, Lombok...) and Dijkstra routing over a 1 degree water grid.
Routes are then straightened with a line-of-sight check on a finer 0.25 degree mask, so no leg crosses land. Everything is pure numpy/scipy."""
import os, json, math, heapq, threading
import numpy as np
from matplotlib.path import Path as MPath
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra, connected_components

HERE = os.path.dirname(os.path.abspath(__file__))
TOPO = [os.path.join(HERE, "static", "vendor", "world.topo.json")]
R_NM = 3440.065
ACCESS = {   # river / bay / canal-side ports: where a ship actually enters open water (lat, lon). Matched on the port name.
    "wuhan": (31.2, 122.6), "nanjing": (31.2, 122.6), "taicang": (31.3, 122.4), "shanghai": (31.1, 122.4), "guangzhou": (22.0, 114.0), "dongguan": (22.0, 114.0), "zhuhai": (21.9, 113.8),
    "tokyo": (34.9, 139.9), "yokohama": (34.9, 139.9), "chiba": (34.9, 139.9), "nagoya": (34.2, 137.0), "osaka": (34.2, 135.0), "kobe": (34.2, 135.0), "hamburg": (54.1, 8.2), "antwerp": (51.45, 3.4),
    "houston": (29.1, -94.6), "new orleans": (28.9, -89.3), "baltimore": (37.0, -75.9), "norfolk": (36.9, -75.9), "montreal": (49.3, -66.0), "palembang": (-2.0, 105.4), "yangon": (15.7, 96.3), "mongla": (21.6, 89.5), "kolkata": (21.6, 88.2),
    "ho chi minh": (10.3, 107.1), "kandla": (22.8, 69.6), "navlakhi": (22.9, 70.0), "dahej": (21.7, 72.0), "hazira": (21.0, 72.4), "ambarli": (40.9, 28.3), "izmir": (38.4, 26.6), "amsterdam": (52.5, 4.3), "london gateway": (51.5, 1.2),
    "oslo": (59.2, 10.6), "douala": (3.9, 9.4), "fuzhou": (26.0, 119.9), "lianyungang": (34.7, 119.6), "port klang": (2.9, 101.0), "belawan": (4.1, 98.6), "rotterdam": (51.95, 3.9), "buenos aires": (-35.0, -57.0), "montevideo": (-35.0, -56.2), "guayaquil": (-2.8, -80.4),
    "seattle": (47.9, -122.7), "tacoma": (47.8, -122.8), "vancouver": (49.2, -123.4), "oakland": (37.8, -122.6), "kuching": (2.0, 110.3), "bangkok": (13.3, 100.6), "haikou": (20.3, 110.3), "port said": (31.4, 32.3), "ain sokhna": (29.6, 32.6), "aqaba": (28.9, 34.9),
}
CORRIDORS = {   # name -> centre line (lat, lon); cells along it (and 0.3 degree either side) are forced to water
    "Suez Canal": [(31.9, 32.3), (31.3, 32.33), (30.6, 32.34), (30.0, 32.55), (29.4, 32.55), (28.9, 32.8), (28.3, 33.2)],
    "Panama Canal": [(9.9, -79.95), (9.35, -79.88), (9.05, -79.6), (8.6, -79.5), (8.0, -79.4)],
    "Strait of Malacca": [(1.1, 104.3), (1.25, 103.7), (1.9, 102.6), (2.7, 101.5), (3.6, 100.5), (4.6, 99.9), (5.6, 99.0), (6.3, 97.5)],
    "Singapore Strait": [(1.2, 104.0), (1.4, 104.6), (2.2, 105.5)],
    "Strait of Gibraltar": [(36.5, -7.0), (36.0, -5.8), (36.0, -5.3), (36.2, -4.5), (36.4, -3.5)],
    "Dover Strait": [(49.6, -6.0), (49.8, -3.0), (50.4, -0.5), (50.9, 1.1), (51.4, 2.1), (52.0, 3.0)],
    "Danish Straits": [(57.7, 7.5), (57.8, 10.3), (57.0, 11.6), (56.2, 12.5), (55.5, 12.8), (55.0, 13.0), (54.9, 14.0), (55.2, 16.0), (56.0, 18.0), (57.5, 19.7), (58.8, 21.0), (59.6, 23.5), (59.9, 26.0), (60.0, 29.0)],
    "Bosphorus": [(39.9, 25.9), (40.2, 26.5), (40.7, 27.7), (41.0, 28.9), (41.4, 29.15), (42.0, 29.4), (42.8, 29.8)],
    "Strait of Hormuz": [(24.5, 58.6), (25.7, 56.8), (26.5, 56.3), (26.4, 54.8), (26.6, 52.8), (27.4, 51.0), (28.8, 49.8), (29.6, 48.9)],
    "Bab-el-Mandeb": [(12.0, 43.9), (12.7, 43.3), (13.6, 42.6)],
    "Sunda Strait": [(-5.7, 105.9), (-6.1, 105.8), (-6.8, 105.3)],
    "Lombok Strait": [(-7.6, 115.7), (-8.4, 115.7), (-9.2, 115.6)],
    "Messina": [(38.4, 15.6), (38.0, 15.6), (37.5, 15.3)],
    "Seto Inland Sea": [(33.9, 135.0), (34.4, 134.9), (34.5, 135.1)],
    "Korea Strait": [(35.0, 129.2), (34.3, 129.8), (33.5, 130.3)],
    "Aegean": [(36.2, 22.8), (36.6, 24.0), (37.5, 24.2), (38.5, 24.6), (39.5, 24.6), (40.0, 25.8)],
    "Adriatic": [(39.8, 18.9), (41.0, 18.7), (42.5, 17.5), (43.8, 15.3), (45.0, 13.5), (45.6, 13.4)],
    "Red Sea": [(27.5, 34.2), (24.0, 36.2), (20.0, 38.4), (16.0, 41.2), (13.5, 42.8)],
    "Gulf of Aden": [(12.5, 43.5), (12.0, 47.0), (12.4, 51.5), (12.0, 54.5)],
    "Persian Gulf": [(24.5, 58.6), (25.5, 56.7), (26.4, 54.5), (27.0, 52.0), (28.0, 50.5), (29.3, 48.8)],
    "Gulf of Mexico": [(24.0, -84.0), (23.0, -87.0), (26.5, -90.0), (28.0, -91.0), (28.0, -95.0), (26.0, -97.0), (21.0, -97.0), (19.0, -95.0)],
    "Baltic South": [(54.2, 18.9), (54.6, 19.5)],
}
BLOCK_GROUPS = {"suez": ["Suez Canal"], "panama": ["Panama Canal"], "malacca": ["Strait of Malacca"], "hormuz": ["Strait of Hormuz"], "bosphorus": ["Bosphorus"]}
CHOKE_LABEL = {"Suez Canal": "Suez Canal", "Panama Canal": "Panama Canal", "Strait of Malacca": "Strait of Malacca", "Strait of Gibraltar": "Gibraltar", "Dover Strait": "English Channel", "Strait of Hormuz": "Hormuz", "Bab-el-Mandeb": "Bab-el-Mandeb", "Bosphorus": "Bosphorus", "Danish Straits": "Danish Straits", "Sunda Strait": "Sunda Strait", "Lombok Strait": "Lombok Strait"}

def _hav(a, b):
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1])); h = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return R_NM * 2 * math.asin(min(1.0, math.sqrt(h)))

def _land_polys():
    p = next((x for x in TOPO if os.path.exists(x)), None)
    if not p: raise FileNotFoundError("world.topo.json (bundled with the app) not found")
    t = json.load(open(p)); sx, sy = t["transform"]["scale"]; tx, ty = t["transform"]["translate"]; arcs = []
    for a in t["arcs"]:
        x = y = 0; pts = []
        for dx, dy in a: x += dx; y += dy; pts.append((x * sx + tx, y * sy + ty))
        arcs.append(pts)
    def ring(idx):
        out = []
        for i in idx:
            seg = arcs[i] if i >= 0 else arcs[~i][::-1]; out += seg if not out else seg[1:]
        return out
    polys = []
    def geoms(o):
        if o["type"] == "GeometryCollection":
            for g in o["geometries"]: yield from geoms(g)
        else: yield o
    for g in geoms(t["objects"]["land"]):
        pl = g["arcs"] if g["type"] == "MultiPolygon" else [g["arcs"]]
        for poly in pl: polys.append([_unwrap_ring(ring(r)) for r in poly])        # [outer, holes...]
    return polys
def _unwrap_ring(pts):
    """Rings that cross the antimeridian jump by ~360 degrees between two points; make the longitude continuous (it may then exceed +-180)."""
    out = [pts[0]]
    for x, y in pts[1:]:
        px = out[-1][0]
        while x - px > 180: x -= 360
        while x - px < -180: x += 360
        out.append((x, y))
    return out

def gc_arr(a, b, step_nm=60.0, include_ends=True):
    """Vectorised great-circle sampling: (lat array, lon array); longitudes continuous with a's."""
    la1, lo1, la2, lo2 = map(math.radians, (a[0], a[1], b[0], b[1]))
    v1 = np.array([math.cos(la1) * math.cos(lo1), math.cos(la1) * math.sin(lo1), math.sin(la1)]); v2 = np.array([math.cos(la2) * math.cos(lo2), math.cos(la2) * math.sin(lo2), math.sin(la2)])
    om = math.acos(max(-1.0, min(1.0, float(v1 @ v2)))); n = max(1, int(om * R_NM / step_nm)); k = np.arange(0 if include_ends else 1, n + 1 if include_ends else n); t = k / n
    if om > 1e-9: w1 = np.sin((1 - t) * om) / math.sin(om); w2 = np.sin(t * om) / math.sin(om)
    else: w1, w2 = 1 - t, t
    v = w1[:, None] * v1[None, :] + w2[:, None] * v2[None, :]; la = np.degrees(np.arcsin(np.clip(v[:, 2], -1, 1))); lo = np.degrees(np.arctan2(v[:, 1], v[:, 0]))
    lo = a[1] + (((lo - a[1]) + 180) % 360 - 180)          # continuous with a's longitude
    return la, lo
def gc_points(a, b, step_nm=60.0, include_ends=True):
    """Points along the great circle from a to b (lat, lon), spaced about step_nm apart; longitudes continuous with a's."""
    la, lo = gc_arr(a, b, step_nm, include_ends); out = list(zip(la.tolist(), lo.tolist()))
    for q in range(1, len(out)):          # keep successive longitudes continuous (no jumps > 180)
        while out[q][1] - out[q - 1][1] > 180: out[q] = (out[q][0], out[q][1] - 360)
        while out[q][1] - out[q - 1][1] < -180: out[q] = (out[q][0], out[q][1] + 360)
    return out

class Sea:
    def __init__(self, step=1.0, fine=0.25, latmin=-78.0, latmax=82.0):
        self.step, self.fine, self.latmin, self.latmax = step, fine, latmin, latmax
        self.lats = np.arange(latmin + step / 2, latmax, step); self.lons = np.arange(-180 + step / 2, 180, step); self.nlat, self.nlon = len(self.lats), len(self.lons)
        self.flats = np.arange(latmin + fine / 2, latmax, fine); self.flons = np.arange(-180 + fine / 2, 180, fine)
        polys = _land_polys(); self.land = self._raster(polys, self.lats, self.lons); self.fland = self._raster(polys, self.flats, self.flons)
        self.corr_cells = {}                                                   # corridor name -> set of coarse node ids
        for name, line in CORRIDORS.items(): self._carve(name, line)
        pol = (self.lats < -52) | (self.lats > 76); drake = (self.lons > -82) & (self.lons < -48)
        self.land[np.ix_(pol & (self.lats < 0), ~drake)] = True; self.land[np.ix_(self.lats > 76, np.ones(self.nlon, bool))] = True
        fpol = (self.flats < -52); fdrake = (self.flons > -82) & (self.flons < -48); self.fland[np.ix_(fpol, ~fdrake)] = True; self.fland[self.flats > 76, :] = True
        self.ocean = ~self.land; self.nid = -np.ones(self.land.shape, int); self.nid[self.ocean] = np.arange(int(self.ocean.sum()))
        self.nodes = np.argwhere(self.ocean); self.n = len(self.nodes); self._build_graph()
        self._cache = {}; self._variants = {}; self._lock = threading.RLock()
    def _raster(self, polys, lats, lons):
        LON, LAT = np.meshgrid(lons, lats); pts = np.c_[LON.ravel(), LAT.ravel()]; inside = np.zeros(len(pts), bool)
        for rings in polys:
            outer = np.array(rings[0]); x0, y0 = outer.min(0); x1, y1 = outer.max(0)
            for off in (0, 360, -360):
                mo = (pts[:, 0] + off >= x0) & (pts[:, 0] + off <= x1) & (pts[:, 1] >= y0) & (pts[:, 1] <= y1)
                if not mo.any(): continue
                idx = np.where(mo)[0]; q = pts[idx] + np.array([off, 0]); ins = MPath(outer).contains_points(q)
                for h in rings[1:]: ins &= ~MPath(np.array(h)).contains_points(q)
                inside[idx[ins]] = True
        return inside.reshape(len(lats), len(lons))
    def _carve(self, name, line):
        cells = set()
        for (a, b) in zip(line[:-1], line[1:]):
            n = max(2, int(_hav(a, b) / 8))
            for t in np.linspace(0, 1, n):
                la, lo = a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t
                i = int((la - self.latmin) / self.step); j = int((lo + 180) / self.step) % self.nlon
                if 0 <= i < self.nlat: self.land[i, j] = False; cells.add((i, j))
                fi = int((la - self.latmin) / self.fine); fj = int((lo + 180) / self.fine) % len(self.flons)
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        a_, b_ = fi + di, (fj + dj) % len(self.flons)
                        if 0 <= a_ < len(self.flats): self.fland[a_, b_] = False
        self.corr_cells[name] = cells
    def _build_graph(self):
        rows, cols, w = [], [], []
        for k, (i, j) in enumerate(self.nodes):
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0: continue
                    a, b = i + di, (j + dj) % self.nlon
                    if not (0 <= a < self.nlat) or not self.ocean[a, b]: continue
                    if di != 0 and dj != 0 and not (self.ocean[i, b] or self.ocean[a, j]): continue         # no diagonal squeeze between two land cells
                    rows.append(k); cols.append(self.nid[a, b]); w.append(_hav((self.lats[i], self.lons[j]), (self.lats[a], self.lons[b])))
        self.A = csr_matrix((w, (rows, cols)), shape=(self.n, self.n))
        nc, lab = connected_components(self.A, directed=False); self.main = lab == np.bincount(lab).argmax(); self.comp = lab
    # ------------------------------------------------------------ helpers
    def is_land(self, la, lo):
        i = int((la - self.latmin) / self.fine); j = int(((lo + 180) % 360) / self.fine)
        return bool(self.fland[i, j]) if 0 <= i < len(self.flats) else True
    def dist_to_water_nm(self, la, lo, maxr=2.0):
        """Distance in nm from a point to the nearest water cell of the fine mask (0 if the point is in water)."""
        i0 = int((la - self.latmin) / self.fine); j0 = int(((lo + 180) % 360) / self.fine); r = int(maxr / self.fine); best = None
        for di in range(-r, r + 1):
            for dj in range(-r, r + 1):
                i, j = i0 + di, (j0 + dj) % len(self.flons)
                if 0 <= i < len(self.flats) and not self.fland[i, j]:
                    d = _hav((la, lo), (self.flats[i], self.flons[j])); best = d if best is None or d < best else best
        return best
    def _clear(self, a, b, ends=0.6):
        la, lo = gc_arr(a, b, 7.0)
        keep = ~(((np.abs(la - a[0]) < ends) & (np.abs(lo - a[1]) < ends)) | ((np.abs(la - b[0]) < ends) & (np.abs(lo - b[1]) < ends)))
        la, lo = la[keep], lo[keep]
        if not len(la): return True
        i = ((la - self.latmin) / self.fine).astype(int); j = (((lo + 180) % 360) / self.fine).astype(int); ok = (i >= 0) & (i < len(self.flats))
        if not ok.all(): return False
        return not bool(self.fland[i, j].any())
    def _near_nodes(self, la, lo, k=12, main_only=True):
        d = (self.lats[self.nodes[:, 0]] - la) ** 2 + ((((self.lons[self.nodes[:, 1]] - lo + 180) % 360) - 180) * math.cos(math.radians(la))) ** 2
        if main_only: d = np.where(self.main, d, 1e9)
        return np.argsort(d)[:k]
    def _snap(self, la, lo):
        near = self._near_nodes(la, lo)
        for k in near:
            p = (float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]]))
            if self._clear((la, lo), p, 0.45): return int(k)
        return int(near[0])
    def _graph_for(self, blocked):
        key = tuple(sorted(blocked))
        if not key: return self.A
        with self._lock:
            if key in self._variants: return self._variants[key]
            ok = np.ones(self.n, bool)
            for g in key:
                for name in BLOCK_GROUPS.get(g, [g]):
                    for (i, j) in self.corr_cells.get(name, ()):
                        if self.nid[i, j] >= 0: ok[self.nid[i, j]] = False
            D = csr_matrix(ok.astype(float)).T if False else None
            A = self.A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros(); self._variants[key] = A; return A
    # ------------------------------------------------------------ public API
    def route(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """o, d = (lat, lon). Returns {"pts": [(lat, lon), ...] (longitudes unwrapped), "nm": float, "via": [names]} or None."""
        key = (round(o[0], 2), round(o[1], 2), round(d[0], 2), round(d[1], 2), oa, da, tuple(sorted(blocked)), tuple((round(s[0]), round(s[1]), round(s[2])) for s in storms))
        with self._lock:
            if key in self._cache: return self._cache[key]
        A = self._graph_for(blocked)
        if storms:
            ok = np.ones(self.n, bool)
            for (sla, slo, rn) in storms:
                dd = np.array([_hav((sla, slo), (self.lats[i], self.lons[j])) for i, j in self.nodes]) if False else None
                la_ = self.lats[self.nodes[:, 0]]; lo_ = self.lons[self.nodes[:, 1]]
                dist = R_NM * np.arccos(np.clip(np.sin(np.radians(la_)) * math.sin(math.radians(sla)) + np.cos(np.radians(la_)) * math.cos(math.radians(sla)) * np.cos(np.radians(lo_ - slo)), -1, 1)); ok &= dist > rn
            A = A.multiply(ok[:, None]).multiply(ok[None, :]).tocsr(); A.eliminate_zeros()
        s, t = self._snap(*(oa or o)), self._snap(*(da or d))
        dist, pred = dijkstra(A, directed=True, indices=s, return_predecessors=True)
        if not np.isfinite(dist[t]): res = None
        else:
            path = [t]
            while path[-1] != s: path.append(pred[path[-1]])
            path.reverse(); pts = [o] + ([oa] if oa else []) + [(float(self.lats[self.nodes[k, 0]]), float(self.lons[self.nodes[k, 1]])) for k in path] + ([da] if da else []) + [d]; pts = self._unwrap(self._straighten(pts)); pts = self._densify(pts)
            res = {"pts": [(round(a, 3), round(b, 3)) for a, b in pts], "nm": round(sum(_hav(pts[i], pts[i + 1]) for i in range(len(pts) - 1)), 1), "via": self._via(pts)}
        with self._lock: self._cache[key] = res
        return res
    def _straighten(self, pts):
        out = [pts[0]]; i = 0
        while i < len(pts) - 1:
            j = len(pts) - 1
            while j > i + 1 and not self._clear(pts[i], pts[j], 0.5): j -= 1
            out.append(pts[j]); i = j
        return out
    @staticmethod
    def _densify(pts, step=90.0):
        out = [pts[0]]
        for a, b in zip(pts[:-1], pts[1:]): out += gc_points(a, b, step)[1:]
        return out
    @staticmethod
    def _unwrap(pts):
        out = [pts[0]]
        for la, lo in pts[1:]:
            prev = out[-1][1]
            while lo - prev > 180: lo -= 360
            while lo - prev < -180: lo += 360
            out.append((la, lo))
        return out
    def _via(self, pts):
        names = []; dense = []
        for a, b in zip(pts[:-1], pts[1:]):
            for t in np.linspace(0, 1, max(2, int(_hav(a, b) / 15))): dense.append((a[0] + (b[0] - a[0]) * t, a[1] + (b[1] - a[1]) * t))
        order = []
        for la, lo in dense:
            i = int((la - self.latmin) / self.step); j = int(((lo + 180) % 360) / self.step) % self.nlon
            for name, cells in self.corr_cells.items():
                if name in CHOKE_LABEL and (i, j) in cells and CHOKE_LABEL[name] not in order: order.append(CHOKE_LABEL[name])
        if any(-47 < la < -30 and 10 < (lo % 360 if lo < 0 else lo) < 35 for la, lo in dense): order.append("Cape of Good Hope")
        if any(la < -50 and (-80 < lo < -55) for la, lo in dense): order.append("Cape Horn / Drake Passage")
        return order
    def options(self, o, d, blocked=(), storms=(), oa=None, da=None):
        """Up to three sea routes: shortest, and the alternatives that avoid Suez / Panama / Malacca when the shortest uses them."""
        base = self.route(o, d, tuple(blocked), storms, oa, da); out = []
        if not base: return out
        def label(r):
            v = r["via"]; return ("via " + " + ".join(v[:3])) if v else "direct sea route"
        out.append({"name": label(base), "pts": base["pts"], "nm": base["nm"], "via": base["via"]})
        for g, nm_ in (("suez", "Suez Canal"), ("panama", "Panama Canal"), ("malacca", "Strait of Malacca")):
            if nm_ in base["via"] and g not in blocked:
                alt = self.route(o, d, tuple(blocked) + (g,), storms, oa, da)
                if alt and abs(alt["nm"] - base["nm"]) > 50 and all(abs(alt["nm"] - x["nm"]) > 50 for x in out): out.append({"name": label(alt) + f" (without {nm_})", "pts": alt["pts"], "nm": alt["nm"], "via": alt["via"]})
        return out

def access_for(name):
    n = (name or "").lower()
    for k, v in ACCESS.items():
        if k in n: return v
    return None
_SEA = None; _SL = threading.Lock(); _VER = "v5"
def _cache_path():
    d = os.environ.get("APP_DATA_DIR") or os.path.dirname(os.path.abspath(__file__)); return os.path.join(d, f"seanav_{_VER}.pkl")
def sea():
    """Singleton. The raster + graph (about 6 s to build) is pickled next to the database, so later starts load in well under a second."""
    global _SEA
    with _SL:
        if _SEA is None:
            import pickle
            p = _cache_path()
            try:
                with open(p, "rb") as f: _SEA = pickle.load(f)
                _SEA._cache = {}; _SEA._variants = {}; _SEA._lock = threading.RLock()
            except Exception:
                _SEA = Sea()
                try:
                    st = {k: v for k, v in _SEA.__dict__.items() if k not in ("_lock", "_cache", "_variants")}
                    class _P: pass
                    o = Sea.__new__(Sea); o.__dict__.update(st)
                    with open(p + ".tmp", "wb") as f: pickle.dump(o, f, protocol=4)
                    os.replace(p + ".tmp", p)
                except Exception: pass
    return _SEA


Writing /content/app/seanav.py


In [22]:
%%writefile /content/app/seed_data.py
import random, sqlite3, datetime
import pandas as pd
from db import get_conn

BASE_PORTS = [
    ("JNPT Nhava Sheva (Mumbai)", "India", 3.4, 4, 18.95, 72.95, "Asia"),
    ("Mundra Port", "India", 2.8, 3, 22.84, 69.70, "Asia"),
    ("Chennai Port", "India", 3.1, 4, 13.10, 80.30, "Asia"),
    ("Tuticorin VOC Port", "India", 2.5, 3, 8.75, 78.18, "Asia"),
    ("Cochin Port", "India", 2.2, 3, 9.96, 76.26, "Asia"),
    ("Visakhapatnam Port", "India", 2.9, 3, 17.68, 83.28, "Asia"),
    ("Kolkata Haldia Port", "India", 3.5, 5, 22.03, 88.11, "Asia"),
    ("Kandla Deendayal Port", "India", 3.0, 4, 23.01, 70.22, "Asia"),
    ("New Mangalore Port", "India", 2.3, 3, 12.92, 74.81, "Asia"),
    ("Paradip Port", "India", 3.2, 4, 20.26, 86.67, "Asia"),
    ("Shanghai Port", "China", 4.2, 5, 31.23, 121.47, "Asia"),
    ("Singapore Port", "Singapore", 1.2, 2, 1.29, 103.85, "Asia"),
    ("Busan Port", "South Korea", 1.8, 3, 35.10, 129.04, "Asia"),
    ("Tokyo Port", "Japan", 2.2, 3, 35.62, 139.77, "Asia"),
    ("Colombo Port", "Sri Lanka", 2.7, 3, 6.94, 79.84, "Asia"),
    ("Dubai Jebel Ali Port", "UAE", 1.9, 2, 25.20, 55.27, "Middle East"),
    ("Rotterdam Port", "Netherlands", 1.5, 2, 51.92, 4.47, "Europe"),
    ("Antwerp Port", "Belgium", 2.6, 3, 51.22, 4.40, "Europe"),
    ("Hamburg Port", "Germany", 2.5, 3, 53.55, 9.99, "Europe"),
    ("Los Angeles Port", "USA", 3.6, 4, 33.74, -118.27, "Americas")
]

def safe_exec(conn, sql, params=()):
    try:
        conn.execute(sql, params)
    except Exception as e:
        pass

def seed_all():
    with get_conn() as conn:
        # 1. Ports
        safe_exec(conn, "DELETE FROM ports;")
        for i, p in enumerate(BASE_PORTS, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO ports (port_id, port_name, country, congestion_index, avg_dwell_days, lat, lon, region) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"PORT-{i:03d}", p[0], p[1], p[2], p[3], p[4], p[5], p[6])
            )

        # 2. User accounts are created by accounts.py (signup, admin dashboard and the seeded Admin); no demo passwords live in the code.

        # 3. Shipments
        safe_exec(conn, "DELETE FROM shipments;")
        carriers_list = ["Maersk Line", "MSC Cargo", "CMA CGM", "COSCO Shipping", "Hapag-Lloyd", "ONE Ocean Express"]
        statuses = ["In Transit", "Customs Hold", "Delivered", "Port Congestion Delay", "Anchorage Pending"]
        cargos = ["Electronics", "Pharmaceuticals", "Automotive Parts", "Textiles", "Heavy Machinery", "Perishables"]

        for i in range(1, 101):
            p1 = BASE_PORTS[i % len(BASE_PORTS)][0]
            p2 = BASE_PORTS[(i+3) % len(BASE_PORTS)][0]
            shp_id = f"SHP-{i:04d}"
            weight = round(random.uniform(500.0, 45000.0), 1)
            dist = round(random.uniform(800.0, 18000.0), 1)
            sev = random.randint(1, 5)
            ch_prob = round(random.uniform(0.02, 0.45), 2)
            cong = round(random.uniform(1.0, 4.8), 1)
            d_risk = round((cong / 5.0) * 0.5 + (ch_prob) * 0.3 + (sev / 5.0) * 0.2, 2)
            co2 = round(weight * dist * 0.00012, 1)
            margin = round(random.uniform(8.5, 28.0), 1)
            dwell = random.randint(1, 8)
            cargo = random.choice(cargos)
            hs = f"HS-{random.randint(8400, 8900)}"

            safe_exec(conn,
                "INSERT OR REPLACE INTO shipments (shipment_id, origin_port, dest_port, carrier, status, weight_kg, distance_km, weather_severity, customs_hold_prob, congestion_index, predicted_delay_risk, co2_emissions_kg, freight_margin, port_dwell_days, cargo_type, hs_code) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (shp_id, p1, p2, random.choice(carriers_list), random.choice(statuses), weight, dist, sev, ch_prob, cong, d_risk, co2, margin, dwell, cargo, hs)
            )

        # 4. Weather Risks
        safe_exec(conn, "DELETE FROM weather_risks;")
        for i in range(1, 21):
            pname = BASE_PORTS[(i - 1) % len(BASE_PORTS)][0]
            sev = random.randint(1, 4)
            fore = "Category 3 Typhoon Warning" if sev >= 3 else "Clear Maritime Conditions"
            w_spd = round(random.uniform(12.0, 58.0), 1)
            wv_ht = round(random.uniform(0.8, 5.5), 1)
            temp = round(random.uniform(14.0, 38.0), 1)
            safe_exec(conn,
                "INSERT OR REPLACE INTO weather_risks (port_name, current_severity, forecast, wind_speed, wave_height, temperature) VALUES (?, ?, ?, ?, ?, ?);",
                (pname, sev, fore, w_spd, wv_ht, temp)
            )

        # 5. Alerts
        safe_exec(conn, "DELETE FROM alerts;")
        categories = ["Customs Hold", "Typhoon Storm", "Port Congestion", "Vessel Mechanical", "Bunker Fuel Surcharge"]
        severities = ["Critical", "High", "Medium", "Low"]
        for i in range(1, 51):
            shp_id = f"SHP-{i:04d}"
            sev = random.choice(severities)
            cat = random.choice(categories)
            msg = f"Alert #{i:03d}: Severe {cat} operational delay reported on {shp_id}."
            safe_exec(conn,
                "INSERT INTO alerts (shipment_id, severity, category, message, date, resolved) VALUES (?, ?, ?, ?, ?, ?);",
                (shp_id, sev, cat, msg, "2024-08-11", 0)
            )

        # 6. Carriers
        safe_exec(conn, "DELETE FROM carriers;")
        for idx, name in enumerate(carriers_list, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO carriers (carrier_id, name, rating, on_time_pct, avg_cost_index, risk_level) VALUES (?, ?, ?, ?, ?, ?);",
                (f"CAR-{idx:03d}", name, round(random.uniform(3.8, 4.9), 2), round(random.uniform(78, 96), 1), round(random.uniform(0.86, 1.18), 2), "Low")
            )

        # 7. Customers
        safe_exec(conn, "DELETE FROM customers;")
        for i in range(1, 25):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customers (customer_id, name, industry, priority_tier, credit_risk) VALUES (?, ?, ?, ?, ?);",
                (f"CUST-{i:03d}", f"Corporate Client {i:03d}", random.choice(["Food Service", "Retail", "QSR", "Hospitality"]), random.choice(["Platinum", "Gold", "Silver"]), round(random.uniform(0.02, 0.18), 2))
            )

        # 8. Freight Quotes
        safe_exec(conn, "DELETE FROM freight_quotes;")
        for i in range(1, 51):
            base = round(random.uniform(1200, 9000), 2)
            margin_pct = round(random.uniform(9, 24), 1)
            final = round(base * (1 + margin_pct / 100), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO freight_quotes (quote_id, shipment_id, customer_id, base_cost, insurance, customs_fee, fuel_surcharge, final_price, margin_pct, status, created_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"QTE-{i:04d}", f"SHP-{random.randint(1, 50):04d}", f"CUST-{random.randint(1, 20):03d}", base, round(base * 0.02, 2), round(random.uniform(100, 600), 2), round(base * 0.08, 2), final, margin_pct, random.choice(["Draft", "Accepted", "Submitted"]), datetime.date.today().isoformat())
            )

        # 9. Customs Tariffs
        safe_exec(conn, "DELETE FROM customs_tariffs;")
        for i, cargo in enumerate(cargos, 1):
            safe_exec(conn,
                "INSERT OR REPLACE INTO customs_tariffs (tariff_id, hs_code, cargo_type, origin_country, destination_country, duty_rate, clearance_risk, required_docs, advisory) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"TAR-{i:03d}", f"HS-{8400 + i * 31}", cargo, "India", "UAE", round(random.uniform(4, 16), 2), round(random.uniform(0.08, 0.42), 2), "Commercial invoice, packing list, bill of lading", "Validate HS code and pre-clear high-risk lanes.")
            )

        # 10. Outlets (FranchiseOps)
        outlet_cities = ["Chennai", "Bengaluru", "Hyderabad", "Mumbai", "Pune", "Delhi", "Kochi", "Coimbatore", "Ahmedabad", "Kolkata"]
        tiers = ["Metro Flagship", "Urban", "Express", "Mall"]
        safe_exec(conn, "DELETE FROM outlets;")
        for i in range(1, 51):
            revenue = round(random.uniform(850000, 6400000), 2)
            cost = round(revenue * random.uniform(0.58, 0.82), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO outlets (outlet_id, outlet_name, location, tier, revenue, operating_costs, customer_satisfaction, staff_headcount) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"OUT-{i:03d}", f"Franchise Outlet {i:03d}", random.choice(outlet_cities), random.choice(tiers), revenue, cost, round(random.uniform(3.2, 4.9), 2), random.randint(12, 55))
            )

        # 11. Staff
        roles = ["Store Manager", "Shift Lead", "Crew", "Chef", "Cashier", "Inventory Associate"]
        safe_exec(conn, "DELETE FROM staff;")
        for i in range(1, 151):
            satisfaction = random.randint(1, 5)
            overtime = round(random.uniform(0, 42), 1)
            attrition = min(0.95, max(0.03, 0.55 - satisfaction * 0.08 + overtime * 0.009 + random.uniform(-0.08, 0.08)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO staff (staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction, age, tenure_years, work_life_balance, predicted_attrition_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"STF-{i:04d}", f"OUT-{random.randint(1, 50):03d}", f"Employee {i:04d}", random.choice(roles), round(random.uniform(18000, 95000), 2), overtime, satisfaction, random.randint(19, 56), random.randint(0, 14), random.randint(1, 5), round(attrition, 2))
            )

        # 12. Inventory
        skus = ["Buns", "Cheese", "Sauce", "Chicken", "Paneer", "Coffee Beans", "Packaging", "Oil", "Frozen Fries", "Dessert Mix"]
        safe_exec(conn, "DELETE FROM inventory;")
        for i in range(1, 151):
            demand = round(random.uniform(20, 420), 1)
            threshold = random.randint(30, 180)
            stock = random.randint(5, 420)
            risk = min(0.95, max(0.02, (threshold - stock) / max(threshold, 1) + random.uniform(0.05, 0.28)))
            safe_exec(conn,
                "INSERT OR REPLACE INTO inventory (record_id, outlet_id, sku_name, category, current_stock, reorder_threshold, weekly_demand, lead_time_days, stockout_risk_prob) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"INV-{i:04d}", f"OUT-{random.randint(1, 50):03d}", random.choice(skus), random.choice(["Food", "Beverage", "Packaging", "Consumable"]), stock, threshold, demand, random.randint(1, 9), round(risk, 2))
            )

        # 13. Marketing
        channels = ["Digital Ads", "Social Media", "Local Print", "Influencer Campaign", "Radio Spots"]
        safe_exec(conn, "DELETE FROM marketing;")
        for i in range(1, 51):
            budget = round(random.uniform(15000, 120000), 2)
            roi = round(random.uniform(1.8, 5.4), 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO marketing (campaign_id, outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);",
                (f"CMP-{i:03d}", f"OUT-{random.randint(1, 50):03d}", f"Campaign {i:03d}", random.choice(channels), budget, roi, random.randint(5000, 80000), random.randint(200, 4500), "2024-01-01", "2024-12-31")
            )

        # 14. Feedback
        safe_exec(conn, "DELETE FROM feedback;")
        comments = ["Great food!", "Slow service", "Clean ambience", "Polite staff", "Average experience"]
        for i in range(1, 101):
            rating = random.randint(1, 5)
            sentiment = round((rating - 3) / 2.0, 2)
            safe_exec(conn,
                "INSERT OR REPLACE INTO feedback (feedback_id, outlet_id, rating, comment, date, sentiment_score) VALUES (?, ?, ?, ?, ?, ?);",
                (f"FB-{i:04d}", f"OUT-{random.randint(1, 50):03d}", rating, random.choice(comments), "2024-08-01", sentiment)
            )

        # 15. Audits
        safe_exec(conn, "DELETE FROM audits;")
        categories_audit = ["Food Safety", "Hygiene & Sanitation", "Fire & Safety", "Financial Compliance"]
        for i in range(1, 51):
            score = round(random.uniform(65, 99), 1)
            status = "Pass" if score >= 85 else ("Conditional Pass" if score >= 75 else "Action Required")
            safe_exec(conn,
                "INSERT OR REPLACE INTO audits (audit_id, outlet_id, audit_date, score, violations, category, status, notes) VALUES (?, ?, ?, ?, ?, ?, ?, ?);",
                (f"AUD-{i:03d}", f"OUT-{random.randint(1, 50):03d}", "2024-08-01", score, int((100-score)/5), random.choice(categories_audit), status, "Audit completed cleanly.")
            )

        conn.commit()


Writing /content/app/seed_data.py


In [23]:
%%writefile /content/app/server.py
"""Web server: FastAPI JSON API + server-sent events + the single-page front end. Models live on the GPU in this process; the simulator and agent cycles run in worker.py."""
import sqlite3, os, sys, io, re, csv, json, time, asyncio, subprocess, threading, importlib, datetime
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config
from config import KIND, DB_PATH
from fastapi import FastAPI, Request, Response, HTTPException, UploadFile, File, Form, Header
from fastapi.responses import JSONResponse, StreamingResponse, FileResponse, HTMLResponse, PlainTextResponse
from fastapi.staticfiles import StaticFiles
import pandas as pd
import numpy as np
import gpu, qa as QA, analyst as AN, live_engine as LE, entities as EN, blocks as B
import views_core as VC
import rbac, accounts, intents
dom = importlib.import_module("domain_franchise" if KIND == "Franchise" else "domain_freight")
V = importlib.import_module("views_franchise" if KIND == "Franchise" else "views_freight")
HERE = os.path.dirname(os.path.abspath(__file__)); STATIC = os.path.join(HERE, "static")
app = FastAPI(title=f"{V.NAME} Platform", docs_url=None, redoc_url=None, openapi_url=None) if os.environ.get("PUBLIC_DOCS") != "1" else FastAPI(title=f"{V.NAME} Platform"); _worker = None; _ready = {"db": False}
import hardening; hardening.install(app, KIND)

@app.middleware("http")
async def sec(request, call_next):
    r = await call_next(request); r.headers["X-Content-Type-Options"] = "nosniff"; r.headers["Referrer-Policy"] = "same-origin"; r.headers["Cache-Control"] = "no-store" if request.url.path.startswith("/api") else r.headers.get("Cache-Control", "public, max-age=300"); return r

def _boot():
    from db import init_db; from seed_data import seed_all
    init_db(); seed_all(); LE.init_live_tables(); accounts.init(); _ready["db"] = True
@app.on_event("startup")
def startup():
    global _worker
    _boot()
    if os.environ.get("NO_WORKER") != "1":
        env = {**os.environ, "APP_KIND": KIND}; _worker = subprocess.Popen([sys.executable, os.path.join(HERE, "worker.py")], env=env, cwd=HERE)
@app.on_event("shutdown")
def shutdown():
    if _worker:
        try: _worker.terminate()
        except Exception: pass

# ------------------------------------------------------------------ auth + context
def user_of(request: Request): return accounts.current_user(request)     # the Milestone 1 login: Bearer JWT -> the real user (401 when missing, expired, deleted or locked)
app.include_router(accounts.router)
app.state.user_of = lambda request: user_of(request)               # late-bound so hardening.py (metrics) always asks the current user_of
def ctx(u, params=None):
    """Identity and permission for one request. Raises 403 for an unknown role or a scoped role with no outlet. scope = the outlet id, or None for all outlets."""
    return {"dom": dom, "kind": KIND, "role": rbac.role_of(u), "email": u["email"], "user": u, "scope": rbac.scope_for(u), "params": params or {}}
def need(c, cap): rbac.need(c, cap)                                 # 403 when the role lacks the capability (see rbac.ROLE_CAPS)
def _catalog(): return list(V.MENU) + list(rbac.EXTRA_MENU)         # the profile and admin pages are drawn by the front end from /api/profile and /api/admin/*
def menu_for(user): return rbac.menu_for(user if isinstance(user, str) else rbac.role_of(user), _catalog())
def ok(msg="Done", **kw): return {"ok": True, "msg": msg, **kw}
def bad(msg): return JSONResponse({"ok": False, "msg": msg}, status_code=400)

@app.get("/api/public")
def public():
    s = LE.get_state("boot", {}) if _ready["db"] else {}
    return {"kind": KIND, "name": V.NAME, "stage": (s or {}).get("stage", "starting"), "gpu": {"cuda": False, "name": None}, "questions": [], "email_enabled": False, "demo": []}
@app.get("/api/me")
def me(request: Request):
    u = user_of(request); c = ctx(u); pub = {k: u.get(k) for k in ("id", "email", "username", "role", "outlet", "created_at", "last_login") if k in u}   # never the password hash
    return {"user": pub, "kind": KIND, "name": V.NAME, "menu": [{"id": i, "label": l, "icon": ic} for i, l, ic in menu_for(u)], "caps": sorted(rbac.caps_of(c["role"])), "scope": c["scope"], "langs": ["English"], "questions": []}
# ------------------------------------------------------------------ live pulse
def pulse(scope=None):
    """Live header numbers. A scoped role only gets its own outlet's tickets and health, never the chain-wide totals."""
    rt = LE.get_state("runtime", {}) or {}; boot = LE.get_state("boot", {}) or {}; d = VC.decision()
    oa = B.Q("SELECT status, severity, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed')" + (" AND target=?" if scope else "") + " GROUP BY status, severity", (scope,) if scope else ())
    n = lambda st, sv=None: int(oa[(oa.status == st) & ((oa.severity == sv) if sv else True)].n.sum()) if len(oa) else 0
    if scope and d: sh = (d.get("ev") or {}).get("health", {}).get(scope); d = {"score": sh, "risk": "High" if sh is not None and sh < 60 else "Medium" if sh is not None and sh < 80 else "Low", "id": d["id"]} if sh is not None else None
    return {"stage": boot.get("stage", "starting"), "ticks": rt.get("ticks", 0), "cycles": rt.get("cycles", 0), "last_cycle": rt.get("last_cycle"), "health": d["score"] if d else None, "risk": d["risk"] if d else None, "dec": d["id"] if d else 0,
            "open": n("open"), "critical": n("open", "Critical"), "proposed": 0, "gpu": {"cuda": True, "name": "", "used": 0, "total": 0, "llm": "off", "llm_name": "", "llm_done": 0, "llm_total": 0}, "worker_alive": bool(rt.get("last_cycle"))}
@app.get("/api/pulse")
def pulse_http(request: Request): c = ctx(user_of(request)); return pulse(c["scope"])
@app.get("/api/stream")
async def stream(request: Request):
    sc = ctx(user_of(request))["scope"]
    async def gen():
        last = None; t0 = time.time()
        while True:
            if await request.is_disconnected(): break
            try: p = await asyncio.to_thread(pulse, sc)
            except Exception as e: p = {"error": str(e)}
            k = json.dumps(p, sort_keys=True, default=str)
            if k != last or time.time() - t0 > 10: last = k; t0 = time.time(); yield f"data: {k}\n\n"
            await asyncio.sleep(1.5)
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# ------------------------------------------------------------------ pages
_TABS = {}
def _tabs_of(c, f):
    if f not in _TABS: _TABS[f] = f(c, None).get("tabs")
    return _TABS[f]
def _page(c, pid, tab):
    if pid == "copilot": return B.page("AI Copilot", "Ask about your data in plain English. Answers are exact database queries.", [{"t": "chat", "examples": _examples(c.get("scope")), "langs": ["English"]}], [], live=0)
    f = V.PAGES.get(pid)
    if not f: raise HTTPException(404, "unknown page")
    o = f(c, tab); _TABS[f] = o.get("tabs"); return o
@app.get("/api/page/{pid}")
def page(pid: str, request: Request, tab: str = "", p: str = ""):
    u = user_of(request); c = ctx(u, json.loads(p) if p else {})
    rbac.need_page(c, pid); rbac.guard_outlets(c, p, tab)                 # 403 for a page outside the role, or for a parameter naming another outlet
    t = time.time(); out = _page(c, pid, tab or None); out["id"] = pid; out["ms"] = round((time.time() - t) * 1000); return JSONResponse(B.clean(out))
@app.get("/api/detail/{eid:path}")
def detail(eid: str, request: Request):
    u = user_of(request); c = ctx(u); need(c, "view"); rbac.guard_outlets(c, eid)
    return JSONResponse(B.clean(V.detail(c, eid)))

# ------------------------------------------------------------------ tickets, cycles, policy
@app.post("/api/run_cycle")
def run_cycle(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); t = time.time(); LE.run_cycle(dom, "manual", ""); LE.audit(u["email"], "manual_cycle", "all"); return ok(f"Agents finished in {time.time() - t:.1f}s")
@app.post("/api/run_agent")
async def run_agent(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run"); a = (await request.json()).get("agent")
    if a not in dom.AGENTS: raise HTTPException(404, "unknown agent")
    await asyncio.to_thread(_run_one, a); LE.audit(u["email"], "manual_agent", a); return ok(f"{a} re-ran")
def _run_one(agent):
    cid = time.strftime("%m%d-%H%M%S") + "-page"; t = time.time(); s = {"cycle_id": cid, "trigger": "page", "request": "", "queue": [], "results": {}}
    if agent == "digest":
        for a in ("outlet", "inventory", "staff", "marketing", "audit"): s["results"][a] = dom.AGENTS[a](s)
    res = dom.AGENTS[agent](s); LE._log_run(cid, "page", res, int((time.time() - t) * 1000)); s["results"][agent] = res; LE.act(dom, {"actions": res.actions}, cid, {agent: res})

# ------------------------------------------------------------------ admin users

# ------------------------------------------------------------------ digital twin

# ------------------------------------------------------------------ data feeds

# ------------------------------------------------------------------ Franchise actions
@app.post("/api/franchise/review")
async def fr_review(request: Request):
    import uuid; u = user_of(request); c = ctx(u); need(c, "review"); d = await request.json(); tx = str(d.get("comment", "")).strip(); rbac.own_outlet(c, d.get("outlet"))
    if not tx: return bad("Write a comment first")
    src = "eng_Latn"; en = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+|\+?\d[\d -]{8,}\d", "[removed]", tx)
    try: from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer; sc = SentimentIntensityAnalyzer().polarity_scores(en)["compound"]
    except Exception: sc = (int(d["rating"]) - 3) / 2
    rt = int(d["rating"])
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO feedback(feedback_id,outlet_id,rating,comment,date,sentiment_score) VALUES(?,?,?,?,?,?)", ("FB-U" + uuid.uuid4().hex[:10], d["outlet"], rt, en, datetime.date.today().isoformat(), .6 * (rt - 3) / 2 + .4 * sc))
    LE.emit("review", d["outlet"], {"rating": rt}); return ok(f"Stored for {d['outlet']} (sentiment {sc:+.2f}{', translated from ' + src if src != 'eng_Latn' else ''}). The Sentiment agent includes it in its next cycle.", refresh=True)
@app.post("/api/franchise/audit")
async def fr_audit(request: Request):
    import uuid; u = user_of(request); c = ctx(u); need(c, "resolve"); d = await request.json(); rbac.own_outlet(c, d.get("outlet")); sc = int(d["score"])
    with LE._DB_LOCK, LE.conn() as cn: cn.execute("INSERT INTO audits(audit_id,outlet_id,audit_date,score,violations,category,status,notes) VALUES(?,?,?,?,?,?,?,?)", ("AUD-U" + uuid.uuid4().hex[:10], d["outlet"], datetime.date.today().isoformat(), sc, int((100 - sc) / 6), d["category"], "Pass" if sc >= 80 else "Warning" if sc >= 60 else "Fail", d.get("notes", "")))
    LE.emit("audit", d["outlet"], {"score": sc}); LE.audit(u["email"], "audit_logged", d["outlet"], str(sc)); await asyncio.to_thread(_run_one, "audit"); return ok("Saved and evaluated by the Audit agent.", refresh=True)

# ------------------------------------------------------------------ Freight actions
@app.post("/api/freight/quote")
async def fq_quote(request: Request):
    u = user_of(request); c = ctx(u); need(c, "quote"); d = await request.json()
    if d["o"] == d["d"]: return bad("Origin and destination must differ.")
    rid = dom.submit_request(d["o"], d["d"], d["cargo"], float(d["weight"]), d["hs"], d["customer"], "English", d.get("docs") or []); await asyncio.to_thread(dom.run_quote, rid); LE.audit(u["email"], "quote_created", str(rid), f"{d['o']}->{d['d']}")
    return ok("Quote generated", blocks=B.clean(V.quote_result(rid)))
@app.post("/api/freight/storm")
async def fq_storm(request: Request):
    PRESET_STORMS = {"Red Sea": (18.0, 40.0), "Arabian Sea": (16.0, 66.0), "Bay of Bengal": (14.0, 88.0), "South China Sea": (14.0, 114.0), "Mediterranean": (35.0, 18.0)}; u = user_of(request); c = ctx(u); need(c, "admin"); d = await request.json() if request.headers.get("content-length", "0") not in ("0", "2") else {}
    la, lo = PRESET_STORMS.get(d.get("area") or "Red Sea"); dom.inject_storm(la, lo, wind=float(d.get("wind", 24))); LE.audit(u["email"], "inject_storm", d.get("area", "Red Sea")); return ok("Storm injected. The weather agent will propose re-routes on its next cycle.", refresh=True)
@app.post("/api/freight/reroute")
async def fq_reroute(request: Request):
    u = user_of(request); c = ctx(u); need(c, "approve"); j = await request.json(); sid = j["shipment"]; opt = j.get("opt")
    idx = int(str(opt).split(".")[0]) - 1 if opt else None; act = {"type": "reroute", "target": sid}
    if idx is not None: act["opt_label"] = str(opt).split(" · ", 1)[-1]; act["opt_idx"] = idx
    with LE._DB_LOCK, LE.conn() as cn: note = dom.apply(act, cn)
    LE.audit(u["email"], "manual_reroute", sid, note); return ok(note or "no change", refresh=True)
@app.post("/api/freight/attach")
async def fq_attach(request: Request):
    u = user_of(request); c = ctx(u); need(c, "quote")
    d = await request.json(); dom.attach_doc(d["ref"], d["doc"], u["email"]); LE.audit(u["email"], "doc_attached", d["ref"], d["doc"]); return ok("Attached. Quotes re-run immediately; shipments clear on the next customs cycle.", refresh=True)
@app.post("/api/freight/readdoc")
async def fq_readdoc(request: Request, file: UploadFile = File(...)):
    u = user_of(request); need(ctx(u), "quote"); data = await file.read(); txt = ""
    try:
        if file.filename.lower().endswith(".pdf"):
            import pdfplumber
            with pdfplumber.open(io.BytesIO(data)) as pdf: txt = "\n".join((pg.extract_text() or "") for pg in pdf.pages)
        else: txt = data.decode("utf-8", errors="ignore")
    except Exception as e: return bad(f"Could not read the file: {e}")
    hs = re.findall(r"HS[- ]?\d{2,8}", txt); q = re.findall(r"QTE-L\d+", txt); wt = re.findall(r"(\d+(?:\.\d+)?)\s*(?:t|tons|kg)\b", txt)
    return ok("Read", blocks=[B.table([{"field": "HS code", "value": ", ".join(dict.fromkeys(hs)) or "-"}, {"field": "Quote reference", "value": ", ".join(dict.fromkeys(q)) or "-"}, {"field": "Weights", "value": ", ".join(wt[:3]) or "-"}], ["field", "value"], title="Fields found", h=170, key="ocr"), B.md("```\n" + txt[:2500] + "\n```")])

# ------------------------------------------------------------------ language, knowledge base

# ------------------------------------------------------------------ Copilot (agents first, then the GPU LLM; NLLB for every language) + executive brief
def _sse(ev, **kw): return f"data: {json.dumps({'ev': ev, **kw}, default=str)}\n\n"
SYSTEM = "You are the operations copilot of an agentic platform. Answer ONLY from the EVIDENCE. Quote numbers exactly as given, never invent any. Be concise: a one-line answer, then up to 5 short bullets with the entity ids. If the evidence does not answer, say so. When you use a numbered source, cite it as [1], [2]."
_BRIEF = {}
def _chat_db():
    c = sqlite3.connect(DB_PATH, timeout=30); c.execute("PRAGMA journal_mode=WAL;"); c.execute("CREATE TABLE IF NOT EXISTS copilot_chats(id TEXT PRIMARY KEY, email TEXT, title TEXT, ts TEXT, data TEXT)"); c.execute("CREATE INDEX IF NOT EXISTS ix_chat_email ON copilot_chats(email, ts)"); return c
@app.get("/api/chats")
async def chats_list(request: Request):
    u = user_of(request); need(ctx(u), "copilot")
    with _chat_db() as c: rows = c.execute("SELECT id, title, ts FROM copilot_chats WHERE email=? ORDER BY ts DESC LIMIT 60", (u["email"],)).fetchall()
    return {"ok": True, "chats": [{"id": r[0], "title": r[1], "ts": r[2]} for r in rows]}
@app.get("/api/chats/{cid}")
async def chats_get(cid: str, request: Request):
    u = user_of(request); need(ctx(u), "copilot")
    with _chat_db() as c: r = c.execute("SELECT title, data FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"])).fetchone()
    if not r: raise HTTPException(404, "chat not found")
    return {"ok": True, "title": r[0], "msgs": json.loads(r[1])}
@app.post("/api/chats/save")
async def chats_save(request: Request):
    u = user_of(request); need(ctx(u), "copilot"); d = await request.json(); cid = re.sub(r"[^a-zA-Z0-9_-]", "", str(d.get("id") or ""))[:40] or ("c" + str(int(time.time() * 1000))); msgs = d.get("msgs") or []
    if not isinstance(msgs, list) or not msgs: return bad("nothing to save")
    msgs = [{"role": m.get("role"), "text": str(m.get("text", ""))[:20000], "en": (str(m["en"])[:20000] if m.get("en") else None), "lang": m.get("lang"), "sql": str(m.get("sql") or "")[:1500]} for m in msgs[-80:]]
    title = next((m["text"] for m in msgs if m["role"] == "user"), "Chat")[:70]; now = datetime.datetime.now().isoformat(timespec="seconds")
    with _chat_db() as c:
        o = c.execute("SELECT email FROM copilot_chats WHERE id=?", (cid,)).fetchone()
        if o and o[0] != u["email"]: raise HTTPException(403, "not your chat")
        c.execute("INSERT OR REPLACE INTO copilot_chats VALUES(?,?,?,?,?)", (cid, u["email"], title, now, json.dumps(msgs)))
    return {"ok": True, "id": cid}
@app.delete("/api/chats/{cid}")
async def chats_del(cid: str, request: Request):
    u = user_of(request); need(ctx(u), "copilot")
    with _chat_db() as c: c.execute("DELETE FROM copilot_chats WHERE id=? AND email=?", (cid, u["email"]))
    return {"ok": True}
@app.post("/api/weather/refresh")
async def wx_refresh(request: Request):
    u = user_of(request); c = ctx(u); need(c, "run")
    if KIND != "Freight": raise HTTPException(404)
    import weather as WX; n, msg = await asyncio.to_thread(WX.refresh); LE.audit(u["email"], "weather_refresh", "ports", msg)
    return ok(("Real weather updated: " if n else "Could not reach Open-Meteo, simulated values stay: ") + msg, refresh=True)

# ------------------------------------------------------------------ reports, exports, profile, invitations


@app.get("/api/health")
def health(): return {"ok": True, "gpu": gpu.status()}
@app.get("/")
def index(): return FileResponse(os.path.join(STATIC, "index.html"), headers={"Cache-Control": "no-store"})
app.mount("/static", StaticFiles(directory=STATIC), name="static")


# ------------------------------------------------------------------ AI Copilot (SQL only: greetings + exact database answers)
EXAMPLES = ["Which outlet has the lowest customer satisfaction?", "How many inventory items need reordering?", "Which campaign has the best ROI?", "Show negative reviews this week", "Show failed audits"]
def _examples(scope=None): return EXAMPLES if not scope else [f"How many staff does {scope} have?", "Show inventory items below reorder level", "Show negative reviews this week", "Show failed audits"]    # a scoped role gets examples about its own outlet
GREET = re.compile(r"^\W*(hi|hello|hey|hii|good (morning|afternoon|evening)|namaste|yo)\b[\w\s!.,]*$", re.I)
THANKS = re.compile(r"^\W*(thanks|thank you|thx|ok thanks|great|cool)\b[\w\s!.,]*$", re.I)
@app.post("/api/copilot")
async def copilot(request: Request):
    u = user_of(request); c = ctx(u); need(c, "copilot"); d = await request.json(); q = str(d.get("q", "")).strip()[:800]
    sc, mask = c["scope"], rbac.masked(c["role"])                         # the Copilot only reads what this role may see: own outlet only for Store Manager and Staff, no salary or personal data for Staff and Auditor
    def _others(txt): return sorted({m.upper() for m in rbac.OUTLET_RE.findall(str(txt))} - {sc}) if sc else []
    def gen():
        t0 = time.time()
        if GREET.match(q): yield _sse("final", text="Hello! I answer questions about your live data with exact database queries. Try one of the examples below.", sql="", ms=0); yield _sse("done"); return
        if THANKS.match(q): yield _sse("final", text="You're welcome! Ask me anything else about your data.", sql="", ms=0); yield _sse("done"); return
        if _others(q): yield _sse("final", text=f"{', '.join(_others(q))} is outside your outlet scope ({sc}). Ask about {sc} instead.", sql="", ms=0); yield _sse("done"); return
        exact = None
        try: exact = AN.answer(KIND, q, sc, VC.decision(), dom, mask) or QA.answer(KIND, q, sc, mask)
        except Exception as e: print("copilot failed:", repr(e), file=sys.stderr)
        if exact and exact.get("answer") and sc and _others(exact["answer"] + (QA.md(exact["df"]) if exact.get("df") is not None else "") + str(exact.get("extra") or "")):
            exact = {"answer": f"That answer would include other outlets, which your role cannot see. Ask about {sc} instead.", "sql": "", "df": None}     # safety net for any intent that forgets the scope
        if exact and exact.get("answer"):
            tbl = (QA.md(exact["df"]) if exact.get("df") is not None else "") + (("\n\n" + exact["extra"]) if exact.get("extra") else "")
            rows = exact.get("rows", 0 if exact.get("df") is None else len(exact["df"])); yield _sse("exact", sql=exact["sql"], rows=rows)
            foot = ("\n\n**%d row%s** returned by a read-only SQL query. Open \"Show the exact query\" to see it." % (rows, "" if rows == 1 else "s")) if exact.get("sql") else ""
            final = exact["answer"] + (("\n\n" + tbl) if tbl else "") + foot; yield _sse("final", text=final, sql=exact.get("sql", ""), ms=round((time.time() - t0) * 1000))
        else: yield _sse("final", text="Sorry, I can only answer from the franchise database (outlets, staff, inventory, campaigns, reviews, audits, tickets and sales), and I could not match that question. Try one of these:\n" + "\n".join("- " + e for e in intents.suggestions(sc, q)), sql="", ms=round((time.time() - t0) * 1000))
        yield _sse("done")
    return StreamingResponse(gen(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})


Writing /content/app/server.py


In [24]:
%%writefile /content/app/validators.py
"""
Validation rules (copied from the Milestone 1 login).

Implements the field validation requirements specified by Infosys Springboard 7.0:
1. Mandatory fields verification
2. Email format rule: >= 2 chars before @, >= 2 chars between @ and dot, >= 2 chars after final dot
3. Password complexity: >= 8 chars, >= 1 uppercase, >= 1 lowercase, >= 1 digit, >= 1 special symbol
4. Password confirmation matching
"""

import re
from typing import Tuple, Dict, Any

SPECIAL_CHARACTERS = set("!@#$%^&*(),.?\":{}|<>~`_-+=/\\[];'")

def validate_mandatory_fields(fields: Dict[str, Any]) -> Tuple[bool, str]:
    """
    Checks that none of the provided form fields are empty or whitespace-only.
    """
    for field_name, value in fields.items():
        if value is None:
            return False, f"'{field_name}' is required."
        if isinstance(value, str) and not value.strip():
            return False, f"'{field_name}' cannot be empty."
    return True, ""

def validate_email_format(email: str) -> Tuple[bool, str]:
    """
    Validates email format against the official Infosys Milestone 1 rule:
    - At least 2 characters before the '@'
    - At least 2 characters between the '@' and the dot
    - At least 2 characters after the final dot
    Example compliant shape: ab@cd.ef
    """
    if not email or not isinstance(email, str):
        return False, "Email address is required."

    email = email.strip()
    if "@" not in email:
        return False, "Email must contain an '@' symbol."

    parts = email.split("@")
    if len(parts) != 2:
        return False, "Email must contain exactly one '@' symbol."

    local_part, domain_part = parts[0], parts[1]

    # Rule 1: At least 2 characters before the @
    if len(local_part) < 2:
        return False, "Email must have at least 2 characters before the '@' symbol."

    if "." not in domain_part:
        return False, "Email domain must contain at least one '.' (dot)."

    domain_subparts = domain_part.split(".")
    # Intermediate domain name (between @ and first dot) must be at least 2 characters
    if len(domain_subparts[0]) < 2:
        return False, "Email domain must have at least 2 characters between '@' and the dot."

    # Final domain extension (after final dot) must be at least 2 characters
    final_extension = domain_subparts[-1]
    if len(final_extension) < 2:
        return False, "Email domain must have at least 2 characters after the final dot."

    # Standard general syntax check
    email_regex = r"^[a-zA-Z0-9_.+-]{2,}@[a-zA-Z0-9-]+(?:\.[a-zA-Z0-9-]+)*\.[a-zA-Z]{2,}$"
    if not re.match(email_regex, email):
        return False, "Invalid email format. Format must match pattern like ab@cd.ef"

    return True, ""

def validate_password_complexity(password: str) -> Tuple[bool, str]:
    """
    Validates password against Infosys complexity requirements:
    - Minimum 8 characters
    - At least one uppercase letter
    - At least one lowercase letter
    - At least one number
    - At least one special symbol
    """
    if not password or not isinstance(password, str):
        return False, "Password is required."

    if len(password) < 8:
        return False, "Password must be at least 8 characters long."

    has_upper = any(c.isupper() for c in password)
    if not has_upper:
        return False, "Password must contain at least one uppercase letter (A-Z)."

    has_lower = any(c.islower() for c in password)
    if not has_lower:
        return False, "Password must contain at least one lowercase letter (a-z)."

    has_digit = any(c.isdigit() for c in password)
    if not has_digit:
        return False, "Password must contain at least one numeric digit (0-9)."

    has_special = any(c in SPECIAL_CHARACTERS for c in password)
    if not has_special:
        return False, "Password must contain at least one special symbol (e.g. !@#$%^&*)."

    return True, ""

def validate_password_confirmation(password: str, confirm_password: str) -> Tuple[bool, str]:
    """
    Validates that password and confirm password match exactly.
    """
    if password != confirm_password:
        return False, "Confirm Password does not match Password."
    return True, ""

def validate_username(username: str) -> Tuple[bool, str]:
    """
    Validates username format (min 3 chars, alphanumeric and underscores).
    """
    if not username or not isinstance(username, str):
        return False, "Username is required."
    username = username.strip()
    if len(username) < 3:
        return False, "Username must be at least 3 characters long."
    if not re.match(r"^[a-zA-Z0-9_]+$", username):
        return False, "Username may only contain letters, numbers, and underscores."
    return True, ""


Writing /content/app/validators.py


In [25]:
%%writefile /content/app/views_core.py
"""Shared page builders: helpers, Command Center (tickets, agents, anomalies), Knowledge Graph."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import blocks as B
import rbac
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, TONE

def can(c, cap): return cap in rbac.caps_of(c["role"])      # same table as need(): a button or form is only drawn for a role that may use it
def latest(agent):
    r = LE.conn().execute("SELECT ts,score,risk,latency_ms,summary,output,cycle_id FROM agent_runs WHERE agent=? ORDER BY run_id DESC LIMIT 1", (agent,)).fetchone()
    if not r: return None
    try: out = json.loads(r[5])
    except Exception: out = {}
    return {"ts": r[0], "score": r[1], "risk": r[2], "ms": r[3], "summary": r[4], "out": out, "cycle": r[6]}
def decision():
    d = Q("SELECT decision_id, ts, score, risk, narrative, evidence FROM decisions ORDER BY decision_id DESC LIMIT 1")
    if not len(d): return None
    try: ev = json.loads(d.evidence[0])
    except Exception: ev = {}
    return {"id": int(d.decision_id[0]), "ts": d.ts[0], "score": float(d.score[0]), "risk": d.risk[0], "narrative": d.narrative[0], "ev": ev}
def scoped(df, c, col="target"):
    return df[df[col].astype(str) == c["scope"]] if c.get("scope") and len(df) else df

# ------------------------------------------------------------------ tickets
def tk_df(c, agent=None, types=None, target=None, statuses=("open", "proposed"), limit=300):
    dom = c["dom"]; sql = "SELECT action_id id, severity, status, type, target, detail, owner, assignee, due, note, ts FROM agent_actions WHERE status IN (%s)" % ",".join("?" * len(statuses)); p = list(statuses)
    ts = types or ([t for t, a in dom.ACTION_AGENT.items() if a == agent] if agent else None)
    if ts: sql += f" AND type IN ({','.join('?' * len(ts))})"; p += list(ts)
    if target: sql += " AND target=?"; p.append(target)
    df = Q(sql + " ORDER BY CASE severity WHEN 'Critical' THEN 0 WHEN 'Warning' THEN 1 ELSE 2 END, action_id DESC LIMIT " + str(limit), p)
    return scoped(df, c)
def ticket_block(c, agent=None, types=None, title="Open tickets", target=None, key=None, limit=60):
    """Read-only list of what the agent has raised (type, target, severity, detail). There is no approval workflow in this version."""
    df = tk_df(c, agent, types, target, limit=limit)
    if not len(df): return B.callout("Nothing raised right now.", "info", title)
    return B.table(df, ["id", "ts", B.pill_col("severity"), "type", "target", "detail"], title=title, h=300, key=key or f"tk_{agent or target or 'all'}", open={"col": "target"} if False else None)
def hero(c, agent, title, blurb):
    r = latest(agent)
    if c.get("scope") and r: r = {**r, "score": None, "risk": "", "summary": "Showing your outlet only."}      # the agent's score and summary are chain-wide and name other outlets
    return B.hero(agent, title, blurb, r, can_run=can(c, "run"))
def cross(c, targets, own, title="Flagged by other agents for these entities"):
    t = [str(x) for x in list(dict.fromkeys(targets))][:60]
    if not t: return None
    df = Q("SELECT type, target, severity, detail FROM agent_actions WHERE status IN ('open','proposed') AND target IN (%s)" % ",".join("?" * len(t)), t)
    df["agent"] = df.type.map(lambda x: c["dom"].ACTION_AGENT.get(x, "engine")); df = df[df.agent != own].head(30)
    if not len(df): return None
    return B.table(df, ["target", "agent", "type", B.pill_col("severity"), "detail"], title=title, h=240, open={"col": "target"}, key="cross_" + own)

# ------------------------------------------------------------------ Command Center

# ------------------------------------------------------------------ Approvals inbox (human in the loop)
SOURCE = {"manual_request": "Approval request", "quote_approval": "Quote approval", "ml_review": "ML second opinion", "twin_mitigation": "Digital Twin"}





# ------------------------------------------------------------------ Knowledge graph
def entity_explorer(c):
    kind, dom = c["kind"], c["dom"]; e = (c.get("params") or {}).get("e")
    ents = Q("SELECT DISTINCT target FROM agent_actions WHERE status IN ('open','proposed') LIMIT 300").target.tolist()
    base = Q("SELECT outlet_id e FROM outlets ORDER BY outlet_id").e.tolist() if kind == "Franchise" else Q("SELECT port_name e FROM ports ORDER BY congestion_index DESC LIMIT 60").e.tolist()
    opts = list(dict.fromkeys(([c["scope"]] if c.get("scope") else ents + base))); e = e or opts[0]
    d = EN.describe(kind, dom, e); nodes = [{"id": e, "kind": "focus", "size": 34, "color": "#ffffff", "tip": "selected", "label": True}]; links = []
    for n, k, tip in d["rel"]: nodes.append({"id": n, "kind": k, "size": 16, "tip": tip, "label": True, "open": n if EN.kind_of(n) != "port" or kind == "Freight" else None}); links.append((e, n))
    for r in d["tickets"][d["tickets"].status.isin(["open", "proposed"])].itertuples():
        n = f"#{r.id} {r.type}"; ag = dom.ACTION_AGENT.get(r.type, "engine"); nodes += [{"id": n, "kind": "ticket", "size": 14, "color": TONE.get(r.severity), "tip": r.detail[:120], "label": True}, {"id": ag, "kind": "agent", "size": 26, "color": TEAL, "tip": "agent", "label": True}]; links += [(e, n), (n, ag)]
    seen = set(); nodes = [n for n in nodes if not (n["id"] in seen or seen.add(n["id"]))]
    sel = {"t": "form", "id": "kg_pick", "fields": [{"name": "e", "label": "Entity", "type": "select", "options": opts, "value": e}], "submit": "Explore", "url": "page:kg:explore", "title": None, "tone": "primary", "inline": True}
    info = B.table([{"field": k, "value": v} for k, v in d["info"].items()], ["field", "value"], title="What the platform knows", h=230, key="kg_info") if d["info"] else B.callout("No master record for this entity.", "info")
    tk = d["tickets"]
    return [sel, B.row(B.card("Facts", info, B.table(tk, ["id", B.pill_col("severity"), B.pill_col("status"), "type", "owner", "detail"], title="Why is it flagged", h=260, key="kg_tk")), B.card("Relations", B.chart(B.graph(nodes, links), h=470, open={"graph": True}, key="kg_ent")), ratio=[2, 3])]


Writing /content/app/views_core.py


In [26]:
%%writefile /content/app/views_franchise.py
"""FranchiseOps AI pages (server-driven blocks)."""
import json, datetime
import numpy as np, pandas as pd
import live_engine as LE, entities as EN
import domain_franchise as dom
import blocks as B
from blocks import Q, clean, L, TEAL, BLUE, AMBER, RED, VIOLET, GREEN, TONE
from views_core import can, latest, decision, ticket_block, tk_df, cross, hero, scoped

NAME = "FranchiseOps AI"
MENU = [('copilot', 'AI Copilot', 'bot'), ('workforce', 'Agent 1 · Workforce', 'users'), ('outlets', 'Agent 2 · Outlets', 'store'), ('inventory', 'Agent 3 · Inventory', 'boxes'), ('marketing', 'Agent 4 · Marketing', 'megaphone'), ('sentiment', 'Agent 5 · Sentiment', 'messages'), ('audit', 'Agent 6 · Audit', 'clipboard')]
def color_h(h): return RED if h < 60 else AMBER if h < 80 else TEAL
def _sc(c, col="outlet_id"): return (f" WHERE {col}=?", (c["scope"],)) if c.get("scope") else ("", ())
def model_card(name, keys):
    m = Q("SELECT metrics, trained_at FROM ml_models WHERE model_name=? ORDER BY model_id DESC LIMIT 1", (name,))
    if not len(m): return None
    try: d = json.loads(m.metrics[0])
    except Exception: d = {}
    return B.kpis(*[(k, d.get(k, "–")) for k in keys], ("trained", (m.trained_at[0] or "")[5:16]))

# ------------------------------------------------------------------ Command Center: Operations
def ops(c):
    d = decision(); health = (d or {}).get("ev", {}).get("health", {}) if d else {}
    g = Q("SELECT g.outlet_id, g.lat, g.lon, o.outlet_name, o.location FROM outlet_geo g JOIN outlets o USING(outlet_id)"); g = g[g.outlet_id == c["scope"]] if c.get("scope") else g
    tk = Q("SELECT target, COUNT(*) n FROM agent_actions WHERE status IN ('open','proposed') GROUP BY target").set_index("target").n; oa = tk_df(c, limit=500)
    pts = [{"id": r.outlet_id, "lat": r.lat, "lon": r.lon, "color": color_h(health.get(r.outlet_id, 70)), "r": 6 + min(10, int(tk.get(r.outlet_id, 0)) * 1.2), "tip": f"<b>{r.outlet_id}</b> {r.outlet_name}<br>{r.location}<br>health <b>{health.get(r.outlet_id, '–')}</b> · {int(tk.get(r.outlet_id, 0))} open tickets", "open": r.outlet_id} for r in g.itertuples() if r.outlet_id in health]
    mp = B.map_([{"type": "points", "id": "outlets", "pts": pts}], h=500, view={"bounds": [[7, 69], [34, 90]]}, key="ops_map", legend=[("Outlet health ≥ 80", TEAL), ("Health 60–79 (watch)", AMBER), ("Health < 60 (critical)", RED), ("Bigger dot = more open tickets", "#8b8d98")])
    hs = pd.Series(health).sort_values() if health else pd.Series(dtype=float); hs = hs[hs.index == c["scope"]] if c.get("scope") else hs
    need = pd.DataFrame({"outlet": hs.index[:10], "health": hs.values[:10]}); need["tickets"] = need.outlet.map(tk).fillna(0).astype(int)
    h = Q("SELECT substr(ts,12,8) t, score FROM decisions ORDER BY decision_id DESC LIMIT 80").iloc[::-1]; sales = Q("SELECT substr(ts,12,8) time, outlet_id, ROUND(amount) amount, source FROM sales ORDER BY sale_id DESC LIMIT 7")
    by_ag = oa.assign(agent=oa.type.map(lambda t: dom.ACTION_AGENT.get(t, "engine"))).groupby("agent").size().sort_values() if len(oa) else pd.Series(dtype=int); ex = dom.kpi_extra()
    bl = [B.kpis(("Franchise health", d["score"] if d else "…", (d or {}).get("risk", "") + " risk", "Critical" if d and d["risk"] == "High" else "Warning" if d and d["risk"] == "Medium" else "Low"), ("Open tickets", int((oa.status == "open").sum()) if len(oa) else 0), ("Critical", int(((oa.severity == "Critical") & (oa.status == "open")).sum()) if len(oa) else 0, None, "Critical"),
                  ("Awaiting approval", int((oa.status == "proposed").sum()) if len(oa) else 0, "agent proposals", "Warning"), (ex[0][0], ex[0][1]), (ex[1][0], ex[1][1]))]
    if d: bl.append({"t": "brief", "text": d["narrative"], "id": d["id"], "can_brief": True})
    bl.append(B.row(B.card("Outlet map · click an outlet", mp), B.col(B.card("Needs attention first", B.table(need, ["outlet", B.num_col("health", "health", "bar", 100), B.num_col("tickets", "tickets", "int")], h=250, open={"col": "outlet"}, key="need")),
                                                                B.card("Health trend", B.chart(B.line(h.t, h.score, ymin=0, ymax=100, colors=[TEAL]), h=150, key="trend"))), ratio=[3, 2]))
    bl.append(B.row(B.card("Live sales stream", B.table(sales, ["time", "outlet_id", B.num_col("amount", "amount", "money"), B.pill_col("source")], h=240, open={"col": "outlet_id"}, key="sales")), B.card("Open tickets by agent", B.chart(B.bar(by_ag.index, by_ag.values, horizontal=True, colors=[BLUE]), h=240, key="byag"))))
    return bl

# ------------------------------------------------------------------ detail drawer
def detail(c, e):
    k = EN.kind_of(e)
    if k != "outlet": return {"title": e, "blocks": [B.callout("No detail view for this entity.", "info")]}
    d = decision(); health = (d or {}).get("ev", {}).get("health", {}); o = Q("SELECT outlet_name, location, revenue, customer_satisfaction csat, staff_headcount hc FROM outlets WHERE outlet_id=?", (e,))
    if not len(o): return {"title": e, "blocks": [B.callout("Unknown outlet.", "warn")]}
    r = o.iloc[0]; bd = {}
    for a in ("outlet", "inventory", "staff", "marketing", "audit", "feedback"):
        v = EN.latest_out(a).get("per_outlet", {}).get(e)
        if v is not None: bd[a] = round(v, 1)
    s = Q("SELECT substr(ts,12,8) t, amount FROM sales WHERE outlet_id=? ORDER BY sale_id DESC LIMIT 50", (e,)).iloc[::-1]
    lo = Q("SELECT sku_name, current_stock stock, weekly_demand wk, lead_time_days lead FROM inventory WHERE outlet_id=? ORDER BY current_stock*1.0/MAX(weekly_demand,1) LIMIT 5", (e,))
    h = health.get(e); bl = [B.kpis(("Health", h if h is not None else "–", None, "Critical" if h is not None and h < 60 else "Warning" if h is not None and h < 80 else "Low"), ("Satisfaction", f"{r.csat:.2f}"), ("Headcount", int(r.hc)), ("Revenue", f"{r.revenue:,.0f}"))]
    if bd: bl.append(B.card("Why: score from each agent", B.chart(B.bar(list(bd), list(bd.values()), horizontal=True, max_=100, itemcolors=[color_h(v) for v in bd.values()]), h=200, key="d_bd"), sub="weights: outlet 30% · inventory 20% · staff 20% · marketing 15% · audit 15%"))
    if len(s): bl.append(B.card("Live sales stream", B.chart(B.line(s.t, s.amount, colors=[TEAL]), h=170, key="d_s")))
    bl.append(B.table(lo, ["sku_name", "stock", "wk", "lead"], title="Lowest-cover SKUs", h=190, key="d_lo"))
    if c.get("scope"): bl.append(B.callout("Stock-transfer suggestions compare you with other outlets, so they are only shown to roles that see the whole chain.", "info"))
    else:
        try:
            import alts; bl += alts.outlet_alternatives(e)
        except Exception as ex: bl.append(B.callout(f"Transfer suggestions unavailable: {ex}", "info"))
    bl.append(ticket_block(c, target=e, title=f"Tickets for {e}", key="d_tk"))
    if can(c, "run"): bl.append(B.button("rerun", "Re-run all agents now", "/api/run_cycle", "ghost"))
    return {"title": f"{e} · {r.outlet_name}", "sub": r.location, "blocks": bl}

# ------------------------------------------------------------------ agent pages
def workforce(c, tab=None):
    w, p = _sc(c); s = Q("SELECT staff_id, outlet_id, name, role, salary, overtime_hrs, job_satisfaction sat, predicted_attrition_prob p FROM staff" + w, p)
    bl = [hero(c, "staff", "Workforce agent", "Attrition risk (LogisticRegression), attendance and overtime for every employee. Feeds 20% of the outlet health score."),
          B.kpis(("Employees", f"{len(s):,}"), ("Avg attrition risk", f"{s.p.mean():.0%}"), ("High risk (>60%)", int((s.p > .6).sum()), None, "Warning"), ("Avg overtime h", round(s.overtime_hrs.mean(), 1)))]
    g = s.groupby("outlet_id").p.mean().sort_values(ascending=False).head(12); at = Q("SELECT status, COUNT(*) n FROM attendance" + w + " GROUP BY status", p)
    bl.append(B.row(B.card("Attrition probability distribution", B.chart(B.hist(s.p, 24, TEAL, pct=True), h=270, key="wf_h")), B.card("Outlets with highest average attrition risk (click)", B.chart(B.bar(g.index, (g.values * 100).round(1), horizontal=True, colors=[AMBER], pct=True), h=270, open={"bar": "y"}, key="wf_b"))))
    t = dom.attrition_table(c.get("scope"), 15); imp = dom.attrition_importance()
    bl.append(B.row(B.card("Live attendance (all recorded shifts)", B.chart(B.pie(at.set_index("status").n.to_dict(), colors={"Present": TEAL, "Late": AMBER, "Absent": RED}), h=270, key="wf_a")),
                    B.card("Highest-risk employees and their main driver", B.table(t[["staff_id", "outlet_id", "name", "role", "prob", "driver"]] if len(t) else pd.DataFrame(), ["staff_id", "outlet_id", "name", "role", B.num_col("prob", "attrition", "pct"), "driver"], h=270, open={"col": "outlet_id"}, key="wf_t"))))
    if imp: ip = pd.Series(imp).sort_values(); bl.append(B.card("What pushes attrition up (positive) or down (negative)", B.chart(B.bar(ip.index, ip.values.round(3), horizontal=True, colors=[VIOLET]), h=240, key="wf_i")))
    mc = model_card("logreg_attrition", ["accuracy", "auc", "source"]); bl += [mc] if mc else []
    x = cross(c, s.groupby("outlet_id").p.mean().sort_values(ascending=False).head(15).index, "staff"); bl += [x] if x else []
    bl.append(ticket_block(c, "staff", title="Open tickets from this agent", key="tk_staff")); return B.page("Workforce", "Agent 1 · attrition, attendance, overtime", bl, live=12)

def outlets(c, tab=None):
    r = latest("outlet"); out = (r or {}).get("out", {}); tiers = pd.Series(out.get("tiers", {})); d = decision(); health = (d or {}).get("ev", {}).get("health", {})
    o = Q("SELECT o.outlet_id, o.outlet_name, o.location, o.revenue, o.customer_satisfaction csat, o.staff_headcount hc, g.lat, g.lon FROM outlets o LEFT JOIN outlet_geo g USING(outlet_id)"); o["tier"] = o.outlet_id.map(tiers); o["health"] = o.outlet_id.map(health); o = o[o.outlet_id == c["scope"]] if c.get("scope") else o
    order = ["Excellent", "Good", "Needs Attention", "Critical"]; col = {"Excellent": TEAL, "Good": BLUE, "Needs Attention": AMBER, "Critical": RED}; tc = o.tier.value_counts().reindex(order).fillna(0)
    bl = [hero(c, "outlet", "Outlet performance agent", "KMeans (k=4) over revenue, orders and customers ranks every outlet. Drives 30% of the health score."),
          B.kpis(("Outlets", len(o)), ("Avg health", f"{o.health.mean():.1f}" if o.health.notna().any() else "…"), ("Critical tier", int((o.tier == "Critical").sum()), None, "Critical"), ("Silhouette", out.get("silhouette", "n/a")))]
    sc = {t: [{"x": a.revenue, "y": a.csat, "label": a.outlet_id, "size": 6 + a.hc / 4} for a in o[o.tier == t].itertuples()] for t in order if (o.tier == t).any()}
    bl.append(B.row(B.card("Outlets per KMeans tier", B.chart(B.bar(order, tc.values.astype(int).tolist(), itemcolors=[col[t] for t in order]), h=290, key="ou_t")), B.card("Revenue vs satisfaction (size = headcount · click a dot)", B.chart(B.scatter(sc, "revenue", "satisfaction", colors=col, sized=True), h=290, open={"scatter": True}, key="ou_s"))))
    bl.append(B.table(o.sort_values("health")[["outlet_id", "outlet_name", "location", "tier", "health", "csat", "hc"]], ["outlet_id", "outlet_name", "location", B.pill_col("tier"), B.num_col("health", "health", "bar", 100), B.num_col("csat", "csat", "num"), B.num_col("hc", "staff", "int")], title="All outlets · click a row for the drill-down", h=360, open={"col": "outlet_id"}, key="ou_tb", search=True))
    x = cross(c, o.sort_values("health").head(15).outlet_id, "outlet"); bl += [x] if x else []
    allo = o.sort_values("outlet_id").outlet_id.tolist(); picks = [p for p in ((c.get("params") or {}).get("cmp") or []) if p in allo][:5] or o.sort_values("health", ascending=False).outlet_id.tolist()[:2] + o.sort_values("health").outlet_id.tolist()[:1]
    names = ["outlet", "inventory", "staff", "marketing", "audit", "feedback"]; ser = {}; rows = []
    for p in picks:
        vals = [round(float(EN.latest_out(a).get("per_outlet", {}).get(p, 0) or 0), 1) for a in names]; ser[p] = vals; rr = o[o.outlet_id == p].iloc[0]; rows.append({"outlet": p, "location": rr.location, "health": health.get(p), "revenue": rr.revenue, "csat": rr.csat, "hc": rr.hc, **dict(zip(names, vals))})
    bl.append({"t": "form", "id": "cmp", "fields": [{"name": "cmp", "label": "Benchmark up to 5 outlets", "type": "multi", "options": allo, "value": picks}], "submit": "Compare", "url": "page:outlets", "title": None, "tone": "ghost", "inline": True})
    bl.append(B.row(B.card("Outlet benchmarking: score from each agent", B.chart(B.radar(names, ser), h=330, key="cmp_r")), B.card("Side by side", B.table(rows, ["outlet", "location", B.num_col("health", "health", "bar", 100)] + [B.num_col(a, a, "num") for a in names] + [B.num_col("revenue", "revenue", "money")], h=330, open={"col": "outlet"}, key="cmp_t")), ratio=[2, 3]))
    bl.append(ticket_block(c, "outlet", ["outlet_review", "health_alert"], title="Open tickets from this agent", key="tk_out")); return B.page("Outlets", "Agent 2 · KMeans tiers and the live health score", bl, live=12)

def inventory(c, tab=None):
    w, p = _sc(c); d = Q("SELECT outlet_id, sku_name, category, current_stock, reorder_threshold mn, weekly_demand wd, lead_time_days lead, stockout_risk_prob risk FROM inventory" + w, p)
    mult = LE.get_state("demand_mult", 1.0) or 1.0; d["cover_days"] = (d.current_stock / (d.wd.clip(lower=1) / 7 * mult)).round(1); d["short"] = d.cover_days < d.lead + 2
    po = Q("SELECT po_id, outlet_id, sku_name, qty, ts, eta_ts, status FROM purchase_orders" + w + " ORDER BY po_id DESC LIMIT 300", p)
    bl = [hero(c, "inventory", "Inventory agent", "Cover days = stock ÷ (forecast demand × live demand multiplier). Below lead time + 2 days raises a reorder and creates a purchase order."),
          B.kpis(("SKUs tracked", f"{len(d):,}"), ("Flagged short", int(d.short.sum()), None, "Warning"), ("POs in transit", int((po.status == "ordered").sum())), ("POs delivered", int((po.status == "delivered").sum())), ("Demand multiplier", f"x{mult}", "weather + holiday feed"))]
    cat = d.groupby("category").short.mean().mul(100).round(1)
    bl.append(B.row(B.card("Stock cover (days)", B.chart(B.hist(d.cover_days, 28, BLUE), h=270, key="iv_h")), B.card("% of SKUs short, by category", B.chart(B.bar(cat.index, cat.values, colors=[AMBER], pct=True), h=270, key="iv_c"))))
    bl.append(B.row(B.card("Most urgent SKUs", B.table(d[d.short].sort_values("cover_days").head(15)[["outlet_id", "sku_name", "current_stock", "cover_days", "lead", "risk"]], ["outlet_id", "sku_name", "current_stock", B.num_col("cover_days", "cover days", "num"), "lead", B.num_col("risk", "stock-out risk", "pct")], h=290, open={"col": "outlet_id"}, key="iv_u")),
                    B.card("Purchase orders created by this agent (they deliver into stock, closing the loop)", B.table(po.head(20), ["po_id", "outlet_id", "sku_name", "qty", B.pill_col("status"), "eta_ts"], h=290, open={"col": "outlet_id"}, key="iv_p"))))
    outs = sorted(d.outlet_id.unique()); sel = (c.get("params") or {}).get("o") or (outs[0] if outs else None); sel = sel if sel in outs else (outs[0] if outs else None); meth = (c.get("params") or {}).get("m") or "Prophet (if installed)"
    bl.append({"t": "form", "id": "inv_fc", "fields": [{"name": "o", "label": "Demand forecast for outlet", "type": "select", "options": outs, "value": sel}, {"name": "m", "label": "Method", "type": "select", "options": ["Prophet (if installed)", "Trend x weekday"], "value": meth}], "submit": "Forecast", "url": "page:inventory", "title": None, "tone": "ghost", "inline": True})
    if sel:
        h, f, how = dom.forecast_series(sel, 14, "trend" if meth.startswith("Trend") else None)
        if len(f): x = list(h.d.astype(str)) + list(f.d.astype(str)); bl.append(B.card(f"{sel}: daily sales and 14-day forecast ({how})", B.chart(B.line(x, {"actual": list(h.a) + [None] * len(f), "forecast": [None] * (len(h) - 1) + [h.a.iloc[-1]] + list(f.a)}, colors=[TEAL, AMBER], dashed=("forecast",), band=([None] * len(h) + list(f.lo), [None] * len(h) + list(f.hi))), h=290, key="iv_f")))
    x = cross(c, d[d.short].outlet_id.unique(), "inventory"); bl += [x] if x else []
    bl.append(ticket_block(c, "inventory", title="Open tickets from this agent", key="tk_inv")); return B.page("Inventory", "Agent 3 · cover days, reorders, forecast", bl, live=10)

def marketing(c, tab=None):
    w, p = _sc(c); m = Q("SELECT outlet_id, campaign_name, channel, budget, actual_roi, reach, conversions, start_date, end_date FROM marketing" + w, p); today = datetime.date.today().isoformat()
    m["status"] = np.where(m.end_date >= today, "live", "ended / paused"); m["roi_pct"] = ((m.actual_roi - 1) * 100).round(0); r = (latest("marketing") or {}).get("out", {})
    bl = [hero(c, "marketing", "Marketing agent", "Campaign ROI = (revenue − budget) ÷ budget, plus KMeans (k=3) customer segments. Campaigns are paused when Inventory reports a shortage."),
          B.kpis(("Campaigns", len(m)), ("Live now", int((m.status == "live").sum())), ("Avg ROI", f"{m.roi_pct.mean():.0f}%"), ("Best channel", r.get("best_channel", "…")))]
    ch = m.groupby("channel").actual_roi.mean().sub(1).mul(100).round(0).sort_values(); seg = r.get("segments", {})
    bl.append(B.row(B.card("Average ROI % by channel", B.chart(B.bar(ch.index, ch.values, horizontal=True, colors=[TEAL], pct=True), h=270, key="mk_c")), B.card("Customer segments (KMeans k=3)", B.chart(B.pie(seg), h=270, key="mk_s")) if seg else B.callout("Segments appear after the first agent run.", "info")))
    bl.append(B.table(m[m.status == "live"].sort_values("roi_pct")[["outlet_id", "campaign_name", "channel", "budget", "roi_pct", "end_date"]].head(30), ["outlet_id", "campaign_name", "channel", B.num_col("budget", "budget", "money"), B.num_col("roi_pct", "ROI %", "num"), "end_date"], title="Live campaigns (lowest ROI first)", h=300, open={"col": "outlet_id"}, key="mk_l"))
    paused = scoped(Q("SELECT target, detail, ts FROM agent_actions WHERE type='pause_promo' ORDER BY action_id DESC LIMIT 8"), c)
    if len(paused): bl.append(B.table(paused, ["target", "detail", "ts"], title="Promotions paused by the cross-agent rule (stock shortage + live campaign)", h=220, open={"col": "target"}, key="mk_p"))
    x = cross(c, m[m.status == "live"].outlet_id.unique(), "marketing"); bl += [x] if x else []
    bl.append(ticket_block(c, "marketing", title="Open tickets from this agent", key="tk_mk")); return B.page("Marketing", "Agent 4 · ROI, segments, campaign control", bl, live=12)

def sentiment(c, tab=None):
    w, p = _sc(c); f = Q("SELECT outlet_id, rating, comment, date, sentiment_score s FROM feedback" + w + " ORDER BY rowid DESC LIMIT 4000", p); r = (latest("feedback") or {}).get("out", {})
    bl = [hero(c, "feedback", "Sentiment agent", "VADER + star rating on every review, with issue categories. A review submitted here is analysed at once and picked up by the next agent cycle."),
          B.kpis(("Reviews analysed", len(f)), ("Avg rating", f"{f.rating.mean():.2f}" if len(f) else "n/a"), ("Outlets scoring < 45", sum(v < 45 for k, v in r.get("per_outlet", {}).items() if not c.get("scope") or k == c["scope"]), None, "Warning"))]
    iss = pd.Series(r.get("issues", {})).sort_values(); rd = f.rating.value_counts().sort_index()
    bl.append(B.row(B.callout("Issue categories are counted across the whole chain, so they are not shown for an outlet role.", "info") if c.get("scope") else B.card("Issue categories in reviews", B.chart(B.bar(iss.index, iss.values, horizontal=True, colors=[AMBER]), h=270, key="se_i")) if len(iss) else B.callout("Issue categories appear after the first run.", "info"), B.card("Rating distribution", B.chart(B.bar(rd.index.astype(int).astype(str), rd.values.astype(int), colors=[TEAL]), h=270, key="se_r"))))
    per = pd.Series(r.get("per_outlet", {})); per = per[per.index == c["scope"]] if c.get("scope") else per.sort_values().head(10)
    if len(per): bl.append(B.table(per.rename("score").reset_index().rename(columns={"index": "outlet"}), ["outlet", B.num_col("score", "score", "bar", 100)], title="Lowest sentiment scores (0-100)", h=250, open={"col": "outlet"}, key="se_l"))
    bl.append(B.table(f.head(14)[["outlet_id", "rating", "comment", "date"]], ["outlet_id", "rating", "comment", "date"], title="Latest reviews", h=300, open={"col": "outlet_id"}, key="se_t"))
    outs = [c["scope"]] if c.get("scope") else Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
    bl.append(B.form("review", [{"name": "outlet", "label": "Outlet", "type": "select", "options": outs, "value": outs[0]}, {"name": "rating", "label": "Rating", "type": "range", "min": 1, "max": 5, "value": 4}, {"name": "comment", "label": "Comment (any language, translated by NLLB)", "type": "textarea"}], "Submit review", "/api/franchise/review", title="Add a review", sub="Stored and analysed immediately; the agent includes it in its next cycle."))
    if can(c, "view"): bl.append(ticket_block(c, "feedback", title="Open tickets from this agent", key="tk_fb"))
    return B.page("Sentiment", "Agent 5 · reviews, VADER, issue categories", bl, live=15)

def audit(c, tab=None):
    w, p = _sc(c); a = Q("SELECT outlet_id, audit_date, score, violations, category, status, notes FROM audits" + w + " ORDER BY audit_date DESC, rowid DESC", p); last = a.drop_duplicates("outlet_id")
    bl = [hero(c, "audit", "Audit & compliance agent", "Latest inspection per outlet. Pass ≥ 80, Warning 60–79, Fail < 60. A fail opens a 72-hour corrective-action ticket and re-plans the Staff and Sentiment agents."),
          B.kpis(("Outlets audited", len(last)), ("Pass", int((last.score >= 80).sum()), None, "Low"), ("Warning", int(((last.score >= 60) & (last.score < 80)).sum()), None, "Warning"), ("Fail", int((last.score < 60).sum()), None, "Critical"))]
    cg = a.groupby("category").score.mean().round(1)
    bl.append(B.row(B.card("Latest audit scores", B.chart(B.hist(last.score, 20, TEAL), h=270, key="au_h")), B.card("Average score by category", B.chart(B.bar(cg.index, cg.values, colors=[BLUE]), h=270, key="au_c"))))
    bl.append(B.table(last.sort_values("score").head(15), ["outlet_id", "audit_date", B.num_col("score", "score", "bar", 100), "violations", "category", B.pill_col("status"), "notes"], title="Lowest-scoring outlets", h=320, open={"col": "outlet_id"}, key="au_l"))
    if can(c, "resolve"):
        outs = [c["scope"]] if c.get("scope") else Q("SELECT outlet_id FROM outlets ORDER BY outlet_id").outlet_id.tolist()
        bl.append(B.form("audit_log", [{"name": "outlet", "label": "Outlet", "type": "select", "options": outs, "value": outs[0]}, {"name": "score", "label": "Score", "type": "range", "min": 0, "max": 100, "value": 82}, {"name": "category", "label": "Category", "type": "select", "options": ["Food Safety", "Hygiene", "Documentation", "Equipment", "Staff Training"], "value": "Food Safety"}, {"name": "notes", "label": "Notes", "type": "text"}], "Save inspection", "/api/franchise/audit", title="Log an inspection", sub="Runs the Audit agent straight away; a fail opens a corrective-action ticket."))
    x = cross(c, last[last.score < 80].outlet_id, "audit"); bl += [x] if x else []
    bl.append(ticket_block(c, "audit", title="Open tickets from this agent", key="tk_au")); return B.page("Audit", "Agent 6 · inspections and compliance", bl, live=15)

PAGES = {"workforce": workforce, "outlets": outlets, "inventory": inventory, "marketing": marketing, "sentiment": sentiment, "audit": audit}


Writing /content/app/views_franchise.py


In [27]:
%%writefile /content/app/weather.py
"""Real weather for every port: Open-Meteo forecast + marine APIs (free, no key), fetched in batches and stored in wx_live.
Falls back to the simulated values already in weather_risks when the network is closed; the page says which one it is showing."""
import time, threading, datetime, json
import live_engine as LE

try: import requests
except Exception: requests = None

FORECAST = "https://api.open-meteo.com/v1/forecast"
MARINE = "https://marine-api.open-meteo.com/v1/marine"
WMO = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Rime fog", 51: "Light drizzle", 53: "Drizzle", 55: "Heavy drizzle", 56: "Freezing drizzle", 57: "Freezing drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 66: "Freezing rain", 67: "Freezing rain",
       71: "Light snow", 73: "Snow", 75: "Heavy snow", 77: "Snow grains", 80: "Rain showers", 81: "Rain showers", 82: "Violent showers", 85: "Snow showers", 86: "Snow showers", 95: "Thunderstorm", 96: "Thunderstorm, hail", 99: "Thunderstorm, heavy hail"}
_get = None   # tests can replace this with a function(url, params) -> json

def ensure():
    c = LE.conn(); c.execute("CREATE TABLE IF NOT EXISTS wx_live(port_name TEXT PRIMARY KEY, ts TEXT, wind_ms REAL, gust_ms REAL, wind_dir REAL, temp_c REAL, precip_mm REAL, wcode INT, wave_m REAL, wave_dir REAL, src TEXT)"); c.commit()

def _fetch(url, params):
    if _get: return _get(url, params)
    r = requests.get(url, params=params, timeout=10); r.raise_for_status(); return r.json()

def _listify(j): return j if isinstance(j, list) else [j]

def refresh(batch=50):
    """Fetch current weather for every port. Returns (rows_updated, message)."""
    ensure(); c = LE.conn(); ports = c.execute("SELECT port_name, lat, lon FROM ports").fetchall()
    if not ports: return 0, "no ports"
    n = 0; err = ""; ts = datetime.datetime.utcnow().isoformat(timespec="seconds")
    for i in range(0, len(ports), batch):
        chunk = ports[i:i + batch]; la = ",".join(f"{p[1]:.3f}" for p in chunk); lo = ",".join(f"{p[2]:.3f}" for p in chunk)
        try:
            f = _listify(_fetch(FORECAST, {"latitude": la, "longitude": lo, "current": "temperature_2m,precipitation,weather_code,wind_speed_10m,wind_direction_10m,wind_gusts_10m", "wind_speed_unit": "ms", "timezone": "UTC"}))
        except Exception as e:
            err = f"{type(e).__name__}: {str(e)[:120]}"; break
        try: m = _listify(_fetch(MARINE, {"latitude": la, "longitude": lo, "current": "wave_height,wave_direction", "timezone": "UTC"}))
        except Exception: m = [{}] * len(chunk)
        for p, a, b in zip(chunk, f, m + [{}] * (len(chunk) - len(m))):
            cur = (a or {}).get("current") or {}
            if cur.get("wind_speed_10m") is None: continue
            mc = (b or {}).get("current") or {}; wave = mc.get("wave_height")
            c.execute("INSERT OR REPLACE INTO wx_live VALUES(?,?,?,?,?,?,?,?,?,?,?)", (p[0], ts, cur.get("wind_speed_10m"), cur.get("wind_gusts_10m"), cur.get("wind_direction_10m"), cur.get("temperature_2m"), cur.get("precipitation") or 0.0, cur.get("weather_code"), wave, mc.get("wave_direction"), "Open-Meteo"))
            wind = float(cur["wind_speed_10m"]); lvl = "High" if wind > 15 else "Medium" if wind >= 8 else "Low"
            c.execute("INSERT OR REPLACE INTO weather_risks(port_name,current_severity,forecast,wind_speed,wave_height,temperature) VALUES(?,?,?,?,?,?)", (p[0], lvl, f"Open-Meteo live {ts[11:16]} UTC · {WMO.get(cur.get('weather_code'), '')}", wind * 3.6, float(wave) if wave is not None else 1.2, cur.get("temperature_2m")))
            n += 1
    c.commit(); LE.set_state("wx_status", {"ts": ts if n else (LE.get_state("wx_status", {}) or {}).get("ts"), "ok": n, "total": len(ports), "err": err, "tried": ts})
    return n, (err or f"{n} ports updated")

def status():
    s = LE.get_state("wx_status", {}) or {}; ok = int(s.get("ok") or 0); age = None
    if s.get("ts"):
        try: age = (datetime.datetime.utcnow() - datetime.datetime.fromisoformat(s["ts"])).total_seconds() / 60
        except Exception: pass
    live = bool(ok) and age is not None and age < 120
    return {"live": live, "ts": s.get("ts"), "age_min": age, "ok": ok, "total": s.get("total"), "err": s.get("err", ""), "tried": s.get("tried")}

def fresh_ports(max_age_min=90):
    """Ports whose real weather is recent: the simulator must not overwrite them."""
    try:
        cut = (datetime.datetime.utcnow() - datetime.timedelta(minutes=max_age_min)).isoformat(timespec="seconds")
        return {r[0] for r in LE.conn().execute("SELECT port_name FROM wx_live WHERE ts>=?", (cut,)).fetchall()}
    except Exception: return set()

_T = {"t": None}
def start(every_min=20):
    if _T["t"]: return
    def loop():
        time.sleep(5)
        while True:
            try: refresh()
            except Exception: pass
            time.sleep(every_min * 60 if status()["live"] else 120)
    _T["t"] = threading.Thread(target=loop, daemon=True); _T["t"].start()

def table():
    import pandas as pd
    ensure()
    return pd.read_sql("SELECT w.*, p.lat, p.lon FROM wx_live w JOIN ports p USING(port_name)", LE.conn())


Writing /content/app/weather.py


In [28]:
%%writefile /content/app/weather_context.py
import requests

CITY_COORDS = {
    "Mumbai": (19.0760, 72.8777), "Delhi": (28.61, 77.21), "Bangalore": (12.97, 77.59),
    "Chennai": (13.0827, 80.2707), "Hyderabad": (17.39, 78.49), "Pune": (18.52, 73.86),
    "Ahmedabad": (23.03, 72.57), "Jaipur": (26.91, 75.79), "Kolkata": (22.5726, 88.3639), "Surat": (21.1702, 72.8311)
}

def get_weather_report(city):
    try:
        lat, lon = CITY_COORDS.get(city, (19.08, 72.88))
        r = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true", timeout=5)
        if r.status_code == 200:
            cw = r.json().get("current_weather", {})
            return {"temp": cw.get("temperature", 28), "wind": cw.get("windspeed", 10), "city": city}
    except: pass
    return {"temp": 28, "wind": 10, "city": city}

def get_city_weather(city):
    return get_weather_report(city)

def get_route_weather_multiplier(origin, dest):
    try:
        w1 = get_weather_report(origin)
        w2 = get_weather_report(dest)
        avg_wind = (w1["wind"] + w2["wind"]) / 2
        return 1.0 + (avg_wind / 100)
    except:
        return 1.0


Writing /content/app/weather_context.py


In [29]:
%%writefile /content/app/worker.py
"""Background worker process: the simulator, the agent cycles, escalation and the feed threads run here so the web server (and the GPU models) never share a GIL with them."""
import os, sys, time, importlib
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config, live_engine as LE
dom = importlib.import_module("domain_franchise" if config.KIND == "Franchise" else "domain_freight")
if __name__ == "__main__":
    rt = LE.Runtime(dom, int(os.environ.get("LIVE_TICK_SEC", 5)), int(os.environ.get("LIVE_CYCLE_SEC", 60)))
    if config.KIND == "Freight":
        import weather as WX; WX.start()
    while True: time.sleep(3600)


Writing /content/app/worker.py


In [30]:
%%writefile /content/app/static/app.js
"use strict";
const $=(s,r=document)=>r.querySelector(s);
const esc=s=>String(s==null?"":s).replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
function h(tag,attrs,...kids){const e=document.createElement(tag);for(const k in(attrs||{})){const v=attrs[k];if(v==null||v===false)continue;if(k==="class")e.className=v;else if(k==="html")e.innerHTML=v;else if(k.startsWith("on"))e.addEventListener(k.slice(2),v);else e.setAttribute(k,v===true?"":v)}for(const c of kids.flat())if(c!=null&&c!==false)e.append(c.nodeType?c:document.createTextNode(c));return e}
const TONE={Critical:"#cd2b31",High:"#cd2b31",Fail:"#cd2b31",rejected:"#cd2b31","On Hold":"#cd2b31",overdue:"#cd2b31",Warning:"#ab6400",Medium:"#ab6400","Pending Approval":"#ab6400",proposed:"#793aaf",Info:"#3451b2",open:"#3451b2",Low:"#0b7a6c",Pass:"#0b7a6c",Approved:"#0b7a6c",approved:"#0b7a6c",resolved:"#0b7a6c","auto-closed":"#6b7080",live:"#0b7a6c",Healthy:"#0b7a6c",Watch:"#ab6400",Rejected:"#6b7080"};
const tone=v=>TONE[v]||(/crit|high|fail|bad|reject|hold/i.test(v)?"#cd2b31":/warn|med|pend/i.test(v)?"#ab6400":/ok|pass|low|good|appr|done|clear|live/i.test(v)?"#0b7a6c":"#6b7080");
const ICON={bot:"M12 8V4H8M4 12a2 2 0 0 1 2-2h12a2 2 0 0 1 2 2v6a2 2 0 0 1-2 2H6a2 2 0 0 1-2-2zM2 14h2M20 14h2M9 15v1M15 15v1",users:"M16 21v-2a4 4 0 0 0-4-4H6a4 4 0 0 0-4 4v2M9 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8M22 21v-2a4 4 0 0 0-3-3.9M16 3.1a4 4 0 0 1 0 7.8",store:"M3 9l1-5h16l1 5M3 9v11h18V9M3 9a3 3 0 0 0 6 0 3 3 0 0 0 6 0 3 3 0 0 0 6 0M9 20v-6h6v6",boxes:"M21 8l-9-5-9 5 9 5zM3 8v8l9 5 9-5V8M12 13v8",megaphone:"M3 11v2a1 1 0 0 0 1 1h3l8 5V5L7 10H4a1 1 0 0 0-1 1zM19 8a5 5 0 0 1 0 8",messages:"M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z",clipboard:"M9 4h6v3H9zM8 5H6a2 2 0 0 0-2 2v13a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V7a2 2 0 0 0-2-2h-2M9 13l2 2 4-4",bell:"M18 8a6 6 0 0 0-12 0c0 7-3 9-3 9h18s-3-2-3-9M13.7 21a2 2 0 0 1-3.4 0",network:"M12 2v6M12 16v6M5 6l5 4M19 6l-5 4M5 18l5-4M19 18l-5-4M12 8a4 4 0 1 0 0 8 4 4 0 0 0 0-8",flask:"M10 2v7L4 20a1 1 0 0 0 1 1h14a1 1 0 0 0 1-1l-6-11V2M8 2h8M7 15h10",upload:"M21 15v4a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2v-4M17 8l-5-5-5 5M12 3v12",shield:"M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10",radar:"M12 12l7-7M12 2a10 10 0 1 0 10 10M12 6a6 6 0 1 0 6 6M12 10a2 2 0 1 0 2 2",route:"M6 19a3 3 0 1 0 0-6 3 3 0 0 0 0 6M18 11a3 3 0 1 0 0-6 3 3 0 0 0 0 6M9 16h6a3 3 0 0 0 0-6H9a3 3 0 0 1 0-6h3",dollar:"M12 2v20M17 5H9.5a3.5 3.5 0 0 0 0 7h5a3.5 3.5 0 0 1 0 7H6",truck:"M1 3h15v13H1zM16 8h4l3 3v5h-7M5.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5M18.5 21a2.5 2.5 0 1 0 0-5 2.5 2.5 0 0 0 0 5",cloud:"M17.5 19a4.5 4.5 0 1 0-1.4-8.8A6 6 0 1 0 5 15a4 4 0 0 0 1 4z",trend:"M3 17l6-6 4 4 8-8M15 7h6v6",chart:"M3 3v18h18M7 15l4-4 3 3 5-6",check:"M22 11.1V12a10 10 0 1 1-5.9-9.1M22 4L12 14l-3-3",user:"M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2M12 11a4 4 0 1 0 0-8 4 4 0 0 0 0 8",file:"M14 2H6a2 2 0 0 0-2 2v16a2 2 0 0 0 2 2h12a2 2 0 0 0 2-2V8zM14 2v6h6M9 13h6M9 17h6"};
const icon=n=>{const s=document.createElementNS("http://www.w3.org/2000/svg","svg");s.setAttribute("viewBox","0 0 24 24");const p=document.createElementNS("http://www.w3.org/2000/svg","path");p.setAttribute("d",ICON[n]||ICON.file);s.append(p);return s};
const S={me:null,page:null,tab:null,params:{},lang:localStorage.getItem("lang")||"English",theme:localStorage.getItem("theme")==="dark"?"dark":"light",pulse:null,tree:null,timer:null,seq:0,boot:null,authScreen:"login",recoveryEmail:"",recoveryOtp:"",otpResendUntil:0,otpCountdown:null,apiNotice:""};
function applyTheme(){document.documentElement.dataset.theme=S.theme;document.querySelectorAll("[data-theme-toggle]").forEach(button=>{const next=S.theme==="dark"?"light":"dark";button.textContent=next==="dark"?"Dark mode":"Light mode";button.setAttribute("aria-label","Switch to "+next+" mode");button.setAttribute("aria-pressed",String(S.theme==="dark"));button.title="Switch to "+next+" mode"})}
function toggleTheme(){S.theme=S.theme==="dark"?"light":"dark";localStorage.setItem("theme",S.theme);applyTheme()}
function themeToggle(className){const next=S.theme==="dark"?"light":"dark";return h("button",{class:className||"btn sm theme-toggle",type:"button","data-theme-toggle":"",onclick:toggleTheme,"aria-label":"Switch to "+next+" mode","aria-pressed":String(S.theme==="dark"),title:"Switch to "+next+" mode"},next==="dark"?"Dark mode":"Light mode")}
applyTheme();
const LANGS_CODE={};
function pageIsAllowed(id){return !!(S.me&&Array.isArray(S.me.menu)&&S.me.menu.some(item=>item&&item.id===id))}
function firstAllowedPage(){const menu=S.me&&Array.isArray(S.me.menu)?S.me.menu:[];const first=menu.find(item=>item&&typeof item.id==="string"&&item.id);return first?first.id:null}
function authToken(){try{return sessionStorage.getItem("fo_token")||localStorage.getItem("fo_token")||""}catch(e){return""}}
function setToken(tok,remember){try{sessionStorage.removeItem("fo_token");localStorage.removeItem("fo_token");if(tok)(remember?localStorage:sessionStorage).setItem("fo_token",tok)}catch(e){}}
async function api(path,opt={}){const o={credentials:"same-origin",...opt};o.headers={...(opt.headers||{})};if(opt.json!==undefined){o.method=o.method||"POST";o.headers["Content-Type"]="application/json";o.body=JSON.stringify(opt.json)}const tk=authToken();if(tk)o.headers.Authorization="Bearer "+tk;
 let r;try{r=await fetch(path,o)}catch{return{ok:false,error:true,networkError:true,msg:"Unable to reach the FranchiseOps AI backend. Check that the application server is running."}}
 let d=null;try{d=await r.json()}catch(e){}
 if(r.status===401&&S.me&&!path.includes("login")){setToken("");S.me=null;S.authScreen="login";S.loginNotice=(d&&(d.detail||d.msg))||"Your session ended. Sign in again.";render();return{ok:false,msg:"Session ended"}}
 if(!d)d={ok:false,msg:r.ok?"The server returned an unreadable response.":"Error "+r.status};
 if(!r.ok){d.ok=false;d.error=true;d.status=r.status;if(!d.msg)d.msg=d.detail||("Error "+r.status)}
 return d}
function toast(msg,bad){const t=h("div",{class:bad?"bad":""},msg);$("#toast").append(t);setTimeout(()=>t.remove(),bad?7000:4500)}
const fmt={num:v=>v==null?"–":(typeof v==="number"?(Math.abs(v)>=1000?v.toLocaleString(undefined,{maximumFractionDigits:1}):+v.toFixed(2)):v),int:v=>v==null?"–":Math.round(v).toLocaleString(),money:v=>v==null?"–":"$"+Math.round(v).toLocaleString()};
function md(t){let s=esc(t||"");s=s.replace(/```([\s\S]*?)```/g,(_,c)=>"<pre>"+c.trim()+"</pre>").replace(/`([^`]+)`/g,"<code>$1</code>").replace(/\*\*([^*]+)\*\*/g,"<b>$1</b>");const L=s.split("\n");let o="",inl=false,tb=null;const row=l=>l.trim().replace(/^\||\|$/g,"").split("|").map(x=>x.trim());const endT=()=>{if(tb){o+='<div class="tbl" style="margin:8px 0"><table><thead><tr>'+row(tb[0]).map(x=>"<th>"+x+"</th>").join("")+"</tr></thead><tbody>"+tb.slice(2).map(r=>"<tr>"+row(r).map(x=>"<td>"+x+"</td>").join("")+"</tr>").join("")+"</tbody></table></div>";tb=null}};for(let i=0;i<L.length;i++){const l=L[i];if(/^\s*\|.*\|\s*$/.test(l)){if(inl){o+="</ul>";inl=false}(tb=tb||[]).push(l);continue}endT();const m=l.match(/^\s*[-*•]\s+(.*)/);if(m){if(!inl){o+="<ul>";inl=true}o+="<li>"+m[1]+"</li>"}else{if(inl){o+="</ul>";inl=false}const hd=l.match(/^(#{1,3})\s+(.*)/);o+=hd?`<h${hd[1].length}>${hd[2]}</h${hd[1].length}>`:(l.trim()?l+"<br>":"")}}endT();if(inl)o+="</ul>";return o}

// ---------------------------------------------------------------- block renderers (each returns {el, update(b)})
function mount(b,key){const f=R[b.t];if(!f){return{el:h("div",{class:"card"},"Unsupported block: "+b.t),update(){}}}const m=f(b,key);m.t=b.t;m.k=b.key||b.id||"";return m}
function mountList(parent,blocks,key){const hs=blocks.map((b,i)=>mount(b,key+"/"+i));hs.forEach(x=>parent.append(x.el));return hs}
function updList(parent,hs,blocks,key){if(hs.length!==blocks.length||hs.some((x,i)=>x.t!==blocks[i].t||x.k!==(blocks[i].key||blocks[i].id||""))){parent.replaceChildren();return mountList(parent,blocks,key)}hs.forEach((x,i)=>x.update(blocks[i]));return hs}
const R={};
R.kpis=b=>{const el=h("div",{class:"kpis"});const up=b=>{el.replaceChildren(...b.items.map(i=>h("div",{class:"kpi "+(i.tone||"")},h("div",{class:"l"},i.label),h("div",{class:"v"},typeof i.value==="number"?fmt.num(i.value):i.value),i.sub?h("div",{class:"s"},i.sub):null)))};up(b);return{el,update:up}};
R.callout=b=>{const el=h("div");const up=b=>{el.className="callout "+(b.tone||"info");el.replaceChildren(h("div",null,b.title?h("b",null,b.title+" "):null,h("span",{html:md(b.text)})))};up(b);return{el,update:up}};
R.md=b=>{const el=h("div",{class:"md"});const up=b=>el.innerHTML=md(b.text);up(b);return{el,update:up}};
R.row=(b,k)=>{const el=h("div",{class:"row"});const sty=r=>el.style.gridTemplateColumns=(r||b.cols.map(()=>1)).map(x=>x+"fr").join(" ");sty(b.ratio);let hs=[];const wrap=[];const up=nb=>{sty(nb.ratio);if(hs.length!==nb.cols.length||hs.some((x,i)=>x.t!==nb.cols[i].t)){el.replaceChildren();hs=nb.cols.map((c,i)=>{const m=mount(c,k+"/"+i);el.append(m.el);return m})}else hs.forEach((x,i)=>x.update(nb.cols[i]))};up(b);return{el,update:up}};
R.col=(b,k)=>{const el=h("div",{class:"col"});let hs=mountList(el,b.blocks,k);return{el,update:nb=>{hs=updList(el,hs,nb.blocks,k)}}};
R.card=(b,k)=>{const el=h("div",{class:"card"});const hd=h("h3"),sub=h("div",{class:"sub"}),body=h("div");el.append(hd,sub,body);let hs=[];const up=(nb,first)=>{hd.textContent=nb.title||"";hd.style.display=nb.title?"":"none";sub.textContent=nb.sub||"";sub.style.display=nb.sub?"":"none";hs=first?mountList(body,nb.blocks,k):updList(body,hs,nb.blocks,k)};up(b,true);return{el,update:nb=>up(nb)}};
R.hero=b=>{const el=h("div",{class:"hero"});const up=b=>{const c=b.risk?tone(b.risk):"#8b8d98";el.replaceChildren(h("div",{class:"sc",style:"color:"+c},b.score==null?"–":Math.round(b.score)),h("div",{style:"flex:1"},h("h2",null,b.title," ",b.risk?h("span",{class:"pill",style:`color:${c};border-color:${c}`},b.risk):null),h("p",null,b.blurb),h("p",{style:"color:#454a58"},b.summary||"No run yet."),h("p",null,b.ts?`last run ${b.ts.slice(11,19)} · ${b.ms} ms`:"")),b.can_run?h("button",{class:"btn primary",onclick:async e=>{e.target.disabled=true;const r=await api("/api/run_agent",{json:{agent:b.agent}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage()}},"Run now"):null)};up(b);return{el,update:up}};
R.brief=b=>{const el=h("div",{class:"card"});const out=h("div",{class:"md"});const up=b=>{out.dataset.def=b.text;if(!out.dataset.ai)out.innerHTML=md(b.text)};el.append(h("h3",null,"Executive briefing"),out,b.can_brief?h("button",{class:"btn sm",style:"margin-top:8px",onclick:async e=>{e.target.disabled=true;out.dataset.ai="1";out.textContent="";let acc="";await sse("/api/brief",{lang:S.lang},ev=>{if(ev.ev==="tok"){acc+=ev.text;out.innerHTML=md(acc)}if(ev.ev==="final")out.innerHTML=md(ev.text)});e.target.disabled=false}},"Write with the GPU model"):null);up(b);return{el,update:up}};
R.button=b=>{const el=h("button",{class:"btn "+(b.tone||"primary"),onclick:async()=>{if(b.payload&&b.payload.download){window.open(b.url,"_blank");return}if(b.confirm&&!confirm(b.confirm))return;el.disabled=true;const r=await api(b.url,{json:b.payload||{}});toast(r.msg||"Done",!r.ok);el.disabled=false;if(r.refresh)reloadPage()}},b.label);return{el,update(){}}};
R.table=b=>{const el=h("div"),title=h("h3",{style:"margin:0 0 6px;font-size:13.5px"}),sub=h("div",{class:"sub"}),srch=h("input",{class:"srch",placeholder:"Search…",oninput:()=>{st.q=srch.value.toLowerCase();draw()}}),box=h("div",{class:"tbl"}),st={sort:null,dir:1,q:""};let cur=b;el.append(title,sub,srch,box);
 const draw=()=>{const b=cur;title.textContent=b.title||"";title.style.display=b.title?"":"none";sub.textContent=b.sub||"";sub.style.display=b.sub?"":"none";srch.style.display=b.search?"":"none";if(b.h)box.style.maxHeight=b.h+"px";
  let rows=b.rows;if(st.q)rows=rows.filter(r=>Object.values(r).some(v=>String(v).toLowerCase().includes(st.q)));if(st.sort)rows=[...rows].sort((x,y)=>{const a=x[st.sort],c=y[st.sort];return(a==null)-(c==null)||(a>c?1:a<c?-1:0)*st.dir});
  const mx={};b.cols.forEach(c=>{if(c.type==="pct"){mx[c.k]=Math.max(...b.rows.map(r=>Math.abs(r[c.k]||0)))<=1.0001?100:1}});
  const th=h("tr",null,...b.cols.map(c=>h("th",{onclick:()=>{st.dir=st.sort===c.k?-st.dir:1;st.sort=c.k;draw()}},c.label+(st.sort===c.k?(st.dir>0?" ▲":" ▼"):""))));
  const tb=rows.slice(0,400).map(r=>{const tr=h("tr",{class:b.open?"cl":""},...b.cols.map(c=>{const v=r[c.k];const t=c.type||"text";let n;
    if(t==="pill"&&v!=null&&v!=="")n=h("span",{class:"pill",style:`color:${tone(v)};border-color:${tone(v)}55;background:${tone(v)}18`},v);
    else if(t==="bar"){const m=c.max||100,p=Math.max(0,Math.min(100,(v||0)/m*100)),col=c.k.match(/risk|prob|congest|ci|index/i)?(p>66?"#e5484d":p>40?"#f5a524":"#12a594"):(p<60?"#e5484d":p<80?"#f5a524":"#12a594");n=h("div",{class:"bar"},h("span",null,v==null?"–":fmt.num(v)),h("i",null,h("b",{style:`width:${p}%;background:${col}`})))}
    else if(t==="pct")n=document.createTextNode(v==null?"–":(+(v*mx[c.k]).toFixed(1))+"%");else if(t==="money")n=document.createTextNode(fmt.money(v));else if(t==="int")n=document.createTextNode(fmt.int(v));else if(t==="num")n=document.createTextNode(fmt.num(v));else n=document.createTextNode(v==null?"":v);
    return h("td",{title:typeof v==="string"&&v.length>40?v:null},n)}));if(b.open)tr.onclick=()=>UI.openEntity(r[b.open.col]);return tr});
  const sc=box.scrollTop;box.replaceChildren(h("table",null,h("thead",null,th),h("tbody",null,...tb)));box.scrollTop=sc;if(!rows.length)box.append(h("div",{class:"sub",style:"padding:12px"},"No rows."))};
 draw();return{el,update:nb=>{cur=nb;draw()}}};
R.tickets=b=>{const el=h("div"),st={f:"all",open:new Set()};let cur=b;const body=h("div"),fl=h("div",{class:"filters"});el.append(h("h3",{style:"margin:0 0 6px;font-size:13.5px"},b.title||"Tickets"),h("div",{class:"sub"},b.sub||""),fl,body);
 const act=async(id,a,json)=>{const r=await api(`/api/ticket/${id}/${a}`,{json:json||{}});toast(r.msg,!r.ok);if(document.activeElement)document.activeElement.blur();st.sig=null;reloadPage(true)};
 const draw=()=>{const items=cur.items;st.sig=JSON.stringify(items)+st.f+[...st.open];const cnt=k=>k==="all"?items.length:k==="proposed"?items.filter(i=>i.status==="proposed").length:k==="overdue"?items.filter(i=>i.overdue).length:items.filter(i=>i.sev===k).length;
  fl.replaceChildren(...["all","proposed","Critical","Warning","Info","overdue"].map(k=>h("span",{class:"chip"+(st.f===k?" on":""),onclick:()=>{st.f=k;draw()}},(k==="proposed"?"awaiting approval":k)+" · "+cnt(k))));
  const list=items.filter(i=>st.f==="all"||(st.f==="proposed"?i.status==="proposed":st.f==="overdue"?i.overdue:i.sev===st.f));
  body.replaceChildren(...list.slice(0,80).map(i=>{const c=tone(i.sev);const t=h("div",{class:"tk "+i.sev+" "+i.status},
   h("div",{class:"h"},h("b",null,"#"+i.id+" "+i.type.replace(/_/g," ")),h("span",{class:"pill",style:`color:${c};border-color:${c}66`},i.sev),i.status==="proposed"?h("span",{class:"pill",style:"color:#8e4ec6;border-color:#8e4ec666"},"awaiting approval"):null,i.overdue?h("span",{class:"pill",style:"color:#e5484d;border-color:#e5484d66"},"overdue"):null,h("a",{href:"#",onclick:e=>{e.preventDefault();UI.openEntity(i.target)}},i.target)),
   h("div",{class:"d"},i.detail),h("div",{class:"m"},h("span",null,"agent: "+i.agent),h("span",null,"owner: "+(i.owner||"–")),h("span",null,"assignee: "+(i.assignee||"unassigned")),h("span",null,"due "+(i.due||"–")),h("span",null,i.ts)),
   i.chain&&i.chain.length?h("div",{class:"m"},h("span",null,"approval chain: "),...i.chain.map(s=>h("span",{class:"pill",style:`color:${s.status==="approved"?"#1a9e6e":s.status==="rejected"?"#e5484d":"#8e4ec6"};border-color:currentColor`},"L"+s.step+" "+s.role+(s.status==="approved"?" ✓ "+s.by:s.status==="rejected"?" ✗":" …")))):null,
   i.status==="proposed"&&i.turn&&!i.can_approve?h("div",{class:"m"},"Waiting for a "+i.turn+" to decide"):null,
   i.note?h("div",{class:"m"},"note: "+i.note):null,
   h("div",{class:"a"},i.status==="proposed"&&i.can_approve?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve & execute"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.quote&&i.can_approve&&i.status!=="proposed"?[h("button",{class:"btn ok sm",onclick:()=>act(i.id,"approve")},"Approve quote"),h("button",{class:"btn danger sm",onclick:()=>act(i.id,"reject")},"Reject")]:null,
    i.can_resolve?[h("button",{class:"btn sm",onclick:()=>act(i.id,"resolve")},"Resolve"),(()=>{const a=h("input",{class:"in",style:"width:170px",placeholder:"assign to email",value:i.assignee||""});return h("span",{style:"display:flex;gap:4px"},a,h("button",{class:"btn sm",onclick:()=>a.value&&act(i.id,"assign",{assignee:a.value})},"Assign"))})()]:null,
    h("button",{class:"btn sm",onclick:()=>{st.open.has(i.id)?st.open.delete(i.id):st.open.add(i.id);draw()}},"Comments ("+i.comments.length+")")),
   st.open.has(i.id)?h("div",{class:"cm"},...i.comments.map(c=>h("div",null,h("b",null,c.by),` ${c.ts}: ${c.body}`)),(()=>{const t=h("input",{class:"in",placeholder:"Write a comment and press Enter",onkeydown:e=>{if(e.key==="Enter"&&t.value.trim())act(i.id,"comment",{body:t.value})}});return t})()):null);return t}));if(!list.length)body.append(h("div",{class:"sub"},"Nothing here."))};
 draw();return{el,update:nb=>{if(el.contains(document.activeElement)&&document.activeElement.tagName==="INPUT"){cur=nb;return}if(JSON.stringify(nb.items)+st.f+[...st.open]===st.sig)return;cur=nb;draw()}}};
// ---- charts
const FN={size3:v=>Math.max(8,Math.min(34,Math.sqrt(Math.abs(v[3]||10))*1.6)),scatterTip:p=>`<b>${esc(p.value[2]||"")}</b><br>${esc(p.seriesName||"")}<br>${p.value[0]}, ${p.value[1]}`,nodeTip:p=>p.dataType==="node"?`<b>${esc(p.name)}</b><br>${esc(p.data.tip||"")}`:"",nodeLabel:p=>p.data.showLabel?p.name:""};
function revive(o){if(Array.isArray(o))return o.map(revive);if(o&&typeof o==="object"){if(o.__fn)return FN[o.__fn];const r={};for(const k in o)r[k]=revive(o[k]);return r}return o}
echarts.registerTheme("lt",{color:["#0f766e","#2563eb","#f59e0b","#7c3aed","#e5484d","#16a34a","#db2777","#0891b2","#64748b"],backgroundColor:"transparent",textStyle:{fontFamily:"Inter, system-ui, sans-serif",color:"#454a58"},legend:{textStyle:{color:"#454a58"}},categoryAxis:{axisLine:{lineStyle:{color:"#d9dce8"}},axisTick:{show:false},axisLabel:{color:"#6b7080"},splitLine:{lineStyle:{color:"#eceef6"}}},valueAxis:{axisLine:{show:false},axisTick:{show:false},axisLabel:{color:"#6b7080"},splitLine:{lineStyle:{color:"#eceef6"}}},tooltip:{backgroundColor:"#fff",borderColor:"#e6e8f0",textStyle:{color:"#1c2024"},extraCssText:"box-shadow:0 8px 24px rgba(20,24,40,.12);border-radius:10px"}});
const CH=[];function lazy(box,fn){let d=0;const t=()=>{if(d)return;if(box.isConnected&&box.clientWidth>0){d=1;fn()}else setTimeout(t,60)};t()}
R.chart=b=>{const el=h("div"),box=h("div",{style:`height:${b.h||290}px`});el.append(h("h3",{style:"margin:0 0 4px;font-size:13.5px;display:"+(b.title?"block":"none")},b.title||""),box);let ch=null,cur=b;
 const init=()=>{ch=echarts.init(box,"lt",{renderer:"canvas"});CH.push(ch);ch.setOption(revive(cur.opt),true);
  ch.on("click",p=>{const o=cur.open;if(!o)return;let id=null;if(o.bar)id=p.name;else if(o.scatter)id=p.value&&p.value[2];else if(o.graph)id=p.data&&p.data.open;else if(p.data&&p.data.open)id=p.data.open;if(id)UI.openEntity(id)});
  if(cur.open)box.style.cursor="pointer"};
 lazy(box,init);
 return{el,update:nb=>{cur=nb;if(ch){const r=JSON.stringify(nb.opt);if(box._h!==r){box._h=r;ch.setOption(revive(nb.opt),{replaceMerge:["series","xAxis","yAxis"]})}}},};};
window.addEventListener("resize",()=>CH.forEach(c=>{try{c.resize()}catch(e){}}));
// ---- map
function shiftGeo(g,sh){const mv=c=>typeof c[0]==="number"?[c[0]+sh,c[1]]:c.map(mv);return{type:"FeatureCollection",features:g.features.map(f=>({type:"Feature",properties:f.properties,geometry:{type:f.geometry.type,coordinates:mv(f.geometry.coordinates)}}))}}
let WORLD=null;async function world(){if(WORLD)return WORLD;try{const t=await(await fetch("/static/vendor/world.topo.json")).json();WORLD=topojson.feature(t,t.objects.countries)}catch(e){WORLD={type:"FeatureCollection",features:[]}}return WORLD}
R.map=b=>{const el=h("div"),box=h("div",{class:"map",style:`height:${b.h||470}px`}),lg=h("div",{class:"lg"});el.append(box,lg);let M=null,grp=null,cur=b;
 const draw=()=>{grp.clearLayers();const SH=[-360,0,360];cur.layers.forEach(L=>{if(L.type==="lines")L.lines.forEach(l=>{if(l.pts.length<2)return;SH.forEach(sh=>{const p=L2.polyline(l.pts.map(q=>[q[0],q[1]+sh]),{color:l.color||"#0f766e",weight:l.w||2,dashArray:l.dash||null,opacity:.9,lineJoin:"round",renderer:M._r});if(l.tip)p.bindTooltip(l.tip,{sticky:true});grp.addLayer(p)})});
   else if(L.type==="circles")L.items.forEach(c=>SH.forEach(sh=>{const k=L2.circle([c.lat,c.lon+sh],{radius:c.r_m,color:c.color||"#e5484d",weight:1,fillOpacity:.18,renderer:M._r});if(c.tip)k.bindTooltip(c.tip);grp.addLayer(k)}));
   else if(L.type==="points")[...L.pts].sort((a,c)=>(a.z||0)-(c.z||0)).forEach(p=>SH.forEach(sh=>{const m=L2.circleMarker([p.lat,p.lon+sh],{radius:p.r||5,color:"#ffffff",weight:1.2,fillColor:p.color||"#0f766e",fillOpacity:.95,renderer:M._r});if(p.tip)m.bindTooltip(p.tip,{sticky:true});const o=p.open||(cur.open?p.id:null)||p.id;if(o)m.on("click",()=>UI.openEntity(o));grp.addLayer(m)}))});
  lg.replaceChildren(...(cur.legend||[]).map(([t,c])=>h("span",null,h("i",{style:"background:"+c}),t)))};
 const L2=L;const go=async()=>{M=L2.map(box,{zoomControl:true,worldCopyJump:true,attributionControl:false,minZoom:1,preferCanvas:true});M._r=L2.canvas({padding:.5});const w=await world();[-360,0,360].forEach(sh=>L2.geoJSON(sh?shiftGeo(w,sh):w,{renderer:M._r,style:{color:"#b9b2a0",weight:.8,fillColor:"#f3efe6",fillOpacity:1},interactive:false}).addTo(M));grp=L2.layerGroup().addTo(M);
  const v=cur.view||{};if(v.bounds)M.fitBounds(v.bounds);else M.setView(v.center||[20,0],v.zoom||2);draw();setTimeout(()=>M.invalidateSize(),50)};
 lazy(box,go);
 return{el,update:nb=>{cur=nb;if(M&&grp)draw()}}};
// ---- forms
R.form=b=>{const el=h("div",{class:b.title||b.inline?"":""});const res=h("div",{class:"res"});const f=h("form",{class:"f"+(b.inline?" inline":"")});let ins={};
 b.fields.forEach(fl=>{const w=h("div",{class:"fld"+(fl.type==="textarea"||fl.type==="checks"||fl.type==="multi"?" wide":"")});if(fl.type!=="hidden")w.append(h("label",null,fl.label||fl.name));let i;
  if(fl.type==="select")i=h("select",{name:fl.name},...fl.options.map(o=>h("option",{value:o,selected:String(o)===String(fl.value)},o)));
  else if(fl.type==="multi"){i=h("select",{name:fl.name,multiple:true,size:5},...fl.options.map(o=>h("option",{value:o,selected:(fl.value||[]).includes(o)},o)))}
  else if(fl.type==="checks"){i=h("div",{class:"chk"},...fl.options.map(o=>h("label",null,h("input",{type:"checkbox",value:o,checked:(fl.value||[]).includes(o)}),o)));i.get=()=>[...i.querySelectorAll("input:checked")].map(x=>x.value)}
  else if(fl.type==="textarea")i=h("textarea",{name:fl.name,placeholder:fl.placeholder||""},fl.value||"");
  else if(fl.type==="range"){const v=h("span",{style:"color:#12a594"},fl.value);i=h("input",{type:"range",name:fl.name,min:fl.min,max:fl.max,step:fl.step||1,value:fl.value,oninput:e=>v.textContent=e.target.value});w.firstChild.append(" ",v)}
  else if(fl.type==="file")i=h("input",{type:"file",name:fl.name,accept:fl.accept||null});
  else if(fl.type==="hidden"){i=h("input",{type:"hidden",name:fl.name,value:fl.value});w.style.display="none"}
  else i=h("input",{type:fl.type==="number"?"number":fl.type==="date"?"date":fl.type==="password"?"password":"text",name:fl.name,value:fl.value==null?"":fl.value,min:fl.min,max:fl.max,step:fl.step||(fl.type==="number"?"any":null),placeholder:fl.placeholder||""});
  i.fl=fl;ins[fl.name]=i;w.append(i);f.append(w)});
 const btn=h("button",{class:"btn "+(b.tone||"primary"),type:"submit"},b.submit||"Submit");f.append(h("div",{class:"fld"},btn));
 f.addEventListener("submit",async e=>{e.preventDefault();const v={};let fd=null;for(const n in ins){const i=ins[n],fl=i.fl;if(fl.type==="file"){if(i.files[0]){fd=fd||new FormData();fd.append(n,i.files[0])}}else if(fl.type==="checks")v[n]=i.get();else if(fl.type==="multi")v[n]=[...i.selectedOptions].map(o=>o.value);else if(fl.type==="number"||fl.type==="range")v[n]=i.value===""?null:+i.value;else v[n]=i.value}
  if(b.url.startsWith("get:")){const q=new URLSearchParams();for(const n in v)if(v[n]!==""&&v[n]!=null)q.set(n,v[n]);window.open(b.url.slice(4)+"?"+q,"_blank");return}
  if(b.url.startsWith("page:")){const[,id,tab]=b.url.split(":");UI.go(id,tab||S.tab,v);return}
  btn.disabled=true;const t0=btn.textContent;btn.textContent="Working…";let r;
  if(fd){for(const n in v)fd.append(n,Array.isArray(v[n])?v[n].join(","):v[n]);r=await api(b.url,{method:"POST",body:fd})}else r=await api(b.url,{json:v});
  btn.disabled=false;btn.textContent=t0;toast(r.msg||(r.ok?"Done":"Failed"),!r.ok);res.replaceChildren();if(r.blocks){const hs=mountList(res,r.blocks,"res");}if(r.refresh)reloadPage(true)});
 if(b.title||b.sub){const c=h("div",{class:"card"},b.title?h("h3",null,b.title):null,b.sub?h("div",{class:"sub"},b.sub):null,f,res);el.append(c)}else el.append(f,res);return{el,update(){}}};
// ---- chat
function setChatState(chat,state,message){
 if(!chat||!chat.isConnected)return;
 let status=chat.querySelector(".chat-status");
 if(!status){status=h("div",{class:"chat-status",role:"status","aria-live":"polite"});chat.querySelector(".cin")?.before(status)}
 chat.dataset.state=state;chat.setAttribute("aria-busy",String(state==="loading"));
 status.className="chat-status"+(state==="error"?" is-error":state==="complete"?" is-complete":"");
 status.textContent=message;
 chat.querySelectorAll(".cin input,.cin button").forEach(control=>{
  if(state==="loading"){if(control.dataset.wasDisabled===undefined)control.dataset.wasDisabled=String(control.disabled);control.disabled=true}
  else if(control.dataset.wasDisabled!==undefined){control.disabled=control.dataset.wasDisabled==="true";delete control.dataset.wasDisabled}
 });
}
function showChatError(chat,message){
 if(!chat||!chat.isConnected)return;
 const bubble=chat.querySelector(".msgs .m.a:last-child");
 if(bubble)bubble.append(h("p",{class:"chat-error",role:"alert"},message));
 setChatState(chat,"error",message);
}
async function sse(url,json,on){
 const chat=url==="/api/copilot"?document.querySelector(".chat"):null;
 if(chat)setChatState(chat,"loading","Connecting to Copilot…");
 let completed=false;
 try{
  const r=await fetch(url,{method:"POST",credentials:"same-origin",headers:{"Content-Type":"application/json",...(authToken()?{Authorization:"Bearer "+authToken()}:{})},body:JSON.stringify(json)});
  if(!r.ok){
   const message=`Copilot could not complete the request (HTTP ${r.status}). Please try again.`;
   if(chat){showChatError(chat,message);return}
   on({ev:"final",text:"Error "+r.status});return;
  }
  if(chat)setChatState(chat,"loading","Copilot is responding…");
  const rd=r.body.getReader(),dec=new TextDecoder();let buf="";
  for(;;){
   const{value,done}=await rd.read();if(done)break;
   buf+=dec.decode(value,{stream:true});let i;
   while((i=buf.indexOf("\n\n"))>=0){
    const m=buf.slice(0,i);buf=buf.slice(i+2);
    if(m.startsWith("data: ")){
     const ev=JSON.parse(m.slice(6));if(ev.ev==="final")completed=true;on(ev);
     if(chat&&ev.ev==="tok")setChatState(chat,"loading","Generating a response…");
    }
   }
  }
  if(chat){
   if(completed)setChatState(chat,"complete","Response complete.");
   else showChatError(chat,"The response stream ended before completion. Please try again.");
  }
 }catch(error){
  if(chat){showChatError(chat,"Connection to Copilot failed. Check your connection and try again.");return}
  throw error;
 }
}
R.chat=b=>{let cid=null,hist=[];const el=h("div",{class:"card chat"}),msgs=h("div",{class:"msgs"}),inp=h("input",{placeholder:"Ask about outlets, staff, inventory, campaigns, reviews or audits",onkeydown:e=>{if(e.key==="Enter")send(inp.value)}});
 const lang=h("select",{class:"in",style:"width:auto",onchange:()=>{S.lang=lang.value;localStorage.setItem("lang",S.lang)}},...b.langs.map(l=>h("option",{value:l,selected:l===S.lang},l)));
 const hsel=h("select",{class:"in",style:"width:auto;max-width:260px",onchange:()=>{if(hsel.value)openChat(hsel.value)}},h("option",{value:""},"Previous chats…"));
 const listChats=async()=>{const r=await api("/api/chats");if(!r.chats)return;hsel.replaceChildren(h("option",{value:""},r.chats.length?`Previous chats (${r.chats.length})`:"No saved chats yet"),...r.chats.map(c=>h("option",{value:c.id,selected:c.id===cid},c.title+" · "+(c.ts||"").replace("T"," ").slice(5,16))));return r.chats};
 const showMsg=m=>{if(m.role==="user")msgs.append(h("div",{class:"m u"},m.text));else{const o=h("div",{class:"md"});o.innerHTML=md(m.text);const a=h("div",{class:"m a"},o);if(m.en){const tg=h("a",{class:"lnk",style:"cursor:pointer;font-size:12px",onclick:()=>{const on=tg._e=!tg._e;o.innerHTML=md(on?m.en:m.text);tg.textContent=on?"Show "+(m.lang||"translation"):"Show English original"}},"Show English original");a.append(h("div",{class:"st"},"Translated with NLLB-200 ",tg))}if(m.sql)a.append(h("details",{class:"cites"},h("summary",null,"Show the exact query"),h("pre",{style:"white-space:pre-wrap;font-size:11px"},m.sql)));msgs.append(a)}};
 const openChat=async id=>{const r=await api("/api/chats/"+encodeURIComponent(id));if(!r.msgs)return;cid=id;hist=r.msgs;msgs.replaceChildren();hist.forEach(showMsg);msgs.lastChild&&msgs.lastChild.scrollIntoView({block:"nearest"});listChats()};
 const newChat=()=>{cid=null;hist=[];msgs.replaceChildren();hsel.value="";inp.focus()};
 const delChat=async()=>{if(!cid||!confirm("Delete this chat?"))return;await api("/api/chats/"+encodeURIComponent(cid),{method:"DELETE"});newChat();listChats()};
 const saveChat=async()=>{const r=await api("/api/chats/save",{json:{id:cid,msgs:hist}});if(r.id){cid=r.id;listChats()}};
 listChats().then(cs=>{if(cs&&cs.length&&!hist.length)openChat(cs[0].id)});
 const send=async q=>{q=q.trim();if(!q)return;inp.value="";hist.push({role:"user",text:q});msgs.append(h("div",{class:"m u"},q));const a=h("div",{class:"m a"}),stg=h("div"),tr=h("div"),out=h("div",{class:"md"});a.append(stg,tr,out);msgs.append(a);let acc="";const sc=()=>{msgs.lastChild.scrollIntoView({block:"nearest"})};
  await sse("/api/copilot",{q,lang:S.lang},ev=>{if(ev.ev==="stage"){stg.append(h("div",{class:"st"},ev.text))}else if(ev.ev==="cites"){const d=h("details",{class:"cites"},h("summary",null,`Sources (${ev.rows.length})`),...ev.rows.map(r=>h("div",{class:"st",title:r.text},h("b",null,`[${r.n}] `),`${r.title} · ${r.source} · ${r.why||r.score}`)));stg.append(d)}else if(ev.ev==="trace"){tr.replaceChildren(h("div",{class:"tbl",style:"margin:6px 0"},h("table",null,h("tbody",null,...ev.rows.map(r=>h("tr",null,h("td",null,r.agent),h("td",null,h("span",{class:"pill",style:`color:${tone(r.risk)};border-color:${tone(r.risk)}55`},r.risk)),h("td",{style:"white-space:normal"},r.finding)))))))}else if(ev.ev==="tok"){acc+=ev.text;out.innerHTML=md(acc)}else if(ev.ev==="final"){out.innerHTML=md(ev.text);hist.push({role:"assistant",text:ev.text,en:ev.en,lang:ev.lang,sql:ev.sql});saveChat();if(ev.en){const tg=h("a",{class:"lnk",style:"cursor:pointer;font-size:12px;margin-left:8px",onclick:()=>{const on=tg._e=!tg._e;out.innerHTML=md(on?ev.en:ev.text);tg.textContent=on?"Show "+(ev.lang||"translation"):"Show English original"}},"Show English original");stg.append(h("div",{class:"st"},"Translated with NLLB-200 ",tg))}if(ev.sql)stg.append(h("details",{class:"cites"},h("summary",null,"Show the exact query"),h("pre",{style:"white-space:pre-wrap;font-size:11px"},ev.sql)));stg.append(h("div",{class:"st"},`done in ${(ev.ms/1000).toFixed(1)}s`+(ev.tok_s?` · ${ev.tok_s} tok/s`:"")))}sc()})};
 el.append(h("div",{style:"display:flex;gap:8px;align-items:center;flex-wrap:wrap"},h("button",{class:"btn sm",onclick:newChat},"+ New chat"),hsel,h("button",{class:"btn sm ghost",onclick:delChat,title:"Delete this chat"},"Delete"),h("b",{style:"margin-left:8px"},"Answer language"),lang,null),msgs,h("div",{class:"filters"},...b.examples.map(e=>h("span",{class:"chip",onclick:()=>send(e)},e))),h("div",{class:"cin"},inp,(()=>{const SR=window.SpeechRecognition||window.webkitSpeechRecognition;if(!SR)return null;const VL={Hindi:"hi-IN",Tamil:"ta-IN",Telugu:"te-IN",Kannada:"kn-IN",Malayalam:"ml-IN",Bengali:"bn-IN",Gujarati:"gu-IN",Marathi:"mr-IN",Punjabi:"pa-IN",Urdu:"ur-IN",Arabic:"ar-SA",French:"fr-FR",German:"de-DE",Spanish:"es-ES",Portuguese:"pt-PT",Chinese:"zh-CN",Japanese:"ja-JP",Korean:"ko-KR",Russian:"ru-RU",Indonesian:"id-ID",Turkish:"tr-TR",Italian:"it-IT",Dutch:"nl-NL",Thai:"th-TH",Vietnamese:"vi-VN"};let rec=null;const mic=h("button",{class:"btn",title:"Speak your question (browser speech recognition)",onclick:()=>{if(rec){rec.stop();return}rec=new SR();const k=Object.keys(VL).find(x=>S.lang.startsWith(x));rec.lang=k?VL[k]:"en-IN";rec.interimResults=true;rec.onresult=e=>{inp.value=[...e.results].map(r=>r[0].transcript).join(" ")};rec.onend=()=>{rec=null;mic.textContent="🎤";if(inp.value.trim())send(inp.value)};mic.textContent="⏺";try{rec.start()}catch(e){rec=null;mic.textContent="🎤"}}},"🎤");return mic})(),h("button",{class:"btn primary",onclick:()=>send(inp.value)},"Ask")));return{el,update(){}}};

// ---------------------------------------------------------------- shell
const UI={};
UI.openEntity=async id=>{if(!id)return;$("#dt").textContent=id;$("#db").replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading…"));$("#drawer").classList.add("on");const d=await api("/api/detail/"+encodeURIComponent(id));if(d.title===undefined){$("#db").replaceChildren(h("div",{class:"callout warn"},d.msg||"Not available"));return}$("#dt").textContent=d.title||id;$("#db").replaceChildren();mountList($("#db"),d.blocks||[],"d");translateUI($("#db"))};
UI.closeDrawer=()=>$("#drawer").classList.remove("on");
UI.go=(id,tab,params)=>{
 const allowed=pageIsAllowed(id),target=allowed?id:firstAllowedPage();
 if(!target){toast("Access unavailable: no pages are available for this account.",true);return}
 S.page=target;S.tab=allowed?(tab||null):null;S.params=allowed?(params||{}):{};
 history.replaceState(null,"","#"+target+(S.tab?":"+S.tab:""));loadPage(false);renderNav();
 if(!allowed){const page=S.me.menu.find(item=>item&&item.id===target);toast(`Access unavailable: ${id||"that page"} is not in your available pages. Redirected to ${page?page.label:target}.`,true)}
};
function fmtTime(t){if(!t)return"Never";const d=new Date(t);return isNaN(d)?String(t):d.toLocaleString(undefined,{year:"numeric",month:"short",day:"numeric",hour:"2-digit",minute:"2-digit"})}
function renderProfilePage(){
 const root=h("div",{class:"profile-page"});
 const status=h("p",{class:"profile-feedback","role":"status","aria-live":"polite"});
 const say=(el,msg,kind)=>{el.className="profile-feedback"+(kind?" is-"+kind:"");el.textContent=msg||""};
 const head=h("section",{class:"profile-header card"}),details=h("div",{class:"profile-details"});
 const username=h("input",{id:"profile-username",name:"username",type:"text",autocomplete:"username",minlength:"3",required:true});
 const editStatus=h("p",{class:"profile-feedback","role":"status","aria-live":"polite"});
 const edit=h("form",{class:"profile-form",hidden:true});
 const editCancel=h("button",{class:"btn",type:"button"},"Cancel"),editSave=h("button",{class:"btn primary",type:"submit"},"Save username");
 edit.append(h("div",{class:"profile-edit-grid"},h("div",{class:"fld"},h("label",{for:"profile-username"},"Username"),username)),h("div",{class:"profile-form-actions"},editCancel,editSave),editStatus);
 const editButton=h("button",{class:"btn sm",type:"button","aria-expanded":"false"},"Change username");
 const currentPassword=h("input",{id:"profile-current-password",name:"current-password",type:"password",autocomplete:"current-password",required:true});
 const newPassword=h("input",{id:"profile-new-password",name:"new-password",type:"password",autocomplete:"new-password",minlength:"8",required:true});
 const confirmPassword=h("input",{id:"profile-confirm-password",name:"confirm-password",type:"password",autocomplete:"new-password",minlength:"8",required:true});
 const passwordStatus=h("p",{class:"profile-feedback","role":"status","aria-live":"polite"});
 const passwordForm=h("form",{class:"profile-form",hidden:true});
 const passwordCancel=h("button",{class:"btn",type:"button"},"Cancel"),passwordSave=h("button",{class:"btn primary",type:"submit"},"Save password");
 passwordForm.append(h("div",{class:"profile-edit-grid"},
  h("div",{class:"fld"},h("label",{for:"profile-current-password"},"Current password"),h("div",{class:"password-field"},currentPassword,passwordToggle(currentPassword))),
  h("div",{class:"fld"},h("label",{for:"profile-new-password"},"New password"),h("div",{class:"password-field"},newPassword,passwordToggle(newPassword))),
  h("div",{class:"fld"},h("label",{for:"profile-confirm-password"},"Confirm new password"),h("div",{class:"password-field"},confirmPassword,passwordToggle(confirmPassword))),
  h("p",{class:"password-guidance profile-password-guidance"},"Use at least 8 characters with uppercase and lowercase letters, a number, and a symbol.")),
  h("div",{class:"profile-form-actions"},passwordCancel,passwordSave),passwordStatus);
 const passwordButton=h("button",{class:"btn sm",type:"button","aria-expanded":"false"},"Change password");
 const criteria=passwordCriteriaForProfile(newPassword);
 let p=null;
 const row=(label,text)=>h("div",{class:"profile-row"},h("div",{class:"profile-label"},label),h("div",{class:"profile-value"},text));
 const draw=()=>{
  const initials=(p.username||p.email||"?").split(/[\s@._-]+/).filter(Boolean).slice(0,2).map(x=>x[0].toUpperCase()).join("")||"?";
  head.replaceChildren(h("div",{class:"profile-avatar","aria-label":"Account initials"},initials),h("div",{class:"profile-header-copy"},h("span",{class:"auth-kicker"},"FRANCHISEOPS AI ACCOUNT"),h("h2",null,p.username),h("p",null,p.email)),h("span",{class:"profile-role"},p.role));
  details.replaceChildren(row("Username",p.username),row("Email address",p.email),row("Role",p.role),row("Outlet",p.outlet||"All outlets"),row("Joined",fmtTime(p.created_at)),row("Last login",fmtTime(p.last_login)));
  username.value=p.username;
 };
 editButton.onclick=()=>{const open=edit.hidden;edit.hidden=!open;editButton.setAttribute("aria-expanded",String(open));say(editStatus,"");if(open)username.focus()};
 editCancel.onclick=()=>{username.value=p.username;say(editStatus,"");edit.hidden=true;editButton.setAttribute("aria-expanded","false")};
 username.addEventListener("input",()=>username.setCustomValidity(""));
 edit.addEventListener("submit",async event=>{event.preventDefault();say(editStatus,"");
  username.setCustomValidity(/^[A-Za-z0-9_]{3,}$/.test(username.value.trim())?"":"Use at least 3 letters, numbers, or underscores.");
  if(!edit.reportValidity()){say(editStatus,"Check the highlighted field.","error");return}
  editSave.disabled=true;const r=await api("/api/profile/username",{json:{username:username.value.trim()}});editSave.disabled=false;
  if(!r.ok){say(editStatus,r.msg||"Could not update the username.","error");return}
  p.username=r.username;S.me.user.username=r.username;draw();say(editStatus,"Username updated.","notice");toast("Username updated.");
 });
 passwordButton.onclick=()=>{const open=passwordForm.hidden;passwordForm.hidden=!open;passwordButton.setAttribute("aria-expanded",String(open));say(passwordStatus,"");if(open)currentPassword.focus()};
 passwordCancel.onclick=()=>{passwordForm.reset();say(passwordStatus,"");passwordForm.hidden=true;passwordButton.setAttribute("aria-expanded","false")};
 for(const f of [currentPassword,newPassword,confirmPassword])f.addEventListener("input",()=>f.setCustomValidity(""));
 passwordForm.addEventListener("submit",async event=>{event.preventDefault();say(passwordStatus,"");
  if(!passwordForm.reportValidity()){say(passwordStatus,"Check the highlighted fields.","error");return}
  if(criteria.some(test=>!test())){say(passwordStatus,"Choose a password that meets the listed requirements.","error");newPassword.focus();return}
  if(newPassword.value!==confirmPassword.value){confirmPassword.setCustomValidity("Passwords do not match.");confirmPassword.reportValidity();say(passwordStatus,"The new passwords do not match.","error");return}
  passwordSave.disabled=true;const r=await api("/api/profile/password",{json:{current_password:currentPassword.value,new_password:newPassword.value,confirm_password:confirmPassword.value}});passwordSave.disabled=false;
  if(!r.ok){say(passwordStatus,r.msg||"Could not change the password.","error");return}
  setToken(r.token,!!localStorage.getItem("fo_token"));passwordForm.reset();passwordForm.hidden=true;passwordButton.setAttribute("aria-expanded","false");say(status,"Password changed. Other sessions were signed out.","notice");toast("Password changed.");
 });
 const account=h("section",{class:"card profile-card","aria-labelledby":"profile-account-title"},h("div",{class:"profile-section-heading"},h("div",null,h("h2",{id:"profile-account-title"},"Account information"),h("p",null,"Your details. Your password hash is never shown."))),details);
 const editCard=h("section",{class:"card profile-card","aria-labelledby":"profile-edit-title"},h("div",{class:"profile-section-heading"},h("div",null,h("h2",{id:"profile-edit-title"},"Username"),h("p",null,"Pick the name shown in the workspace.")),editButton),edit);
 const securityCard=h("section",{class:"card profile-card","aria-labelledby":"profile-security-title"},h("div",{class:"profile-section-heading"},h("div",null,h("h2",{id:"profile-security-title"},"Security"),h("p",null,"Enter your current password to set a new one.")),passwordButton),passwordForm,status);
 root.append(head,account,editCard,securityCard);
 head.replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading your profile…"));
 api("/api/profile").then(r=>{if(!r.profile){head.replaceChildren(h("div",{class:"callout error"},r.msg||"Could not load your profile."));return}p=r.profile;draw()});
 return root;
}
function passwordCriteriaForProfile(input){
 return[()=>input.value.length>=8,()=>/[A-Z]/.test(input.value),()=>/[a-z]/.test(input.value),()=>/[0-9]/.test(input.value),()=>/[!@#$%^&*(),.?":{}|<>~`_+=/\\[\];'-]/.test(input.value)];
}
function pickOutlet(username,role,outlets){
 return new Promise(resolve=>{
  const sel=h("select",{class:"in","aria-label":"Outlet"},...outlets.map(x=>h("option",{value:x},x)));
  const done=v=>{ov.remove();document.removeEventListener("keydown",kd,true);resolve(v)};
  const kd=e=>{if(e.key==="Escape")done(null)};
  const ov=h("div",{class:"kpal",role:"dialog","aria-modal":"true","aria-label":"Choose an outlet",onclick:e=>{if(e.target===ov)done(null)}},
   h("div",{class:"card",style:"padding:20px;display:grid;gap:12px;max-width:420px"},h("h3",{style:"margin:0"},"Choose the outlet"),
    h("p",{class:"sub",style:"margin:0"},username+" becomes a "+role+", who works at one outlet. Which outlet?"),sel,
    h("div",{class:"profile-form-actions"},h("button",{class:"btn",type:"button",onclick:()=>done(null)},"Cancel"),h("button",{class:"btn primary",type:"button",onclick:()=>done(sel.value)},"Apply"))));
  document.addEventListener("keydown",kd,true);document.body.append(ov);sel.focus();
 });
}
function renderAdminDashboard(){
 const root=h("div",{class:"admin-dashboard"});
 let data=null,query="",pendingDelete=null;
 const kpis=h("section",{class:"admin-kpis","aria-label":"Administration overview"});
 const roleCard=h("section",{class:"card admin-card","aria-labelledby":"admin-roles-title"});
 const userSection=h("section",{class:"card admin-card","aria-labelledby":"admin-users-title"});
 const activitySection=h("section",{class:"card admin-card","aria-labelledby":"admin-activity-title"});
 const matrixSection=h("section",{class:"card admin-card","aria-labelledby":"admin-matrix-title"});
 const formStatus=h("p",{class:"profile-feedback","role":"status","aria-live":"polite"});
 const say=(msg,kind)=>{formStatus.className="profile-feedback"+(kind?" is-"+kind:"");formStatus.textContent=msg||""};
 const act=async(path,body)=>{const r=await api(path,{json:body||{}});if(r.ok){toast(r.msg);await load()}else toast(r.msg||"That action failed.",true);return r};
 const kpi=(label,value,sub,tone)=>h("article",{class:"kpi "+(tone||"")},h("div",{class:"l"},label),h("div",{class:"v"},String(value)),h("div",{class:"s"},sub));
 const table=(heads,rows,empty)=>h("div",{class:"tbl admin-table"},h("table",null,h("thead",null,h("tr",null,...heads.map(x=>h("th",null,x)))),h("tbody",null,...(rows.length?rows:[h("tr",null,h("td",{colspan:String(heads.length)},h("div",{class:"admin-empty",role:"status"},empty)))]))));
 // ---- add user form
 const email=h("input",{id:"admin-user-email",name:"email",type:"email",autocomplete:"off",required:true});
 const username=h("input",{id:"admin-user-username",name:"username",type:"text",autocomplete:"off",minlength:"3",required:true});
 const pass=h("input",{id:"admin-user-password",name:"password",type:"password",autocomplete:"new-password",minlength:"8",required:true});
 const role=h("select",{id:"admin-user-role",name:"role",required:true});
 const outlet=h("select",{id:"admin-user-outlet",name:"outlet"});
 const outletField=h("div",{class:"fld",hidden:true},h("label",{for:"admin-user-outlet"},"Outlet"),outlet);
 const addForm=h("form",{class:"admin-add-form",hidden:true,novalidate:true,"aria-labelledby":"admin-add-title"});
 const syncOutlet=()=>{const need=data&&data.scoped_roles.includes(role.value);outletField.hidden=!need;outlet.required=!!need;if(!need)outlet.value=""};
 role.addEventListener("change",syncOutlet);
 addForm.append(h("div",{class:"admin-editor-heading"},h("div",null,h("h3",{id:"admin-add-title"},"Add user"),h("p",null,"The user signs in with the temporary password and should change it in their profile."))),
  h("div",{class:"admin-form-grid"},h("div",{class:"fld"},h("label",{for:"admin-user-email"},"Email address"),email),h("div",{class:"fld"},h("label",{for:"admin-user-username"},"Username"),username),
   h("div",{class:"fld"},h("label",{for:"admin-user-password"},"Temporary password"),h("div",{class:"password-field"},pass,passwordToggle(pass))),h("div",{class:"fld"},h("label",{for:"admin-user-role"},"Role"),role),outletField),
  formStatus,h("div",{class:"admin-form-footer"},h("span",{class:"sub"},"The same email and password rules as signup apply."),h("div",{class:"profile-form-actions"},
   h("button",{class:"btn",type:"button",onclick:()=>{addForm.reset();say("");addForm.hidden=true;syncOutlet()}},"Close"),h("button",{class:"btn primary",type:"submit"},"Create user"))));
 addForm.addEventListener("submit",async event=>{event.preventDefault();say("");
  if(!addForm.reportValidity()){say("Check the highlighted fields.","error");return}
  const r=await api("/api/admin/users",{json:{email:email.value.trim(),username:username.value.trim(),password:pass.value,role:role.value,outlet:outlet.value}});
  if(!r.ok){say(r.msg||"Could not create the user.","error");return}
  toast(r.msg);addForm.reset();addForm.hidden=true;syncOutlet();await load();
 });
 const addButton=h("button",{class:"btn primary",type:"button",onclick:()=>{addForm.hidden=false;email.focus()}},"Add user");
 const search=h("input",{class:"admin-search",type:"search",placeholder:"Search users by email, username, role or outlet","aria-label":"Search users",oninput:()=>{query=search.value.trim().toLowerCase();drawUsers()}});
 // ---- users table
 const roleCell=u=>{
  const me=data.me===u.id;const sel=h("select",{class:"in admin-role-select","aria-label":"Role of "+u.username,disabled:me,title:me?"You cannot change your own role.":""},...data.roles.map(r=>h("option",{value:r,selected:r===u.role},r)));
  sel.addEventListener("change",async()=>{
   let o="";if(data.scoped_roles.includes(sel.value)){o=u.outlet||u.saved_outlet||"";if(!o){o=await pickOutlet(u.username,sel.value,data.outlets);if(!o){sel.value=u.role;return}}}
   const r=await act("/api/admin/users/"+u.id+"/role",{role:sel.value,outlet:o});if(!r.ok)sel.value=u.role;
  });return sel};
 const outletCell=u=>{
  if(!data.scoped_roles.includes(u.role))return h("span",{class:"sub",style:"margin:0"},"All outlets");
  const sel=h("select",{class:"in admin-role-select","aria-label":"Outlet of "+u.username},...data.outlets.map(x=>h("option",{value:x,selected:x===u.outlet},x)));
  sel.addEventListener("change",async()=>{const r=await act("/api/admin/users/"+u.id+"/role",{role:u.role,outlet:sel.value});if(!r.ok)sel.value=u.outlet});return sel};
 const actionsCell=u=>{
  const me=data.me===u.id;
  if(pendingDelete===u.id)return h("div",{class:"admin-row-actions"},h("span",{class:"sub",style:"margin:0 6px 0 0"},"Delete "+u.username+"?"),h("button",{class:"btn sm danger",type:"button",onclick:async()=>{pendingDelete=null;await act("/api/admin/users/"+u.id+"/delete",{confirm:true})}},"Confirm delete"),h("button",{class:"btn sm",type:"button",onclick:()=>{pendingDelete=null;drawUsers()}},"Cancel"));
  return h("div",{class:"admin-row-actions"},
   h("button",{class:"btn sm",type:"button",disabled:me&&!u.locked,title:me?"You cannot lock your own account.":"",onclick:()=>act("/api/admin/users/"+u.id+(u.locked?"/unlock":"/lock"))},u.locked?"Unlock":"Lock"),
   h("button",{class:"btn sm danger",type:"button",disabled:me,title:me?"You cannot delete your own account.":"",onclick:()=>{pendingDelete=u.id;drawUsers()}},"Delete"));
 };
 const drawUsers=()=>{
  if(!data)return;const rows=data.users.filter(u=>!query||[u.email,u.username,u.role,u.outlet||""].join(" ").toLowerCase().includes(query)).map(u=>
   h("tr",null,h("td",null,u.email),h("td",null,u.username+(data.me===u.id?" (you)":"")),h("td",null,roleCell(u)),h("td",null,outletCell(u)),
    h("td",null,h("span",{class:"pill",style:"color:"+(u.locked?"#cd2b31":"#0b7a6c")+";border-color:currentColor"},u.locked?"Locked":"Active")),h("td",null,fmtTime(u.created_at)),h("td",null,fmtTime(u.last_login)),h("td",null,actionsCell(u))));
  userTable.replaceChildren(table(["Email","Username","Role","Outlet","Status","Created","Last login","Actions"],rows,query?"No users match your search.":"No users yet."));
 };
 const userTable=h("div",null);
 userSection.append(h("div",{class:"admin-section-heading"},h("div",null,h("h2",{id:"admin-users-title"},"User management"),h("p",null,"Promote or demote with the role drop-down. A locked user is signed out at once.")),addButton),h("div",{class:"admin-user-toolbar"},search),addForm,userTable);
 const draw=()=>{
  const k=data.kpis;
  kpis.replaceChildren(kpi("Total users",k.total,"registered accounts"),kpi("Active users",k.active,"can sign in"),kpi("Locked users",k.locked,"blocked from signing in",k.locked?"Warning":""),kpi("Roles in use",Object.values(k.per_role).filter(n=>n>0).length,"of "+data.roles.length+" roles"));
  roleCard.replaceChildren(h("div",{class:"admin-section-heading"},h("div",null,h("h2",{id:"admin-roles-title"},"Users per role"),h("p",null,"How many accounts hold each role."))),
   h("div",{class:"admin-row-actions"},...data.roles.map(r=>h("span",{class:"chip"},r+": ",h("b",null,String(k.per_role[r]||0))))));
  role.replaceChildren(h("option",{value:""},"Choose a role"),...data.roles.map(r=>h("option",{value:r},r)));
  outlet.replaceChildren(h("option",{value:""},"Choose an outlet"),...data.outlets.map(x=>h("option",{value:x},x)));syncOutlet();
  drawUsers();
  activitySection.replaceChildren(h("div",{class:"admin-section-heading"},h("div",null,h("h2",{id:"admin-activity-title"},"Activity log"),h("p",null,"Latest "+data.activity.length+" actions: who, what and when."))),
   table(["Time","Who","Action","Target","Detail"],data.activity.map(a=>h("tr",null,h("td",null,fmtTime(a.ts)),h("td",null,a.actor),h("td",null,h("span",{class:"pill",style:"color:#3451b2;border-color:currentColor"},String(a.event).replace(/_/g," "))),h("td",null,a.target),h("td",null,a.detail))),"No activity recorded yet."));
  const cols=Object.keys(data.matrix[0]);
  matrixSection.replaceChildren(h("div",{class:"admin-section-heading"},h("div",null,h("h2",{id:"admin-matrix-title"},"Roles and privileges"),h("p",null,"What each role can open and do. Enforced on the server, not only in the menu."))),
   table(cols,data.matrix.map(r=>h("tr",null,...cols.map(c=>h("td",{style:c==="Role"?"font-weight:600":"text-align:center;color:"+(r[c]==="✓"?"#0b7a6c":"#cd2b31")},r[c])))),"No roles."));
 };
 const load=async()=>{
  const d=await api("/api/admin/overview");
  if(!d.users){root.replaceChildren(h("div",{class:"callout error"},d.msg||"Could not load the admin dashboard."));return}
  data=d;data.me=S.me&&S.me.user&&S.me.user.id;draw();
 };
 root.append(kpis,roleCard,userSection,activitySection,matrixSection);
 kpis.replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading the dashboard…"));load();
 return root;
}
async function loadPage(silent){const seq=++S.seq;clearTimeout(S.timer);if(S.page==="profile"||S.page==="admin"){S.tab=null;S.params={};S.live=0;const profile=S.page==="profile";$("#ttl").textContent=profile?"My Profile":"Admin Dashboard";$("#sub").textContent=profile?"Account details and security":"Users, access, and recent activity";document.title=(profile?"My Profile":"Admin Dashboard")+" · "+S.me.name;$("#tabs").replaceChildren();$("#tabs").style.display="none";$("#acts").replaceChildren();$("#blocks").replaceChildren(profile?renderProfilePage():renderAdminDashboard());return}const q=new URLSearchParams();if(S.tab)q.set("tab",S.tab);if(Object.keys(S.params).length)q.set("p",JSON.stringify(S.params));
 if(!silent){S.tree=null;$("#blocks").replaceChildren(h("div",{class:"boot"},h("div",{class:"spin"}),"Loading…"))}
 const t=performance.now();const d=await api(`/api/page/${S.page}?`+q);if(seq!==S.seq)return;
 if(d.blocks===undefined){$("#blocks").replaceChildren(h("div",{class:"callout error",role:"alert"},h("span",null,h("b",null,d.status===403?"Access denied (403). ":d.status===404?"Not found. ":"Could not load this page. "),d.msg||"")));return}
 S.ms=Math.round(performance.now()-t);$("#ttl").textContent=d.title;$("#sub").textContent=d.sub||"";document.title=d.title+" · "+S.me.name;
 const tabs=$("#tabs");tabs.replaceChildren(...(d.tabs||[]).map(x=>h("b",{class:(S.tab||d.tabs[0].id)===x.id?"on":"",onclick:()=>{S.params={};UI.go(S.page,x.id)}},x.label)));tabs.style.display=d.tabs?"":"none";
 $("#acts").replaceChildren(...(d.actions||[]).map(a=>h("button",{class:"btn "+(a.tone||"primary"),onclick:async e=>{e.target.disabled=true;const r=await api(a.url,{json:{}});toast(r.msg,!r.ok);e.target.disabled=false;reloadPage(true)}},a.label)));
 const box=$("#blocks");const focus=document.activeElement&&box.contains(document.activeElement)&&["INPUT","TEXTAREA","SELECT"].includes(document.activeElement.tagName);
 if(!S.tree||silent===false||!S.tree.ok){box.replaceChildren();S.tree={hs:mountList(box,d.blocks,"p"),ok:1,id:S.page,tab:S.tab};translateUI(box)}else S.tree.hs=updList(box,S.tree.hs,d.blocks,"p");
 S.live=d.live||0;sched()}
function sched(){clearTimeout(S.timer);if(S.live&&S.lang==="English"||S.live&&true)S.timer=setTimeout(()=>{if(!document.hidden)loadPage(true);else sched()},S.live*1000)}
function reloadPage(soft){loadPage(!!soft)}
async function translateUI(root){if(S.lang==="English")return;const nodes=[];const w=document.createTreeWalker(root,NodeFilter.SHOW_TEXT);let n;while((n=w.nextNode())){const t=n.nodeValue.trim();if(t.length>2&&/[A-Za-z]{3}/.test(t)&&!n.parentElement.closest("script,style,pre,code,.m.u,.mono,.noTr,input,select,option")&&!n._tr&&!(n.parentElement.closest("td")&&t.split(/\s+/).length<4)&&!/^[A-Z]{2,6}-?[A-Z]?\d+$/.test(t))nodes.push(n)}
 const list=nodes.slice(0,300);if(!list.length)return;const r=await api("/api/translate_batch",{json:{texts:list.map(n=>n.nodeValue.trim()),lang:S.lang}});if(r.out)list.forEach((n,i)=>{if(r.out[i]){n.nodeValue=r.out[i];n._tr=1}})}
const GROUPS=[["Assistant",["copilot","approvals"]],["Agents",["workforce","outlets","inventory","marketing","sentiment","audit","route","quotes","carriers","weather","margin","customs"]],["Insights",["twin","reports","notifications"]],["Platform",["feeds","admin","profile"]]];
function renderNav(){const nav=$("#nav");if(!nav)return;const menu=S.me&&Array.isArray(S.me.menu)?S.me.menu.filter(m=>m&&pageIsAllowed(m.id)):[];const items=[];const used=new Set();const mk=m=>h("div",{class:"nav"+(m.id===S.page?" on":"")+(m.id==="command"?" cmd":""),"data-id":m.id,onclick:()=>UI.go(m.id)},icon(m.icon),h("span",null,m.label.replace(/^Agent \d+ · /,"")),h("em",{class:"bd",style:"display:none"}));
 GROUPS.forEach(([g,ids])=>{const ms=menu.filter(m=>ids.includes(m.id));if(!ms.length)return;items.push(h("div",{class:"ngrp"},g));ms.forEach(m=>{used.add(m.id);items.push(mk(m))})});
 menu.filter(m=>!used.has(m.id)&&m.id!=="command").forEach(m=>items.push(mk(m)));const c=menu.find(m=>m.id==="command");if(c)items.push(mk(c));
 nav.replaceChildren(...items);updBadges()}
// ---- command palette (Ctrl/Cmd+K): pages, entities, ask the copilot
UI.palette=()=>{if($(".kpal"))return;let sel=0,rows=[];const inp=h("input",{placeholder:"Jump to a page, open an entity (e.g. OUT-007, QTE-0012, SHP-…), or ask the Copilot…"}),lst=h("div",{class:"lst"}),ov=h("div",{class:"kpal",onclick:e=>{if(e.target===ov)close()}},h("div",null,inp,lst));
 const close=()=>{ov.remove();document.removeEventListener("keydown",kd,true)};
 const build=()=>{const q=inp.value.trim(),ql=q.toLowerCase();rows=[];S.me.menu.filter(m=>pageIsAllowed(m.id)&&(!ql||m.label.toLowerCase().includes(ql)||m.id.includes(ql))).forEach(m=>rows.push({g:"Pages",t:m.label.replace(/^Agent \d+ · /,""),s:"go to",f:()=>UI.go(m.id)}));
  if(/^[A-Za-z]{2,5}-?[A-Za-z]?\d+$/.test(q))rows.unshift({g:"Open",t:q.toUpperCase(),s:"entity",f:()=>UI.openEntity(q.toUpperCase())});
  if(q.length>2)rows.push({g:"Ask",t:"Ask the Copilot: "+q,s:"agents + RAG",f:()=>{UI.go("copilot");setTimeout(()=>{const i=document.querySelector(".cin input");if(i){i.value=q;i.dispatchEvent(new KeyboardEvent("keydown",{key:"Enter"}))}},700)}});
  if(sel>=rows.length)sel=rows.length-1;
  let last="";lst.replaceChildren(...rows.flatMap((r,i)=>{const e=[];if(r.g!==last){last=r.g;e.push(h("div",{class:"gl"},r.g))}e.push(h("div",{class:"it"+(i===sel?" on":""),onclick:()=>{close();r.f()}},r.t,h("small",null,r.s)));return e}))};
 const kd=e=>{if(e.key==="Escape"){close()}else if(e.key==="ArrowDown"){sel=Math.min(rows.length-1,sel+1);build();e.preventDefault()}else if(e.key==="ArrowUp"){sel=Math.max(0,sel-1);build();e.preventDefault()}else if(e.key==="Enter"&&document.activeElement===inp){const r=rows[sel];if(r){close();r.f()}}};
 document.addEventListener("keydown",kd,true);inp.addEventListener("input",()=>{sel=0;build()});document.body.append(ov);build();inp.focus()};
document.addEventListener("keydown",e=>{if((e.ctrlKey||e.metaKey)&&e.key.toLowerCase()==="k"&&S.me){e.preventDefault();UI.palette()}});
function updBadges(){const p=S.pulse;if(!p)return;const cc=$('.nav[data-id="command"] .bd');if(cc){const n=p.critical;cc.style.display=n?"":"none";cc.textContent=n;cc.className="bd"}const ap=$('.nav[data-id="approvals"] .bd');if(ap){const n=p.proposed;ap.style.display=n?"":"none";ap.textContent=n;ap.className="bd a"}
 const g=p.gpu||{};$("#gpu").style.display="none";
 $("#hl").innerHTML=`<span class="dot g live"></span>live · health ${p.health==null?"–":Math.round(p.health)} · open ${p.open} · crit ${p.critical} · cycle ${p.cycles}`;$("#warn").style.display="none";if(!p.worker_alive)$("#hl").innerHTML+=' · <b style="color:#e5484d">worker down</b>'}
function openStream(){if(S.es)S.es.close();const es=new EventSource("/api/stream");S.es=es;es.onmessage=e=>{const p=JSON.parse(e.data);if(p.error)return;const old=S.pulse;S.pulse=p;updBadges();if(old&&p.dec!==old.dec&&S.live)loadPage(true)};es.onerror=()=>{}}
// ---------------------------------------------------------------- auth screens
function passwordToggle(input){
 const button=h("button",{class:"password-toggle",type:"button","aria-label":"Show password","aria-pressed":"false",onclick:()=>{const show=input.type==="password";input.type=show?"text":"password";button.textContent=show?"Hide":"Show";button.setAttribute("aria-label",show?"Hide password":"Show password");button.setAttribute("aria-pressed",String(show));input.focus()}},"Show");
 return button;
}
function authSubmitButton(label,busyLabel){
 const spinner=h("span",{class:"login-spinner","aria-hidden":"true",hidden:true}),text=h("span",null,label);
 const button=h("button",{class:"btn primary login-submit",type:"submit","aria-busy":"false"},spinner,text);
 return{button,setLoading(loading){button.disabled=loading;button.setAttribute("aria-busy",String(loading));spinner.hidden=!loading;text.textContent=loading?busyLabel:label}};
}
function passwordCriteria(input){
 return[()=>input.value.length>=8,()=>/[A-Z]/.test(input.value),()=>/[a-z]/.test(input.value),()=>/[0-9]/.test(input.value),()=>/[!@#$%^&*(),.?":{}|<>~`_+=/\\[\];'-]/.test(input.value)];
}
function updatePasswordStrength(input,meter,fill,label,criteria){
 const score=criteria.filter(test=>test()).length;
 const strength=input.value?["","Very weak","Weak","Fair","Good","Strong"][score]:"Enter a password";
 label.textContent=strength;fill.style.width=(score*20)+"%";
 fill.dataset.level=score===0?"empty":score<3?"weak":score<5?"medium":"strong";
 meter.setAttribute("aria-valuenow",String(score));meter.setAttribute("aria-valuetext",strength);
}
function authShell(form,label){
 return h("section",{class:"login","aria-label":label},
  h("aside",{class:"lp","aria-label":"FranchiseOps AI overview"},
   h("div",{class:"login-brand"},h("span",{class:"login-mark","aria-hidden":"true"},icon("radar")),h("span",null,"FranchiseOps",h("b",null," AI"))),
   h("div",{class:"login-message"},h("span",{class:"login-eyebrow"},"FRANCHISE MANAGEMENT PLATFORM"),h("h2",null,"Run every location with greater clarity."),h("p",null,"Bring franchise operations, performance insights, and AI-powered assistance into one workspace.")),
   h("ul",{class:"login-benefits"},h("li",null,h("span",{class:"benefit-icon","aria-hidden":"true"},"01"),h("span",null,h("b",null,"See the bigger picture"),h("small",null,"A clearer view across your franchise network."))),h("li",null,h("span",{class:"benefit-icon","aria-hidden":"true"},"02"),h("span",null,h("b",null,"Turn signals into action"),h("small",null,"Insights from six focused operations agents."))),h("li",null,h("span",{class:"benefit-icon","aria-hidden":"true"},"03"),h("span",null,h("b",null,"Ask your operations Copilot"),h("small",null,"Explore live operational data in plain language.")))),
   h("div",{class:"login-footnote"},"A unified workspace for franchise teams.")),
  h("main",{class:"auth-panel"},themeToggle("btn sm theme-toggle auth-theme-toggle"),form));
}
function renderLogin(){
 clearInterval(S.otpCountdown);S.otpCountdown=null;
 const app=$("#app");app.classList.add("auth-shell");S.authScreen="login";
 const status=h("p",{class:"login-status",role:"status","aria-live":"polite"},S.apiNotice);
 if(S.apiNotice)status.classList.add("is-error");
 if(S.loginNotice){status.textContent=S.loginNotice;status.className="login-status is-notice";S.loginNotice=""}
 const email=h("input",{id:"login-identity",name:"identity",type:"text",autocomplete:"username",autocapitalize:"none",spellcheck:"false",required:true});
 const password=h("input",{id:"login-password",name:"password",type:"password",autocomplete:"current-password",required:true});
 const toggle=passwordToggle(password);
 const submitLabel=h("span",null,"Sign in"),spinner=h("span",{class:"login-spinner","aria-hidden":"true",hidden:true});
 const submit=h("button",{id:"login-submit",class:"btn primary login-submit",type:"submit","aria-busy":"false"},spinner,submitLabel);
 UI.setLoginLoading=loading=>{submit.disabled=loading;submit.setAttribute("aria-busy",String(loading));spinner.hidden=!loading;submitLabel.textContent=loading?"Signing in...":"Sign in"};
 email.addEventListener("input",()=>email.setCustomValidity(""));
 password.addEventListener("input",()=>password.setCustomValidity(""));
 const unavailable=async event=>{
  event.preventDefault();status.className="login-status";status.textContent="";
  email.setCustomValidity(email.value.trim()?"":"Enter your email or username.");
  password.setCustomValidity(password.value?"":"Enter your password.");
  if(!event.currentTarget.reportValidity()){status.className="login-status is-error";status.textContent="Enter your email or username and password.";return}
  UI.setLoginLoading(true);
  const r=await api("/api/auth/login",{json:{identity:email.value.trim(),password:password.value}});
  UI.setLoginLoading(false);
  if(r.ok&&r.token){setToken(r.token,$("#remember-me").checked);S.me=null;S.apiNotice="";render();return}
  status.className="login-status is-error";status.textContent=r.msg||"Could not sign in.";password.value="";password.focus();
 };
 const showSignup=event=>{event.preventDefault();S.authScreen="signup";renderSignup()};
 const showForgot=event=>{event.preventDefault();S.authScreen="forgot";renderForgotPassword()};
 const form=h("form",{class:"auth auth-form",novalidate:true,onsubmit:unavailable},
  h("div",{class:"auth-heading"},h("span",{class:"auth-kicker"},"FRANCHISE OPERATIONS INTELLIGENCE"),h("h1",null,"Welcome back"),h("p",null,"Sign in to continue to your operations workspace.")),
  h("div",{class:"fld"},h("label",{for:"login-identity"},"Email or username"),email),
  h("div",{class:"fld"},h("label",{for:"login-password"},"Password"),h("div",{class:"password-field"},password,toggle)),
  h("div",{class:"login-options"},h("label",{class:"remember",for:"remember-me"},h("input",{id:"remember-me",name:"remember",type:"checkbox"}),h("span",null,"Remember me")),h("a",{href:"#forgot-password",onclick:showForgot},"Forgot password?")),
  submit,status,
  h("p",{class:"signup-prompt"},"New to FranchiseOps AI? ",h("a",{href:"#signup",onclick:showSignup},"Create an account")));
 app.replaceChildren(authShell(form,"FranchiseOps AI sign in"));
}
function renderSignup(){
 clearInterval(S.otpCountdown);S.otpCountdown=null;
 const app=$("#app");app.classList.add("auth-shell");S.authScreen="signup";
 const status=h("p",{class:"signup-status","role":"status","aria-live":"polite"});
 const email=h("input",{id:"signup-email",name:"email",type:"email",autocomplete:"email",autocapitalize:"none",spellcheck:"false",required:true});
 const username=h("input",{id:"signup-username",name:"username",type:"text",autocomplete:"username",autocapitalize:"none",spellcheck:"false",minlength:"3",required:true});
 const role=h("select",{id:"signup-role",name:"role",required:true,"aria-describedby":"signup-access-guidance"},h("option",{value:""},"Loading roles…"));
 const outlet=h("select",{id:"signup-outlet",name:"outlet","aria-describedby":"signup-access-guidance"},h("option",{value:""},"Loading outlets…"));
 const outletField=h("div",{class:"fld",hidden:true},h("label",{for:"signup-outlet"},"Your outlet"),outlet);
 let scoped=[];
 const syncOutlet=()=>{const need=scoped.includes(role.value);outletField.hidden=!need;outlet.required=need;if(!need)outlet.value=""};
 role.addEventListener("change",syncOutlet);
 api("/api/auth/options").then(o=>{
  if(!o.roles){status.className="signup-status is-error";status.textContent=o.msg||"Could not load the role list. Refresh the page.";return}
  scoped=o.scoped_roles||[];
  role.replaceChildren(h("option",{value:""},"Choose your role"),...o.roles.map(r=>h("option",{value:r},r)));
  outlet.replaceChildren(h("option",{value:""},"Choose your outlet"),...o.outlets.map(x=>h("option",{value:x},x)));syncOutlet();
 });
 const password=h("input",{id:"signup-password",name:"password",type:"password",autocomplete:"new-password",minlength:"8",required:true,"aria-describedby":"signup-strength signup-guidance"});
 const confirm=h("input",{id:"signup-confirm",name:"confirm-password",type:"password",autocomplete:"new-password",minlength:"8",required:true});
 const strengthText=h("span",{class:"strength-label","aria-hidden":"true"},"Enter a password");
 const strengthFill=h("span",{class:"strength-fill"});
 const criteria=passwordCriteria(password),meter=h("div",{id:"signup-strength",class:"strength-meter","role":"progressbar","aria-label":"Password strength estimate","aria-valuemin":"0","aria-valuemax":"5","aria-valuenow":"0","aria-valuetext":"Enter a password"},h("span",{class:"strength-track"},strengthFill),strengthText);
 const updateStrength=()=>updatePasswordStrength(password,meter,strengthFill,strengthText,criteria);
 password.addEventListener("input",()=>{password.setCustomValidity("");updateStrength()});
 username.addEventListener("input",()=>username.setCustomValidity(""));
 confirm.addEventListener("input",()=>confirm.setCustomValidity(""));
 const togglePassword=passwordToggle(password),toggleConfirm=passwordToggle(confirm);
 const submit=authSubmitButton("Create account","Creating account...");
 const validationMessage=message=>{status.className="signup-status is-error";status.textContent=message};
 const onSubmit=async event=>{
  event.preventDefault();status.className="signup-status";status.textContent="";
  username.setCustomValidity(/^[A-Za-z0-9_]{3,}$/.test(username.value.trim())?"":"Use at least 3 letters, numbers, or underscores for your username.");
  if(!event.currentTarget.reportValidity()){validationMessage("Please check the highlighted fields.");return}
  if(!role.value){validationMessage("Choose your role.");role.focus();return}
  if(criteria.map(test=>test()).some(ok=>!ok)){validationMessage("Use at least 8 characters with uppercase and lowercase letters, a number, and a symbol.");password.focus();return}
  if(password.value!==confirm.value){confirm.setCustomValidity("Passwords do not match.");confirm.reportValidity();validationMessage("The passwords do not match.");confirm.focus();return}
  submit.setLoading(true);
  const r=await api("/api/auth/signup",{json:{email:email.value.trim(),username:username.value.trim(),password:password.value,confirm_password:confirm.value,role:role.value,outlet:outlet.value}});
  submit.setLoading(false);
  if(r.ok){S.loginNotice="Account created. Sign in with your new account.";S.authScreen="login";renderLogin();return}
  validationMessage(r.msg||"Could not create the account.");
 };
 const backToLogin=event=>{event.preventDefault();S.authScreen="login";renderLogin()};
 const form=h("form",{class:"auth auth-form signup-form",novalidate:true,onsubmit:onSubmit},
  h("div",{class:"auth-heading"},h("span",{class:"auth-kicker"},"FRANCHISE MANAGEMENT PLATFORM"),h("h1",null,"Create your account"),h("p",null,"Set up your access to the FranchiseOps AI workspace.")),
  h("p",{id:"signup-access-guidance",class:"auth-info"},"Choose the role you work in; it decides which pages you can open. Store Managers and Staff are limited to their own outlet. An administrator can change your role later, and Admin accounts are created by an administrator only."),
  h("div",{class:"fld"},h("label",{for:"signup-email"},"Email address"),email),
  h("div",{class:"fld"},h("label",{for:"signup-username"},"Username"),username),
  h("div",{class:"fld"},h("label",{for:"signup-role"},"Your role"),role),
  outletField,
  h("div",{class:"fld"},h("label",{for:"signup-password"},"Password"),h("div",{class:"password-field"},password,togglePassword),
   meter,
   h("p",{id:"signup-guidance",class:"password-guidance"},"Aim for 8+ characters with uppercase and lowercase letters, a number, and a symbol.")),
  h("div",{class:"fld"},h("label",{for:"signup-confirm"},"Confirm password"),h("div",{class:"password-field"},confirm,toggleConfirm)),
  submit.button,
  status,
  h("p",{class:"signup-prompt"},"Already have an account? ",h("a",{href:"#login",onclick:backToLogin},"Return to login")));
 app.replaceChildren(authShell(form,"Create a FranchiseOps AI account"));
}
function renderForgotPassword(){
 clearInterval(S.otpCountdown);S.otpCountdown=null;
 const app=$("#app");app.classList.add("auth-shell");S.authScreen="forgot";
 const status=h("p",{class:"auth-feedback","role":"status","aria-live":"polite"});
 const email=h("input",{id:"recovery-email",name:"email",type:"email",autocomplete:"email",autocapitalize:"none",spellcheck:"false",required:true,value:S.recoveryEmail});
 const submit=authSubmitButton("Send code","Sending...");
 const onSubmit=async event=>{
  event.preventDefault();status.className="auth-feedback";status.textContent="";
  email.setCustomValidity(/^[^\s@]+@[^\s@]+\.[^\s@]+$/.test(email.value.trim())?"":"Enter a valid email address.");
  if(!event.currentTarget.reportValidity()){status.classList.add("is-error");status.textContent="Please enter a valid email address.";return}
  S.recoveryEmail=email.value.trim();submit.setLoading(true);
  const r=await api("/api/auth/forgot",{json:{email:S.recoveryEmail}});submit.setLoading(false);
  if(!r.ok){status.classList.add("is-error");status.textContent=r.msg||"Could not send the code.";return}
  S.recoveryOtp="";S.otpResendUntil=Date.now()+30000;S.otpNotice=r.msg;S.authScreen="otp";renderOtpVerification();
 };
 const back=event=>{event.preventDefault();S.authScreen="login";renderLogin()};
 email.addEventListener("input",()=>{email.setCustomValidity("");S.recoveryEmail=email.value});
 const form=h("form",{class:"auth auth-form recovery-form",novalidate:true,onsubmit:onSubmit},
  h("div",{class:"auth-heading"},h("span",{class:"auth-kicker"},"ACCOUNT RECOVERY"),h("h1",null,"Forgot your password?"),h("p",null,"Enter your account email and we will send you a 6-digit verification code.")),
  h("div",{class:"fld"},h("label",{for:"recovery-email"},"Email address"),email),
  submit.button,status,
  h("p",{class:"signup-prompt"},h("a",{href:"#login",onclick:back},"Back to login")));
 app.replaceChildren(authShell(form,"Recover your FranchiseOps AI account"));
}
function renderOtpVerification(){
 clearInterval(S.otpCountdown);S.otpCountdown=null;
 const app=$("#app");app.classList.add("auth-shell");S.authScreen="otp";
 if(!S.otpResendUntil)S.otpResendUntil=Date.now()+30000;
 const status=h("p",{class:"auth-feedback","role":"status","aria-live":"polite"});
 if(S.otpNotice){status.classList.add("is-notice");status.textContent=S.otpNotice;S.otpNotice=""}
 const emailText=h("b",null,S.recoveryEmail||"your email");
 const code=h("input",{id:"recovery-otp",class:"otp-code",name:"otp",type:"text",inputmode:"numeric",pattern:"[0-9]{6}",minlength:"6",maxlength:"6",autocomplete:"one-time-code",autocapitalize:"off",spellcheck:"false",required:true,value:S.recoveryOtp,"aria-describedby":"otp-guidance"});
 const resend=h("button",{class:"btn otp-resend",type:"button",disabled:true,onclick:async()=>{
  resend.disabled=true;const r=await api("/api/auth/forgot",{json:{email:S.recoveryEmail}});
  status.className="auth-feedback "+(r.ok?"is-notice":"is-error");status.textContent=r.msg||"Could not resend the code.";
  S.otpResendUntil=Date.now()+30000;if(!S.otpCountdown)S.otpCountdown=setInterval(updateCountdown,1000);updateCountdown();
 }},"Resend code");
 const countdown=h("span",{id:"otp-countdown",class:"otp-countdown","aria-live":"off"});
 const submit=authSubmitButton("Verify code","Verifying...");
 const updateCountdown=()=>{
  const remaining=Math.max(0,Math.ceil((S.otpResendUntil-Date.now())/1000));
  resend.disabled=remaining>0;
  resend.textContent=remaining>0?"Resend code ("+String(Math.floor(remaining/60)).padStart(2,"0")+":"+String(remaining%60).padStart(2,"0")+")":"Resend code";
  countdown.textContent=remaining>0?"You can ask for a new code in "+remaining+" seconds.":"You can ask for a new code now.";
  if(!remaining){clearInterval(S.otpCountdown);S.otpCountdown=null}
 };
 updateCountdown();
 if(S.otpResendUntil>Date.now())S.otpCountdown=setInterval(updateCountdown,1000);
 const onSubmit=async event=>{
  event.preventDefault();status.className="auth-feedback";status.textContent="";
  const value=code.value.trim();
  code.setCustomValidity(!value?"Enter the six-digit code.":/^\d{6}$/.test(value)?"":"Enter exactly six digits.");
  if(!event.currentTarget.reportValidity()){
   code.setAttribute("aria-invalid","true");status.classList.add("is-error");status.textContent=!value?"Enter the six-digit code.":"Enter exactly six digits.";return;
  }
  submit.setLoading(true);const r=await api("/api/auth/verify",{json:{email:S.recoveryEmail,otp:value}});submit.setLoading(false);
  if(!r.ok){code.setAttribute("aria-invalid","true");status.classList.add("is-error");status.textContent=r.msg||"That code is wrong or has expired.";return}
  S.recoveryOtp=value;code.removeAttribute("aria-invalid");S.authScreen="reset";renderResetPassword();
 };
 code.addEventListener("input",()=>{S.recoveryOtp=code.value;code.setCustomValidity("");code.removeAttribute("aria-invalid")});
 const back=event=>{event.preventDefault();S.recoveryOtp=code.value;S.authScreen="forgot";renderForgotPassword()};
 const form=h("form",{class:"auth auth-form recovery-form otp-form",novalidate:true,onsubmit:onSubmit},
  h("div",{class:"auth-heading"},h("span",{class:"auth-kicker"},"ACCOUNT RECOVERY"),h("h1",null,"Verify your email"),h("p",null,"Enter the six-digit verification code sent to ",emailText,".")),
  h("div",{class:"fld"},h("label",{for:"recovery-otp"},"Six-digit verification code"),code,h("p",{id:"otp-guidance",class:"password-guidance"},"The code is valid for 10 minutes. Five wrong codes cancel it.")),
  h("div",{class:"otp-resend-row"},resend,countdown),
  submit.button,status,
  h("p",{class:"signup-prompt"},h("a",{href:"#forgot-password",onclick:back},"Back to email")));
 app.replaceChildren(authShell(form,"Verify your FranchiseOps AI recovery code"));
}
function renderResetPassword(){
 clearInterval(S.otpCountdown);S.otpCountdown=null;
 const app=$("#app");app.classList.add("auth-shell");S.authScreen="reset";
 const status=h("p",{class:"auth-feedback","role":"status","aria-live":"polite"});
 const password=h("input",{id:"reset-password",name:"password",type:"password",autocomplete:"new-password",minlength:"8",required:true,"aria-describedby":"reset-strength reset-guidance"});
 const confirm=h("input",{id:"reset-confirm",name:"confirm-password",type:"password",autocomplete:"new-password",minlength:"8",required:true});
 const strengthText=h("span",{class:"strength-label","aria-hidden":"true"},"Enter a password"),strengthFill=h("span",{class:"strength-fill"});
 const meter=h("div",{id:"reset-strength",class:"strength-meter","role":"progressbar","aria-label":"New password strength estimate","aria-valuemin":"0","aria-valuemax":"5","aria-valuenow":"0","aria-valuetext":"Enter a password"},h("span",{class:"strength-track"},strengthFill),strengthText);
 const criteria=passwordCriteria(password),submit=authSubmitButton("Reset password","Resetting...");
 const updateStrength=()=>updatePasswordStrength(password,meter,strengthFill,strengthText,criteria);
 password.addEventListener("input",()=>{password.setCustomValidity("");updateStrength()});
 confirm.addEventListener("input",()=>confirm.setCustomValidity(""));
 const onSubmit=async event=>{
  event.preventDefault();status.className="auth-feedback";status.textContent="";
  if(!event.currentTarget.reportValidity()){status.classList.add("is-error");status.textContent="Please check the highlighted fields.";return}
  if(criteria.some(test=>!test())){password.setCustomValidity("Use at least 8 characters with uppercase and lowercase letters, a number, and a symbol.");password.reportValidity();status.classList.add("is-error");status.textContent="Choose a password with at least 8 characters, uppercase and lowercase letters, a number, and a symbol.";password.focus();return}
  if(password.value!==confirm.value){confirm.setCustomValidity("Passwords do not match.");confirm.reportValidity();status.classList.add("is-error");status.textContent="The passwords do not match.";confirm.focus();return}
  submit.setLoading(true);const r=await api("/api/auth/reset",{json:{email:S.recoveryEmail,otp:S.recoveryOtp,password:password.value,confirm_password:confirm.value}});submit.setLoading(false);
  if(!r.ok){status.classList.add("is-error");status.textContent=r.msg||"Could not reset the password.";return}
  S.recoveryEmail="";S.recoveryOtp="";S.otpResendUntil=0;S.loginNotice=r.msg;S.authScreen="login";renderLogin();
 };
 const back=event=>{event.preventDefault();S.authScreen="login";renderLogin()};
 const backToOtp=event=>{event.preventDefault();S.authScreen="otp";renderOtpVerification()};
 const form=h("form",{class:"auth auth-form recovery-form",novalidate:true,onsubmit:onSubmit},
  h("div",{class:"auth-heading"},h("span",{class:"auth-kicker"},"ACCOUNT RECOVERY"),h("h1",null,"Choose a new password"),h("p",null,"Enter and confirm your new password.")),
  h("div",{class:"fld"},h("label",{for:"reset-password"},"New password"),h("div",{class:"password-field"},password,passwordToggle(password)),
   meter,h("p",{id:"reset-guidance",class:"password-guidance"},"Use at least 8 characters with uppercase and lowercase letters, a number, and a symbol.")),
  h("div",{class:"fld"},h("label",{for:"reset-confirm"},"Confirm new password"),h("div",{class:"password-field"},confirm,passwordToggle(confirm))),
  submit.button,status,
  S.recoveryEmail?h("p",{class:"auth-secondary-link"},h("a",{href:"#otp-verification",onclick:backToOtp},"Back to code verification")):null,
  h("p",{class:"signup-prompt"},h("a",{href:"#login",onclick:back},"Back to login")));
 app.replaceChildren(authShell(form,"Reset your FranchiseOps AI password"));
}
async function render(){const app=$("#app");if(S.es){S.es.close();S.es=null}
 if(!S.me)app.replaceChildren(h("div",{class:"boot",role:"status"},h("div",{class:"spin","aria-hidden":"true"}),"Connecting to FranchiseOps AI…"));
 const pub=await api("/api/public");S.pub=pub;S.apiNotice=pub.error&&pub.status!==401?pub.msg:"";
 if(!S.me&&authToken()){const m=await api("/api/me");if(m.user){S.me=m;S.apiNotice=""}else if(m.error&&m.status!==401&&!S.apiNotice)S.apiNotice=m.msg}
 if(!S.me){if(S.authScreen==="signup")renderSignup();else if(S.authScreen==="forgot")renderForgotPassword();else if(S.authScreen==="otp")renderOtpVerification();else if(S.authScreen==="reset")renderResetPassword();else renderLogin();return}
 if(Array.isArray(S.me.menu)&&!S.me.menu.some(m=>m.id==="profile"))S.me.menu=[...S.me.menu,{id:"profile",label:"My Profile",icon:"user"}];
 if(Array.isArray(S.me.menu)&&!S.me.menu.some(m=>m.id==="admin")&&String(S.me.user.role||"").toLowerCase()==="admin")S.me.menu=[...S.me.menu,{id:"admin",label:"Admin Dashboard",icon:"shield"}];
 app.classList.remove("auth-shell");
 app.replaceChildren(h("div",{class:"side"},h("div",{class:"brand"},h("i",null,icon("radar")),h("div",null,S.me.name,h("small",null,S.me.user.role+" · "+S.me.user.email))),h("div",{class:"ksearch",onclick:()=>UI.palette()},"Search or ask…",h("kbd",null,"Ctrl K")),h("div",{id:"nav"}),h("div",{class:"sp"}),h("div",{class:"sidefoot"},h("div",{class:"who"},h("b",null,S.me.user.username||S.me.user.email),h("small",null,S.me.user.role+(S.me.scope?" · "+S.me.scope:""))),h("button",{class:"btn sm signout",type:"button",onclick:()=>UI.signOut()},"Sign out"))),
  h("div",{class:"main"},h("div",{id:"warn",class:"warnbar",style:"display:none"},"No CUDA GPU detected: models run on CPU and will be slow. In Colab choose Runtime → Change runtime type → T4 GPU, then restart."),
   h("div",{class:"top"},h("div",null,h("h1",{id:"ttl"}),h("div",{class:"sub",id:"sub"})),h("div",{class:"sp"}),h("div",{id:"acts",style:"display:flex;gap:8px"}),themeToggle(),h("span",{class:"chip",id:"hl"},"connecting…"),h("span",{class:"chip",id:"gpu"})),
   h("div",{class:"content"},h("div",{class:"tabs",id:"tabs"}),h("div",{id:"pg"},h("div",{class:"blocks",id:"blocks"})))));

 renderNav();const hs=(location.hash||"").slice(1).split(":");UI.go(hs[0]||firstAllowedPage(),hs[1]);openStream();api("/api/pulse").then(p=>{if(!p.error){S.pulse=p;updBadges()}})}
window.addEventListener("hashchange",()=>{if(!S.me)return;const hs=(location.hash||"").slice(1).split(":");UI.go(hs[0]||firstAllowedPage(),hs[1])});
UI.signOut=async()=>{await api("/api/auth/logout",{json:{}});setToken("");if(S.es){S.es.close();S.es=null}clearTimeout(S.timer);S.me=null;S.tree=null;S.authScreen="login";S.loginNotice="You have been signed out.";render()};
window.UI=UI;render();


Writing /content/app/static/app.js


In [31]:
%%writefile /content/app/static/index.html
<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>FranchiseOps AI</title>
<link rel="icon" href="data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='8' fill='%235b5bd6'/%3E%3Cpath d='M8 20l5-6 4 4 7-9' stroke='%23fff' stroke-width='3' fill='none' stroke-linecap='round'/%3E%3C/svg%3E">
<link rel="stylesheet" href="/static/vendor/leaflet.css"><link rel="stylesheet" href="/static/style.css"></head>
<body><div id="app"></div><div id="toast"></div><div id="drawer"><div class="dh"><b id="dt"></b><button class="x" aria-label="Close drawer" onclick="UI.closeDrawer()">Close</button></div><div id="db"></div></div>
<script src="/static/vendor/echarts.min.js"></script><script src="/static/vendor/leaflet.js"></script><script src="/static/vendor/topojson-client.min.js"></script><script src="/static/app.js"></script></body></html>


Writing /content/app/static/index.html


In [32]:
%%writefile /content/app/static/style.css
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-wght-normal.woff2) format("woff2");unicode-range:U+0000-00FF,U+0131,U+0152-0153,U+02BB-02BC,U+02C6,U+02DA,U+02DC,U+2000-206F,U+20AC,U+2122,U+2191,U+2193,U+2212,U+2215,U+FEFF,U+FFFD}
@font-face{font-family:"Inter";font-style:normal;font-weight:100 900;font-display:swap;src:url(/static/vendor/fonts/inter-latin-ext-wght-normal.woff2) format("woff2");unicode-range:U+0100-02BA,U+02BD-02C5,U+02C7-02CC,U+02CE-02D7,U+02DD-02FF,U+0304,U+0308,U+0329,U+1D00-1DBF,U+1E00-1E9F,U+1EF2-1EFF,U+2020,U+20A0-20AB,U+20AD-20C0,U+2113,U+2C60-2C7F,U+A720-A7FF}
:root{--bg:#f1f5f9;--card:#fff;--line:#e2e8f0;--line2:#cbd5e1;--tx:#0f172a;--tx2:#334155;--mut:#64748b;--pri:#0f766e;--pri2:#115e59;--pri-bg:#e6f4f1;--teal:#0d9488;--teal-bg:#e6f7f4;--blue:#2563eb;--blue-bg:#eaf1fe;--amber:#f59e0b;--amber-bg:#fff6e0;--red:#dc2626;--red-bg:#feeceb;--vio:#7c3aed;--vio-bg:#f3edfe;--green:#16a34a;--r:14px;--sh:0 1px 2px rgba(15,23,42,.05),0 6px 18px rgba(15,23,42,.06)}
:root{color-scheme:light;--surface:#fff;--surface-soft:#f8fafc;--surface-input:#fff;--surface-hover:#f1f5f9;--border-soft:#eef2f6;--toast-bg:#0f172a;--toast-tx:#fff}
:root[data-theme="dark"]{color-scheme:dark;--bg:#11151d;--card:#1b222d;--line:#303a48;--line2:#465263;--tx:#edf1f7;--tx2:#c6cfdb;--mut:#9aa8b8;--pri:#a4a4ff;--pri2:#bebeff;--pri-bg:#292b45;--teal:#53d5c2;--teal-bg:#173b3a;--blue:#8eb2ff;--blue-bg:#1b2c47;--amber:#ffd166;--amber-bg:#3a301b;--red:#ff8589;--red-bg:#42252b;--vio:#d2a3ff;--vio-bg:#342541;--green:#70d9a2;--surface:#1b222d;--surface-soft:#202936;--surface-input:#171e28;--surface-hover:#252e3b;--border-soft:#2b3542;--toast-bg:#e8edf5;--toast-tx:#171b22;--sh:0 1px 2px rgba(0,0,0,.2),0 8px 22px rgba(0,0,0,.22)}
*{box-sizing:border-box}html,body{margin:0;height:100%;background:var(--bg);color:var(--tx);font:14px/1.5 "Inter",ui-sans-serif,system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;font-feature-settings:"cv11","ss01";-webkit-font-smoothing:antialiased}
button,input,select,textarea{font:inherit;color:inherit}a{color:var(--pri);text-decoration:none}a:hover{text-decoration:underline}
#app{display:flex;height:100vh}
.side{width:248px;flex:none;background:#fff;border-right:1px solid var(--line);display:flex;flex-direction:column;padding:14px 12px;overflow:auto}
.brand{display:flex;gap:11px;align-items:center;padding:4px 6px 14px;font-weight:700;font-size:15px;letter-spacing:-.01em}.brand i{width:34px;height:34px;border-radius:11px;background:linear-gradient(135deg,var(--pri),#14b8a6);display:grid;place-items:center;color:#fff;box-shadow:0 4px 12px rgba(15,118,110,.35)}.brand i svg{width:18px;height:18px;stroke:#fff;fill:none;stroke-width:2}
.brand small{display:block;color:var(--mut);font-weight:500;font-size:11px;max-width:150px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.ksearch{display:flex;align-items:center;gap:8px;margin:0 2px 10px;padding:8px 10px;border:1px solid var(--line);border-radius:10px;color:var(--mut);cursor:pointer;font-size:12.5px;background:var(--bg)}.ksearch:hover{border-color:var(--line2)}.ksearch kbd{margin-left:auto;font:600 10.5px "Inter";border:1px solid var(--line2);border-radius:5px;padding:1px 5px;background:#fff;color:var(--mut)}
.ngrp{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:#9a9eb0;padding:12px 10px 4px}
.nav{display:flex;align-items:center;gap:10px;padding:8px 10px;border-radius:10px;color:var(--tx2);cursor:pointer;border:1px solid transparent;margin-bottom:1px;user-select:none;font-weight:500;transition:background .12s}
.nav:hover{background:#f1f2f8}.nav.on{background:var(--pri-bg);color:var(--pri2);font-weight:600}
.nav svg{width:17px;height:17px;flex:none;stroke:currentColor;fill:none;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round;opacity:.85}.nav .bd{margin-left:auto;background:var(--red);color:#fff;border-radius:99px;font-size:10.5px;padding:1px 7px;font-weight:700}.nav .bd.a{background:var(--amber);color:#3d2600}
.nav.cmd{margin-top:10px;background:linear-gradient(135deg,#0f766e,#0d9488);color:#fff;font-weight:600;box-shadow:0 6px 16px rgba(15,118,110,.28)}.nav.cmd:hover{filter:brightness(1.05)}.nav.cmd.on{background:linear-gradient(135deg,#115e59,#0f766e);color:#fff}
.main{flex:1;min-width:0;min-height:0;display:flex;flex-direction:column}
.top{display:flex;align-items:center;gap:12px;padding:12px 24px;border-bottom:1px solid var(--line);background:rgba(255,255,255,.85);backdrop-filter:blur(10px);flex-wrap:wrap}
.top h1{font-size:19px;margin:0;letter-spacing:-.015em}.top .sub{color:var(--mut);font-size:12.5px;margin:0}.sp{flex:1}
.chip{display:inline-flex;gap:6px;align-items:center;padding:4px 11px;border-radius:99px;border:1px solid var(--line);background:#fff;font-size:12px;white-space:nowrap;color:var(--tx2)}
.dot{width:8px;height:8px;border-radius:50%;background:#b4b8c8}.dot.g{background:var(--green);box-shadow:0 0 0 3px rgba(48,164,108,.18)}.dot.r{background:var(--red);box-shadow:0 0 0 3px rgba(229,72,77,.16)}.dot.a{background:var(--amber)}
.live{animation:pl 1.6s infinite}@keyframes pl{50%{opacity:.4}}
.content{flex:1;min-height:0;overflow:auto;padding:20px 24px 70px}
.tabs{display:flex;gap:2px;border-bottom:1px solid var(--line);margin-bottom:16px;flex-wrap:wrap}.tabs b{padding:9px 14px;cursor:pointer;color:var(--mut);border-bottom:2px solid transparent;font-weight:600;margin-bottom:-1px}.tabs b:hover{color:var(--tx)}.tabs b.on{color:var(--pri2);border-color:var(--pri)}
.blocks>*{margin-bottom:16px}
.card{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:16px;box-shadow:var(--sh);min-width:0}.card>h3{margin:0 0 2px;font-size:14px;letter-spacing:-.005em}.card>.sub,.sub{color:var(--mut);font-size:12.5px;margin-bottom:8px}
.row{display:grid;gap:16px}.row>*{min-width:0}.col>*{margin-bottom:16px}.col>*:last-child{margin-bottom:0}
@media(max-width:1000px){.row{grid-template-columns:1fr!important}.side{width:64px}.nav span,.brand div,.side .foot,.ngrp,.ksearch{display:none}}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:14px}.kpi{background:var(--card);border:1px solid var(--line);border-radius:var(--r);padding:14px 16px;position:relative;overflow:hidden;box-shadow:var(--sh)}
.kpi:before{content:"";position:absolute;left:0;top:14px;bottom:14px;width:3px;border-radius:0 3px 3px 0;background:var(--teal)}.kpi.Critical:before,.kpi.High:before{background:var(--red)}.kpi.Warning:before,.kpi.Medium:before{background:var(--amber)}.kpi.Info:before{background:var(--blue)}
.kpi .l{color:var(--mut);font-size:11.5px;font-weight:600;text-transform:uppercase;letter-spacing:.05em}.kpi .v{font-size:26px;font-weight:700;margin-top:3px;letter-spacing:-.02em}.kpi .s{color:var(--mut);font-size:12px}
.pill{display:inline-block;padding:1px 9px;border-radius:99px;font-size:11.5px;font-weight:600;border:1px solid}
.tbl{overflow:auto;border:1px solid var(--line);border-radius:12px;background:#fff}.tbl table{width:100%;border-collapse:collapse;font-size:12.8px}.tbl th{position:sticky;top:0;background:#fafbfe;text-align:left;padding:9px 12px;color:var(--mut);font-weight:600;cursor:pointer;white-space:nowrap;text-transform:capitalize;z-index:1;border-bottom:1px solid var(--line)}
.tbl td{padding:8px 12px;border-top:1px solid #eff0f6;max-width:380px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.tbl tr.cl{cursor:pointer}.tbl tr.cl:hover td{background:var(--pri-bg)}
.bar{display:flex;align-items:center;gap:8px;min-width:110px}.bar i{display:block;height:6px;border-radius:4px;flex:1;background:#eceef6;overflow:hidden}.bar i b{display:block;height:100%;border-radius:4px}
.srch{margin:0 0 8px;padding:7px 11px;border-radius:9px;border:1px solid var(--line2);background:#fff;width:240px}
.callout{padding:11px 14px;border-radius:12px;border:1px solid;display:flex;gap:10px;color:var(--tx2)}.callout.info{border-color:#cfd9fb;background:var(--blue-bg)}.callout.warn,.callout.warning{border-color:#f6d98f;background:var(--amber-bg)}.callout.error,.callout.critical,.callout.bad{border-color:#f5bcbe;background:var(--red-bg)}.callout.good,.callout.ok,.callout.success{border-color:#b6e3da;background:var(--teal-bg)}
.md{line-height:1.65}.md code,.md pre{background:#f1f2f8;border-radius:6px;padding:1px 5px;font-family:ui-monospace,"JetBrains Mono",monospace;font-size:12px}.md pre{padding:10px;overflow:auto;white-space:pre-wrap}.md h1,.md h2,.md h3{margin:.7em 0 .3em;letter-spacing:-.01em}.md ul{margin:.3em 0;padding-left:20px}.md table{border-collapse:collapse;margin:.5em 0}.md td,.md th{border:1px solid var(--line);padding:5px 10px}
.btn{border:1px solid var(--line2);background:#fff;padding:7px 14px;border-radius:10px;cursor:pointer;font-weight:600;color:var(--tx2);transition:all .12s}.btn:hover{background:#f4f5fb;border-color:#c9cde0}
.btn.primary{background:var(--pri);color:#fff;border-color:var(--pri);box-shadow:0 2px 8px rgba(15,118,110,.3)}.btn.primary:hover{background:var(--pri2)}.btn.danger{background:var(--red-bg);border-color:#f5bcbe;color:#c62a2f}.btn.ok{background:var(--teal-bg);border-color:#b6e3da;color:#0b7a6c}.btn.ghost{background:transparent}.btn.sm{padding:3px 10px;font-size:12px}.btn:disabled{opacity:.5;cursor:wait}
form.f{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:12px;align-items:end}form.f.inline{grid-template-columns:repeat(auto-fit,minmax(160px,max-content))}
.fld label{display:block;color:var(--mut);font-size:12px;font-weight:500;margin-bottom:4px}.fld input,.fld select,.fld textarea,.in{width:100%;padding:8px 11px;border-radius:10px;border:1px solid var(--line2);background:#fff;transition:border .12s,box-shadow .12s}.fld input:focus,.fld select:focus,.fld textarea:focus,.in:focus,.srch:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(15,118,110,.15)}.fld textarea{min-height:80px;grid-column:1/-1}.fld.wide{grid-column:1/-1}.fld input[type=range]{padding:0;accent-color:var(--pri)}
.chk{display:flex;flex-wrap:wrap;gap:6px}.chk label{display:flex;gap:6px;align-items:center;margin:0;color:var(--tx2);padding:3px 10px;border:1px solid var(--line2);border-radius:99px;font-size:12px;background:#fff}
.res{margin-top:12px}
.tk{border:1px solid var(--line);border-left:4px solid var(--blue);border-radius:12px;padding:12px 14px;margin-bottom:10px;background:#fff;box-shadow:var(--sh)}.tk.Critical{border-left-color:var(--red)}.tk.Warning{border-left-color:var(--amber)}.tk.proposed{background:var(--vio-bg)}
.tk .h{display:flex;gap:8px;align-items:center;flex-wrap:wrap}.tk .d{margin:6px 0;color:var(--tx2)}.tk .m{color:var(--mut);font-size:11.8px;display:flex;gap:10px;flex-wrap:wrap}.tk .a{display:flex;gap:6px;margin-top:8px;flex-wrap:wrap;align-items:center}.tk .cm{margin-top:8px;border-top:1px dashed var(--line2);padding-top:6px;font-size:12.5px}
.filters{display:flex;gap:6px;margin-bottom:12px;flex-wrap:wrap;align-items:center}.filters .chip{cursor:pointer}.filters .chip:hover{border-color:var(--pri)}.filters .chip.on{border-color:var(--pri);background:var(--pri-bg);color:var(--pri2);font-weight:600}
.hero{display:flex;gap:18px;align-items:center;background:linear-gradient(120deg,#e6f4f1,#e8f6f4);border:1px solid #d5e5e2;border-radius:16px;padding:16px 20px}.hero .sc{font-size:36px;font-weight:800;min-width:84px;text-align:center;letter-spacing:-.03em}.hero h2{margin:0;font-size:16px}.hero p{margin:2px 0;color:var(--mut);font-size:12.5px}
.map{border-radius:12px;overflow:hidden;border:1px solid var(--line);background:#cfe5f7}.leaflet-container{background:#cfe5f7;font:inherit}.lg{display:flex;gap:12px;font-size:12px;color:var(--mut);margin-top:8px;flex-wrap:wrap}.lg i{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:4px}
.leaflet-tooltip{background:#fff;color:var(--tx);border:1px solid var(--line);border-radius:8px;box-shadow:var(--sh)}
#toast{position:fixed;right:18px;bottom:18px;display:flex;flex-direction:column;gap:8px;z-index:99}#toast div{background:#1c2024;color:#fff;padding:10px 16px;border-radius:12px;max-width:380px;box-shadow:0 10px 30px rgba(20,24,40,.25);animation:up .25s}#toast div.bad{background:#c62a2f}@keyframes up{from{transform:translateY(10px);opacity:0}}
#drawer{z-index:2000!important;position:fixed;right:0;top:0;bottom:0;width:min(720px,96vw);background:#fff;border-left:1px solid var(--line);transform:translateX(105%);transition:.25s;z-index:50;display:flex;flex-direction:column;box-shadow:-20px 0 60px rgba(20,24,40,.12)}#drawer.on{transform:none}
#drawer .dh{display:flex;justify-content:space-between;padding:14px 18px;border-bottom:1px solid var(--line);font-size:16px}#db{padding:16px;overflow:auto;flex:1;background:var(--bg)}.x{background:none;border:0;color:var(--mut);cursor:pointer;font-size:16px}
#app.auth-shell{display:block;height:auto;min-height:100vh}
.login{display:grid;grid-template-columns:minmax(0,1.08fr) minmax(420px,.92fr);width:100%;min-height:100vh;background:#fff}
.lp{position:relative;isolation:isolate;overflow:hidden;background:radial-gradient(800px 500px at 10% 5%,rgba(45,212,191,.42),transparent 62%),linear-gradient(145deg,#0b1f3a,#115e59 56%,#147f80);color:#fff;padding:clamp(32px,6vw,82px);display:flex;flex-direction:column;justify-content:center;gap:clamp(30px,5vh,52px)}
.lp:after{content:"";position:absolute;z-index:-1;width:380px;height:380px;right:-170px;bottom:-170px;border:1px solid rgba(255,255,255,.16);border-radius:50%;box-shadow:0 0 0 44px rgba(255,255,255,.035),0 0 0 90px rgba(255,255,255,.025)}
.login-brand{display:flex;align-items:center;gap:11px;font-size:17px;font-weight:650;letter-spacing:-.02em}.login-brand b{font-weight:500;color:#a8f3e8}
.login-mark{display:grid;place-items:center;width:40px;height:40px;border:1px solid rgba(255,255,255,.25);border-radius:13px;background:rgba(255,255,255,.13);box-shadow:0 8px 24px rgba(15,20,70,.2)}
.login-mark svg{width:23px;height:23px;stroke:#fff;fill:none;stroke-width:1.7;stroke-linecap:round;stroke-linejoin:round}
.login-message{max-width:590px}.login-eyebrow,.auth-kicker{font-size:10.5px;font-weight:700;letter-spacing:.14em}.login-eyebrow{color:#a8f3e8}
.lp h2{font-size:clamp(34px,4vw,50px);line-height:1.08;margin:14px 0 14px;letter-spacing:-.04em;max-width:560px}.lp p{margin:0;color:rgba(255,255,255,.8);max-width:510px;font-size:15px;line-height:1.75}
.lp .login-benefits{list-style:none;margin:0;padding:0;display:grid;gap:18px;max-width:510px}
.lp .login-benefits li{display:flex;align-items:center;gap:13px;color:rgba(255,255,255,.94)}
.lp .login-benefits li:before{display:none}
.benefit-icon{display:grid;place-items:center;flex:none;width:38px;height:38px;border:1px solid rgba(255,255,255,.2);border-radius:12px;background:rgba(255,255,255,.1);color:#a8f3e8;font-size:10px;font-weight:700;letter-spacing:.04em}
.login-benefits b,.login-benefits small{display:block}.login-benefits b{font-size:13px;font-weight:600}.login-benefits small{margin-top:2px;color:rgba(255,255,255,.68);font-size:12px}
.login-footnote{color:rgba(255,255,255,.55);font-size:11px}
.auth-panel{display:flex;align-items:center;justify-content:center;padding:40px 30px;background:linear-gradient(180deg,#fff,#fbfbfe)}
.auth{margin:auto;width:min(440px,100%);padding:24px;border:0;box-shadow:none}
.auth-form{display:grid;gap:18px}
.auth-heading{margin-bottom:4px}.auth-heading h1{margin:9px 0 5px;letter-spacing:-.035em;font-size:30px}.auth-heading p{margin:0;color:var(--mut);font-size:13.5px}
.auth-kicker{color:var(--pri);font-size:9.5px}
.auth .fld{margin:0}.auth .fld label{margin-bottom:7px;color:var(--tx2);font-size:12.5px;font-weight:600}
.auth .fld input{min-height:46px;padding:10px 12px;border-color:var(--line2);border-radius:10px;background:#fff}
.auth .fld input:focus-visible,.auth .fld input:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(15,118,110,.14)}
.password-field{position:relative;display:flex;align-items:center}.password-field input{padding-right:68px!important}
.password-toggle{position:absolute;right:9px;padding:5px 7px;border:0;border-radius:6px;background:transparent;color:var(--pri2);font-size:12px;font-weight:650;cursor:pointer}
.password-toggle:hover{background:var(--pri-bg)}.password-toggle:focus-visible,.auth a:focus-visible,.remember input:focus-visible,.login-submit:focus-visible{outline:3px solid rgba(15,118,110,.36);outline-offset:3px}
.login-options{display:flex;align-items:center;justify-content:space-between;gap:12px;margin-top:-3px;font-size:12px}
.remember{display:inline-flex;align-items:center;gap:8px;color:var(--tx2);cursor:pointer}.remember input{width:16px;height:16px;margin:0;accent-color:var(--pri)}
.auth a{color:var(--pri2);font-weight:600;text-decoration:none}.auth a:hover{text-decoration:underline}
.login-submit{display:flex;align-items:center;justify-content:center;gap:9px;width:100%;min-height:46px;margin-top:2px;font-size:13px}
.login-submit:disabled{cursor:wait;opacity:.72}
.login-spinner{width:16px;height:16px;border:2px solid rgba(255,255,255,.45);border-top-color:#fff;border-radius:50%;animation:sp .75s linear infinite}
.login-spinner[hidden]{display:none}
.login-status{min-height:18px;margin:-12px 0 -7px;color:#a22d36;font-size:12px}.signup-prompt{margin:0;text-align:center;color:var(--mut);font-size:12.5px}
.signup-form{gap:13px}.signup-form .auth-heading{margin-bottom:1px}.signup-form .auth-heading h1{font-size:27px}
.signup-status{min-height:18px;margin:-5px 0 -3px;color:var(--mut);font-size:12px;line-height:1.5}.signup-status.is-error{color:#a22d36}.signup-status.is-notice{padding:9px 11px;border:1px solid #cfd9fb;border-radius:9px;background:var(--blue-bg);color:#3451b2}
.strength-meter{display:flex;align-items:center;gap:10px;margin-top:8px;color:var(--mut);font-size:11px}
.strength-track{display:block;height:5px;flex:1;overflow:hidden;border-radius:99px;background:var(--line)}
.strength-fill{display:block;width:0;height:100%;border-radius:inherit;background:var(--red);transition:width .18s ease,background .18s ease}
.strength-fill[data-level="medium"]{background:var(--amber)}.strength-fill[data-level="strong"]{background:var(--green)}
.strength-label{min-width:76px;text-align:right}
.password-guidance{margin:5px 0 0;color:var(--mut);font-size:10.5px;line-height:1.45}
.recovery-form{gap:17px}
.auth-info{padding:11px 13px;border:1px solid var(--line);border-radius:10px;background:var(--bg);color:var(--tx2);font-size:12px;line-height:1.55}
.auth-feedback{min-height:18px;margin:-7px 0 -4px;color:var(--mut);font-size:12px;line-height:1.5}
.auth-feedback.is-error{color:#a22d36}.auth-feedback.is-notice{padding:10px 12px;border:1px solid #cfd9fb;border-radius:9px;background:var(--blue-bg);color:#3451b2}
.auth-secondary-link{margin:-5px 0 0;text-align:center;color:var(--mut);font-size:11.5px}
.demo{display:flex;gap:6px;flex-wrap:wrap;margin-top:14px}.demo .chip{cursor:pointer}.demo .chip:hover{border-color:var(--pri);color:var(--pri2)}
@media(max-width:860px){.login{grid-template-columns:1fr}.lp{display:none}}
@media(max-width:500px){.auth-panel{padding:22px 14px}.auth{padding:18px 10px}.signup-form{gap:12px}.signup-form .auth-heading h1{font-size:25px}.login-options{font-size:11.5px}}
.boot{text-align:center;padding:60px;color:var(--mut)}.spin{width:28px;height:28px;border:3px solid var(--line);border-top-color:var(--pri);border-radius:50%;animation:sp .8s linear infinite;margin:0 auto 12px}@keyframes sp{to{transform:rotate(360deg)}}
.profile-page{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px;max-width:1120px;margin:0 auto}
.profile-page>[hidden],.profile-form[hidden]{display:none!important}
.profile-header{grid-column:1/-1;display:flex;align-items:center;gap:18px;padding:22px}
.profile-avatar{display:grid;place-items:center;flex:none;width:64px;height:64px;border:1px solid #dcdcff;border-radius:20px;background:var(--pri-bg);color:var(--pri2);font-size:22px;font-weight:700;letter-spacing:-.04em}
.profile-header-copy{min-width:0;flex:1}.profile-header-copy h2{margin:4px 0 1px;font-size:21px;letter-spacing:-.025em}.profile-header-copy p{margin:0;color:var(--mut);overflow-wrap:anywhere}
.profile-role{padding:5px 11px;border:1px solid var(--line);border-radius:99px;background:var(--bg);color:var(--tx2);font-size:11.5px;font-weight:650;white-space:nowrap}
.profile-card{min-width:0}.profile-section-heading{display:flex;align-items:center;justify-content:space-between;gap:14px;margin-bottom:16px}.profile-section-heading h2{margin:0;font-size:15px;letter-spacing:-.01em}.profile-section-heading p{margin:3px 0 0;color:var(--mut);font-size:12px}
.profile-details{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));column-gap:26px}
.profile-row{min-width:0;padding:12px 0;border-top:1px solid var(--line)}.profile-label{margin-bottom:4px;color:var(--mut);font-size:11px;font-weight:600}.profile-value{overflow-wrap:anywhere;color:var(--tx);font-weight:550}.profile-value.unavailable{color:var(--mut);font-weight:400}
.profile-pending{display:flex;align-items:center;gap:9px;margin-top:12px;padding:10px 12px;border:1px solid #cfd9fb;border-radius:10px;background:var(--blue-bg);color:#3451b2;font-size:11.5px;line-height:1.5}
.profile-form{display:grid;gap:14px}.profile-edit-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px 16px}.profile-form .fld{min-width:0}.profile-form .fld label{display:block;margin-bottom:5px;color:var(--tx2);font-size:12px;font-weight:600}.profile-form .fld input{width:100%;min-height:42px;padding:9px 11px;border:1px solid var(--line2);border-radius:10px;background:#fff}.profile-form .fld input:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(15,118,110,.14)}
.profile-form-actions{display:flex;justify-content:flex-end;gap:8px}.profile-feedback{min-height:0;margin:0;color:var(--mut);font-size:12px;line-height:1.5}.profile-feedback.is-error{color:#a22d36}.profile-feedback.is-notice{padding:9px 11px;border:1px solid #cfd9fb;border-radius:9px;background:var(--blue-bg);color:#3451b2}
.profile-password-guidance{grid-column:1/-1;margin:0}
@media(max-width:700px){.profile-page{grid-template-columns:1fr}.profile-header{grid-column:auto;align-items:flex-start;flex-wrap:wrap;padding:18px}.profile-header-copy{flex-basis:calc(100% - 84px)}.profile-role{margin-left:82px}.profile-details,.profile-edit-grid{grid-template-columns:1fr}.profile-section-heading{align-items:flex-start}.profile-section-heading .btn{flex:none}.profile-password-guidance{grid-column:auto}}
.admin-dashboard{display:grid;gap:16px;max-width:1440px;margin:0 auto}.admin-integration-note{align-items:center;font-size:12px}
.admin-kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:14px}.admin-kpis .kpi{min-width:0}.admin-kpis .kpi .v{color:var(--mut)}
.admin-card{min-width:0}.admin-section-heading{display:flex;align-items:center;justify-content:space-between;gap:16px;margin-bottom:15px}.admin-section-heading h2{margin:0;font-size:15px;letter-spacing:-.01em}.admin-section-heading p{margin:3px 0 0;color:var(--mut);font-size:12px}
.admin-user-toolbar{display:flex;align-items:center;justify-content:space-between;gap:12px;margin-bottom:12px}.admin-search{width:min(360px,100%);min-height:38px;padding:8px 11px;border:1px solid var(--line2);border-radius:9px;background:#fff}.admin-search:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(15,118,110,.14)}
.admin-row-actions{display:flex;gap:6px;flex-wrap:wrap}.admin-row-actions .btn:disabled{opacity:.55;cursor:not-allowed}.btn.danger{color:#a22d36;border-color:#f0c7ca}.admin-empty{padding:18px 12px;color:var(--mut);font-size:12.5px;text-align:center;white-space:normal}.admin-table{min-width:0}.admin-table table{min-width:680px}.admin-table td[colspan]{white-space:normal}
.admin-pending-note{margin-top:12px;color:var(--mut);font-size:11.5px;line-height:1.55}
.admin-delete-preview{margin-top:10px;border-top:1px solid var(--line);padding-top:10px;color:var(--tx2);font-size:12px}.admin-delete-preview summary{width:fit-content;color:var(--pri2);font-weight:600;cursor:pointer}.admin-delete-copy{display:flex;align-items:center;justify-content:space-between;gap:12px;margin-top:8px;padding:11px 12px;border:1px solid #f0c7ca;border-radius:10px;background:#fffafb}.admin-delete-copy p{margin:0;color:var(--mut);line-height:1.5}.admin-delete-copy .btn:disabled{opacity:.55;cursor:not-allowed}
.admin-add-form{display:grid;gap:14px;margin:0 0 14px;padding:15px;border:1px solid var(--line);border-radius:12px;background:#fafbfe}.admin-add-form[hidden]{display:none}.admin-editor-heading h3{margin:0;font-size:13px}.admin-editor-heading p{margin:3px 0 0;color:var(--mut);font-size:11.5px}
.admin-form-grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:12px 16px}.admin-form-grid .fld{min-width:0}.admin-form-grid label{display:block;margin-bottom:5px;color:var(--tx2);font-size:12px;font-weight:600}.admin-form-grid input,.admin-form-grid select{width:100%;min-height:40px;padding:8px 10px;border:1px solid var(--line2);border-radius:9px;background:#fff}.admin-form-grid :disabled{background:#f1f2f8;color:var(--mut);cursor:not-allowed}
.admin-form-footer{display:flex;align-items:center;justify-content:space-between;gap:10px}.admin-form-footer .sub{margin:0}
@media(max-width:900px){.admin-kpis{grid-template-columns:repeat(2,minmax(0,1fr))}}
@media(max-width:600px){.admin-section-heading{align-items:flex-start}.admin-user-toolbar{align-items:stretch;flex-direction:column}.admin-search{width:100%}.admin-row-actions{justify-content:flex-start}.admin-form-grid{grid-template-columns:1fr}.admin-form-footer{align-items:flex-start;flex-direction:column-reverse}.admin-form-footer .btn{align-self:flex-end}.admin-delete-copy{align-items:flex-start;flex-direction:column}}
@media(max-width:420px){.admin-kpis{grid-template-columns:1fr 1fr}.admin-kpis .kpi{padding:12px}.admin-kpis .kpi .v{font-size:22px}.admin-kpis .kpi .s{font-size:11px}}
.warnbar{background:var(--amber-bg);border-bottom:1px solid #f6d98f;color:#7a4b00;padding:7px 24px;font-size:12.5px}
.chat{display:flex;flex-direction:column;gap:12px}.msgs{display:flex;flex-direction:column;gap:12px;min-height:280px}.m{max-width:88%;padding:11px 15px;border-radius:16px;border:1px solid var(--line)}.m.u{align-self:flex-end;background:var(--pri);color:#fff;border-color:var(--pri);border-bottom-right-radius:5px}.m.a{background:#fff;box-shadow:var(--sh);border-bottom-left-radius:5px}.st{color:var(--mut);font-size:12px}.st:before{content:"› "}.cites{margin:4px 0;font-size:12px;color:var(--mut)}.cites summary{cursor:pointer;color:var(--pri2);font-weight:600}
.cin{display:flex;gap:8px}.cin input{flex:1;padding:11px 15px;border-radius:12px;border:1px solid var(--line2);background:#fff}.cin input:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px rgba(15,118,110,.15)}
.kpal{position:fixed;inset:0;background:rgba(20,24,40,.35);backdrop-filter:blur(2px);z-index:90;display:flex;justify-content:center;padding-top:12vh}.kpal>div{width:min(620px,94vw);background:#fff;border-radius:16px;box-shadow:0 30px 80px rgba(20,24,40,.3);overflow:hidden;align-self:flex-start}.kpal input{width:100%;border:0;border-bottom:1px solid var(--line);padding:16px 18px;font-size:15px;outline:0}.kpal .it{display:flex;gap:10px;padding:10px 18px;cursor:pointer;align-items:center;color:var(--tx2)}.kpal .it.on{background:var(--pri-bg);color:var(--pri2)}.kpal .it small{margin-left:auto;color:var(--mut)}.kpal .lst{max-height:50vh;overflow:auto;padding:6px 0}.kpal .gl{font-size:10.5px;font-weight:700;letter-spacing:.07em;text-transform:uppercase;color:#9a9eb0;padding:8px 18px 2px}
::-webkit-scrollbar{width:9px;height:9px}::-webkit-scrollbar-thumb{background:#d5d8e6;border-radius:9px}
@media(max-width:760px){#app{flex-direction:column-reverse}.side{width:100%;flex-direction:row;padding:4px;overflow-x:auto;border-right:0;border-top:1px solid var(--line);position:sticky;bottom:0;z-index:20;gap:2px}.side .brand,.side .foot,.side .sp,.ngrp,.ksearch{display:none}#nav{display:flex;gap:2px}.nav{flex-direction:column;gap:2px;padding:6px 10px;font-size:10px;min-width:62px;text-align:center;margin:0}.nav.cmd{margin-top:0}.nav span{display:block!important;white-space:nowrap;max-width:70px;overflow:hidden;text-overflow:ellipsis}.nav .bd{position:absolute;margin:0}.nav{position:relative}.content{padding:12px 10px 20px}.top{padding:8px 10px}.top h1{font-size:15px}.kpis{grid-template-columns:repeat(2,1fr)}.hero{flex-direction:column;align-items:flex-start}#drawer{width:100vw}.tbl td{max-width:200px}.srch{width:100%}.chip{font-size:11px}}
@media(max-width:760px){.top #hl{min-width:0;max-width:100%;overflow:hidden;white-space:nowrap}}
.top>div:first-child{min-width:0;flex:1 1 280px}.top .sub{white-space:nowrap;overflow:hidden;text-overflow:ellipsis;max-width:100%}

.leaflet-container{isolation:isolate;overflow:hidden!important}.mapw,.map,.leafmap{isolation:isolate;position:relative;z-index:0}#drawer.on{z-index:2000}.kpal{z-index:3000!important}#toast{z-index:3500!important}
.side{background:var(--surface)}
.top{background:color-mix(in srgb,var(--surface) 90%,transparent)}
.chip,.ksearch kbd,.tbl,.srch,.btn,.fld input,.fld select,.fld textarea,.in,.chk label,.tk,.m.a,.cin input,.leaflet-tooltip{background-color:var(--surface);color:var(--tx2)}
.tbl th{background:var(--surface-soft);border-bottom-color:var(--line)}
.tbl td{border-top-color:var(--border-soft)}
.bar i,.md code,.md pre{background:var(--surface-hover)}
.btn:hover,.ksearch:hover{background:var(--surface-hover);border-color:var(--line2)}
.fld input,.fld select,.fld textarea,.in,.srch,.cin input{background:var(--surface-input);color:var(--tx)}
.fld input::placeholder,.fld textarea::placeholder,.in::placeholder,.srch::placeholder,.cin input::placeholder{color:var(--mut)}
.callout.info{border-color:var(--line2)}.callout.warn,.callout.warning{border-color:var(--amber)}.callout.error,.callout.critical,.callout.bad{border-color:var(--red)}.callout.good,.callout.ok,.callout.success{border-color:var(--teal)}
.tk{background-color:var(--card)}.tk.proposed{background:var(--vio-bg)}.m.a{box-shadow:var(--sh)}
.hero{background:linear-gradient(120deg,var(--pri-bg),var(--teal-bg));border-color:var(--line2)}
.auth-panel{position:relative;background:linear-gradient(180deg,var(--surface),var(--bg))}
.login{background:var(--surface)}
.auth .fld input,.profile-form .fld input,.admin-form-grid input,.admin-form-grid select,.admin-search{background:var(--surface-input);color:var(--tx);border-color:var(--line2)}
.auth-info,.admin-add-form{background:var(--surface-soft);border-color:var(--line)}
.profile-role,.profile-role,.profile-header .profile-role{background:var(--surface-soft);color:var(--tx2);border-color:var(--line)}
.profile-pending,.profile-feedback.is-notice,.auth-feedback.is-notice,.signup-status.is-notice{background:var(--blue-bg);border-color:var(--line2);color:var(--blue)}
.admin-delete-copy{background:color-mix(in srgb,var(--red-bg) 35%,var(--surface));border-color:var(--line2)}
.admin-form-grid :disabled{background:var(--surface-hover);color:var(--mut)}
.kpal>div{background:var(--surface);color:var(--tx)}.kpal input{background:var(--surface-input);color:var(--tx);border-bottom-color:var(--line)}.kpal .it{color:var(--tx2)}.kpal .it.on{background:var(--pri-bg);color:var(--pri2)}
#drawer{background:var(--surface);border-left-color:var(--line)}#db{background:var(--bg)}#drawer .dh{border-bottom-color:var(--line)}
#toast div{background:var(--toast-bg);color:var(--toast-tx)}#toast div.bad{background:#a91f32;color:#fff}
::-webkit-scrollbar-thumb{background:var(--line2)}
.theme-toggle{white-space:nowrap;min-height:34px}
.auth-theme-toggle{position:absolute;top:18px;right:18px;z-index:1}
.theme-toggle:focus-visible{outline:3px solid color-mix(in srgb,var(--pri) 55%,transparent);outline-offset:3px}
:root[data-theme="dark"] .auth-feedback.is-error,:root[data-theme="dark"] .signup-status.is-error,:root[data-theme="dark"] .login-status,:root[data-theme="dark"] .profile-feedback.is-error{color:#ff9a9e}
:root[data-theme="dark"] .btn.danger{background:var(--red-bg);border-color:#75414a;color:#ffb0b3}
:root[data-theme="dark"] .btn.ok{background:var(--teal-bg);border-color:#28615b;color:#8ce4ce}
:root[data-theme="dark"] .admin-delete-copy{border-color:#75414a}
@media(max-width:760px){.top .theme-toggle{padding:5px 8px;font-size:11px}.auth-theme-toggle{top:12px;right:12px}}
:root{--amber-tx:#7a4b00;--amber-border:#f6d98f}
:root[data-theme="dark"]{--amber-tx:#ffd166;--amber-border:#725d2f}
.nav:hover{background:var(--surface-hover)}
.warnbar{border-color:var(--amber-border);color:var(--amber-tx)}
.otp-form .otp-code{width:min(100%,320px);min-height:56px;padding:10px 14px;text-align:center;font-size:23px;font-weight:700;letter-spacing:.42em;font-variant-numeric:tabular-nums}
.otp-form .otp-code:focus{outline:0;border-color:var(--pri);box-shadow:0 0 0 3px color-mix(in srgb,var(--pri) 18%,transparent)}
.otp-form .otp-code[aria-invalid="true"]{border-color:var(--red)}
.otp-resend-row{display:flex;align-items:center;justify-content:space-between;gap:12px}
.otp-resend{flex:none}
.otp-countdown{color:var(--mut);font-size:11.5px;text-align:right}
.otp-resend:disabled{cursor:not-allowed;opacity:.65}
@media(max-width:500px){.otp-form .otp-code{min-height:52px;font-size:21px;letter-spacing:.34em}.otp-resend-row{align-items:flex-start;flex-direction:column}.otp-countdown{text-align:left}}
:root{--chat-user-bg:var(--pri);--chat-user-tx:#fff;--chat-user-border:var(--pri);--chat-action-bg:var(--pri);--chat-action-hover:var(--pri2)}
:root[data-theme="dark"]{--chat-user-bg:#363a70;--chat-user-tx:#f4f3ff;--chat-user-border:#545994;--chat-action-bg:#5659b8;--chat-action-hover:#6669c8}
.chat{gap:16px;padding:18px;background:var(--card)}
.chat>div:first-child{padding-bottom:14px;border-bottom:1px solid var(--line);align-items:center}
.chat>div:first-child b{color:var(--tx);font-size:12px}
.chat>div:first-child .sub{flex-basis:100%;margin-left:2px!important}
.chat>div:first-child .in{min-height:36px;padding:6px 10px}
.chat .msgs{min-height:300px;max-height:min(58vh,680px);overflow-y:auto;overscroll-behavior:contain;padding:18px;gap:14px;border:1px solid var(--line);border-radius:14px;background:var(--surface-soft);scroll-behavior:smooth}
.chat .m{max-width:min(86%,760px);padding:13px 16px;line-height:1.6;overflow-wrap:anywhere;white-space:normal}
.chat .m.u{background:var(--chat-user-bg);color:var(--chat-user-tx);border-color:var(--chat-user-border);border-radius:16px 16px 5px 16px}
.chat .cin .btn.primary{background:var(--chat-action-bg);border-color:var(--chat-action-bg);color:#fff}
.chat .cin .btn.primary:hover{background:var(--chat-action-hover);border-color:var(--chat-action-hover)}
.chat .m.a{background:var(--surface);color:var(--tx);border-color:var(--line);border-radius:16px 16px 16px 5px}
.chat .m.a .md{min-width:0}
.chat .m.a .md pre{max-width:100%}
.chat .m.a .tbl{max-width:100%}
.chat .filters{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,260px),1fr));gap:8px;margin:0}
.chat .filters:before{content:"Suggested questions";grid-column:1/-1;color:var(--mut);font-size:11px;font-weight:650;letter-spacing:.04em}
.chat .filters .chip{display:flex;align-items:center;min-height:40px;padding:8px 12px;border:1px solid var(--line);border-radius:11px;background:var(--surface-soft);color:var(--tx2);font-size:12px;line-height:1.45;white-space:normal;overflow-wrap:anywhere;transition:border-color .15s,background .15s,color .15s}
.chat .filters .chip:hover{border-color:var(--pri);background:var(--pri-bg);color:var(--pri2)}
.chat .cin{align-items:stretch;padding:10px;border:1px solid var(--line);border-radius:14px;background:var(--surface-soft)}
.chat .cin input{min-width:0;min-height:44px;padding:10px 12px;border-radius:10px}
.chat .cin .btn{min-width:44px}
.chat .cin .btn.primary{min-width:76px}
.chat-status{display:flex;align-items:center;gap:8px;min-height:20px;margin-top:-8px;color:var(--mut);font-size:12px}
.chat-status:empty{display:none}
.chat-status.is-error{color:var(--red)}
.chat-status.is-complete{color:var(--teal)}
.chat[data-state="loading"] .chat-status:before{content:"";width:13px;height:13px;border:2px solid var(--line2);border-top-color:var(--pri);border-radius:50%;animation:chat-spin .7s linear infinite}
.chat-error{margin:10px 0 0;padding:9px 11px;border:1px solid var(--red);border-radius:9px;background:var(--red-bg);color:var(--red);font-size:12px}
@keyframes chat-spin{to{transform:rotate(360deg)}}
@media(max-width:760px){
 .chat{gap:12px;padding:12px}
 .chat>div:first-child{gap:7px!important}
 .chat>div:first-child b{margin-left:0!important}
 .chat>div:first-child .sub{flex-basis:100%}
 .chat .msgs{min-height:240px;max-height:52vh;padding:12px}
 .chat .m{max-width:96%;padding:11px 12px}
 .chat .filters{grid-template-columns:1fr}
 .chat .cin{gap:6px;padding:7px}
 .chat .cin .btn{padding:7px 10px}
}
/* sign out, notices and admin table details (RBAC integration) */
.login-status.is-notice{color:#0b7a6c}:root[data-theme="dark"] .login-status.is-notice{color:#70d9a2}
.sidefoot{display:flex;align-items:center;justify-content:space-between;gap:8px;padding:10px 6px 2px;margin-top:8px;border-top:1px solid var(--line)}
.sidefoot .who{min-width:0}.sidefoot .who b{display:block;font-size:13px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}.sidefoot .who small{display:block;color:var(--mut);font-size:11px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.admin-kpis .kpi .v{color:var(--tx)}
.admin-role-select{width:auto;min-width:140px;padding:5px 8px}.admin-table td{vertical-align:middle;white-space:nowrap}
@media(max-width:1000px){.sidefoot .who{display:none}.sidefoot{justify-content:center}}
/* ---- light theme: midnight-navy sidebar with a teal accent (slate + teal + amber) ---- */
:root:not([data-theme="dark"]) .side{background:#0f172a;border-right-color:#0f172a;color:#cbd5e1}
:root:not([data-theme="dark"]) .side .brand{color:#fff}
:root:not([data-theme="dark"]) .side .brand small{color:#94a3b8}
:root:not([data-theme="dark"]) .brand i{background:linear-gradient(135deg,#14b8a6,#0f766e);box-shadow:0 4px 14px rgba(20,184,166,.35)}
:root:not([data-theme="dark"]) .ksearch{background:rgba(255,255,255,.06);border-color:rgba(255,255,255,.1);color:#94a3b8}
:root:not([data-theme="dark"]) .ksearch:hover{border-color:rgba(255,255,255,.22)}
:root:not([data-theme="dark"]) .ksearch kbd{background:rgba(255,255,255,.08);border-color:rgba(255,255,255,.16);color:#cbd5e1}
:root:not([data-theme="dark"]) .ngrp{color:#64748b}
:root:not([data-theme="dark"]) .nav{color:#cbd5e1}
:root:not([data-theme="dark"]) .nav:hover{background:rgba(255,255,255,.07);color:#fff}
:root:not([data-theme="dark"]) .nav.on{background:rgba(20,184,166,.16);color:#5eead4}
:root:not([data-theme="dark"]) .sidefoot{border-top-color:rgba(255,255,255,.12)}
:root:not([data-theme="dark"]) .sidefoot .who b{color:#fff}
:root:not([data-theme="dark"]) .sidefoot .who small{color:#94a3b8}
:root:not([data-theme="dark"]) .side .signout{background:transparent;color:#e2e8f0;border-color:rgba(255,255,255,.25)}
:root:not([data-theme="dark"]) .side .signout:hover{background:rgba(255,255,255,.1)}
@media(max-width:760px){:root:not([data-theme="dark"]) .side{border-top-color:#1e293b}}


Writing /content/app/static/style.css


In [33]:
import base64, io, tarfile
B64 = (
'...'
'QX0V+gAwFgA='
)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz').extractall('/content/app/static')
print('vendor files ready')

vendor files ready


/tmp/ipykernel_876/2855760141.py:7303: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(fileobj=io.BytesIO(base64.b64decode(B64)), mode='r:gz').extractall('/content/app/static')


In [34]:
import os, subprocess, sys, time, urllib.request
os.chdir('/content/app')
os.environ['APP_KIND'] = 'Franchise'
os.environ['RATE_LIMIT_OFF'] = '1'
subprocess.run('fuser -k 8000/tcp', shell=True, capture_output=True)
log = open('/content/app/server.log', 'w')
proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'server:app', '--host', '0.0.0.0', '--port', '8000'], stdout=log, stderr=log, cwd='/content/app')
for _ in range(90):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/health', timeout=2); print('FranchiseOps AI is running'); break
    except Exception: time.sleep(2)
else:
    print(open('/content/app/server.log').read()[-3000:])

FranchiseOps AI is running


In [35]:
import os
from IPython.display import HTML, display
links = []
# 1) ngrok: a normal public https link that works in any browser (Colab's own link is often blocked or answers 404)
try:
    from pyngrok import ngrok
    token = os.environ.get('NGROK_AUTHTOKEN')
    if not token:
        from google.colab import userdata
        token = userdata.get('NGROK_AUTHTOKEN')
    token = (token or '').strip()
    ngrok.kill()
    from pyngrok import conf
    conf.get_default().auth_token = token          # used when ngrok starts
    try:
        ngrok.set_auth_token(token)
    except Exception as e:                           # pyngrok raises here even when ngrok saved the token, so carry on and connect
        print('ngrok token note (usually harmless):', str(e)[:90])
    links.append(("Open the app", ngrok.connect(8000).public_url, "ngrok link. If ngrok shows a notice page, click Visit Site."))
except Exception as e:
    print('ngrok link unavailable (add the NGROK_AUTHTOKEN secret):', e)
# 2) Colab's own proxy link as a backup
try:
    from google.colab import output
    links.append(("Backup: Colab link", output.eval_js("google.colab.kernel.proxyPort(8000, {'cache': false})"), "Works only in a browser signed in to the same Google account."))
except Exception as e:
    print('Colab proxy link unavailable:', e)
for label, url, note in links:
    display(HTML(f'<p><a href="{url}" target="_blank" style="font-size:18px">{label}</a><br><small>{url}<br>{note}</small></p>'))